<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/03_Statistical_%26_Data_Characterization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# NOTEBOOK 03 — STATISTICAL & DATA CHARACTERIZATION
# ==============================================================================
#
# PURPOSE
# -------
# This notebook constructs the statistical reference layer required by the
# SPP-GAN framework.
#
# The notebook characterizes:
#   1. Dataset-level properties
#   2. Feature types
#   3. Numerical distributions
#   4. Categorical distributions
#   5. Missingness
#   6. Cardinality and entropy
#   7. Pearson dependency
#   8. Spearman dependency
#   9. Categorical dependency
#  10. Feature-level statistical profiles
#  11. Dataset-level statistical profiles
#  12. SPP-GAN statistical reference
#  13. Machine-readable statistical guidance
#
# IMPORTANT METHODOLOGICAL POLICY
# --------------------------------
# Statistical reference information is derived from the TRAINING split only.
#
# Validation and test data are NOT used to construct:
#   - distributions
#   - frequencies
#   - entropy
#   - correlations
#   - dependency matrices
#   - SPP-GAN statistical guidance
#
# Notebook 02 remains frozen.
#
# ==============================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import math
import os
import warnings
import hashlib

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("=" * 100)
print("NOTEBOOK 03 — STATISTICAL & DATA CHARACTERIZATION")
print("=" * 100)

print("\nScope:")
print("  • Training-only statistical characterization")
print("  • Publication-grade statistical reference construction")
print("  • RAM-safe Colab implementation")
print("  • No model training")
print("  • No synthetic data generation")
print("  • No validation/test leakage")

NOTEBOOK 03 — STATISTICAL & DATA CHARACTERIZATION

Scope:
  • Training-only statistical characterization
  • Publication-grade statistical reference construction
  • RAM-safe Colab implementation
  • No model training
  • No synthetic data generation
  • No validation/test leakage


In [2]:
# ==============================================================================
# SECTION 2 — LOAD PROCESSED DATA
# ==============================================================================
#
# PURPOSE
# -------
# Load the canonical TRAINING datasets persisted and validated by Notebook 02.
#
# IMPORTANT
# ---------
# Notebook 02 is COMPLETE and FROZEN.
#
# Notebook 03 must NOT depend on Notebook 02 Python objects existing in the
# current Colab runtime.
#
# Therefore, this section reconstructs the Notebook 03 input layer exclusively
# from persisted Notebook 02 artifacts.
#
# METHODOLOGICAL POLICY
# ---------------------
# 1. ONLY TRAINING data are loaded for statistical characterization.
# 2. Validation and test data are NOT used for statistical reference
#    construction.
# 3. Notebook 02 persisted artifacts are the authoritative upstream source.
# 4. Native TRAIN datasets are used for semantic statistical characterization.
# 5. The target remains available in the statistical characterization universe.
# 6. Explicit identifiers are excluded from statistical feature analysis.
# 7. __original_row_id__ is audit-only and excluded from statistical analysis.
# 8. No preprocessing is fitted in this section.
# 9. No imputation is performed in this section.
# 10. No validation/test statistics are constructed.
# 11. No synthetic data are generated.
# 12. No model is trained.
#
# ==============================================================================


from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import warnings

import numpy as np
import pandas as pd


warnings.filterwarnings("ignore")


print("=" * 100)
print("SECTION 2 — LOAD PROCESSED DATA")
print("=" * 100)


# ==============================================================================
# 2.1 GOOGLE DRIVE MOUNT
# ==============================================================================

print("\n[2.1] Validating Google Drive mount")
print("-" * 100)


try:

    from google.colab import drive

except ImportError as exc:

    raise RuntimeError(
        "Google Colab Drive interface could not be imported.\n"
        "This notebook is designed for Google Colab."
    ) from exc


DRIVE_ROOT = Path("/content/drive")
MYDRIVE_ROOT = DRIVE_ROOT / "MyDrive"


if not MYDRIVE_ROOT.exists():

    print("Google Drive is not mounted.")
    print("Mounting Google Drive...")

    drive.mount("/content/drive")


if not MYDRIVE_ROOT.exists():

    raise FileNotFoundError(
        "Google Drive mount failed.\n"
        f"Expected directory:\n{MYDRIVE_ROOT}"
    )


print("✓ Google Drive is mounted.")
print(f"✓ MyDrive : {MYDRIVE_ROOT}")


# ==============================================================================
# 2.2 PROJECT PATH CONFIGURATION
# ==============================================================================

print("\n[2.2] Configuring project paths")
print("-" * 100)


PROJECT_ROOT = (
    MYDRIVE_ROOT
    / "SPP_GAN_Research"
)


NB02_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_02"
)


NB03_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_03"
)


NB03_RESULTS_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebook_03"
)


print(f"Project root : {PROJECT_ROOT}")
print(f"Notebook 02  : {NB02_ROOT}")
print(f"Notebook 03  : {NB03_ROOT}")


# ==============================================================================
# 2.3 PROJECT ROOT VALIDATION
# ==============================================================================

print("\n[2.3] Validating project root")
print("-" * 100)


if not PROJECT_ROOT.is_dir():

    raise FileNotFoundError(
        "SPP-GAN project root does not exist.\n"
        f"Expected:\n{PROJECT_ROOT}"
    )


print("✓ SPP-GAN project root found.")


# ==============================================================================
# 2.4 NOTEBOOK 02 FROZEN ARTIFACT VALIDATION
# ==============================================================================

print("\n[2.4] Validating frozen Notebook 02 artifacts")
print("-" * 100)


if not NB02_ROOT.is_dir():

    raise FileNotFoundError(
        "Notebook 02 processed-data directory was not found.\n"
        f"Expected:\n{NB02_ROOT}"
    )


NB02_NATIVE_ROOT = (
    NB02_ROOT
    / "native"
)


NB02_SCHEMAS_ROOT = (
    NB02_ROOT
    / "schemas"
)


NB02_FEATURE_MAPPING_ROOT = (
    NB02_ROOT
    / "feature_mapping"
)


NB02_NATIVE_MANIFEST = (
    NB02_NATIVE_ROOT
    / "native_dataset_manifest.csv"
)


NB02_FINAL_INTEGRITY_MANIFEST = (
    NB02_SCHEMAS_ROOT
    / "section_24_final_integrity_manifest.json"
)


NB02_COMPLETION_METADATA = (
    NB02_SCHEMAS_ROOT
    / "notebook_02_completion_metadata.json"
)


REQUIRED_NB02_ARTIFACTS = {

    "native_root":
        NB02_NATIVE_ROOT,

    "schemas_root":
        NB02_SCHEMAS_ROOT,

    "feature_mapping_root":
        NB02_FEATURE_MAPPING_ROOT,

    "native_dataset_manifest":
        NB02_NATIVE_MANIFEST,

    "final_integrity_manifest":
        NB02_FINAL_INTEGRITY_MANIFEST,

    "completion_metadata":
        NB02_COMPLETION_METADATA,

}


for artifact_name, artifact_path in REQUIRED_NB02_ARTIFACTS.items():

    if not artifact_path.exists():

        raise FileNotFoundError(
            f"Required Notebook 02 artifact is missing.\n"
            f"Artifact : {artifact_name}\n"
            f"Path     : {artifact_path}"
        )


print("✓ Notebook 02 root verified.")
print("✓ Native dataset root verified.")
print("✓ Schema root verified.")
print("✓ Feature mapping root verified.")
print("✓ Native dataset manifest verified.")
print("✓ Final integrity manifest verified.")
print("✓ Completion metadata verified.")


# ==============================================================================
# 2.5 VERIFY NOTEBOOK 02 COMPLETION / FROZEN STATUS
# ==============================================================================

print("\n[2.5] Verifying Notebook 02 completion/frozen status")
print("-" * 100)


# ------------------------------------------------------------------------------
# Load persisted Notebook 02 completion metadata
# ------------------------------------------------------------------------------

with open(
    NB02_COMPLETION_METADATA,
    "r",
    encoding="utf-8"
) as f:

    NB02_COMPLETION = json.load(f)


# ------------------------------------------------------------------------------
# Load persisted Notebook 02 final integrity manifest
# ------------------------------------------------------------------------------

with open(
    NB02_FINAL_INTEGRITY_MANIFEST,
    "r",
    encoding="utf-8"
) as f:

    NB02_INTEGRITY = json.load(f)


# ==============================================================================
# 2.5.1 NOTEBOOK 02 SECTION 23 STATUS
# ==============================================================================

SECTION_23_STATUS = (
    NB02_COMPLETION.get(
        "section_23_status",
        None
    )
)


if SECTION_23_STATUS is not True:

    raise RuntimeError(
        "Notebook 02 Section 23 validation gate is not PASS.\n"
        f"Detected section_23_status: {SECTION_23_STATUS!r}\n\n"
        f"Completion metadata:\n{NB02_COMPLETION_METADATA}"
    )


print(
    "✓ Notebook 02 Section 23 validation status : PASS"
)


# ==============================================================================
# 2.5.2 NOTEBOOK 02 SECTION 24 FINAL INTEGRITY STATUS
# ==============================================================================

SECTION_24_STATUS = (
    str(
        NB02_INTEGRITY.get(
            "status",
            ""
        )
    ).upper()
)


if SECTION_24_STATUS != "PASS":

    raise RuntimeError(
        "Notebook 02 Section 24 final integrity status is not PASS.\n"
        f"Detected status: {SECTION_24_STATUS!r}\n\n"
        f"Final integrity manifest:\n"
        f"{NB02_FINAL_INTEGRITY_MANIFEST}"
    )


print(
    "✓ Notebook 02 Section 24 final integrity status : PASS"
)


# ==============================================================================
# 2.5.3 NOTEBOOK 02 COMPLETION GATE
# ==============================================================================

#
# Section 23 PASS + Section 24 PASS means the persisted Notebook 02
# upstream artifacts have passed the frozen integrity pipeline.
#
# Notebook 03 therefore accepts the persisted Notebook 02 artifacts as its
# authoritative upstream source.
#

NB02_UPSTREAM_STATUS = "FROZEN"


print(
    "✓ Notebook 02 upstream artifact status : FROZEN"
)

print(
    "✓ Persisted Notebook 02 artifacts accepted as authoritative input."
)

# ==============================================================================
# 2.6 CANONICAL DATASET REGISTRY
# ==============================================================================

print("\n[2.6] Establishing canonical dataset registry")
print("-" * 100)


DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]


TARGET_COLUMNS = {

    "adult_income":
        "income",

    "bank_marketing":
        "y",

    "diabetes_130us":
        "readmitted",

}


IDENTIFIER_COLUMNS = {

    "adult_income":
        [],

    "bank_marketing":
        [],

    "diabetes_130us":
        [
            "encounter_id",
            "patient_nbr",
        ],

}


PROVENANCE_COLUMN = "__original_row_id__"


print(
    f"✓ Dataset count : {len(DATASET_IDS)}"
)

print(
    f"✓ Dataset IDs   : {DATASET_IDS}"
)


# ==============================================================================
# 2.7 LOAD PERSISTED NOTEBOOK 02 NATIVE MANIFEST
# ==============================================================================

print("\n[2.7] Loading Notebook 02 native dataset manifest")
print("-" * 100)


NATIVE_MANIFEST_DF = pd.read_csv(
    NB02_NATIVE_MANIFEST
)


if NATIVE_MANIFEST_DF.empty:

    raise RuntimeError(
        "Notebook 02 native dataset manifest is empty."
    )


print(
    f"✓ Native manifest rows : "
    f"{len(NATIVE_MANIFEST_DF):,}"
)


# ==============================================================================
# 2.8 VALIDATE MANIFEST DATASET/SPLIT COVERAGE
# ==============================================================================

print("\n[2.8] Validating native manifest coverage")
print("-" * 100)


manifest_columns_lower = {

    str(column).lower():
        column

    for column in NATIVE_MANIFEST_DF.columns

}


required_manifest_fields = [
    "dataset_id",
    "split",
]


missing_manifest_fields = [

    field
    for field in required_manifest_fields
    if field not in manifest_columns_lower

]


if missing_manifest_fields:

    raise RuntimeError(
        "Native dataset manifest does not contain required fields.\n"
        f"Missing: {missing_manifest_fields}\n"
        f"Available: {list(NATIVE_MANIFEST_DF.columns)}"
    )


dataset_manifest_column = (
    manifest_columns_lower["dataset_id"]
)


split_manifest_column = (
    manifest_columns_lower["split"]
)


manifest_dataset_ids = set(
    NATIVE_MANIFEST_DF[
        dataset_manifest_column
    ].astype(str)
)


if manifest_dataset_ids != set(DATASET_IDS):

    raise RuntimeError(
        "Native manifest dataset coverage mismatch.\n"
        f"Expected : {set(DATASET_IDS)}\n"
        f"Found    : {manifest_dataset_ids}"
    )


manifest_splits = set(
    NATIVE_MANIFEST_DF[
        split_manifest_column
    ]
    .astype(str)
    .str.lower()
)


if "train" not in manifest_splits:

    raise RuntimeError(
        "Native manifest does not contain a TRAIN split."
    )


print(
    "✓ All three canonical datasets present."
)

print(
    f"✓ Available splits : {sorted(manifest_splits)}"
)


# ==============================================================================
# 2.9 RESOLVE TRAIN FILES FROM NATIVE MANIFEST
# ==============================================================================

print("\n[2.9] Resolving canonical TRAIN files")
print("-" * 100)


def resolve_manifest_path(row):
    """
    Resolve a persisted Notebook 02 artifact path from a manifest row.
    """

    candidate_values = []


    for column in NATIVE_MANIFEST_DF.columns:

        column_name = str(column).lower()


        if any(
            token in column_name
            for token in [
                "path",
                "file",
                "artifact",
            ]
        ):

            value = row[column]


            if pd.notna(value):

                candidate_values.append(
                    str(value)
                )


    for value in candidate_values:

        candidate = Path(value)


        if candidate.is_file():

            return candidate


        project_relative = (
            PROJECT_ROOT / value
        )


        if project_relative.is_file():

            return project_relative


        nb02_relative = (
            NB02_ROOT / value
        )


        if nb02_relative.is_file():

            return nb02_relative


    return None


TRAIN_DATASET_PATHS = {}


for dataset_id in DATASET_IDS:

    dataset_rows = NATIVE_MANIFEST_DF[
        (
            NATIVE_MANIFEST_DF[
                dataset_manifest_column
            ].astype(str)
            == dataset_id
        )
        &
        (
            NATIVE_MANIFEST_DF[
                split_manifest_column
            ]
            .astype(str)
            .str.lower()
            == "train"
        )
    ]


    if len(dataset_rows) != 1:

        raise RuntimeError(
            f"Expected exactly one TRAIN manifest entry for "
            f"'{dataset_id}', found {len(dataset_rows)}."
        )


    train_path = resolve_manifest_path(
        dataset_rows.iloc[0]
    )


    if train_path is None:

        raise FileNotFoundError(
            f"Could not resolve persisted TRAIN file for "
            f"'{dataset_id}' from native manifest."
        )


    TRAIN_DATASET_PATHS[
        dataset_id
    ] = train_path


    print(
        f"  ✓ {dataset_id:<20} : {train_path}"
    )


# ==============================================================================
# 2.10 DATAFRAME LOADER
# ==============================================================================

print("\n[2.10] Preparing dataframe loader")
print("-" * 100)


def load_dataframe(path):

    path = Path(path)


    if not path.is_file():

        raise FileNotFoundError(
            f"Data file does not exist:\n{path}"
        )


    suffix = path.suffix.lower()


    if suffix == ".parquet":

        return pd.read_parquet(path)


    if suffix == ".csv":

        return pd.read_csv(path)


    if suffix in {
        ".pkl",
        ".pickle",
    }:

        return pd.read_pickle(path)


    if suffix == ".feather":

        return pd.read_feather(path)


    raise ValueError(
        f"Unsupported dataframe format:\n{path}"
    )


# ==============================================================================
# 2.11 LOAD TRAINING DATA ONLY
# ==============================================================================

print("\n[2.11] Loading TRAINING datasets only")
print("-" * 100)


TRAIN_STATISTICAL_DATASETS = {}


for dataset_id in DATASET_IDS:

    train_path = TRAIN_DATASET_PATHS[
        dataset_id
    ]


    df = load_dataframe(
        train_path
    )


    if not isinstance(
        df,
        pd.DataFrame
    ):

        raise TypeError(
            f"Persisted TRAIN artifact for '{dataset_id}' "
            "is not a pandas DataFrame."
        )


    if df.empty:

        raise ValueError(
            f"Persisted TRAIN dataframe for '{dataset_id}' is empty."
        )


    TRAIN_STATISTICAL_DATASETS[
        dataset_id
    ] = df


    print(
        f"  ✓ {dataset_id:<20} | "
        f"Rows = {len(df):>8,} | "
        f"Columns = {len(df.columns):>4}"
    )


# ==============================================================================
# 2.12 TARGET VALIDATION
# ==============================================================================

print("\n[2.12] Validating target columns")
print("-" * 100)


for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[
        dataset_id
    ]


    target_column = TARGET_COLUMNS[
        dataset_id
    ]


    if target_column not in df.columns:

        raise RuntimeError(
            f"Target column '{target_column}' is missing from "
            f"the persisted TRAIN dataset '{dataset_id}'."
        )


    print(
        f"  ✓ {dataset_id:<20} | "
        f"Target = {target_column}"
    )


# ==============================================================================
# 2.13 VALIDATE IDENTIFIER AND PROVENANCE POLICY
# ==============================================================================

print("\n[2.13] Validating identifier and provenance policy")
print("-" * 100)


for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    expected_identifiers = IDENTIFIER_COLUMNS.get(dataset_id, [])

    # --------------------------------------------------------------------------
    # Provenance validation
    # --------------------------------------------------------------------------

    if "__original_row_id__" not in df.columns:

        raise RuntimeError(
            f"Required provenance column '__original_row_id__' is missing "
            f"from '{dataset_id}' training dataset."
        )

    # --------------------------------------------------------------------------
    # Identifier validation
    #
    # IMPORTANT:
    # Notebook 02 Section 18 deliberately excludes identifier columns from
    # the native generative schema.
    #
    # Therefore identifiers are NOT expected to be physically present in
    # the persisted native training datasets consumed by Notebook 03.
    # --------------------------------------------------------------------------

    present_identifiers = [
        col
        for col in expected_identifiers
        if col in df.columns
    ]

    unexpected_identifier_columns = [
        col
        for col in expected_identifiers
        if col in df.columns
    ]

    # Since Notebook 02 explicitly excludes identifiers from the native
    # generative schema, no identifier should be present here.

    if present_identifiers:

        raise RuntimeError(
            f"Identifier leakage detected in persisted native training "
            f"dataset '{dataset_id}'.\n"
            f"Identifiers unexpectedly present: {present_identifiers}\n\n"
            "Notebook 02 native generative datasets must exclude "
            "identifier columns."
        )

    # --------------------------------------------------------------------------
    # Verify provenance is audit-only
    # --------------------------------------------------------------------------

    provenance_position = df.columns.get_loc("__original_row_id__")

    if provenance_position < 0:

        raise RuntimeError(
            f"Unable to locate provenance column for '{dataset_id}'."
        )

    # --------------------------------------------------------------------------
    # Report
    # --------------------------------------------------------------------------

    if expected_identifiers:

        identifier_status = (
            f"expected identifiers excluded from native data "
            f"({expected_identifiers})"
        )

    else:

        identifier_status = "no configured identifiers"

    print(
        f"  ✓ {dataset_id:<20} | "
        f"Identifiers = {identifier_status} | "
        f"Provenance = present — audit only"
    )


print(
    "\n✓ Identifier/provenance policy is consistent with frozen "
    "Notebook 02 native generative datasets."
)
# ==============================================================================
# 2.14 DEFINE STATISTICAL FEATURE UNIVERSE
# ==============================================================================

print("\n[2.14] Defining statistical characterization universe")
print("-" * 100)


STATISTICAL_FEATURE_COLUMNS = {}


for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[
        dataset_id
    ]


    target_column = TARGET_COLUMNS[
        dataset_id
    ]


    identifier_columns = set(
        IDENTIFIER_COLUMNS[
            dataset_id
        ]
    )


    excluded_columns = (
        identifier_columns
        | {
            PROVENANCE_COLUMN
        }
    )


    statistical_columns = [

        column
        for column in df.columns
        if column not in excluded_columns

    ]


    if target_column not in statistical_columns:

        raise RuntimeError(
            f"Target '{target_column}' was excluded from the "
            f"statistical characterization universe for '{dataset_id}'."
        )


    if any(
        column in statistical_columns
        for column in identifier_columns
    ):

        raise RuntimeError(
            f"Identifier leakage detected in statistical feature "
            f"universe for '{dataset_id}'."
        )


    if PROVENANCE_COLUMN in statistical_columns:

        raise RuntimeError(
            f"Provenance leakage detected in statistical feature "
            f"universe for '{dataset_id}'."
        )


    STATISTICAL_FEATURE_COLUMNS[
        dataset_id
    ] = statistical_columns


    print(
        f"  ✓ {dataset_id:<20} | "
        f"Statistical variables = "
        f"{len(statistical_columns):>3}"
    )


# ==============================================================================
# 2.15 STATISTICAL REFERENCE REGISTRY
# ==============================================================================

STATISTICAL_REFERENCE_REGISTRY = {

    dataset_id: {

        "dataset_id":
            dataset_id,

        "source_artifact":
            str(
                TRAIN_DATASET_PATHS[
                    dataset_id
                ]
            ),

        "source_split":
            "train",

        "statistical_feature_columns":
            STATISTICAL_FEATURE_COLUMNS[
                dataset_id
            ],

        "target_column":
            TARGET_COLUMNS[
                dataset_id
            ],

        "identifier_columns_excluded":
            list(
                IDENTIFIER_COLUMNS[
                    dataset_id
                ]
            ),

        "provenance_column_excluded":
            PROVENANCE_COLUMN,

        "validation_used":
            False,

        "test_used":
            False,

    }

    for dataset_id in DATASET_IDS
}


# ==============================================================================
# 2.16 FINAL SECTION 2 SUMMARY
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 2 — LOAD PROCESSED DATA SUMMARY")
print("=" * 100)


print(
    f"\nProject root     : {PROJECT_ROOT}"
)

print(
    f"Notebook 02 root : {NB02_ROOT}"
)

print(
    f"Notebook 03 root : {NB03_ROOT}"
)


print("\nStatistical source policy:")

print(
    "  • Source split       : TRAIN only"
)

print(
    "  • Validation used    : NO"
)

print(
    "  • Test used          : NO"
)

print(
    "  • Synthetic data     : NO"
)

print(
    "  • Model training     : NO"
)


print("\nLoaded TRAIN datasets:")


for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[
        dataset_id
    ]


    print(
        f"  {dataset_id:<20} | "
        f"Rows = {len(df):>8,} | "
        f"Columns = {len(df.columns):>4} | "
        f"Statistical variables = "
        f"{len(STATISTICAL_FEATURE_COLUMNS[dataset_id]):>3} | "
        f"Target = {TARGET_COLUMNS[dataset_id]}"
    )


# ==============================================================================
# FINAL GATE
# ==============================================================================

print("\n" + "-" * 100)
print("SECTION 2 FINAL VALIDATION")
print("-" * 100)

print("✓ Google Drive mounted")
print("✓ SPP-GAN project root verified")
print("✓ Notebook 02 processed root verified")
print("✓ Notebook 02 completion metadata verified")
print("✓ Notebook 02 final integrity manifest verified")
print("✓ Notebook 02 frozen upstream artifacts accepted")
print("✓ Native dataset manifest loaded")
print("✓ Canonical dataset registry established")
print("✓ TRAIN files resolved from Notebook 02 persisted artifacts")
print("✓ TRAIN datasets loaded")
print("✓ Target columns verified")
print("✓ Explicit identifiers excluded from statistical analysis")
print("✓ Provenance excluded from statistical analysis")
print("✓ Statistical feature universe established")
print("✓ Validation data excluded")
print("✓ Test data excluded")
print("✓ No synthetic data generated")
print("✓ No model training performed")

print("\nSECTION 2 STATUS: PASS")

SECTION 2 — LOAD PROCESSED DATA

[2.1] Validating Google Drive mount
----------------------------------------------------------------------------------------------------
Google Drive is not mounted.
Mounting Google Drive...
Mounted at /content/drive
✓ Google Drive is mounted.
✓ MyDrive : /content/drive/MyDrive

[2.2] Configuring project paths
----------------------------------------------------------------------------------------------------
Project root : /content/drive/MyDrive/SPP_GAN_Research
Notebook 02  : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
Notebook 03  : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_03

[2.3] Validating project root
----------------------------------------------------------------------------------------------------
✓ SPP-GAN project root found.

[2.4] Validating frozen Notebook 02 artifacts
----------------------------------------------------------------------------------------------------
✓ Notebook 02 root verifi

In [3]:
# ==============================================================================
# SECTION 3 — VALIDATE INPUT SCHEMAS
# ==============================================================================

print("=" * 100)
print("SECTION 3 — VALIDATE INPUT SCHEMAS")
print("=" * 100)


INPUT_SCHEMA_REPORTS = []
INPUT_SCHEMA_SUMMARY = []


# ==============================================================================
# 3.1 — Validate persisted Notebook 02 schema contracts
# ==============================================================================

print("\n[3.1] Loading persisted Notebook 02 schema contracts")
print("-" * 100)


# ------------------------------------------------------------------------------
# Locate persisted preprocessing schemas
#
# Notebook 02 Section 20 persists:
#
# schemas/<dataset_id>/preprocessing_schema.json
#
# These files are authoritative for the feature/generative schema contract.
# ------------------------------------------------------------------------------

PERSISTED_SCHEMA_ROOT = (
    NB02_ROOT / "schemas"
)


if not PERSISTED_SCHEMA_ROOT.exists():

    raise RuntimeError(
        "Notebook 02 schema root does not exist:\n"
        f"{PERSISTED_SCHEMA_ROOT}"
    )


NB02_PERSISTED_SCHEMA_CONTRACTS = {}


for dataset_id in DATASET_IDS:

    schema_path = (
        PERSISTED_SCHEMA_ROOT
        / dataset_id
        / "preprocessing_schema.json"
    )

    if not schema_path.exists():

        raise RuntimeError(
            f"Persisted preprocessing schema not found for "
            f"'{dataset_id}':\n"
            f"{schema_path}"
        )


    with open(
        schema_path,
        "r",
        encoding="utf-8"
    ) as f:

        schema = json.load(f)


    NB02_PERSISTED_SCHEMA_CONTRACTS[
        dataset_id
    ] = schema


    print(
        f"  ✓ {dataset_id:20s} : "
        f"{schema_path}"
    )


print(
    "\n✓ Persisted Notebook 02 schema contracts loaded."
)


# ==============================================================================
# 3.2 — Validate schema contract structure
# ==============================================================================

print("\n[3.2] Validating persisted schema contract structure")
print("-" * 100)


REQUIRED_MODELING_SCHEMA_KEYS = [
    "preprocessing_columns",
    "all_columns",
    "numeric_columns",
    "categorical_columns",
    "generative_columns",
    "target_column",
    "identifier_columns_excluded",
    "provenance_column",
]


REQUIRED_TARGET_POLICY_KEYS = [
    "retained_in_generative_schema",
    "excluded_from_preprocessor_input",
    "excluded_from_transformed_features",
    "raw_target_manually_appended",
]


for dataset_id in DATASET_IDS:

    schema = NB02_PERSISTED_SCHEMA_CONTRACTS[
        dataset_id
    ]


    if "modeling_schema" not in schema:

        raise RuntimeError(
            f"'modeling_schema' missing from persisted schema "
            f"for '{dataset_id}'."
        )


    modeling_schema = schema[
        "modeling_schema"
    ]


    missing_modeling_keys = [
        key
        for key in REQUIRED_MODELING_SCHEMA_KEYS
        if key not in modeling_schema
    ]


    if missing_modeling_keys:

        raise RuntimeError(
            f"Persisted schema for '{dataset_id}' is missing "
            f"required modeling keys:\n"
            f"{missing_modeling_keys}"
        )


    if "target_policy" not in schema:

        raise RuntimeError(
            f"'target_policy' missing from persisted schema "
            f"for '{dataset_id}'."
        )


    target_policy = schema[
        "target_policy"
    ]


    missing_target_policy_keys = [
        key
        for key in REQUIRED_TARGET_POLICY_KEYS
        if key not in target_policy
    ]


    if missing_target_policy_keys:

        raise RuntimeError(
            f"Persisted target policy for '{dataset_id}' "
            f"is missing required keys:\n"
            f"{missing_target_policy_keys}"
        )


    print(
        f"  ✓ {dataset_id:20s} : schema contract structure valid"
    )


# ==============================================================================
# 3.3 — Dataset-level input schema validation
# ==============================================================================

print("\n[3.3] Validating TRAIN input schemas against Notebook 02")
print("-" * 100)


for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[
        dataset_id
    ]

    schema = NB02_PERSISTED_SCHEMA_CONTRACTS[
        dataset_id
    ]

    modeling_schema = schema[
        "modeling_schema"
    ]

    target_policy = schema[
        "target_policy"
    ]


    target = modeling_schema[
        "target_column"
    ]


    identifiers = list(
        modeling_schema[
            "identifier_columns_excluded"
        ]
    )


    provenance_column = modeling_schema[
        "provenance_column"
    ]


    expected_preprocessing_columns = list(
        modeling_schema[
            "all_columns"
        ]
    )


    expected_generative_columns = list(
        modeling_schema[
            "generative_columns"
        ]
    )


    actual_columns = list(
        df.columns
    )


    failures = []


    # ==========================================================================
    # 3.3.1 — Basic dataframe validation
    # ==========================================================================

    if not isinstance(
        df,
        pd.DataFrame
    ):

        failures.append(
            "training_input_is_not_dataframe"
        )


    if len(df) == 0:

        failures.append(
            "training_input_is_empty"
        )


    # ==========================================================================
    # 3.3.2 — Duplicate column validation
    # ==========================================================================

    duplicate_columns = (
        df.columns[
            df.columns.duplicated()
        ].tolist()
    )


    if duplicate_columns:

        failures.append(
            "duplicate_columns"
        )


    # ==========================================================================
    # 3.3.3 — Provenance validation
    # ==========================================================================

    if provenance_column not in actual_columns:

        failures.append(
            "provenance_missing"
        )


    # ==========================================================================
    # 3.3.4 — Target validation
    # ==========================================================================

    if target not in actual_columns:

        failures.append(
            "target_missing"
        )


    # ==========================================================================
    # 3.3.5 — Identifier exclusion validation
    # ==========================================================================

    identifiers_present = [
        identifier
        for identifier in identifiers
        if identifier in actual_columns
    ]


    if identifiers_present:

        for identifier in identifiers_present:

            failures.append(
                f"identifier_present_in_native_data:{identifier}"
            )


    # ==========================================================================
    # 3.3.6 — Exact native schema validation
    #
    # Frozen Notebook 02 Section 18 contract:
    #
    #     native schema
    #     =
    #     provenance + generative columns
    # ==========================================================================

    expected_native_columns = (
        [provenance_column]
        +
        expected_generative_columns
    )


    missing_expected_columns = [
        column
        for column in expected_native_columns
        if column not in actual_columns
    ]


    unexpected_columns = [
        column
        for column in actual_columns
        if column not in expected_native_columns
    ]


    if missing_expected_columns:

        failures.append(
            "missing_expected_native_columns"
        )


    if unexpected_columns:

        failures.append(
            "unexpected_native_columns"
        )


    if actual_columns != expected_native_columns:

        failures.append(
            "native_column_order_mismatch"
        )


    # ==========================================================================
    # 3.3.7 — Target policy validation
    # ==========================================================================

    if target not in expected_generative_columns:

        failures.append(
            "target_missing_from_generative_schema"
        )


    if target in expected_preprocessing_columns:

        failures.append(
            "target_present_in_preprocessing_schema"
        )


    if not target_policy[
        "retained_in_generative_schema"
    ]:

        failures.append(
            "target_policy_not_retained_in_generative_schema"
        )


    if not target_policy[
        "excluded_from_preprocessor_input"
    ]:

        failures.append(
            "target_policy_not_excluded_from_preprocessor_input"
        )


    if not target_policy[
        "excluded_from_transformed_features"
    ]:

        failures.append(
            "target_policy_not_excluded_from_transformed_features"
        )


    if target_policy[
        "raw_target_manually_appended"
    ]:

        failures.append(
            "raw_target_manually_appended"
        )


    # ==========================================================================
    # 3.3.8 — Identifier policy validation
    # ==========================================================================

    for identifier in identifiers:

        if identifier in expected_generative_columns:

            failures.append(
                f"identifier_present_in_generative_schema:{identifier}"
            )


        if identifier in expected_preprocessing_columns:

            failures.append(
                f"identifier_present_in_preprocessing_schema:{identifier}"
            )


    # ==========================================================================
    # 3.3.9 — Provenance policy validation
    # ==========================================================================

    if provenance_column in expected_generative_columns:

        failures.append(
            "provenance_present_in_generative_schema"
        )


    if provenance_column in expected_preprocessing_columns:

        failures.append(
            "provenance_present_in_preprocessing_schema"
        )


    # ==========================================================================
    # 3.3.10 — Generative/preprocessing schema relationship
    #
    # Notebook 02 contract:
    #
    #     generative columns
    #     =
    #     preprocessing columns + target
    # ==========================================================================

    expected_generatives_from_preprocessing = (
        expected_preprocessing_columns
        +
        [target]
    )


    if (
        set(expected_generative_columns)
        !=
        set(expected_generatives_from_preprocessing)
    ):

        failures.append(
            "generative_schema_not_equal_to_preprocessing_plus_target"
        )


    # ==========================================================================
    # 3.3.11 — Statistical modeling universe
    #
    # Target remains part of statistical characterization.
    # Provenance and identifiers are excluded.
    # ==========================================================================

    modeling_columns = [
        column
        for column in actual_columns
        if column != provenance_column
        and column not in identifiers
    ]


    if target not in modeling_columns:

        failures.append(
            "target_not_in_statistical_modeling_columns"
        )


    if provenance_column in modeling_columns:

        failures.append(
            "provenance_in_statistical_modeling_columns"
        )


    for identifier in identifiers:

        if identifier in modeling_columns:

            failures.append(
                f"identifier_in_statistical_modeling_columns:{identifier}"
            )


    # ==========================================================================
    # 3.3.12 — Numeric-value integrity
    # ==========================================================================

    numeric_columns = []

    infinite_numeric_columns = []


    if isinstance(
        df,
        pd.DataFrame
    ):

        numeric_columns = (
            df[
                modeling_columns
            ]
            .select_dtypes(
                include=[np.number]
            )
            .columns
            .tolist()
        )


        for column in numeric_columns:

            values = pd.to_numeric(
                df[column],
                errors="coerce"
            )


            numeric_array = values.to_numpy(
                dtype=float
            )


            if np.isinf(
                numeric_array
            ).any():

                infinite_numeric_columns.append(
                    column
                )


    if infinite_numeric_columns:

        failures.append(
            "infinite_numeric_values"
        )


    # ==========================================================================
    # 3.3.13 — Final dataset status
    # ==========================================================================

    status = (
        "PASS"
        if not failures
        else "FAIL"
    )


    # ==========================================================================
    # 3.3.14 — Detailed report
    # ==========================================================================

    INPUT_SCHEMA_REPORTS.append({

        "dataset_id":
            dataset_id,

        "status":
            status,

        "failures":
            failures,

        "actual_native_columns":
            actual_columns,

        "expected_native_columns":
            expected_native_columns,

        "missing_expected_columns":
            missing_expected_columns,

        "unexpected_columns":
            unexpected_columns,

        "expected_preprocessing_columns":
            expected_preprocessing_columns,

        "expected_generative_columns":
            expected_generative_columns,

        "numeric_columns":
            numeric_columns,

        "infinite_numeric_columns":
            infinite_numeric_columns,

        "target_column":
            target,

        "identifier_columns":
            identifiers,

        "provenance_column":
            provenance_column,

    })


    # ==========================================================================
    # 3.3.15 — Compact summary
    # ==========================================================================

    INPUT_SCHEMA_SUMMARY.append({

        "dataset_id":
            dataset_id,

        "rows":
            len(df),

        "total_columns":
            len(actual_columns),

        "expected_native_columns":
            len(expected_native_columns),

        "modeling_columns":
            len(modeling_columns),

        "preprocessing_columns":
            len(expected_preprocessing_columns),

        "generative_columns":
            len(expected_generative_columns),

        "numeric_columns":
            len(numeric_columns),

        "target_column":
            target,

        "identifier_columns_excluded":
            ", ".join(identifiers)
            if identifiers
            else "",

        "provenance_present":
            provenance_column in actual_columns,

        "schema_exact_match":
            actual_columns == expected_native_columns,

        "target_in_generative_schema":
            target in expected_generative_columns,

        "target_in_preprocessing_schema":
            target in expected_preprocessing_columns,

        "status":
            status,

        "failures":
            "; ".join(failures),

    })


    print(
        f"  {dataset_id:20s} : {status} | "
        f"Native = {len(actual_columns):4d} | "
        f"Generative = {len(expected_generative_columns):3d} | "
        f"Preprocessing = {len(expected_preprocessing_columns):3d} | "
        f"Numeric = {len(numeric_columns):3d}"
    )


# ==============================================================================
# 3.4 — Create validation dataframes
# ==============================================================================

INPUT_SCHEMA_SUMMARY_DF = pd.DataFrame(
    INPUT_SCHEMA_SUMMARY
)


INPUT_SCHEMA_REPORT_DF = pd.DataFrame(
    INPUT_SCHEMA_REPORTS
)


# ==============================================================================
# 3.5 — Global validation gate
# ==============================================================================

failed_schema_datasets = (
    INPUT_SCHEMA_SUMMARY_DF[
        INPUT_SCHEMA_SUMMARY_DF["status"] != "PASS"
    ]["dataset_id"]
    .tolist()
)


if failed_schema_datasets:

    print("\n" + "=" * 100)
    print("SECTION 3 — SCHEMA VALIDATION FAILURE")
    print("=" * 100)

    for dataset_id in failed_schema_datasets:

        print(
            f"  ✗ {dataset_id}"
        )

    raise RuntimeError(
        "Notebook 02 input schema validation failed for: "
        +
        ", ".join(
            failed_schema_datasets
        )
    )


# ==============================================================================
# 3.6 — Final Section 3 validation summary
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 3 FINAL VALIDATION")
print("=" * 100)

print(
    "✓ Persisted Notebook 02 schema contracts loaded."
)

print(
    "✓ All TRAIN datasets are non-empty."
)

print(
    "✓ Native schemas exactly match the frozen Notebook 02 schema."
)

print(
    "✓ Target columns are present and retained in the generative schema."
)

print(
    "✓ Targets are excluded from the generic preprocessing schema."
)

print(
    "✓ Explicit identifiers are excluded from native generative data."
)

print(
    "✓ Provenance is retained only for auditability."
)

print(
    "✓ Provenance is excluded from statistical modeling variables."
)

print(
    "✓ No duplicate columns detected."
)

print(
    "✓ No infinite numeric values detected."
)

print(
    "✓ Statistical modeling universe is correctly defined."
)

print(
    "\nSECTION 3 STATUS: PASS"
)

SECTION 3 — VALIDATE INPUT SCHEMAS

[3.1] Loading persisted Notebook 02 schema contracts
----------------------------------------------------------------------------------------------------
  ✓ adult_income         : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas/adult_income/preprocessing_schema.json
  ✓ bank_marketing       : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas/bank_marketing/preprocessing_schema.json
  ✓ diabetes_130us       : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas/diabetes_130us/preprocessing_schema.json

✓ Persisted Notebook 02 schema contracts loaded.

[3.2] Validating persisted schema contract structure
----------------------------------------------------------------------------------------------------
  ✓ adult_income         : schema contract structure valid
  ✓ bank_marketing       : schema contract structure valid
  ✓ diabetes_130us       : schema contract structure valid



In [4]:
# ==============================================================================
# SECTION 4 — DATASET-LEVEL CHARACTERIZATION
# ==============================================================================

print("=" * 100)
print("SECTION 4 — DATASET-LEVEL CHARACTERIZATION")
print("=" * 100)


DATASET_CHARACTERIZATION_RECORDS = []


# ==============================================================================
# 4.1 — Dataset-level characterization
# ==============================================================================

for dataset_id in DATASET_IDS:

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    schema = NB02_PERSISTED_SCHEMA_CONTRACTS[
        dataset_id
    ]

    modeling_schema = schema[
        "modeling_schema"
    ]

    target_policy = schema[
        "target_policy"
    ]


    # --------------------------------------------------------------------------
    # Authoritative Notebook 02 schema
    # --------------------------------------------------------------------------

    target = modeling_schema[
        "target_column"
    ]

    identifiers = list(
        modeling_schema[
            "identifier_columns_excluded"
        ]
    )

    provenance_column = modeling_schema[
        "provenance_column"
    ]

    preprocessing_columns = list(
        modeling_schema[
            "all_columns"
        ]
    )

    generative_columns = list(
        modeling_schema[
            "generative_columns"
        ]
    )


    # --------------------------------------------------------------------------
    # Statistical modeling universe
    #
    # Target is retained.
    # Provenance and identifiers are excluded.
    # --------------------------------------------------------------------------

    modeling_columns = [
        column
        for column in df.columns
        if column != provenance_column
        and column not in identifiers
    ]


    # --------------------------------------------------------------------------
    # Semantic numeric / categorical characterization
    # --------------------------------------------------------------------------

    numeric_columns = (
        df[modeling_columns]
        .select_dtypes(
            include=[np.number]
        )
        .columns
        .tolist()
    )


    categorical_columns = [
        column
        for column in modeling_columns
        if column not in numeric_columns
    ]


    # --------------------------------------------------------------------------
    # Target characterization
    # --------------------------------------------------------------------------

    target_series = df[target]

    target_non_null = int(
        target_series.notna().sum()
    )

    target_missing = int(
        target_series.isna().sum()
    )

    target_cardinality = int(
        target_series.nunique(
            dropna=True
        )
    )


    target_missing_rate = (
        target_missing / len(df)
        if len(df) > 0
        else np.nan
    )


    # --------------------------------------------------------------------------
    # Target semantic type
    # --------------------------------------------------------------------------

    if pd.api.types.is_numeric_dtype(
        target_series
    ):

        target_dtype_group = "numeric"

    else:

        target_dtype_group = "categorical"


    # --------------------------------------------------------------------------
    # Dataset memory
    # --------------------------------------------------------------------------

    memory_mb = (
        df.memory_usage(
            deep=True
        ).sum()
        /
        (1024 ** 2)
    )


    # --------------------------------------------------------------------------
    # Characterization record
    # --------------------------------------------------------------------------

    DATASET_CHARACTERIZATION_RECORDS.append({

        # ----------------------------------------------------------------------
        # Identity
        # ----------------------------------------------------------------------

        "dataset_id":
            dataset_id,

        # ----------------------------------------------------------------------
        # Dataset size
        # ----------------------------------------------------------------------

        "training_rows":
            int(len(df)),

        "native_columns":
            int(len(df.columns)),

        "generative_columns":
            int(len(generative_columns)),

        "preprocessing_columns":
            int(len(preprocessing_columns)),

        "modeling_columns":
            int(len(modeling_columns)),

        # ----------------------------------------------------------------------
        # Feature composition
        # ----------------------------------------------------------------------

        "numeric_columns":
            int(len(numeric_columns)),

        "categorical_columns":
            int(len(categorical_columns)),

        "identifier_columns_excluded":
            int(len(identifiers)),

        "provenance_present":
            bool(
                provenance_column in df.columns
            ),

        # ----------------------------------------------------------------------
        # Target characterization
        # ----------------------------------------------------------------------

        "target_column":
            target,

        "target_dtype_group":
            target_dtype_group,

        "target_non_null":
            target_non_null,

        "target_missing":
            target_missing,

        "target_missing_rate":
            round(
                float(target_missing_rate),
                6
            ),

        "target_cardinality":
            target_cardinality,

        # ----------------------------------------------------------------------
        # Audit / reproducibility metadata
        # ----------------------------------------------------------------------

        "memory_mb":
            round(
                float(memory_mb),
                3
            ),

    })


# ==============================================================================
# 4.2 — Create characterization dataframe
# ==============================================================================

DATASET_CHARACTERIZATION_DF = pd.DataFrame(
    DATASET_CHARACTERIZATION_RECORDS
)


# ==============================================================================
# 4.3 — Structural consistency checks
# ==============================================================================

print("\n[4.3] Validating dataset-level characterization")
print("-" * 100)


for dataset_id in DATASET_IDS:

    row = DATASET_CHARACTERIZATION_DF[
        DATASET_CHARACTERIZATION_DF[
            "dataset_id"
        ] == dataset_id
    ].iloc[0]


    schema = NB02_PERSISTED_SCHEMA_CONTRACTS[
        dataset_id
    ]

    modeling_schema = schema[
        "modeling_schema"
    ]


    # --------------------------------------------------------------------------
    # Expected schema counts
    # --------------------------------------------------------------------------

    expected_generatives = len(
        modeling_schema[
            "generative_columns"
        ]
    )

    expected_preprocessing = len(
        modeling_schema[
            "all_columns"
        ]
    )


    expected_native = (
        expected_generatives + 1
    )


    if int(row["native_columns"]) != expected_native:

        raise RuntimeError(
            f"Native column-count mismatch for "
            f"'{dataset_id}'. "
            f"Expected {expected_native}, "
            f"found {row['native_columns']}."
        )


    if int(row["generative_columns"]) != expected_generatives:

        raise RuntimeError(
            f"Generative column-count mismatch for "
            f"'{dataset_id}'."
        )


    if int(row["preprocessing_columns"]) != expected_preprocessing:

        raise RuntimeError(
            f"Preprocessing column-count mismatch for "
            f"'{dataset_id}'."
        )


    # --------------------------------------------------------------------------
    # Feature arithmetic
    # --------------------------------------------------------------------------

    if (
        int(row["numeric_columns"])
        +
        int(row["categorical_columns"])
        !=
        int(row["modeling_columns"])
    ):

        raise RuntimeError(
            f"Numeric/categorical feature-count mismatch "
            f"for '{dataset_id}'."
        )


    # --------------------------------------------------------------------------
    # Target validation
    # --------------------------------------------------------------------------

    target = modeling_schema[
        "target_column"
    ]

    if row["target_column"] != target:

        raise RuntimeError(
            f"Target mismatch for '{dataset_id}'."
        )


    if not bool(
        row["provenance_present"]
    ):

        raise RuntimeError(
            f"Provenance column missing from "
            f"'{dataset_id}'."
        )


    # --------------------------------------------------------------------------
    # Target policy consistency
    # --------------------------------------------------------------------------

    if not target_policy.get(
        "retained_in_generative_schema",
        False
    ):

        raise RuntimeError(
            f"Persisted target policy is inconsistent "
            f"for '{dataset_id}'."
        )


    if target_policy.get(
        "excluded_from_preprocessor_input",
        False
    ) is not True:

        raise RuntimeError(
            f"Target preprocessing exclusion policy is "
            f"inconsistent for '{dataset_id}'."
        )


    print(
        f"  ✓ {dataset_id:20s} | "
        f"Rows = {int(row['training_rows']):7d} | "
        f"Generative = {int(row['generative_columns']):3d} | "
        f"Numeric = {int(row['numeric_columns']):3d} | "
        f"Categorical = {int(row['categorical_columns']):3d} | "
        f"Target = {target}"
    )


# ==============================================================================
# 4.4 — Display dataset-level characterization
# ==============================================================================

print("\n" + "=" * 100)
print("DATASET-LEVEL CHARACTERIZATION SUMMARY")
print("=" * 100)


display_columns = [
    "dataset_id",
    "training_rows",
    "native_columns",
    "generative_columns",
    "preprocessing_columns",
    "modeling_columns",
    "numeric_columns",
    "categorical_columns",
    "identifier_columns_excluded",
    "target_column",
    "target_dtype_group",
    "target_non_null",
    "target_missing",
    "target_missing_rate",
    "target_cardinality",
    "memory_mb",
]


print(
    DATASET_CHARACTERIZATION_DF[
        display_columns
    ].to_string(
        index=False
    )
)


# ==============================================================================
# 4.5 — Final Section 4 status
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 4 FINAL VALIDATION")
print("=" * 100)

print(
    "✓ Dataset characterization uses TRAIN data only."
)

print(
    "✓ No validation or test data were accessed."
)

print(
    "✓ No synthetic data were accessed."
)

print(
    "✓ Dataset dimensions agree with persisted Notebook 02 schemas."
)

print(
    "✓ Generative and preprocessing feature counts are consistent."
)

print(
    "✓ Numeric and categorical feature counts are internally consistent."
)

print(
    "✓ Target columns are retained in the statistical characterization universe."
)

print(
    "✓ Target missingness and cardinality were characterized."
)

print(
    "✓ Provenance is retained only for auditability."
)

print(
    "✓ Identifiers are excluded from characterization."
)

print(
    "\nSECTION 4 STATUS: PASS"
)

SECTION 4 — DATASET-LEVEL CHARACTERIZATION

[4.3] Validating dataset-level characterization
----------------------------------------------------------------------------------------------------
  ✓ adult_income         | Rows =   34189 | Generative =  15 | Numeric =   6 | Categorical =   9 | Target = income
  ✓ bank_marketing       | Rows =   31647 | Generative =  17 | Numeric =   7 | Categorical =  10 | Target = y
  ✓ diabetes_130us       | Rows =   71236 | Generative =  48 | Numeric =  11 | Categorical =  37 | Target = readmitted

DATASET-LEVEL CHARACTERIZATION SUMMARY
    dataset_id  training_rows  native_columns  generative_columns  preprocessing_columns  modeling_columns  numeric_columns  categorical_columns  identifier_columns_excluded target_column target_dtype_group  target_non_null  target_missing  target_missing_rate  target_cardinality  memory_mb
  adult_income          34189              16                  15                     14                15                6        

In [5]:
# ==============================================================================
# SECTION 5 — FEATURE-TYPE CHARACTERIZATION
# ==============================================================================
#
# PURPOSE
# -------
# Characterize the statistical/data type and structural properties of every
# native generative variable using TRAINING data only.
#
# IMPORTANT:
# - Notebook 02 remains frozen.
# - Numeric/categorical preprocessing schema comes from the persisted
#   Notebook 02 schema contract.
# - The target is NOT part of Notebook 02 preprocessing_columns,
#   numeric_columns, or categorical_columns.
# - The target is characterized separately from preprocessing features.
# - Identifier columns are excluded from the modeling/generative universe.
# - Provenance is retained only for auditability.
#
# OUTPUTS
# -------
# FEATURE_TYPE_CHARACTERIZATION_DF
# FEATURE_TYPE_SUMMARY_DF
#
# ==============================================================================

print("=" * 100)
print("SECTION 5 — FEATURE-TYPE CHARACTERIZATION")
print("=" * 100)


# ==============================================================================
# 5.1 — REQUIRED OBJECT VALIDATION
# ==============================================================================

required_objects = [
    "DATASET_IDS",
    "TRAIN_STATISTICAL_DATASETS",
    "NB02_PERSISTED_SCHEMA_CONTRACTS",
    "PROVENANCE_COLUMN",
]

missing_objects = [
    obj
    for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required Section 5 objects are missing:\n"
        + "\n".join(
            f"  - {obj}"
            for obj in missing_objects
        )
        + "\n\nExecute Sections 2–4 before Section 5."
    )


# ==============================================================================
# 5.2 — INITIALIZE OUTPUT CONTAINERS
# ==============================================================================

FEATURE_TYPE_RECORDS = []
FEATURE_TYPE_SUMMARY_RECORDS = []


# ==============================================================================
# 5.3 — DATASET-WISE FEATURE CHARACTERIZATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    # --------------------------------------------------------------------------
    # TRAINING DATA ONLY
    # --------------------------------------------------------------------------

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    if not isinstance(df, pd.DataFrame):
        raise TypeError(
            f"{dataset_id}: TRAIN_STATISTICAL_DATASETS entry "
            "is not a pandas DataFrame."
        )

    # --------------------------------------------------------------------------
    # Load frozen Notebook 02 schema contract
    # --------------------------------------------------------------------------

    schema_contract = NB02_PERSISTED_SCHEMA_CONTRACTS[dataset_id]

    modeling_schema = schema_contract["modeling_schema"]

    preprocessing_columns = list(
        modeling_schema["preprocessing_columns"]
    )

    numeric_columns_schema = list(
        modeling_schema["numeric_columns"]
    )

    categorical_columns_schema = list(
        modeling_schema["categorical_columns"]
    )

    generative_columns = list(
        modeling_schema["generative_columns"]
    )

    target_column = modeling_schema["target_column"]

    identifier_columns = list(
        modeling_schema["identifier_columns_excluded"]
    )

    provenance_column = modeling_schema["provenance_column"]

    # --------------------------------------------------------------------------
    # Required columns
    # --------------------------------------------------------------------------

    if target_column not in df.columns:
        raise RuntimeError(
            f"{dataset_id}: target column '{target_column}' "
            "is missing from native training data."
        )

    if provenance_column not in df.columns:
        raise RuntimeError(
            f"{dataset_id}: provenance column "
            f"'{provenance_column}' is missing."
        )

    # --------------------------------------------------------------------------
    # Identifiers must remain excluded from native generative data
    # --------------------------------------------------------------------------

    unexpected_identifiers = [
        column
        for column in identifier_columns
        if column in df.columns
    ]

    if unexpected_identifiers:
        raise RuntimeError(
            f"{dataset_id}: identifier columns unexpectedly present "
            f"in native training data: {unexpected_identifiers}"
        )

    # --------------------------------------------------------------------------
    # Native generative/modeling universe
    # --------------------------------------------------------------------------

    modeling_columns = [
        column
        for column in df.columns
        if column != provenance_column
        and column not in identifier_columns
    ]

    # --------------------------------------------------------------------------
    # Validate exact native schema
    # --------------------------------------------------------------------------

    expected_native_columns = [
        provenance_column
    ] + generative_columns

    if df.columns.tolist() != expected_native_columns:
        raise RuntimeError(
            f"{dataset_id}: native training column schema/order mismatch.\n"
            f"Observed : {df.columns.tolist()}\n"
            f"Expected : {expected_native_columns}"
        )

    # --------------------------------------------------------------------------
    # Generative schema = preprocessing features + target
    # --------------------------------------------------------------------------

    expected_generative_columns = (
        preprocessing_columns + [target_column]
    )

    if generative_columns != expected_generative_columns:
        raise RuntimeError(
            f"{dataset_id}: generative schema mismatch.\n"
            f"Observed : {generative_columns}\n"
            f"Expected : {expected_generative_columns}"
        )

    # ==============================================================================
    # 5.3.1 — VERIFY PREPROCESSING FEATURE TYPES
    # ==============================================================================

    observed_numeric_preprocessing = (
        df[preprocessing_columns]
        .select_dtypes(include=[np.number])
        .columns
        .tolist()
    )

    observed_categorical_preprocessing = [
        column
        for column in preprocessing_columns
        if column not in observed_numeric_preprocessing
    ]

    if set(observed_numeric_preprocessing) != set(
        numeric_columns_schema
    ):
        raise RuntimeError(
            f"{dataset_id}: numeric preprocessing feature mismatch.\n"
            f"Observed : {observed_numeric_preprocessing}\n"
            f"Expected : {numeric_columns_schema}"
        )

    if set(observed_categorical_preprocessing) != set(
        categorical_columns_schema
    ):
        raise RuntimeError(
            f"{dataset_id}: categorical preprocessing feature mismatch.\n"
            f"Observed : {observed_categorical_preprocessing}\n"
            f"Expected : {categorical_columns_schema}"
        )

    # ==============================================================================
    # 5.3.2 — FEATURE-LEVEL CHARACTERIZATION
    # ==============================================================================

    for feature_order, feature in enumerate(modeling_columns):

        series = df[feature]

        is_target = feature == target_column

        # ----------------------------------------------------------------------
        # Feature type
        #
        # Target is intentionally handled separately because Notebook 02's
        # numeric_columns/categorical_columns describe preprocessing features
        # only.
        # ----------------------------------------------------------------------

        if is_target:

            if pd.api.types.is_numeric_dtype(series):
                feature_type = "numeric"
            else:
                feature_type = "categorical"

            schema_role = "target"
            type_source = "native_target_dtype"

        else:

            if feature in numeric_columns_schema:

                feature_type = "numeric"

            elif feature in categorical_columns_schema:

                feature_type = "categorical"

            else:

                raise RuntimeError(
                    f"Feature '{feature}' in dataset '{dataset_id}' "
                    "is not represented in the persisted Notebook 02 "
                    "preprocessing numeric/categorical schema."
                )

            schema_role = "preprocessing_feature"
            type_source = "notebook_02_persisted_schema"

        # ----------------------------------------------------------------------
        # Counts
        # ----------------------------------------------------------------------

        training_rows = int(len(series))

        non_missing_count = int(
            series.notna().sum()
        )

        missing_count = int(
            series.isna().sum()
        )

        if training_rows > 0:
            missing_rate = (
                missing_count / training_rows
            )
        else:
            missing_rate = np.nan

        # ----------------------------------------------------------------------
        # Cardinality
        # ----------------------------------------------------------------------

        unique_count = int(
            series.nunique(dropna=True)
        )

        if non_missing_count > 0:
            unique_ratio = (
                unique_count / non_missing_count
            )
        else:
            unique_ratio = np.nan

        # ----------------------------------------------------------------------
        # Dominant value
        # ----------------------------------------------------------------------

        dominant_frequency = np.nan
        dominant_value_share = np.nan

        non_missing = series.dropna()

        if len(non_missing) > 0:

            value_counts = non_missing.value_counts()

            if not value_counts.empty:

                dominant_frequency = int(
                    value_counts.iloc[0]
                )

                dominant_value_share = (
                    dominant_frequency / len(non_missing)
                )

        # ----------------------------------------------------------------------
        # Constant / near-constant
        # ----------------------------------------------------------------------

        is_constant = unique_count <= 1

        if pd.isna(dominant_value_share):
            is_near_constant = False
        else:
            is_near_constant = (
                dominant_value_share >= 0.95
            )

        # ==============================================================================
        # NUMERIC CHARACTERIZATION
        # ==============================================================================

        if feature_type == "numeric":

            numeric_series = pd.to_numeric(
                series,
                errors="coerce"
            )

            numeric_non_missing = (
                numeric_series.dropna()
            )

            if len(numeric_non_missing) > 0:

                finite_mask = np.isfinite(
                    numeric_non_missing.to_numpy()
                )

                finite_values = (
                    numeric_non_missing.iloc[
                        np.flatnonzero(finite_mask)
                    ]
                )

            else:

                finite_values = numeric_non_missing

            non_finite_count = int(
                len(numeric_non_missing)
                - len(finite_values)
            )

            if len(finite_values) > 0:

                minimum = float(
                    finite_values.min()
                )

                maximum = float(
                    finite_values.max()
                )

                mean_value = float(
                    finite_values.mean()
                )

                median_value = float(
                    finite_values.median()
                )

                if len(finite_values) > 1:
                    std_value = float(
                        finite_values.std(
                            ddof=1
                        )
                    )
                else:
                    std_value = 0.0

                if len(finite_values) > 2:
                    skewness = float(
                        finite_values.skew()
                    )
                else:
                    skewness = np.nan

                if len(finite_values) > 3:
                    kurtosis = float(
                        finite_values.kurtosis()
                    )
                else:
                    kurtosis = np.nan

            else:

                minimum = np.nan
                maximum = np.nan
                mean_value = np.nan
                median_value = np.nan
                std_value = np.nan
                skewness = np.nan
                kurtosis = np.nan

        # ==============================================================================
        # CATEGORICAL CHARACTERIZATION
        # ==============================================================================

        else:

            minimum = np.nan
            maximum = np.nan
            mean_value = np.nan
            median_value = np.nan
            std_value = np.nan
            skewness = np.nan
            kurtosis = np.nan
            non_finite_count = np.nan

        if feature_type == "categorical":

            categorical_cardinality = int(
                series.dropna().nunique()
            )

        else:

            categorical_cardinality = np.nan

        # ----------------------------------------------------------------------
        # Store feature record
        # ----------------------------------------------------------------------

        FEATURE_TYPE_RECORDS.append({

            "dataset_id": dataset_id,

            "feature_order": int(
                feature_order
            ),

            "feature": feature,

            "feature_type": feature_type,

            "schema_role": schema_role,

            "type_source": type_source,

            "native_dtype": str(
                series.dtype
            ),

            "is_target": bool(
                is_target
            ),

            "is_preprocessing_feature": bool(
                feature in preprocessing_columns
            ),

            "is_generative_feature": bool(
                feature in generative_columns
            ),

            "training_rows": training_rows,

            "non_missing_count": non_missing_count,

            "missing_count": missing_count,

            "missing_rate": (
                round(
                    float(missing_rate),
                    8
                )
                if not pd.isna(missing_rate)
                else np.nan
            ),

            "unique_count": unique_count,

            "unique_ratio": (
                round(
                    float(unique_ratio),
                    8
                )
                if not pd.isna(unique_ratio)
                else np.nan
            ),

            "is_constant": bool(
                is_constant
            ),

            "dominant_frequency": (
                dominant_frequency
                if not pd.isna(dominant_frequency)
                else np.nan
            ),

            "dominant_value_share": (
                round(
                    float(dominant_value_share),
                    8
                )
                if not pd.isna(dominant_value_share)
                else np.nan
            ),

            "is_near_constant": bool(
                is_near_constant
            ),

            "minimum": minimum,

            "maximum": maximum,

            "mean": mean_value,

            "median": median_value,

            "std": std_value,

            "skewness": skewness,

            "kurtosis": kurtosis,

            "non_finite_count": non_finite_count,

            "categorical_cardinality": (
                categorical_cardinality
                if not pd.isna(categorical_cardinality)
                else np.nan
            ),
        })


# ==============================================================================
# 5.4 — CREATE FEATURE-LEVEL DATAFRAME
# ==============================================================================

FEATURE_TYPE_CHARACTERIZATION_DF = pd.DataFrame(
    FEATURE_TYPE_RECORDS
)

if FEATURE_TYPE_CHARACTERIZATION_DF.empty:
    raise RuntimeError(
        "FEATURE_TYPE_CHARACTERIZATION_DF is empty."
    )


# ==============================================================================
# 5.5 — OUTPUT SCHEMA VALIDATION
# ==============================================================================

required_output_columns = [
    "dataset_id",
    "feature_order",
    "feature",
    "feature_type",
    "schema_role",
    "type_source",
    "native_dtype",
    "is_target",
    "is_preprocessing_feature",
    "is_generative_feature",
    "training_rows",
    "non_missing_count",
    "missing_count",
    "missing_rate",
    "unique_count",
    "unique_ratio",
    "is_constant",
    "is_near_constant",
]

missing_output_columns = [
    column
    for column in required_output_columns
    if column not in FEATURE_TYPE_CHARACTERIZATION_DF.columns
]

if missing_output_columns:
    raise RuntimeError(
        "Feature-type characterization is missing required columns:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_output_columns
        )
    )


# ==============================================================================
# 5.6 — DATASET-WISE SCHEMA INTEGRITY VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    dataset_rows = (
        FEATURE_TYPE_CHARACTERIZATION_DF[
            FEATURE_TYPE_CHARACTERIZATION_DF[
                "dataset_id"
            ] == dataset_id
        ]
    )

    schema_contract = (
        NB02_PERSISTED_SCHEMA_CONTRACTS[
            dataset_id
        ]
    )

    modeling_schema = (
        schema_contract["modeling_schema"]
    )

    expected_preprocessing = list(
        modeling_schema["preprocessing_columns"]
    )

    expected_numeric = list(
        modeling_schema["numeric_columns"]
    )

    expected_categorical = list(
        modeling_schema["categorical_columns"]
    )

    expected_generative = list(
        modeling_schema["generative_columns"]
    )

    expected_target = (
        modeling_schema["target_column"]
    )

    # --------------------------------------------------------------------------
    # Feature count
    # --------------------------------------------------------------------------

    if len(dataset_rows) != len(expected_generative):

        raise RuntimeError(
            f"{dataset_id}: feature count mismatch. "
            f"Observed={len(dataset_rows)}, "
            f"Expected={len(expected_generative)}."
        )

    # --------------------------------------------------------------------------
    # Feature identity
    # --------------------------------------------------------------------------

    observed_features = set(
        dataset_rows["feature"]
    )

    expected_features = set(
        expected_generative
    )

    if observed_features != expected_features:

        missing_features = sorted(
            expected_features - observed_features
        )

        unexpected_features = sorted(
            observed_features - expected_features
        )

        raise RuntimeError(
            f"{dataset_id}: feature identity mismatch.\n"
            f"Missing    : {missing_features}\n"
            f"Unexpected : {unexpected_features}"
        )

    # --------------------------------------------------------------------------
    # Target validation
    # --------------------------------------------------------------------------

    target_rows = dataset_rows[
        dataset_rows["is_target"]
    ]

    if len(target_rows) != 1:

        raise RuntimeError(
            f"{dataset_id}: expected exactly one target record; "
            f"found {len(target_rows)}."
        )

    observed_target = target_rows.iloc[0]["feature"]

    if observed_target != expected_target:

        raise RuntimeError(
            f"{dataset_id}: target mismatch.\n"
            f"Observed : {observed_target}\n"
            f"Expected : {expected_target}"
        )

    # --------------------------------------------------------------------------
    # Target must NOT be a preprocessing feature
    # --------------------------------------------------------------------------

    target_is_preprocessing = bool(
        target_rows.iloc[0][
            "is_preprocessing_feature"
        ]
    )

    if target_is_preprocessing:

        raise RuntimeError(
            f"{dataset_id}: target '{expected_target}' "
            "was incorrectly marked as a preprocessing feature."
        )

    # --------------------------------------------------------------------------
    # Numeric preprocessing features
    # --------------------------------------------------------------------------

    numeric_mask = (
        dataset_rows["is_preprocessing_feature"]
        &
        (
            dataset_rows["feature_type"]
            == "numeric"
        )
    )

    observed_numeric = set(
        dataset_rows.loc[
            numeric_mask,
            "feature"
        ]
    )

    if observed_numeric != set(
        expected_numeric
    ):

        raise RuntimeError(
            f"{dataset_id}: numeric preprocessing feature mismatch.\n"
            f"Observed : {sorted(observed_numeric)}\n"
            f"Expected : {sorted(expected_numeric)}"
        )

    # --------------------------------------------------------------------------
    # Categorical preprocessing features
    # --------------------------------------------------------------------------

    categorical_mask = (
        dataset_rows["is_preprocessing_feature"]
        &
        (
            dataset_rows["feature_type"]
            == "categorical"
        )
    )

    observed_categorical = set(
        dataset_rows.loc[
            categorical_mask,
            "feature"
        ]
    )

    if observed_categorical != set(
        expected_categorical
    ):

        raise RuntimeError(
            f"{dataset_id}: categorical preprocessing feature mismatch.\n"
            f"Observed : {sorted(observed_categorical)}\n"
            f"Expected : {sorted(expected_categorical)}"
        )

    # --------------------------------------------------------------------------
    # Generative feature coverage
    # --------------------------------------------------------------------------

    observed_generative = set(
        dataset_rows.loc[
            dataset_rows["is_generative_feature"],
            "feature"
        ]
    )

    if observed_generative != set(
        expected_generative
    ):

        raise RuntimeError(
            f"{dataset_id}: generative feature coverage mismatch."
        )

    # --------------------------------------------------------------------------
    # Every generative variable has exactly one type
    # --------------------------------------------------------------------------

    typed_features = set(
        dataset_rows.loc[
            dataset_rows["feature_type"].isin(
                ["numeric", "categorical"]
            ),
            "feature"
        ]
    )

    if typed_features != expected_features:

        raise RuntimeError(
            f"{dataset_id}: not every generative feature "
            "has a valid numeric/categorical type."
        )


# ==============================================================================
# 5.7 — DATASET-LEVEL SUMMARY
# ==============================================================================

for dataset_id in DATASET_IDS:

    dataset_rows = (
        FEATURE_TYPE_CHARACTERIZATION_DF[
            FEATURE_TYPE_CHARACTERIZATION_DF[
                "dataset_id"
            ] == dataset_id
        ]
    )

    preprocessing_rows = (
        dataset_rows[
            dataset_rows[
                "is_preprocessing_feature"
            ]
        ]
    )

    numeric_rows = (
        preprocessing_rows[
            preprocessing_rows[
                "feature_type"
            ] == "numeric"
        ]
    )

    categorical_rows = (
        preprocessing_rows[
            preprocessing_rows[
                "feature_type"
            ] == "categorical"
        ]
    )

    target_rows = (
        dataset_rows[
            dataset_rows[
                "is_target"
            ]
        ]
    )

    FEATURE_TYPE_SUMMARY_RECORDS.append({

        "dataset_id": dataset_id,

        "training_rows": int(
            dataset_rows[
                "training_rows"
            ].iloc[0]
        ),

        "generative_variables": int(
            len(dataset_rows)
        ),

        "preprocessing_features": int(
            len(preprocessing_rows)
        ),

        "numeric_features": int(
            len(numeric_rows)
        ),

        "categorical_features": int(
            len(categorical_rows)
        ),

        "target_feature": str(
            target_rows[
                "feature"
            ].iloc[0]
        ),

        "target_type": str(
            target_rows[
                "feature_type"
            ].iloc[0]
        ),

        "target_native_dtype": str(
            target_rows[
                "native_dtype"
            ].iloc[0]
        ),

        "constant_features": int(
            dataset_rows[
                "is_constant"
            ].sum()
        ),

        "near_constant_features": int(
            dataset_rows[
                "is_near_constant"
            ].sum()
        ),

        "features_with_missing_values": int(
            (
                dataset_rows[
                    "missing_count"
                ] > 0
            ).sum()
        ),

        "maximum_cardinality": int(
            dataset_rows[
                "unique_count"
            ].max()
        ),
    })


FEATURE_TYPE_SUMMARY_DF = pd.DataFrame(
    FEATURE_TYPE_SUMMARY_RECORDS
)


# ==============================================================================
# 5.8 — DUPLICATE RECORD VALIDATION
# ==============================================================================

duplicate_mask = (
    FEATURE_TYPE_CHARACTERIZATION_DF[
        ["dataset_id", "feature"]
    ]
    .duplicated(keep=False)
)

if duplicate_mask.any():

    duplicate_rows = (
        FEATURE_TYPE_CHARACTERIZATION_DF[
            duplicate_mask
        ]
    )

    raise RuntimeError(
        "Duplicate dataset-feature records detected:\n"
        + duplicate_rows.to_string(index=False)
    )


# ==============================================================================
# 5.9 — DISPLAY DATASET SUMMARY
# ==============================================================================

print("\nDATASET-LEVEL FEATURE-TYPE SUMMARY")
print("-" * 100)

print(
    FEATURE_TYPE_SUMMARY_DF.to_string(
        index=False
    )
)


# ==============================================================================
# 5.10 — DISPLAY FEATURE-LEVEL CHARACTERIZATION
# ==============================================================================

print("\nFEATURE-LEVEL CHARACTERIZATION")
print("-" * 100)

display(
    FEATURE_TYPE_CHARACTERIZATION_DF[
        [
            "dataset_id",
            "feature_order",
            "feature",
            "feature_type",
            "schema_role",
            "type_source",
            "native_dtype",
            "is_target",
            "training_rows",
            "missing_count",
            "missing_rate",
            "unique_count",
            "unique_ratio",
            "is_constant",
            "is_near_constant",
        ]
    ]
)


# ==============================================================================
# 5.11 — FINAL VALIDATION COUNTS
# ==============================================================================

total_records = len(
    FEATURE_TYPE_CHARACTERIZATION_DF
)

total_datasets = (
    FEATURE_TYPE_CHARACTERIZATION_DF[
        "dataset_id"
    ].nunique()
)

total_preprocessing = int(
    FEATURE_TYPE_CHARACTERIZATION_DF[
        "is_preprocessing_feature"
    ].sum()
)

total_numeric_preprocessing = int(
    (
        FEATURE_TYPE_CHARACTERIZATION_DF[
            "is_preprocessing_feature"
        ]
        &
        (
            FEATURE_TYPE_CHARACTERIZATION_DF[
                "feature_type"
            ]
            == "numeric"
        )
    ).sum()
)

total_categorical_preprocessing = int(
    (
        FEATURE_TYPE_CHARACTERIZATION_DF[
            "is_preprocessing_feature"
        ]
        &
        (
            FEATURE_TYPE_CHARACTERIZATION_DF[
                "feature_type"
            ]
            == "categorical"
        )
    ).sum()
)

total_targets = int(
    FEATURE_TYPE_CHARACTERIZATION_DF[
        "is_target"
    ].sum()
)

total_constant = int(
    FEATURE_TYPE_CHARACTERIZATION_DF[
        "is_constant"
    ].sum()
)

total_near_constant = int(
    FEATURE_TYPE_CHARACTERIZATION_DF[
        "is_near_constant"
    ].sum()
)

total_missing_features = int(
    (
        FEATURE_TYPE_CHARACTERIZATION_DF[
            "missing_count"
        ] > 0
    ).sum()
)


# ==============================================================================
# 5.12 — FINAL STATUS
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 5 VALIDATION")
print("=" * 100)

print(
    f"Datasets characterized        : {total_datasets}"
)

print(
    f"Feature records               : {total_records:,}"
)

print(
    f"Preprocessing feature records : {total_preprocessing:,}"
)

print(
    f"Numeric preprocessing records : "
    f"{total_numeric_preprocessing:,}"
)

print(
    f"Categorical preprocessing     : "
    f"{total_categorical_preprocessing:,}"
)

print(
    f"Target records                : {total_targets:,}"
)

print(
    f"Constant features             : {total_constant:,}"
)

print(
    f"Near-constant features        : {total_near_constant:,}"
)

print(
    f"Features with missing values  : "
    f"{total_missing_features:,}"
)

print("\n✓ TRAIN-only characterization verified.")
print("✓ Frozen Notebook 02 schema contract verified.")
print("✓ Numeric preprocessing feature identities verified.")
print("✓ Categorical preprocessing feature identities verified.")
print("✓ Target characterized independently from preprocessing schema.")
print("✓ Target excluded from preprocessing-feature classification.")
print("✓ Generative feature coverage verified.")
print("✓ No validation/test data accessed.")
print("✓ No synthetic data accessed.")
print("\nSECTION 5 STATUS: PASS")
print("=" * 100)

SECTION 5 — FEATURE-TYPE CHARACTERIZATION

DATASET-LEVEL FEATURE-TYPE SUMMARY
----------------------------------------------------------------------------------------------------
    dataset_id  training_rows  generative_variables  preprocessing_features  numeric_features  categorical_features target_feature target_type target_native_dtype  constant_features  near_constant_features  features_with_missing_values  maximum_cardinality
  adult_income          34189                    15                      14                 6                     8         income categorical              object                  0                       1                             3                22452
bank_marketing          31647                    17                      16                 7                     9              y categorical              object                  0                       1                             0                 6278
diabetes_130us          71236                    4

,dataset_id,feature_order,feature,feature_type,schema_role,type_source,native_dtype,is_target,training_rows,missing_count,missing_rate,unique_count,unique_ratio,is_constant,is_near_constant
0,adult_income,0,age,numeric,preprocessing_feature,notebook_02_persisted_schema,int64,False,34189,0,0.000000,73,0.002135,False,False
1,adult_income,1,workclass,categorical,preprocessing_feature,notebook_02_persisted_schema,object,False,34189,1992,0.058264,8,0.000248,False,False
2,adult_income,2,fnlwgt,numeric,preprocessing_feature,notebook_02_persisted_schema,int64,False,34189,0,0.000000,22452,0.656702,False,False
3,adult_income,3,education,categorical,preprocessing_feature,notebook_02_persisted_schema,object,False,34189,0,0.000000,16,0.000468,False,False
4,adult_income,4,education_num,numeric,preprocessing_feature,notebook_02_persisted_schema,int64,False,34189,0,0.000000,16,0.000468,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,diabetes_130us,43,metformin-rosiglitazone,categorical,preprocessing_feature,notebook_02_persisted_schema,object,False,71236,0,0.000000,2,0.000028,False,True
76,diabetes_130us,44,metformin-pioglitazone,categorical,preprocessing_feature,notebook_02_persisted_schema,object,False,71236,0,0.000000,2,0.000028,False,True
77,diabetes_130us,45,change,categorical,preprocessing_feature,notebook_02_persisted_schema,object,False,71236,0,0.000000,2,0.000028,False,False
78,diabetes_130us,46,diabetesMed,categorical,preprocessing_feature,notebook_02_persisted_schema,object,False,71236,0,0.000000,2,0.000028,False,False



SECTION 5 VALIDATION
Datasets characterized        : 3
Feature records               : 80
Preprocessing feature records : 77
Numeric preprocessing records : 24
Categorical preprocessing     : 53
Target records                : 3
Constant features             : 4
Near-constant features        : 19
Features with missing values  : 5

✓ TRAIN-only characterization verified.
✓ Frozen Notebook 02 schema contract verified.
✓ Numeric preprocessing feature identities verified.
✓ Categorical preprocessing feature identities verified.
✓ Target characterized independently from preprocessing schema.
✓ Target excluded from preprocessing-feature classification.
✓ Generative feature coverage verified.
✓ No validation/test data accessed.
✓ No synthetic data accessed.

SECTION 5 STATUS: PASS


In [6]:
# ==============================================================================
# SECTION 6 — NUMERICAL DESCRIPTIVE STATISTICS
# ==============================================================================
#
# PURPOSE
# -------
# Compute numerical descriptive statistics for the numerical preprocessing
# features using TRAINING data only.
#
# METHODOLOGICAL POLICY
# ---------------------
# 1. TRAINING data only.
# 2. No validation/test data access.
# 3. No synthetic data access.
# 4. Numerical feature identity follows the frozen Notebook 02 schema.
# 5. Native, unencoded training representation is used.
# 6. No preprocessing is refitted or modified.
# 7. Missing values are characterized, not imputed here.
# 8. Statistics are descriptive and do not perform model evaluation.
#
# OUTPUT
# ------
# NUMERICAL_STATISTICS_DF
#
# ==============================================================================

print("=" * 100)
print("SECTION 6 — NUMERICAL DESCRIPTIVE STATISTICS")
print("=" * 100)


# ==============================================================================
# 6.1 — REQUIRED OBJECT VALIDATION
# ==============================================================================

required_objects = [
    "DATASET_IDS",
    "TRAIN_STATISTICAL_DATASETS",
    "NB02_PERSISTED_SCHEMA_CONTRACTS",
    "FEATURE_TYPE_CHARACTERIZATION_DF",
]

missing_objects = [
    obj
    for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required Section 6 objects are missing:\n"
        + "\n".join(
            f"  - {obj}"
            for obj in missing_objects
        )
        + "\n\nExecute Sections 2–5 before Section 6."
    )


# ==============================================================================
# 6.2 — INITIALIZE OUTPUT CONTAINER
# ==============================================================================

NUMERICAL_STATISTICS_RECORDS = []


# ==============================================================================
# 6.3 — DATASET-WISE NUMERICAL CHARACTERIZATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    # --------------------------------------------------------------------------
    # TRAINING DATA ONLY
    # --------------------------------------------------------------------------

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    if not isinstance(df, pd.DataFrame):
        raise TypeError(
            f"{dataset_id}: training statistical dataset "
            "is not a pandas DataFrame."
        )

    # --------------------------------------------------------------------------
    # Load frozen Notebook 02 schema
    # --------------------------------------------------------------------------

    schema_contract = (
        NB02_PERSISTED_SCHEMA_CONTRACTS[
            dataset_id
        ]
    )

    modeling_schema = (
        schema_contract[
            "modeling_schema"
        ]
    )

    expected_numeric_columns = list(
        modeling_schema[
            "numeric_columns"
        ]
    )

    expected_preprocessing_columns = list(
        modeling_schema[
            "preprocessing_columns"
        ]
    )

    target_column = (
        modeling_schema[
            "target_column"
        ]
    )

    # ==============================================================================
    # 6.3.1 — VERIFY NUMERICAL FEATURES AGAINST SECTION 5
    # ==============================================================================

    section5_rows = (
        FEATURE_TYPE_CHARACTERIZATION_DF[
            FEATURE_TYPE_CHARACTERIZATION_DF[
                "dataset_id"
            ] == dataset_id
        ]
    )

    if section5_rows.empty:
        raise RuntimeError(
            f"{dataset_id}: no Section 5 feature-characterization "
            "records found."
        )

    section5_numeric_columns = (
        section5_rows.loc[
            (
                section5_rows[
                    "feature_type"
                ] == "numeric"
            )
            &
            (
                section5_rows[
                    "is_preprocessing_feature"
                ]
            ),
            "feature"
        ]
        .tolist()
    )

    if set(section5_numeric_columns) != set(
        expected_numeric_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: Section 5 numerical feature identity "
            "does not match the frozen Notebook 02 schema.\n"
            f"Section 5 : {section5_numeric_columns}\n"
            f"Notebook 02: {expected_numeric_columns}"
        )

    # --------------------------------------------------------------------------
    # Target must not accidentally enter preprocessing statistics
    # --------------------------------------------------------------------------

    if target_column in expected_numeric_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' appears in "
            "Notebook 02 numeric preprocessing features. "
            "This violates the frozen target policy."
        )

    # ==============================================================================
    # 6.3.2 — COMPUTE NUMERICAL DESCRIPTIVE STATISTICS
    # ==============================================================================

    for feature in expected_numeric_columns:

        if feature not in df.columns:
            raise RuntimeError(
                f"{dataset_id}: numerical feature '{feature}' "
                "is missing from training data."
            )

        # ----------------------------------------------------------------------
        # Convert only for safe numerical statistical computation.
        # Original dataframe is not modified.
        # ----------------------------------------------------------------------

        s = pd.to_numeric(
            df[feature],
            errors="coerce"
        )

        non_missing = s.dropna()

        # ----------------------------------------------------------------------
        # Empty numerical variable
        # ----------------------------------------------------------------------

        if len(non_missing) == 0:

            NUMERICAL_STATISTICS_RECORDS.append({

                "dataset_id": dataset_id,

                "feature": feature,

                "count": 0,

                "missing": int(
                    s.isna().sum()
                ),

                "missing_rate": float(
                    s.isna().mean()
                ),

                "mean": np.nan,

                "std": np.nan,

                "min": np.nan,

                "q01": np.nan,

                "q05": np.nan,

                "q25": np.nan,

                "median": np.nan,

                "q75": np.nan,

                "q95": np.nan,

                "q99": np.nan,

                "max": np.nan,

                "skewness": np.nan,

                "kurtosis": np.nan,

                "n_unique": 0,

                "unique_ratio": np.nan,

                "zero_count": 0,

                "zero_rate": np.nan,

                "negative_count": 0,

                "negative_rate": np.nan,

                "positive_count": 0,

                "positive_rate": np.nan,
            })

            continue

        # ----------------------------------------------------------------------
        # Finite-value validation
        # ----------------------------------------------------------------------

        finite_mask = np.isfinite(
            non_missing.to_numpy()
        )

        finite_values = (
            non_missing.iloc[
                np.flatnonzero(
                    finite_mask
                )
            ]
        )

        non_finite_count = int(
            len(non_missing)
            - len(finite_values)
        )

        if non_finite_count > 0:

            raise RuntimeError(
                f"{dataset_id}: numerical feature '{feature}' "
                f"contains {non_finite_count} non-finite values."
            )

        # ----------------------------------------------------------------------
        # Quantiles
        # ----------------------------------------------------------------------

        q = finite_values.quantile(
            [
                0.01,
                0.05,
                0.25,
                0.50,
                0.75,
                0.95,
                0.99,
            ]
        )

        # ----------------------------------------------------------------------
        # Counts
        # ----------------------------------------------------------------------

        count = int(
            finite_values.count()
        )

        missing = int(
            s.isna().sum()
        )

        missing_rate = float(
            s.isna().mean()
        )

        n_unique = int(
            finite_values.nunique()
        )

        unique_ratio = (
            n_unique / count
            if count > 0
            else np.nan
        )

        # ----------------------------------------------------------------------
        # Zero / sign characterization
        # ----------------------------------------------------------------------

        zero_count = int(
            (finite_values == 0).sum()
        )

        negative_count = int(
            (finite_values < 0).sum()
        )

        positive_count = int(
            (finite_values > 0).sum()
        )

        zero_rate = (
            zero_count / count
            if count > 0
            else np.nan
        )

        negative_rate = (
            negative_count / count
            if count > 0
            else np.nan
        )

        positive_rate = (
            positive_count / count
            if count > 0
            else np.nan
        )

        # ----------------------------------------------------------------------
        # Dispersion / distribution statistics
        # ----------------------------------------------------------------------

        if count > 1:

            std_value = float(
                finite_values.std(
                    ddof=1
                )
            )

        else:

            std_value = 0.0

        if count > 2:

            skewness = float(
                finite_values.skew()
            )

        else:

            skewness = np.nan

        if count > 3:

            kurtosis = float(
                finite_values.kurtosis()
            )

        else:

            kurtosis = np.nan

        # ----------------------------------------------------------------------
        # Store record
        # ----------------------------------------------------------------------

        NUMERICAL_STATISTICS_RECORDS.append({

            "dataset_id": dataset_id,

            "feature": feature,

            "count": count,

            "missing": missing,

            "missing_rate": missing_rate,

            "mean": float(
                finite_values.mean()
            ),

            "std": std_value,

            "min": float(
                finite_values.min()
            ),

            "q01": float(
                q.loc[0.01]
            ),

            "q05": float(
                q.loc[0.05]
            ),

            "q25": float(
                q.loc[0.25]
            ),

            "median": float(
                q.loc[0.50]
            ),

            "q75": float(
                q.loc[0.75]
            ),

            "q95": float(
                q.loc[0.95]
            ),

            "q99": float(
                q.loc[0.99]
            ),

            "max": float(
                finite_values.max()
            ),

            "skewness": skewness,

            "kurtosis": kurtosis,

            "n_unique": n_unique,

            "unique_ratio": unique_ratio,

            "zero_count": zero_count,

            "zero_rate": zero_rate,

            "negative_count": negative_count,

            "negative_rate": negative_rate,

            "positive_count": positive_count,

            "positive_rate": positive_rate,
        })


# ==============================================================================
# 6.4 — CREATE OUTPUT DATAFRAME
# ==============================================================================

NUMERICAL_STATISTICS_DF = pd.DataFrame(
    NUMERICAL_STATISTICS_RECORDS
)


# ==============================================================================
# 6.5 — OUTPUT VALIDATION
# ==============================================================================

required_statistics_columns = [
    "dataset_id",
    "feature",
    "count",
    "missing",
    "missing_rate",
    "mean",
    "std",
    "min",
    "q01",
    "q05",
    "q25",
    "median",
    "q75",
    "q95",
    "q99",
    "max",
    "skewness",
    "kurtosis",
    "n_unique",
    "unique_ratio",
    "zero_count",
    "zero_rate",
    "negative_count",
    "negative_rate",
    "positive_count",
    "positive_rate",
]

missing_statistics_columns = [
    column
    for column in required_statistics_columns
    if column not in NUMERICAL_STATISTICS_DF.columns
]

if missing_statistics_columns:

    raise RuntimeError(
        "Numerical statistics output is missing required columns:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_statistics_columns
        )
    )


# ==============================================================================
# 6.6 — FEATURE COVERAGE VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    observed_features = set(
        NUMERICAL_STATISTICS_DF.loc[
            NUMERICAL_STATISTICS_DF[
                "dataset_id"
            ] == dataset_id,
            "feature"
        ]
    )

    expected_features = set(
        NB02_PERSISTED_SCHEMA_CONTRACTS[
            dataset_id
        ][
            "modeling_schema"
        ][
            "numeric_columns"
        ]
    )

    if observed_features != expected_features:

        missing_features = sorted(
            expected_features
            - observed_features
        )

        unexpected_features = sorted(
            observed_features
            - expected_features
        )

        raise RuntimeError(
            f"{dataset_id}: numerical statistics feature coverage mismatch.\n"
            f"Missing    : {missing_features}\n"
            f"Unexpected : {unexpected_features}"
        )


# ==============================================================================
# 6.7 — DUPLICATE VALIDATION
# ==============================================================================

duplicate_mask = (
    NUMERICAL_STATISTICS_DF[
        [
            "dataset_id",
            "feature"
        ]
    ]
    .duplicated(
        keep=False
    )
)

if duplicate_mask.any():

    raise RuntimeError(
        "Duplicate dataset-feature numerical statistics detected:\n"
        +
        NUMERICAL_STATISTICS_DF[
            duplicate_mask
        ].to_string(index=False)
    )


# ==============================================================================
# 6.8 — STATISTICAL SANITY CHECKS
# ==============================================================================

if (
    NUMERICAL_STATISTICS_DF[
        "missing_rate"
    ]
    .dropna()
    .lt(0)
    .any()
    or
    NUMERICAL_STATISTICS_DF[
        "missing_rate"
    ]
    .dropna()
    .gt(1)
    .any()
):

    raise RuntimeError(
        "Invalid missing_rate detected."
    )


if (
    NUMERICAL_STATISTICS_DF[
        "unique_ratio"
    ]
    .dropna()
    .lt(0)
    .any()
    or
    NUMERICAL_STATISTICS_DF[
        "unique_ratio"
    ]
    .dropna()
    .gt(1)
    .any()
):

    raise RuntimeError(
        "Invalid unique_ratio detected."
    )


if (
    NUMERICAL_STATISTICS_DF[
        "zero_rate"
    ]
    .dropna()
    .lt(0)
    .any()
    or
    NUMERICAL_STATISTICS_DF[
        "zero_rate"
    ]
    .dropna()
    .gt(1)
    .any()
):

    raise RuntimeError(
        "Invalid zero_rate detected."
    )


# ==============================================================================
# 6.9 — DISPLAY RESULTS
# ==============================================================================

print(
    f"\nNumerical feature statistics generated: "
    f"{len(NUMERICAL_STATISTICS_DF):,} rows"
)

print(
    f"Datasets characterized: "
    f"{NUMERICAL_STATISTICS_DF['dataset_id'].nunique()}"
)

print(
    f"Unique numerical features: "
    f"{NUMERICAL_STATISTICS_DF['feature'].nunique()}"
)

print("\nNUMERICAL DESCRIPTIVE STATISTICS")
print("-" * 100)

display(
    NUMERICAL_STATISTICS_DF
)


# ==============================================================================
# 6.10 — FINAL STATUS
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 6 VALIDATION")
print("=" * 100)

print("✓ Training data only.")
print("✓ Frozen Notebook 02 numerical feature schema verified.")
print("✓ Section 5 numerical feature identities verified.")
print("✓ Native, unencoded representation used.")
print("✓ Missingness characterized without imputation.")
print("✓ Quantile statistics computed.")
print("✓ Distributional moments computed.")
print("✓ Numerical feature coverage verified.")
print("✓ Duplicate dataset-feature records absent.")
print("✓ Statistical sanity checks passed.")
print("✓ No validation/test data accessed.")
print("✓ No synthetic data accessed.")

print("\nSECTION 6 STATUS: PASS")
print("=" * 100)

SECTION 6 — NUMERICAL DESCRIPTIVE STATISTICS

Numerical feature statistics generated: 24 rows
Datasets characterized: 3
Unique numerical features: 23

NUMERICAL DESCRIPTIVE STATISTICS
----------------------------------------------------------------------------------------------------


,dataset_id,feature,count,missing,missing_rate,mean,std,min,q01,q05,...,skewness,kurtosis,n_unique,unique_ratio,zero_count,zero_rate,negative_count,negative_rate,positive_count,positive_rate
0,adult_income,age,34189,0,0.0,38.679956,13.751078,17.0,17.00,19.0,...,0.560003,-0.167669,73,0.002135,0,0.000000,0,0.000000,34189,1.000000
1,adult_income,fnlwgt,34189,0,0.0,189718.707099,106299.973216,12285.0,26993.52,39529.4,...,1.517084,6.786484,22452,0.656702,0,0.000000,0,0.000000,34189,1.000000
2,adult_income,education_num,34189,0,0.0,10.069935,2.569360,1.0,3.00,5.0,...,-0.315443,0.638256,16,0.000468,0,0.000000,0,0.000000,34189,1.000000
3,adult_income,capital_gain,34189,0,0.0,1072.672146,7464.812462,0.0,0.00,0.0,...,11.911273,152.743010,120,0.003510,31375,0.917693,0,0.000000,2814,0.082307
4,adult_income,capital_loss,34189,0,0.0,86.581240,400.006883,0.0,0.00,0.0,...,4.582842,20.116260,95,0.002779,32601,0.953552,0,0.000000,1588,0.046448
5,adult_income,hours_per_week,34189,0,0.0,40.384480,12.411311,1.0,8.00,18.0,...,0.249085,2.977173,92,0.002691,0,0.000000,0,0.000000,34189,1.000000
6,bank_marketing,age,31647,0,0.0,40.923626,10.633585,18.0,23.00,27.0,...,0.688231,0.331053,77,0.002433,0,0.000000,0,0.000000,31647,1.000000
7,bank_marketing,balance,31647,0,0.0,1360.214712,3058.385160,-8019.0,-625.00,-171.7,...,8.577498,148.673931,6278,0.198376,2421,0.076500,2637,0.083325,26589,0.840174
8,bank_marketing,day,31647,0,0.0,15.785951,8.347214,1.0,2.00,3.0,...,0.095803,-1.066752,31,0.000980,0,0.000000,0,0.000000,31647,1.000000
9,bank_marketing,duration,31647,0,0.0,257.623061,258.547124,0.0,11.00,34.0,...,3.243001,19.843934,1470,0.046450,2,0.000063,0,0.000000,31645,0.999937



SECTION 6 VALIDATION
✓ Training data only.
✓ Frozen Notebook 02 numerical feature schema verified.
✓ Section 5 numerical feature identities verified.
✓ Native, unencoded representation used.
✓ Missingness characterized without imputation.
✓ Quantile statistics computed.
✓ Distributional moments computed.
✓ Numerical feature coverage verified.
✓ Duplicate dataset-feature records absent.
✓ Statistical sanity checks passed.
✓ No validation/test data accessed.
✓ No synthetic data accessed.

SECTION 6 STATUS: PASS


In [7]:
# ==============================================================================
# SECTION 7 — CATEGORICAL DESCRIPTIVE STATISTICS
# ==============================================================================

print("=" * 100)
print("SECTION 7 — CATEGORICAL DESCRIPTIVE STATISTICS")
print("=" * 100)

# ------------------------------------------------------------------------------
# Required upstream objects
# ------------------------------------------------------------------------------

REQUIRED_SECTION_7_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_STATISTICAL_DATASETS",
    "NB02_PERSISTED_SCHEMA_CONTRACTS",
    "FEATURE_TYPE_CHARACTERIZATION_DF",
]

_missing_section_7_objects = [
    name
    for name in REQUIRED_SECTION_7_OBJECTS
    if name not in globals()
]

if _missing_section_7_objects:
    raise RuntimeError(
        "Section 7 cannot proceed because required upstream objects are missing: "
        f"{_missing_section_7_objects}"
    )

# ------------------------------------------------------------------------------
# Reset output container
# ------------------------------------------------------------------------------

CATEGORICAL_STATISTICS_RECORDS = []

# ------------------------------------------------------------------------------
# Process each dataset
# ------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print(f"\nProcessing dataset: {dataset_id}")

    # --------------------------------------------------------------------------
    # Load training-only statistical data
    # --------------------------------------------------------------------------

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    # --------------------------------------------------------------------------
    # Load frozen Notebook 02 schema contract
    # --------------------------------------------------------------------------

    schema_contract = NB02_PERSISTED_SCHEMA_CONTRACTS[dataset_id]

    modeling_schema = schema_contract["modeling_schema"]

    expected_categorical_columns = list(
        modeling_schema["categorical_columns"]
    )

    expected_numeric_columns = list(
        modeling_schema["numeric_columns"]
    )

    preprocessing_columns = list(
        modeling_schema["preprocessing_columns"]
    )

    generative_columns = list(
        modeling_schema["generative_columns"]
    )

    target_column = modeling_schema["target_column"]

    provenance_column = modeling_schema["provenance_column"]

    identifier_columns_excluded = list(
        modeling_schema["identifier_columns_excluded"]
    )

    # --------------------------------------------------------------------------
    # Validate persisted schema
    # --------------------------------------------------------------------------

    if target_column in expected_categorical_columns:
        raise RuntimeError(
            f"Target '{target_column}' in dataset '{dataset_id}' "
            "must not appear in Notebook 02 categorical preprocessing columns."
        )

    if target_column in expected_numeric_columns:
        raise RuntimeError(
            f"Target '{target_column}' in dataset '{dataset_id}' "
            "must not appear in Notebook 02 numeric preprocessing columns."
        )

    if provenance_column in expected_categorical_columns:
        raise RuntimeError(
            f"Provenance column '{provenance_column}' in dataset "
            f"'{dataset_id}' must not be a categorical preprocessing feature."
        )

    identifier_overlap = set(
        identifier_columns_excluded
    ).intersection(
        expected_categorical_columns
    )

    if identifier_overlap:
        raise RuntimeError(
            f"Excluded identifier columns incorrectly appear as categorical "
            f"preprocessing features for '{dataset_id}': "
            f"{sorted(identifier_overlap)}"
        )

    # --------------------------------------------------------------------------
    # Validate feature partition
    # --------------------------------------------------------------------------

    if set(expected_numeric_columns).intersection(
        expected_categorical_columns
    ):
        raise RuntimeError(
            f"Numeric/categorical feature overlap detected for '{dataset_id}'."
        )

    if set(expected_numeric_columns).union(
        expected_categorical_columns
    ) != set(preprocessing_columns):
        raise RuntimeError(
            f"Numeric + categorical preprocessing feature partition does not "
            f"match Notebook 02 preprocessing columns for '{dataset_id}'."
        )

    # --------------------------------------------------------------------------
    # Validate native generative schema
    # --------------------------------------------------------------------------

    expected_native_columns = [
        provenance_column
    ] + generative_columns

    if list(df.columns) != expected_native_columns:
        raise RuntimeError(
            f"Native column order/schema mismatch for '{dataset_id}'.\n"
            f"Expected: {expected_native_columns}\n"
            f"Observed: {list(df.columns)}"
        )

    # --------------------------------------------------------------------------
    # Validate generative schema
    # --------------------------------------------------------------------------

    expected_generative_columns = (
        preprocessing_columns
        + [target_column]
    )

    if generative_columns != expected_generative_columns:
        raise RuntimeError(
            f"Generative schema mismatch for '{dataset_id}'.\n"
            f"Expected: {expected_generative_columns}\n"
            f"Observed: {generative_columns}"
        )

    # --------------------------------------------------------------------------
    # Validate categorical feature presence
    # --------------------------------------------------------------------------

    missing_categorical_columns = [
        feature
        for feature in expected_categorical_columns
        if feature not in df.columns
    ]

    if missing_categorical_columns:
        raise RuntimeError(
            f"Categorical features missing from native training data "
            f"for '{dataset_id}': {missing_categorical_columns}"
        )

    # --------------------------------------------------------------------------
    # Cross-check against Section 5
    #
    # Section 5 is descriptive, but Notebook 02 remains authoritative.
    # --------------------------------------------------------------------------

    section_5_categorical_columns = (
        FEATURE_TYPE_CHARACTERIZATION_DF[
            (FEATURE_TYPE_CHARACTERIZATION_DF["dataset_id"] == dataset_id) &
            (
                FEATURE_TYPE_CHARACTERIZATION_DF["feature_type"]
                == "categorical"
            ) &
            (
                FEATURE_TYPE_CHARACTERIZATION_DF["is_preprocessing_feature"]
                == True
            )
        ]["feature"]
        .tolist()
    )

    if set(section_5_categorical_columns) != set(
        expected_categorical_columns
    ):
        raise RuntimeError(
            f"Section 5 categorical feature identity does not match the "
            f"frozen Notebook 02 schema for '{dataset_id}'.\n"
            f"Notebook 02: {expected_categorical_columns}\n"
            f"Section 5:   {section_5_categorical_columns}"
        )

    # --------------------------------------------------------------------------
    # Compute categorical statistics
    # --------------------------------------------------------------------------

    for feature in expected_categorical_columns:

        s = df[feature]

        total = int(len(s))

        missing_count = int(
            s.isna().sum()
        )

        non_missing = s.dropna()

        non_missing_count = int(
            len(non_missing)
        )

        # ----------------------------------------------------------------------
        # Category counts
        # ----------------------------------------------------------------------

        value_counts = (
            non_missing
            .astype("object")
            .value_counts(
                dropna=False,
                normalize=False
            )
        )

        if non_missing_count > 0:

            top_value = value_counts.index[0]

            top_count = int(
                value_counts.iloc[0]
            )

            top_frequency = float(
                top_count / non_missing_count
            )

        else:

            top_value = None
            top_count = 0
            top_frequency = np.nan

        # ----------------------------------------------------------------------
        # Cardinality
        # ----------------------------------------------------------------------

        n_unique = int(
            non_missing.nunique(
                dropna=True
            )
        )

        unique_ratio = float(
            n_unique / non_missing_count
        ) if non_missing_count > 0 else np.nan

        # ----------------------------------------------------------------------
        # Rare-category information
        #
        # Useful for identifying high-cardinality / sparse categorical
        # variables that may affect synthetic-data generation.
        # ----------------------------------------------------------------------

        if non_missing_count > 0:

            category_frequencies = (
                value_counts / non_missing_count
            )

            rare_category_count = int(
                (category_frequencies < 0.01).sum()
            )

            singleton_category_count = int(
                (value_counts == 1).sum()
            )

            min_category_frequency = float(
                category_frequencies.min()
            )

        else:

            rare_category_count = 0
            singleton_category_count = 0
            min_category_frequency = np.nan

        # ----------------------------------------------------------------------
        # Constant / near-constant categorical feature
        # ----------------------------------------------------------------------

        is_constant = (
            n_unique <= 1
        )

        is_near_constant = (
            top_frequency >= 0.95
            if non_missing_count > 0
            else False
        )

        # ----------------------------------------------------------------------
        # Store record
        # ----------------------------------------------------------------------

        CATEGORICAL_STATISTICS_RECORDS.append({

            "dataset_id": dataset_id,

            "feature": feature,

            "count": total,

            "non_missing": non_missing_count,

            "missing": missing_count,

            "missing_rate": float(
                missing_count / total
            ) if total > 0 else np.nan,

            "n_unique": n_unique,

            "unique_ratio": unique_ratio,

            "top_category": (
                str(top_value)
                if top_value is not None
                else None
            ),

            "top_count": top_count,

            "top_frequency": top_frequency,

            "min_category_frequency": (
                min_category_frequency
            ),

            "rare_category_count": (
                rare_category_count
            ),

            "singleton_category_count": (
                singleton_category_count
            ),

            "is_constant": bool(
                is_constant
            ),

            "is_near_constant": bool(
                is_near_constant
            ),

            "native_dtype": str(
                s.dtype
            ),

            "is_target": False,

            "is_preprocessing_feature": True,

            "is_generative_feature": True,

            "schema_role": "categorical_preprocessing_feature",

        })

# ==============================================================================
# Create DataFrame
# ==============================================================================

CATEGORICAL_STATISTICS_DF = pd.DataFrame(
    CATEGORICAL_STATISTICS_RECORDS
)

# ==============================================================================
# Validate output
# ==============================================================================

REQUIRED_SECTION_7_COLUMNS = [
    "dataset_id",
    "feature",
    "count",
    "non_missing",
    "missing",
    "missing_rate",
    "n_unique",
    "unique_ratio",
    "top_category",
    "top_count",
    "top_frequency",
    "min_category_frequency",
    "rare_category_count",
    "singleton_category_count",
    "is_constant",
    "is_near_constant",
    "native_dtype",
    "is_target",
    "is_preprocessing_feature",
    "is_generative_feature",
    "schema_role",
]

missing_output_columns = [
    column
    for column in REQUIRED_SECTION_7_COLUMNS
    if column not in CATEGORICAL_STATISTICS_DF.columns
]

if missing_output_columns:
    raise RuntimeError(
        "Section 7 output is missing required columns: "
        f"{missing_output_columns}"
    )

# ------------------------------------------------------------------------------
# Expected coverage
# ------------------------------------------------------------------------------

expected_categorical_feature_count = sum(
    len(
        NB02_PERSISTED_SCHEMA_CONTRACTS[dataset_id][
            "modeling_schema"
        ]["categorical_columns"]
    )
    for dataset_id in DATASET_IDS
)

observed_categorical_feature_count = len(
    CATEGORICAL_STATISTICS_DF
)

if observed_categorical_feature_count != (
    expected_categorical_feature_count
):
    raise RuntimeError(
        "Categorical statistics coverage mismatch.\n"
        f"Expected rows: {expected_categorical_feature_count}\n"
        f"Observed rows: {observed_categorical_feature_count}"
    )

# ------------------------------------------------------------------------------
# Duplicate check
# ------------------------------------------------------------------------------

duplicate_rows = (
    CATEGORICAL_STATISTICS_DF
    .duplicated(
        subset=["dataset_id", "feature"]
    )
    .sum()
)

if duplicate_rows != 0:
    raise RuntimeError(
        f"Duplicate dataset-feature rows detected: {duplicate_rows}"
    )

# ------------------------------------------------------------------------------
# Missing-rate validation
# ------------------------------------------------------------------------------

if (
    CATEGORICAL_STATISTICS_DF["missing_rate"]
    .dropna()
    .lt(0)
    .any()
    or
    CATEGORICAL_STATISTICS_DF["missing_rate"]
    .dropna()
    .gt(1)
    .any()
):
    raise RuntimeError(
        "Invalid categorical missing_rate values detected."
    )

# ------------------------------------------------------------------------------
# Frequency validation
# ------------------------------------------------------------------------------

for column in [
    "top_frequency",
    "min_category_frequency",
]:

    valid_values = (
        CATEGORICAL_STATISTICS_DF[column]
        .dropna()
    )

    if (
        (valid_values < 0).any()
        or
        (valid_values > 1).any()
    ):
        raise RuntimeError(
            f"Invalid frequency values detected in '{column}'."
        )

# ------------------------------------------------------------------------------
# Cardinality validation
# ------------------------------------------------------------------------------

if (
    CATEGORICAL_STATISTICS_DF["n_unique"]
    < 0
).any():

    raise RuntimeError(
        "Negative categorical cardinality detected."
    )

# ------------------------------------------------------------------------------
# Target exclusion validation
# ------------------------------------------------------------------------------

if CATEGORICAL_STATISTICS_DF["is_target"].any():
    raise RuntimeError(
        "Target feature incorrectly included in categorical "
        "preprocessing statistics."
    )

# ------------------------------------------------------------------------------
# Display
# ------------------------------------------------------------------------------

print(
    "\nCategorical feature statistics generated:"
)

print(
    f"  Datasets                  : {len(DATASET_IDS)}"
)

print(
    f"  Categorical features      : "
    f"{observed_categorical_feature_count}"
)

print(
    f"  Expected categorical      : "
    f"{expected_categorical_feature_count}"
)

print(
    f"  Duplicate dataset-feature : {duplicate_rows}"
)

print(
    "\nCategorical descriptive statistics:"
)

print(
    CATEGORICAL_STATISTICS_DF.to_string(
        index=False
    )
)

# ==============================================================================
# Section 7 completion gate
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 7 VALIDATION")
print("=" * 100)

print("✓ Statistics computed from TRAINING data only")
print("✓ Categorical feature identity taken from frozen Notebook 02 schema")
print("✓ Section 5 categorical classification cross-checked")
print("✓ Target excluded from preprocessing statistics")
print("✓ Identifier/provenance columns excluded")
print("✓ Missingness statistics validated")
print("✓ Cardinality statistics validated")
print("✓ Category-frequency statistics validated")
print("✓ Duplicate dataset-feature rows checked")
print("✓ Expected categorical feature coverage verified")

print("\nSECTION 7 STATUS: PASS")
print("=" * 100)

SECTION 7 — CATEGORICAL DESCRIPTIVE STATISTICS

Processing dataset: adult_income

Processing dataset: bank_marketing

Processing dataset: diabetes_130us

Categorical feature statistics generated:
  Datasets                  : 3
  Categorical features      : 53
  Expected categorical      : 53
  Duplicate dataset-feature : 0

Categorical descriptive statistics:
    dataset_id                  feature  count  non_missing  missing  missing_rate  n_unique  unique_ratio       top_category  top_count  top_frequency  min_category_frequency  rare_category_count  singleton_category_count  is_constant  is_near_constant native_dtype  is_target  is_preprocessing_feature  is_generative_feature                       schema_role
  adult_income                workclass  34189        32197     1992      0.058264         8      0.000248            Private      23655       0.734696                0.000155                    2                         0        False             False       object      Fals

In [8]:
# ==============================================================================
# SECTION 8 — MISSINGNESS CHARACTERIZATION
# ==============================================================================

print("=" * 100)
print("SECTION 8 — MISSINGNESS CHARACTERIZATION")
print("=" * 100)

# ------------------------------------------------------------------------------
# Required upstream objects
# ------------------------------------------------------------------------------

REQUIRED_SECTION_8_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_STATISTICAL_DATASETS",
    "NB02_PERSISTED_SCHEMA_CONTRACTS",
    "FEATURE_TYPE_CHARACTERIZATION_DF",
]

_missing_section_8_objects = [
    name
    for name in REQUIRED_SECTION_8_OBJECTS
    if name not in globals()
]

if _missing_section_8_objects:
    raise RuntimeError(
        "Section 8 cannot proceed because required upstream objects "
        f"are missing: {_missing_section_8_objects}"
    )

# ------------------------------------------------------------------------------
# Reset output containers
# ------------------------------------------------------------------------------

MISSINGNESS_RECORDS = []

# ==============================================================================
# DATASET-LEVEL PROCESSING
# ==============================================================================

for dataset_id in DATASET_IDS:

    print(f"\nProcessing dataset: {dataset_id}")

    # --------------------------------------------------------------------------
    # Load canonical TRAINING data only
    # --------------------------------------------------------------------------

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    # --------------------------------------------------------------------------
    # Load frozen Notebook 02 schema contract
    # --------------------------------------------------------------------------

    schema_contract = NB02_PERSISTED_SCHEMA_CONTRACTS[dataset_id]

    modeling_schema = schema_contract["modeling_schema"]

    preprocessing_columns = list(
        modeling_schema["preprocessing_columns"]
    )

    generative_columns = list(
        modeling_schema["generative_columns"]
    )

    numeric_columns = list(
        modeling_schema["numeric_columns"]
    )

    categorical_columns = list(
        modeling_schema["categorical_columns"]
    )

    target_column = modeling_schema["target_column"]

    provenance_column = modeling_schema["provenance_column"]

    identifier_columns_excluded = list(
        modeling_schema["identifier_columns_excluded"]
    )

    # ==========================================================================
    # SCHEMA VALIDATION
    # ==========================================================================

    # --------------------------------------------------------------------------
    # Expected native schema
    # --------------------------------------------------------------------------

    expected_native_columns = (
        [provenance_column]
        + generative_columns
    )

    if list(df.columns) != expected_native_columns:
        raise RuntimeError(
            f"Native schema mismatch for '{dataset_id}'.\n"
            f"Expected: {expected_native_columns}\n"
            f"Observed: {list(df.columns)}"
        )

    # --------------------------------------------------------------------------
    # Expected generative schema
    # --------------------------------------------------------------------------

    expected_generative_columns = (
        preprocessing_columns
        + [target_column]
    )

    if generative_columns != expected_generative_columns:
        raise RuntimeError(
            f"Generative schema mismatch for '{dataset_id}'.\n"
            f"Expected: {expected_generative_columns}\n"
            f"Observed: {generative_columns}"
        )

    # --------------------------------------------------------------------------
    # Numeric + categorical partition
    # --------------------------------------------------------------------------

    if set(numeric_columns).intersection(
        categorical_columns
    ):
        raise RuntimeError(
            f"Numeric/categorical feature overlap detected "
            f"for '{dataset_id}'."
        )

    if (
        set(numeric_columns).union(categorical_columns)
        != set(preprocessing_columns)
    ):
        raise RuntimeError(
            f"Numeric + categorical features do not exactly reproduce "
            f"Notebook 02 preprocessing columns for '{dataset_id}'."
        )

    # --------------------------------------------------------------------------
    # Target exclusion
    # --------------------------------------------------------------------------

    if target_column in preprocessing_columns:
        raise RuntimeError(
            f"Target '{target_column}' incorrectly appears in "
            f"preprocessing columns for '{dataset_id}'."
        )

    if target_column in numeric_columns:
        raise RuntimeError(
            f"Target '{target_column}' incorrectly appears in numeric "
            f"preprocessing columns for '{dataset_id}'."
        )

    if target_column in categorical_columns:
        raise RuntimeError(
            f"Target '{target_column}' incorrectly appears in categorical "
            f"preprocessing columns for '{dataset_id}'."
        )

    # --------------------------------------------------------------------------
    # Provenance exclusion
    # --------------------------------------------------------------------------

    if provenance_column in preprocessing_columns:
        raise RuntimeError(
            f"Provenance column '{provenance_column}' incorrectly appears "
            f"in preprocessing columns for '{dataset_id}'."
        )

    # --------------------------------------------------------------------------
    # Identifier exclusion
    # --------------------------------------------------------------------------

    identifier_overlap = set(
        identifier_columns_excluded
    ).intersection(
        preprocessing_columns
    )

    if identifier_overlap:
        raise RuntimeError(
            f"Excluded identifier columns incorrectly appear in preprocessing "
            f"columns for '{dataset_id}': "
            f"{sorted(identifier_overlap)}"
        )

    # ==========================================================================
    # CROSS-CHECK SECTION 5
    # ==========================================================================

    section_5_features = set(
        FEATURE_TYPE_CHARACTERIZATION_DF[
            (
                FEATURE_TYPE_CHARACTERIZATION_DF["dataset_id"]
                == dataset_id
            )
            &
            (
                FEATURE_TYPE_CHARACTERIZATION_DF[
                    "is_preprocessing_feature"
                ]
                == True
            )
        ]["feature"]
    )

    if section_5_features != set(preprocessing_columns):
        raise RuntimeError(
            f"Section 5 preprocessing feature universe does not match "
            f"Notebook 02 for '{dataset_id}'.\n"
            f"Notebook 02 count: {len(preprocessing_columns)}\n"
            f"Section 5 count:   {len(section_5_features)}"
        )

    # ==========================================================================
    # FEATURE-LEVEL MISSINGNESS
    # ==========================================================================

    total_rows = int(len(df))

    if total_rows == 0:
        raise RuntimeError(
            f"Training dataset '{dataset_id}' contains zero rows."
        )

    for feature in preprocessing_columns:

        s = df[feature]

        missing_count = int(
            s.isna().sum()
        )

        observed_count = int(
            total_rows - missing_count
        )

        missing_rate = float(
            missing_count / total_rows
        )

        observed_rate = float(
            observed_count / total_rows
        )

        # ----------------------------------------------------------------------
        # Cross-check against Section 5.
        #
        # Section 5 stores missing_rate rounded to 8 decimal places.
        # Therefore compare at the same precision rather than requiring
        # artificial full-precision equality.
        # ----------------------------------------------------------------------

        section_5_row = FEATURE_TYPE_CHARACTERIZATION_DF[
            (
                FEATURE_TYPE_CHARACTERIZATION_DF["dataset_id"]
                == dataset_id
            )
            &
            (
                FEATURE_TYPE_CHARACTERIZATION_DF["feature"]
                == feature
            )
        ]

        if len(section_5_row) != 1:
            raise RuntimeError(
                f"Expected exactly one Section 5 record for "
                f"'{dataset_id}' / '{feature}', "
                f"found {len(section_5_row)}."
            )

        section_5_missing_rate = float(
            section_5_row["missing_rate"].iloc[0]
        )

        if round(missing_rate, 8) != round(
            section_5_missing_rate,
            8
        ):
            raise RuntimeError(
                f"Missing-rate mismatch for "
                f"'{dataset_id}' / '{feature}'.\n"
                f"Section 8: {missing_rate}\n"
                f"Section 5: {section_5_missing_rate}"
            )

        # ----------------------------------------------------------------------
        # Descriptive missingness level
        # ----------------------------------------------------------------------

        if missing_rate == 0.0:
            missingness_level = "none"

        elif missing_rate < 0.05:
            missingness_level = "low"

        elif missing_rate < 0.20:
            missingness_level = "moderate"

        elif missing_rate < 0.50:
            missingness_level = "high"

        else:
            missingness_level = "very_high"

        # ----------------------------------------------------------------------
        # Store feature-level record
        # ----------------------------------------------------------------------

        MISSINGNESS_RECORDS.append({

            "dataset_id": dataset_id,

            "feature": feature,

            "count": total_rows,

            "missing_count": missing_count,

            "observed_count": observed_count,

            "missing_rate": missing_rate,

            "observed_rate": observed_rate,

            "missingness_level": missingness_level,

            "is_numeric": bool(
                feature in numeric_columns
            ),

            "is_categorical": bool(
                feature in categorical_columns
            ),

            "is_target": False,

            "is_preprocessing_feature": True,

            "is_generative_feature": True,

            "schema_role": "preprocessing_feature",

        })

# ==============================================================================
# CREATE FEATURE-LEVEL DATAFRAME
# ==============================================================================

MISSINGNESS_DF = pd.DataFrame(
    MISSINGNESS_RECORDS
)

# ------------------------------------------------------------------------------
# Required output columns
# ------------------------------------------------------------------------------

REQUIRED_MISSINGNESS_COLUMNS = [
    "dataset_id",
    "feature",
    "count",
    "missing_count",
    "observed_count",
    "missing_rate",
    "observed_rate",
    "missingness_level",
    "is_numeric",
    "is_categorical",
    "is_target",
    "is_preprocessing_feature",
    "is_generative_feature",
    "schema_role",
]

missing_output_columns = [
    column
    for column in REQUIRED_MISSINGNESS_COLUMNS
    if column not in MISSINGNESS_DF.columns
]

if missing_output_columns:
    raise RuntimeError(
        "MISSINGNESS_DF is missing required columns: "
        f"{missing_output_columns}"
    )

# ==============================================================================
# COVERAGE VALIDATION
# ==============================================================================

expected_feature_count = sum(
    len(
        NB02_PERSISTED_SCHEMA_CONTRACTS[dataset_id][
            "modeling_schema"
        ]["preprocessing_columns"]
    )
    for dataset_id in DATASET_IDS
)

observed_feature_count = len(
    MISSINGNESS_DF
)

if observed_feature_count != expected_feature_count:
    raise RuntimeError(
        "Missingness feature coverage mismatch.\n"
        f"Expected: {expected_feature_count}\n"
        f"Observed: {observed_feature_count}"
    )

# ------------------------------------------------------------------------------
# Duplicate dataset-feature check
# ------------------------------------------------------------------------------

duplicate_count = int(
    MISSINGNESS_DF
    .duplicated(
        subset=["dataset_id", "feature"]
    )
    .sum()
)

if duplicate_count != 0:
    raise RuntimeError(
        f"Duplicate dataset-feature rows detected: "
        f"{duplicate_count}"
    )

# ==============================================================================
# NUMERICAL SANITY CHECKS
# ==============================================================================

# ------------------------------------------------------------------------------
# Missing-rate bounds
# ------------------------------------------------------------------------------

if (
    MISSINGNESS_DF["missing_rate"].lt(0).any()
    or
    MISSINGNESS_DF["missing_rate"].gt(1).any()
):
    raise RuntimeError(
        "Invalid missing_rate values detected."
    )

# ------------------------------------------------------------------------------
# Observed-rate bounds
# ------------------------------------------------------------------------------

if (
    MISSINGNESS_DF["observed_rate"].lt(0).any()
    or
    MISSINGNESS_DF["observed_rate"].gt(1).any()
):
    raise RuntimeError(
        "Invalid observed_rate values detected."
    )

# ------------------------------------------------------------------------------
# Count consistency
# ------------------------------------------------------------------------------

count_consistency = (
    MISSINGNESS_DF["missing_count"]
    +
    MISSINGNESS_DF["observed_count"]
    ==
    MISSINGNESS_DF["count"]
)

if not count_consistency.all():
    raise RuntimeError(
        "Missing + observed counts do not equal total counts."
    )

# ------------------------------------------------------------------------------
# Rate consistency
# ------------------------------------------------------------------------------

rate_consistency = np.isclose(
    (
        MISSINGNESS_DF["missing_rate"]
        +
        MISSINGNESS_DF["observed_rate"]
    ),
    1.0,
    rtol=0.0,
    atol=1e-12
)

if not rate_consistency.all():
    raise RuntimeError(
        "Missing + observed rates do not equal 1."
    )

# ------------------------------------------------------------------------------
# Target exclusion
# ------------------------------------------------------------------------------

if MISSINGNESS_DF["is_target"].any():
    raise RuntimeError(
        "Target unexpectedly appears in preprocessing "
        "missingness characterization."
    )

# ==============================================================================
# DATASET-LEVEL SUMMARY
# ==============================================================================

MISSINGNESS_SUMMARY_DF = (
    MISSINGNESS_DF
    .groupby(
        "dataset_id",
        as_index=False
    )
    .agg(
        features=(
            "feature",
            "count"
        ),

        features_with_missing=(
            "missing_rate",
            lambda x: int(
                (x > 0).sum()
            )
        ),

        features_without_missing=(
            "missing_rate",
            lambda x: int(
                (x == 0).sum()
            )
        ),

        maximum_feature_missing_rate=(
            "missing_rate",
            "max"
        ),

        mean_feature_missing_rate=(
            "missing_rate",
            "mean"
        ),

        median_feature_missing_rate=(
            "missing_rate",
            "median"
        ),

        total_missing_values=(
            "missing_count",
            "sum"
        ),

        total_observed_values=(
            "observed_count",
            "sum"
        ),
    )
)

# ==============================================================================
# SUMMARY BY FEATURE TYPE
# ==============================================================================

MISSINGNESS_BY_TYPE_DF = (
    MISSINGNESS_DF
    .assign(
        feature_type=np.where(
            MISSINGNESS_DF["is_numeric"],
            "numeric",
            "categorical"
        )
    )
    .groupby(
        ["dataset_id", "feature_type"],
        as_index=False
    )
    .agg(
        features=(
            "feature",
            "count"
        ),

        features_with_missing=(
            "missing_rate",
            lambda x: int(
                (x > 0).sum()
            )
        ),

        mean_missing_rate=(
            "missing_rate",
            "mean"
        ),

        maximum_missing_rate=(
            "missing_rate",
            "max"
        ),
    )
)

# ==============================================================================
# DISPLAY RESULTS
# ==============================================================================

print(
    "\nFeature-level missingness characterization:"
)

print(
    MISSINGNESS_DF.to_string(
        index=False
    )
)

print(
    "\nDataset-level missingness summary:"
)

print(
    MISSINGNESS_SUMMARY_DF.to_string(
        index=False
    )
)

print(
    "\nMissingness summary by feature type:"
)

print(
    MISSINGNESS_BY_TYPE_DF.to_string(
        index=False
    )
)

# ==============================================================================
# SECTION 8 COMPLETION GATE
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 8 VALIDATION")
print("=" * 100)

print("✓ Characterization uses TRAINING data only")
print("✓ No artificial MCAR/MAR/MNAR masking performed")
print("✓ Frozen Notebook 02 schema used as feature-universe authority")
print("✓ Section 5 missingness values cross-checked")
print("✓ Cross-check uses consistent 8-decimal precision")
print("✓ Target excluded from preprocessing missingness analysis")
print("✓ Identifier/provenance columns excluded")
print("✓ Feature coverage verified")
print("✓ Duplicate dataset-feature rows checked")
print("✓ Missing/observed counts validated")
print("✓ Missing/observed rates validated")
print("✓ Dataset-level summary generated")
print("✓ Numeric/categorical missingness summary generated")

print("\nSECTION 8 STATUS: PASS")
print("=" * 100)

SECTION 8 — MISSINGNESS CHARACTERIZATION

Processing dataset: adult_income

Processing dataset: bank_marketing

Processing dataset: diabetes_130us

Feature-level missingness characterization:
    dataset_id                  feature  count  missing_count  observed_count  missing_rate  observed_rate missingness_level  is_numeric  is_categorical  is_target  is_preprocessing_feature  is_generative_feature           schema_role
  adult_income                      age  34189              0           34189      0.000000       1.000000              none        True           False      False                      True                   True preprocessing_feature
  adult_income                workclass  34189           1992           32197      0.058264       0.941736          moderate       False            True      False                      True                   True preprocessing_feature
  adult_income                   fnlwgt  34189              0           34189      0.000000       1.000

In [9]:
# ==============================================================================
# SECTION 9 — CARDINALITY / ENTROPY ANALYSIS
# ==============================================================================

print("=" * 100)
print("SECTION 9 — CARDINALITY / ENTROPY ANALYSIS")
print("=" * 100)

# ------------------------------------------------------------------------------
# Required upstream objects
# ------------------------------------------------------------------------------

REQUIRED_SECTION_9_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_STATISTICAL_DATASETS",
    "NB02_PERSISTED_SCHEMA_CONTRACTS",
    "FEATURE_TYPE_CHARACTERIZATION_DF",
    "CATEGORICAL_STATISTICS_DF",
]

_missing_section_9_objects = [
    name
    for name in REQUIRED_SECTION_9_OBJECTS
    if name not in globals()
]

if _missing_section_9_objects:
    raise RuntimeError(
        "Section 9 cannot proceed because required upstream objects "
        f"are missing: {_missing_section_9_objects}"
    )

# ------------------------------------------------------------------------------
# Reset output container
# ------------------------------------------------------------------------------

CARDINALITY_ENTROPY_RECORDS = []


# ==============================================================================
# HELPER — NORMALIZED SHANNON ENTROPY
# ==============================================================================

def normalized_entropy(series):
    """
    Compute Shannon entropy and normalized Shannon entropy
    for a categorical series.

    Missing values are excluded from the probability distribution.

    Returns
    -------
    tuple
        (
            entropy_bits,
            normalized_entropy,
            n_observed_categories
        )
    """

    s = series.dropna()

    if len(s) == 0:
        return np.nan, np.nan, 0

    probabilities = (
        s.astype("object")
        .value_counts(
            normalize=True,
            dropna=True
        )
        .to_numpy(dtype=float)
    )

    if len(probabilities) == 0:
        return np.nan, np.nan, 0

    # --------------------------------------------------------------------------
    # Numerical safety
    # --------------------------------------------------------------------------

    probabilities = probabilities[
        np.isfinite(probabilities)
        &
        (probabilities > 0)
    ]

    if len(probabilities) == 0:
        return np.nan, np.nan, 0

    # --------------------------------------------------------------------------
    # Shannon entropy in bits
    # --------------------------------------------------------------------------

    entropy = float(
        -np.sum(
            probabilities
            *
            np.log2(
                np.clip(
                    probabilities,
                    1e-15,
                    None
                )
            )
        )
    )

    cardinality = int(
        len(probabilities)
    )

    # --------------------------------------------------------------------------
    # Constant categorical feature
    # --------------------------------------------------------------------------

    if cardinality <= 1:
        return entropy, 0.0, cardinality

    # --------------------------------------------------------------------------
    # Maximum possible entropy for observed cardinality
    # --------------------------------------------------------------------------

    max_entropy = float(
        math.log2(cardinality)
    )

    if max_entropy <= 0:
        normalized = 0.0
    else:
        normalized = float(
            entropy / max_entropy
        )

    # Numerical clipping for floating-point safety
    normalized = float(
        np.clip(
            normalized,
            0.0,
            1.0
        )
    )

    return entropy, normalized, cardinality


# ==============================================================================
# DATASET-LEVEL PROCESSING
# ==============================================================================

for dataset_id in DATASET_IDS:

    print(
        f"\nProcessing dataset: {dataset_id}"
    )

    # --------------------------------------------------------------------------
    # Training-only canonical data
    # --------------------------------------------------------------------------

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    # --------------------------------------------------------------------------
    # Frozen Notebook 02 schema
    # --------------------------------------------------------------------------

    schema_contract = NB02_PERSISTED_SCHEMA_CONTRACTS[
        dataset_id
    ]

    modeling_schema = schema_contract[
        "modeling_schema"
    ]

    preprocessing_columns = list(
        modeling_schema[
            "preprocessing_columns"
        ]
    )

    numeric_columns = list(
        modeling_schema[
            "numeric_columns"
        ]
    )

    categorical_columns = list(
        modeling_schema[
            "categorical_columns"
        ]
    )

    target_column = modeling_schema[
        "target_column"
    ]

    provenance_column = modeling_schema[
        "provenance_column"
    ]

    # ==========================================================================
    # SCHEMA VALIDATION
    # ==========================================================================

    expected_generative_columns = (
        preprocessing_columns
        + [target_column]
    )

    expected_native_columns = (
        [provenance_column]
        + expected_generative_columns
    )

    if list(df.columns) != expected_native_columns:
        raise RuntimeError(
            f"Native schema mismatch for '{dataset_id}'.\n"
            f"Expected: {expected_native_columns}\n"
            f"Observed: {list(df.columns)}"
        )

    # --------------------------------------------------------------------------
    # Numeric/categorical partition
    # --------------------------------------------------------------------------

    if set(numeric_columns).intersection(
        categorical_columns
    ):
        raise RuntimeError(
            f"Numeric/categorical overlap detected "
            f"for '{dataset_id}'."
        )

    if (
        set(numeric_columns).union(
            categorical_columns
        )
        != set(preprocessing_columns)
    ):
        raise RuntimeError(
            f"Numeric + categorical feature partition does not match "
            f"Notebook 02 preprocessing columns for '{dataset_id}'."
        )

    # --------------------------------------------------------------------------
    # Target exclusion
    # --------------------------------------------------------------------------

    if target_column in preprocessing_columns:
        raise RuntimeError(
            f"Target '{target_column}' incorrectly appears in "
            f"preprocessing columns for '{dataset_id}'."
        )

    # ==========================================================================
    # CROSS-CHECK SECTION 5
    # ==========================================================================

    section_5_categorical = set(
        FEATURE_TYPE_CHARACTERIZATION_DF[
            (
                FEATURE_TYPE_CHARACTERIZATION_DF[
                    "dataset_id"
                ]
                == dataset_id
            )
            &
            (
                FEATURE_TYPE_CHARACTERIZATION_DF[
                    "feature_type"
                ]
                == "categorical"
            )
            &
            (
                FEATURE_TYPE_CHARACTERIZATION_DF[
                    "is_preprocessing_feature"
                ]
                == True
            )
        ]["feature"]
    )

    if section_5_categorical != set(
        categorical_columns
    ):
        raise RuntimeError(
            f"Section 5 categorical feature identity does not match "
            f"Notebook 02 for '{dataset_id}'."
        )

    # ==========================================================================
    # CROSS-CHECK SECTION 7
    # ==========================================================================

    section_7_categorical = set(
        CATEGORICAL_STATISTICS_DF[
            CATEGORICAL_STATISTICS_DF[
                "dataset_id"
            ]
            == dataset_id
        ]["feature"]
    )

    if section_7_categorical != set(
        categorical_columns
    ):
        raise RuntimeError(
            f"Section 7 categorical feature identity does not match "
            f"Notebook 02 for '{dataset_id}'."
        )

    # ==========================================================================
    # FEATURE-LEVEL CARDINALITY / ENTROPY
    # ==========================================================================

    for feature in categorical_columns:

        s = df[feature]

        total_count = int(
            len(s)
        )

        missing_count = int(
            s.isna().sum()
        )

        non_missing = s.dropna()

        non_missing_count = int(
            len(non_missing)
        )

        # ----------------------------------------------------------------------
        # Cardinality
        # ----------------------------------------------------------------------

        cardinality = int(
            non_missing.nunique(
                dropna=True
            )
        )

        # ----------------------------------------------------------------------
        # Shannon entropy
        # ----------------------------------------------------------------------

        entropy_bits, entropy_normalized, entropy_categories = (
            normalized_entropy(s)
        )

        if cardinality != entropy_categories:
            raise RuntimeError(
                f"Cardinality/entropy category mismatch for "
                f"'{dataset_id}' / '{feature}'."
            )

        # ----------------------------------------------------------------------
        # Category frequencies
        # ----------------------------------------------------------------------

        if non_missing_count > 0:

            value_counts = (
                non_missing
                .astype("object")
                .value_counts(
                    normalize=False
                )
            )

            probabilities = (
                value_counts
                / non_missing_count
            )

            top_frequency = float(
                probabilities.iloc[0]
            )

            rare_category_count = int(
                (probabilities < 0.01).sum()
            )

            singleton_category_count = int(
                (value_counts == 1).sum()
            )

        else:

            top_frequency = np.nan
            rare_category_count = 0
            singleton_category_count = 0

        # ----------------------------------------------------------------------
        # Cardinality ratio
        #
        # Measures number of observed categories relative to available
        # non-missing observations.
        # ----------------------------------------------------------------------

        cardinality_ratio = (
            float(
                cardinality / non_missing_count
            )
            if non_missing_count > 0
            else np.nan
        )

        # ----------------------------------------------------------------------
        # Entropy interpretation
        # ----------------------------------------------------------------------

        if pd.isna(entropy_normalized):

            entropy_level = "undefined"

        elif entropy_normalized < 0.25:

            entropy_level = "low"

        elif entropy_normalized < 0.50:

            entropy_level = "moderate"

        elif entropy_normalized < 0.75:

            entropy_level = "high"

        else:

            entropy_level = "very_high"

        # ----------------------------------------------------------------------
        # Store record
        # ----------------------------------------------------------------------

        CARDINALITY_ENTROPY_RECORDS.append({

            "dataset_id": dataset_id,

            "feature": feature,

            "cardinality": cardinality,

            "cardinality_ratio": cardinality_ratio,

            "shannon_entropy_bits": entropy_bits,

            "normalized_entropy": entropy_normalized,

            "entropy_level": entropy_level,

            "top_frequency": top_frequency,

            "rare_category_count": rare_category_count,

            "singleton_category_count": singleton_category_count,

            "count": total_count,

            "non_missing_count": non_missing_count,

            "missing_count": missing_count,

            "missing_rate": float(
                missing_count / total_count
            ) if total_count > 0 else np.nan,

            "is_numeric": False,

            "is_categorical": True,

            "is_target": False,

            "is_preprocessing_feature": True,

            "is_generative_feature": True,

            "schema_role": "categorical_preprocessing_feature",

        })


# ==============================================================================
# CREATE OUTPUT DATAFRAME
# ==============================================================================

CARDINALITY_ENTROPY_DF = pd.DataFrame(
    CARDINALITY_ENTROPY_RECORDS
)

# ==============================================================================
# OUTPUT VALIDATION
# ==============================================================================

REQUIRED_SECTION_9_COLUMNS = [
    "dataset_id",
    "feature",
    "cardinality",
    "cardinality_ratio",
    "shannon_entropy_bits",
    "normalized_entropy",
    "entropy_level",
    "top_frequency",
    "rare_category_count",
    "singleton_category_count",
    "count",
    "non_missing_count",
    "missing_count",
    "missing_rate",
    "is_numeric",
    "is_categorical",
    "is_target",
    "is_preprocessing_feature",
    "is_generative_feature",
    "schema_role",
]

missing_output_columns = [
    column
    for column in REQUIRED_SECTION_9_COLUMNS
    if column not in CARDINALITY_ENTROPY_DF.columns
]

if missing_output_columns:
    raise RuntimeError(
        "Section 9 output is missing required columns: "
        f"{missing_output_columns}"
    )

# ==============================================================================
# COVERAGE VALIDATION
# ==============================================================================

expected_categorical_count = sum(
    len(
        NB02_PERSISTED_SCHEMA_CONTRACTS[
            dataset_id
        ][
            "modeling_schema"
        ][
            "categorical_columns"
        ]
    )
    for dataset_id in DATASET_IDS
)

observed_record_count = len(
    CARDINALITY_ENTROPY_DF
)

if observed_record_count != expected_categorical_count:
    raise RuntimeError(
        "Cardinality/entropy coverage mismatch.\n"
        f"Expected: {expected_categorical_count}\n"
        f"Observed: {observed_record_count}"
    )

# ==============================================================================
# DUPLICATE CHECK
# ==============================================================================

duplicate_count = int(
    CARDINALITY_ENTROPY_DF
    .duplicated(
        subset=[
            "dataset_id",
            "feature"
        ]
    )
    .sum()
)

if duplicate_count != 0:
    raise RuntimeError(
        f"Duplicate dataset-feature records detected: "
        f"{duplicate_count}"
    )

# ==============================================================================
# CARDINALITY VALIDATION
# ==============================================================================

if (
    CARDINALITY_ENTROPY_DF[
        "cardinality"
    ]
    < 0
).any():

    raise RuntimeError(
        "Negative cardinality values detected."
    )

# ==============================================================================
# ENTROPY VALIDATION
# ==============================================================================

valid_entropy = (
    CARDINALITY_ENTROPY_DF[
        "normalized_entropy"
    ]
    .dropna()
)

if (
    valid_entropy < 0
).any() or (
    valid_entropy > 1
).any():

    raise RuntimeError(
        "Normalized entropy values outside [0,1] detected."
    )

valid_raw_entropy = (
    CARDINALITY_ENTROPY_DF[
        "shannon_entropy_bits"
    ]
    .dropna()
)

if (
    valid_raw_entropy < 0
).any():

    raise RuntimeError(
        "Negative Shannon entropy detected."
    )

# ==============================================================================
# FREQUENCY VALIDATION
# ==============================================================================

valid_top_frequency = (
    CARDINALITY_ENTROPY_DF[
        "top_frequency"
    ]
    .dropna()
)

if (
    valid_top_frequency < 0
).any() or (
    valid_top_frequency > 1
).any():

    raise RuntimeError(
        "Invalid top-frequency values detected."
    )

# ==============================================================================
# TARGET EXCLUSION
# ==============================================================================

if CARDINALITY_ENTROPY_DF[
    "is_target"
].any():

    raise RuntimeError(
        "Target unexpectedly included in categorical "
        "cardinality/entropy analysis."
    )

# ==============================================================================
# DATASET SUMMARY
# ==============================================================================

CARDINALITY_ENTROPY_SUMMARY_DF = (
    CARDINALITY_ENTROPY_DF
    .groupby(
        "dataset_id",
        as_index=False
    )
    .agg(

        categorical_features=(
            "feature",
            "count"
        ),

        mean_cardinality=(
            "cardinality",
            "mean"
        ),

        maximum_cardinality=(
            "cardinality",
            "max"
        ),

        mean_normalized_entropy=(
            "normalized_entropy",
            "mean"
        ),

        minimum_normalized_entropy=(
            "normalized_entropy",
            "min"
        ),

        maximum_normalized_entropy=(
            "normalized_entropy",
            "max"
        ),

        high_entropy_features=(
            "normalized_entropy",
            lambda x: int(
                (x >= 0.75).sum()
            )
        ),

        low_entropy_features=(
            "normalized_entropy",
            lambda x: int(
                (x < 0.25).sum()
            )
        ),

        total_rare_categories=(
            "rare_category_count",
            "sum"
        ),

        total_singleton_categories=(
            "singleton_category_count",
            "sum"
        ),
    )
)

# ==============================================================================
# DISPLAY RESULTS
# ==============================================================================

print(
    f"\nCategorical cardinality/entropy records: "
    f"{observed_record_count}"
)

print(
    "\nFeature-level cardinality/entropy characterization:"
)

print(
    CARDINALITY_ENTROPY_DF.to_string(
        index=False
    )
)

print(
    "\nDataset-level cardinality/entropy summary:"
)

print(
    CARDINALITY_ENTROPY_SUMMARY_DF.to_string(
        index=False
    )
)

# ==============================================================================
# SECTION 9 COMPLETION GATE
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 9 VALIDATION")
print("=" * 100)

print("✓ Training data used exclusively")
print("✓ Frozen Notebook 02 schema used as feature authority")
print("✓ Categorical preprocessing features characterized")
print("✓ Section 5 categorical feature identity cross-checked")
print("✓ Section 7 categorical feature identity cross-checked")
print("✓ Target excluded")
print("✓ Shannon entropy computed from observed categorical values")
print("✓ Normalized entropy constrained to [0,1]")
print("✓ Cardinality coverage verified")
print("✓ Duplicate dataset-feature rows checked")
print("✓ Frequency values validated")
print("✓ Dataset-level summary generated")

print("\nSECTION 9 STATUS: PASS")
print("=" * 100)

SECTION 9 — CARDINALITY / ENTROPY ANALYSIS

Processing dataset: adult_income

Processing dataset: bank_marketing

Processing dataset: diabetes_130us

Categorical cardinality/entropy records: 53

Feature-level cardinality/entropy characterization:
    dataset_id                  feature  cardinality  cardinality_ratio  shannon_entropy_bits  normalized_entropy entropy_level  top_frequency  rare_category_count  singleton_category_count  count  non_missing_count  missing_count  missing_rate  is_numeric  is_categorical  is_target  is_preprocessing_feature  is_generative_feature                       schema_role
  adult_income                workclass            8           0.000248              1.428644            0.476215      moderate       0.734696                    2                         0  34189              32197           1992      0.058264       False            True      False                      True                   True categorical_preprocessing_feature
  adult_income     

In [10]:
# ==============================================================================
# SECTION 10 — DISTRIBUTION CHARACTERIZATION
# ==============================================================================

print("=" * 100)
print("SECTION 10 — DISTRIBUTION CHARACTERIZATION")
print("=" * 100)

# ------------------------------------------------------------------------------
# Required upstream objects
# ------------------------------------------------------------------------------

REQUIRED_SECTION_10_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_STATISTICAL_DATASETS",
    "NB02_PERSISTED_SCHEMA_CONTRACTS",
    "FEATURE_TYPE_CHARACTERIZATION_DF",
    "CATEGORICAL_STATISTICS_DF",
    "CARDINALITY_ENTROPY_DF",
]

_missing_section_10_objects = [
    name
    for name in REQUIRED_SECTION_10_OBJECTS
    if name not in globals()
]

if _missing_section_10_objects:
    raise RuntimeError(
        "Section 10 cannot proceed because required upstream objects "
        f"are missing: {_missing_section_10_objects}"
    )

# ------------------------------------------------------------------------------
# Reset output container
# ------------------------------------------------------------------------------

DISTRIBUTION_RECORDS = []


# ==============================================================================
# DATASET-LEVEL PROCESSING
# ==============================================================================

for dataset_id in DATASET_IDS:

    print(
        f"\nProcessing dataset: {dataset_id}"
    )

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    # --------------------------------------------------------------------------
    # Frozen Notebook 02 schema
    # --------------------------------------------------------------------------

    schema_contract = NB02_PERSISTED_SCHEMA_CONTRACTS[
        dataset_id
    ]

    modeling_schema = schema_contract[
        "modeling_schema"
    ]

    preprocessing_columns = list(
        modeling_schema[
            "preprocessing_columns"
        ]
    )

    numeric_columns = list(
        modeling_schema[
            "numeric_columns"
        ]
    )

    categorical_columns = list(
        modeling_schema[
            "categorical_columns"
        ]
    )

    target_column = modeling_schema[
        "target_column"
    ]

    provenance_column = modeling_schema[
        "provenance_column"
    ]

    # ==========================================================================
    # SCHEMA VALIDATION
    # ==========================================================================

    expected_generative_columns = (
        preprocessing_columns
        + [target_column]
    )

    expected_native_columns = (
        [provenance_column]
        + expected_generative_columns
    )

    if list(df.columns) != expected_native_columns:
        raise RuntimeError(
            f"Native schema mismatch for '{dataset_id}'.\n"
            f"Expected: {expected_native_columns}\n"
            f"Observed: {list(df.columns)}"
        )

    if target_column in preprocessing_columns:
        raise RuntimeError(
            f"Target '{target_column}' incorrectly appears in "
            f"preprocessing columns for '{dataset_id}'."
        )

    if (
        set(numeric_columns).intersection(
            categorical_columns
        )
    ):
        raise RuntimeError(
            f"Numeric/categorical overlap detected for '{dataset_id}'."
        )

    if (
        set(numeric_columns).union(
            categorical_columns
        )
        != set(preprocessing_columns)
    ):
        raise RuntimeError(
            f"Numeric + categorical partition does not match "
            f"preprocessing columns for '{dataset_id}'."
        )

    # ==========================================================================
    # CROSS-CHECK SECTION 5
    # ==========================================================================

    section_5_features = set(
        FEATURE_TYPE_CHARACTERIZATION_DF[
            (
                FEATURE_TYPE_CHARACTERIZATION_DF[
                    "dataset_id"
                ]
                == dataset_id
            )
            &
            (
                FEATURE_TYPE_CHARACTERIZATION_DF[
                    "is_preprocessing_feature"
                ]
                == True
            )
        ]["feature"]
    )

    if section_5_features != set(
        preprocessing_columns
    ):
        raise RuntimeError(
            f"Section 5 feature universe mismatch for '{dataset_id}'."
        )

    # ==========================================================================
    # CROSS-CHECK SECTION 7
    # ==========================================================================

    section_7_features = set(
        CATEGORICAL_STATISTICS_DF[
            CATEGORICAL_STATISTICS_DF[
                "dataset_id"
            ]
            == dataset_id
        ]["feature"]
    )

    if section_7_features != set(
        categorical_columns
    ):
        raise RuntimeError(
            f"Section 7 categorical feature identity mismatch "
            f"for '{dataset_id}'."
        )

    # ==========================================================================
    # CROSS-CHECK SECTION 9
    # ==========================================================================

    section_9_features = set(
        CARDINALITY_ENTROPY_DF[
            CARDINALITY_ENTROPY_DF[
                "dataset_id"
            ]
            == dataset_id
        ]["feature"]
    )

    if section_9_features != set(
        categorical_columns
    ):
        raise RuntimeError(
            f"Section 9 categorical feature identity mismatch "
            f"for '{dataset_id}'."
        )

    # ==========================================================================
    # FEATURE-LEVEL DISTRIBUTION CHARACTERIZATION
    # ==========================================================================

    for feature in preprocessing_columns:

        s = df[feature]

        total_count = int(
            len(s)
        )

        missing_count = int(
            s.isna().sum()
        )

        missing_rate = (
            float(
                missing_count / total_count
            )
            if total_count > 0
            else np.nan
        )

        non_missing = s.dropna()

        n_non_missing = int(
            len(non_missing)
        )

        n_unique = int(
            non_missing.nunique(
                dropna=True
            )
        )

        # ----------------------------------------------------------------------
        # Base record
        # ----------------------------------------------------------------------

        record = {
            "dataset_id": dataset_id,
            "feature": feature,
            "distribution_type": (
                "numeric"
                if feature in numeric_columns
                else "categorical"
            ),
            "count": total_count,
            "non_missing_count": n_non_missing,
            "missing_count": missing_count,
            "missing_rate": missing_rate,
            "n_unique": n_unique,
            "unique_ratio": (
                float(
                    n_unique / n_non_missing
                )
                if n_non_missing > 0
                else np.nan
            ),
            "is_target": False,
            "is_preprocessing_feature": True,
            "is_generative_feature": True,
            "schema_role": (
                "numeric_preprocessing_feature"
                if feature in numeric_columns
                else "categorical_preprocessing_feature"
            ),
        }

        # ==========================================================================
        # NUMERIC DISTRIBUTION
        # ==========================================================================

        if feature in numeric_columns:

            x = pd.to_numeric(
                non_missing,
                errors="coerce"
            )

            # Remove non-finite values defensively
            x = x[
                np.isfinite(x)
            ]

            if len(x) == 0:

                record.update({
                    "mean": np.nan,
                    "median": np.nan,
                    "std": np.nan,
                    "min": np.nan,
                    "q01": np.nan,
                    "q05": np.nan,
                    "q25": np.nan,
                    "q50": np.nan,
                    "q75": np.nan,
                    "q95": np.nan,
                    "q99": np.nan,
                    "max": np.nan,
                    "iqr": np.nan,
                    "skewness": np.nan,
                    "kurtosis": np.nan,
                    "coefficient_of_variation": np.nan,
                    "zero_count": 0,
                    "zero_rate": np.nan,
                    "negative_count": 0,
                    "negative_rate": np.nan,
                    "positive_count": 0,
                    "positive_rate": np.nan,
                    "shape_class": "undefined",
                    "concentration_class": np.nan,
                })

            else:

                mean_value = float(
                    x.mean()
                )

                median_value = float(
                    x.median()
                )

                std_value = float(
                    x.std()
                )

                q01 = float(
                    x.quantile(0.01)
                )

                q05 = float(
                    x.quantile(0.05)
                )

                q25 = float(
                    x.quantile(0.25)
                )

                q75 = float(
                    x.quantile(0.75)
                )

                q95 = float(
                    x.quantile(0.95)
                )

                q99 = float(
                    x.quantile(0.99)
                )

                min_value = float(
                    x.min()
                )

                max_value = float(
                    x.max()
                )

                iqr = float(
                    q75 - q25
                )

                skewness = float(
                    x.skew()
                )

                kurtosis = float(
                    x.kurtosis()
                )

                # ------------------------------------------------------------------
                # Coefficient of variation
                # ------------------------------------------------------------------

                if (
                    np.isfinite(mean_value)
                    and abs(mean_value) > 1e-15
                ):
                    coefficient_of_variation = float(
                        std_value / abs(mean_value)
                    )
                else:
                    coefficient_of_variation = np.nan

                # ------------------------------------------------------------------
                # Sign / zero characterization
                # ------------------------------------------------------------------

                zero_count = int(
                    (x == 0).sum()
                )

                negative_count = int(
                    (x < 0).sum()
                )

                positive_count = int(
                    (x > 0).sum()
                )

                numeric_count = len(x)

                zero_rate = float(
                    zero_count / numeric_count
                )

                negative_rate = float(
                    negative_count / numeric_count
                )

                positive_rate = float(
                    positive_count / numeric_count
                )

                # ------------------------------------------------------------------
                # Descriptive shape classification
                #
                # This is based on skewness only and is NOT a formal normality
                # test.
                # ------------------------------------------------------------------

                if not np.isfinite(skewness):

                    shape_class = "undefined"

                elif abs(skewness) < 0.5:

                    shape_class = "approximately_symmetric"

                elif abs(skewness) < 1.0:

                    shape_class = "moderately_skewed"

                else:

                    shape_class = "highly_skewed"

                record.update({

                    "mean": mean_value,
                    "median": median_value,
                    "std": std_value,
                    "min": min_value,
                    "q01": q01,
                    "q05": q05,
                    "q25": q25,
                    "q50": median_value,
                    "q75": q75,
                    "q95": q95,
                    "q99": q99,
                    "max": max_value,
                    "iqr": iqr,
                    "skewness": skewness,
                    "kurtosis": kurtosis,
                    "coefficient_of_variation":
                        coefficient_of_variation,
                    "zero_count": zero_count,
                    "zero_rate": zero_rate,
                    "negative_count": negative_count,
                    "negative_rate": negative_rate,
                    "positive_count": positive_count,
                    "positive_rate": positive_rate,
                    "shape_class": shape_class,
                    "concentration_class": np.nan,
                })

        # ==========================================================================
        # CATEGORICAL DISTRIBUTION
        # ==========================================================================

        else:

            counts = (
                non_missing
                .astype("object")
                .value_counts(
                    normalize=False
                )
            )

            probabilities = (
                counts / n_non_missing
                if n_non_missing > 0
                else pd.Series(
                    dtype=float
                )
            )

            if len(probabilities) > 0:

                top_frequency = float(
                    probabilities.iloc[0]
                    / n_non_missing
                )

                rare_category_count = int(
                    (
                        probabilities / n_non_missing
                        < 0.01
                    ).sum()
                )

                singleton_category_count = int(
                    (
                        probabilities == 1
                    ).sum()
                )

                # ------------------------------------------------------------------
                # Concentration classification
                # ------------------------------------------------------------------

                if top_frequency >= 0.90:

                    concentration_class = (
                        "very_high_concentration"
                    )

                elif top_frequency >= 0.75:

                    concentration_class = (
                        "high_concentration"
                    )

                elif top_frequency >= 0.50:

                    concentration_class = (
                        "moderate_concentration"
                    )

                else:

                    concentration_class = (
                        "low_concentration"
                    )

                top_category = str(
                    probabilities.index[0]
                )

            else:

                top_frequency = np.nan
                rare_category_count = 0
                singleton_category_count = 0
                concentration_class = "undefined"
                top_category = np.nan

            # ----------------------------------------------------------------------
            # Pull entropy statistics from frozen Section 9 output
            # ----------------------------------------------------------------------

            section_9_row = CARDINALITY_ENTROPY_DF[
                (
                    CARDINALITY_ENTROPY_DF[
                        "dataset_id"
                    ]
                    == dataset_id
                )
                &
                (
                    CARDINALITY_ENTROPY_DF[
                        "feature"
                    ]
                    == feature
                )
            ]

            if len(section_9_row) != 1:

                raise RuntimeError(
                    f"Expected exactly one Section 9 record for "
                    f"'{dataset_id}' / '{feature}', "
                    f"found {len(section_9_row)}."
                )

            section_9_row = (
                section_9_row.iloc[0]
            )

            # ----------------------------------------------------------------------
            # Cross-check cardinality
            # ----------------------------------------------------------------------

            section_9_cardinality = int(
                section_9_row[
                    "cardinality"
                ]
            )

            if section_9_cardinality != n_unique:
                raise RuntimeError(
                    f"Cardinality mismatch for "
                    f"'{dataset_id}' / '{feature}'."
                )

            record.update({

                "mean": np.nan,
                "median": np.nan,
                "std": np.nan,
                "min": np.nan,
                "q01": np.nan,
                "q05": np.nan,
                "q25": np.nan,
                "q50": np.nan,
                "q75": np.nan,
                "q95": np.nan,
                "q99": np.nan,
                "max": np.nan,
                "iqr": np.nan,
                "skewness": np.nan,
                "kurtosis": np.nan,
                "coefficient_of_variation": np.nan,
                "zero_count": np.nan,
                "zero_rate": np.nan,
                "negative_count": np.nan,
                "negative_rate": np.nan,
                "positive_count": np.nan,
                "positive_rate": np.nan,
                "shape_class": np.nan,

                "top_category": top_category,
                "top_category_frequency": top_frequency,
                "number_of_categories": n_unique,
                "shannon_entropy_bits": float(
                    section_9_row[
                        "shannon_entropy_bits"
                    ]
                )
                if pd.notna(
                    section_9_row[
                        "shannon_entropy_bits"
                    ]
                )
                else np.nan,

                "normalized_entropy": float(
                    section_9_row[
                        "normalized_entropy"
                    ]
                )
                if pd.notna(
                    section_9_row[
                        "normalized_entropy"
                    ]
                )
                else np.nan,

                "rare_category_count":
                    rare_category_count,

                "singleton_category_count":
                    singleton_category_count,

                "concentration_class":
                    concentration_class,
            })

        # --------------------------------------------------------------------------
        # Append final feature record
        # --------------------------------------------------------------------------

        DISTRIBUTION_RECORDS.append(
            record
        )


# ==============================================================================
# CREATE DISTRIBUTION DATAFRAME
# ==============================================================================

DISTRIBUTION_DF = pd.DataFrame(
    DISTRIBUTION_RECORDS
)

print(
    f"\nDistribution characterization records: "
    f"{len(DISTRIBUTION_DF)}"
)


# ==============================================================================
# OUTPUT SCHEMA VALIDATION
# ==============================================================================

REQUIRED_SECTION_10_COLUMNS = [

    "dataset_id",
    "feature",
    "distribution_type",

    "count",
    "non_missing_count",
    "missing_count",
    "missing_rate",

    "n_unique",
    "unique_ratio",

    "mean",
    "median",
    "std",
    "min",

    "q01",
    "q05",
    "q25",
    "q50",
    "q75",
    "q95",
    "q99",

    "max",
    "iqr",

    "skewness",
    "kurtosis",
    "coefficient_of_variation",

    "zero_count",
    "zero_rate",

    "negative_count",
    "negative_rate",

    "positive_count",
    "positive_rate",

    "shape_class",

    "top_category",
    "top_category_frequency",
    "number_of_categories",

    "shannon_entropy_bits",
    "normalized_entropy",

    "rare_category_count",
    "singleton_category_count",

    "concentration_class",

    "is_target",
    "is_preprocessing_feature",
    "is_generative_feature",
    "schema_role",
]

missing_output_columns = [
    column
    for column in REQUIRED_SECTION_10_COLUMNS
    if column not in DISTRIBUTION_DF.columns
]

if missing_output_columns:

    raise RuntimeError(
        "Section 10 output is missing required columns: "
        f"{missing_output_columns}"
    )


# ==============================================================================
# COVERAGE VALIDATION
# ==============================================================================

expected_feature_count = sum(
    len(
        NB02_PERSISTED_SCHEMA_CONTRACTS[
            dataset_id
        ][
            "modeling_schema"
        ][
            "preprocessing_columns"
        ]
    )
    for dataset_id in DATASET_IDS
)

observed_feature_count = len(
    DISTRIBUTION_DF
)

if observed_feature_count != expected_feature_count:

    raise RuntimeError(
        "Distribution characterization coverage mismatch.\n"
        f"Expected: {expected_feature_count}\n"
        f"Observed: {observed_feature_count}"
    )


# ==============================================================================
# DUPLICATE CHECK
# ==============================================================================

duplicate_count = int(
    DISTRIBUTION_DF
    .duplicated(
        subset=[
            "dataset_id",
            "feature"
        ]
    )
    .sum()
)

if duplicate_count != 0:

    raise RuntimeError(
        f"Duplicate dataset-feature records detected: "
        f"{duplicate_count}"
    )


# ==============================================================================
# TARGET EXCLUSION
# ==============================================================================

if DISTRIBUTION_DF[
    "is_target"
].any():

    raise RuntimeError(
        "Target unexpectedly included in Section 10."
    )


# ==============================================================================
# MISSING-RATE VALIDATION
# ==============================================================================

if (
    DISTRIBUTION_DF[
        "missing_rate"
    ]
    .dropna()
    .lt(0)
    .any()
    or
    DISTRIBUTION_DF[
        "missing_rate"
    ]
    .dropna()
    .gt(1)
    .any()
):

    raise RuntimeError(
        "Invalid missing-rate values detected."
    )


# ==============================================================================
# NUMERIC VALIDATION
# ==============================================================================

numeric_output = DISTRIBUTION_DF[
    DISTRIBUTION_DF[
        "distribution_type"
    ]
    == "numeric"
]

if len(numeric_output) > 0:

    numeric_measure_columns = [
        "mean",
        "median",
        "std",
        "min",
        "q01",
        "q05",
        "q25",
        "q50",
        "q75",
        "q95",
        "q99",
        "max",
        "iqr",
        "skewness",
        "kurtosis",
    ]

    for column in numeric_measure_columns:

        values = numeric_output[
            column
        ].dropna()

        if not np.isfinite(
            values.to_numpy(
                dtype=float
            )
        ).all():

            raise RuntimeError(
                f"Non-finite numeric values detected in "
                f"'{column}'."
            )

    # --------------------------------------------------------------------------
    # Quantile ordering
    # --------------------------------------------------------------------------

    quantile_pairs = [
        ("q01", "q05"),
        ("q05", "q25"),
        ("q25", "q50"),
        ("q50", "q75"),
        ("q75", "q95"),
        ("q95", "q99"),
    ]

    for lower, upper in quantile_pairs:

        invalid = (
            numeric_output[lower].notna()
            &
            numeric_output[upper].notna()
            &
            (
                numeric_output[lower]
                >
                numeric_output[upper]
            )
        )

        if invalid.any():

            raise RuntimeError(
                f"Quantile ordering violation: "
                f"{lower} > {upper}."
            )


# ==============================================================================
# CATEGORICAL VALIDATION
# ==============================================================================

categorical_output = DISTRIBUTION_DF[
    DISTRIBUTION_DF[
        "distribution_type"
    ]
    == "categorical"
]

if len(categorical_output) > 0:

    entropy_values = (
        categorical_output[
            "normalized_entropy"
        ]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if (
        len(entropy_values) > 0
        and
        (
            (entropy_values < 0).any()
            or
            (entropy_values > 1).any()
        )
    ):

        raise RuntimeError(
            "Categorical normalized entropy outside [0,1]."
        )

    frequency_values = (
        categorical_output[
            "top_category_frequency"
        ]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if (
        len(frequency_values) > 0
        and
        (
            (frequency_values < 0).any()
            or
            (frequency_values > 1).any()
        )
    ):

        raise RuntimeError(
            "Invalid categorical top-frequency values detected."
        )


# ==============================================================================
# DATASET-LEVEL SUMMARY
# ==============================================================================

DISTRIBUTION_SUMMARY_RECORDS = []

for dataset_id in DATASET_IDS:

    dataset_df = DISTRIBUTION_DF[
        DISTRIBUTION_DF[
            "dataset_id"
        ]
        == dataset_id
    ]

    numeric_df = dataset_df[
        dataset_df[
            "distribution_type"
        ]
        == "numeric"
    ]

    categorical_df = dataset_df[
        dataset_df[
            "distribution_type"
        ]
        == "categorical"
    ]

    DISTRIBUTION_SUMMARY_RECORDS.append({

        "dataset_id": dataset_id,

        "total_features": int(
            len(dataset_df)
        ),

        "numeric_features": int(
            len(numeric_df)
        ),

        "categorical_features": int(
            len(categorical_df)
        ),

        "features_with_missing": int(
            (
                dataset_df[
                    "missing_count"
                ]
                > 0
            ).sum()
        ),

        "mean_numeric_skewness": (
            float(
                numeric_df[
                    "skewness"
                ].mean()
            )
            if len(numeric_df) > 0
            else np.nan
        ),

        "mean_numeric_kurtosis": (
            float(
                numeric_df[
                    "kurtosis"
                ].mean()
            )
            if len(numeric_df) > 0
            else np.nan
        ),

        "highly_skewed_numeric_features": int(
            (
                numeric_df[
                    "shape_class"
                ]
                == "highly_skewed"
            ).sum()
        ),

        "mean_categorical_cardinality": (
            float(
                categorical_df[
                    "number_of_categories"
                ].mean()
            )
            if len(categorical_df) > 0
            else np.nan
        ),

        "maximum_categorical_cardinality": (
            int(
                categorical_df[
                    "number_of_categories"
                ].max()
            )
            if len(categorical_df) > 0
            else 0
        ),

        "mean_normalized_categorical_entropy": (
            float(
                categorical_df[
                    "normalized_entropy"
                ].mean()
            )
            if len(categorical_df) > 0
            else np.nan
        ),

        "high_concentration_categorical_features": int(
            (
                categorical_df[
                    "concentration_class"
                ].isin(
                    [
                        "high_concentration",
                        "very_high_concentration",
                    ]
                )
            ).sum()
        ),
    })


DISTRIBUTION_SUMMARY_DF = pd.DataFrame(
    DISTRIBUTION_SUMMARY_RECORDS
)


# ==============================================================================
# DISPLAY RESULTS
# ==============================================================================

print(
    "\nDistribution characterization summary:"
)

print(
    DISTRIBUTION_SUMMARY_DF.to_string(
        index=False
    )
)


# ==============================================================================
# SECTION 10 COMPLETION GATE
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 10 VALIDATION")
print("=" * 100)

print("✓ Training data used exclusively")
print("✓ Frozen Notebook 02 schema used as feature authority")
print("✓ Preprocessing feature universe validated")
print("✓ Target excluded")
print("✓ Provenance excluded from characterization")
print("✓ Numeric distribution descriptors generated")
print("✓ Categorical distribution descriptors generated")
print("✓ Section 5 feature identity cross-checked")
print("✓ Section 7 categorical identity cross-checked")
print("✓ Section 9 entropy/cardinality identity cross-checked")
print("✓ Quantile ordering validated")
print("✓ Numeric finiteness validated")
print("✓ Entropy range validated")
print("✓ Frequency range validated")
print("✓ Coverage validated")
print("✓ Duplicate dataset-feature rows checked")
print("✓ Dataset-level distribution summary generated")

print("\nSECTION 10 STATUS: PASS")
print("=" * 100)

SECTION 10 — DISTRIBUTION CHARACTERIZATION

Processing dataset: adult_income

Processing dataset: bank_marketing

Processing dataset: diabetes_130us

Distribution characterization records: 77

Distribution characterization summary:
    dataset_id  total_features  numeric_features  categorical_features  features_with_missing  mean_numeric_skewness  mean_numeric_kurtosis  highly_skewed_numeric_features  mean_categorical_cardinality  maximum_categorical_cardinality  mean_normalized_categorical_entropy  high_concentration_categorical_features
  adult_income              14                 6                     8                      3               3.084141              30.515586                               3                     12.375000                               41                             0.625974                                        0
bank_marketing              16                 7                     9                      0               4.013944              46.716901   

In [11]:
# ==============================================================================
# SECTION 11 — PEARSON CORRELATION
# ==============================================================================

print("=" * 100)
print("SECTION 11 — PEARSON CORRELATION")
print("=" * 100)

# ==============================================================================
# 11.1 REQUIRED OBJECTS
# ==============================================================================

REQUIRED_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_STATISTICAL_DATASETS",
    "NB02_PERSISTED_SCHEMA_CONTRACTS",
    "FEATURE_TYPE_CHARACTERIZATION_DF",
]

missing_objects = [
    obj for obj in REQUIRED_OBJECTS
    if obj not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 11 cannot start.\n"
        f"Missing required objects: {missing_objects}"
    )

# ==============================================================================
# 11.2 RESET OUTPUT CONTAINERS
# ==============================================================================

PEARSON_MATRICES = {}
PEARSON_CONSTANT_FEATURES = {}
PEARSON_NUMERIC_COLUMNS = {}

PEARSON_LONG_RECORDS = []
PEARSON_FEATURE_SUMMARY_RECORDS = []
PEARSON_DATASET_SUMMARY_RECORDS = []

# ==============================================================================
# 11.3 HELPER — CORRELATION STRENGTH
# ==============================================================================

def classify_correlation_strength(abs_r):

    if pd.isna(abs_r):
        return "undefined"

    if abs_r < 0.10:
        return "negligible"

    if abs_r < 0.30:
        return "weak"

    if abs_r < 0.50:
        return "moderate"

    if abs_r < 0.70:
        return "strong"

    return "very_strong"


# ==============================================================================
# 11.4 DATASET-LEVEL PEARSON CORRELATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    print(f"\nProcessing dataset: {dataset_id}")

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    schema = NB02_PERSISTED_SCHEMA_CONTRACTS[dataset_id]
    modeling_schema = schema["modeling_schema"]

    # --------------------------------------------------------------------------
    # Frozen NB02 schema
    # --------------------------------------------------------------------------

    preprocessing_columns = list(
        modeling_schema["preprocessing_columns"]
    )

    numeric_columns = list(
        modeling_schema["numeric_columns"]
    )

    categorical_columns = list(
        modeling_schema["categorical_columns"]
    )

    generative_columns = list(
        modeling_schema["generative_columns"]
    )

    target_column = modeling_schema["target_column"]

    identifier_columns = list(
        modeling_schema.get(
            "identifier_columns_excluded",
            []
        )
    )

    provenance_column = modeling_schema.get(
        "provenance_column",
        PROVENANCE_COLUMN
    )

    # --------------------------------------------------------------------------
    # Validate native schema
    # --------------------------------------------------------------------------

    expected_native_columns = (
        [provenance_column]
        + generative_columns
    )

    if list(df.columns) != expected_native_columns:
        raise RuntimeError(
            f"Native schema mismatch for {dataset_id}."
        )

    # --------------------------------------------------------------------------
    # Validate target policy
    # --------------------------------------------------------------------------

    if target_column not in generative_columns:
        raise RuntimeError(
            f"Target {target_column} missing from generative schema "
            f"for {dataset_id}."
        )

    if target_column in preprocessing_columns:
        raise RuntimeError(
            f"Target {target_column} incorrectly included in "
            f"preprocessing columns for {dataset_id}."
        )

    if target_column in numeric_columns:
        raise RuntimeError(
            f"Target {target_column} incorrectly included in "
            f"numeric preprocessing columns for {dataset_id}."
        )

    if target_column in categorical_columns:
        raise RuntimeError(
            f"Target {target_column} incorrectly included in "
            f"categorical preprocessing columns for {dataset_id}."
        )

    # --------------------------------------------------------------------------
    # Validate identifier / provenance policy
    # --------------------------------------------------------------------------

    for identifier in identifier_columns:

        if identifier in df.columns:
            raise RuntimeError(
                f"Excluded identifier {identifier} is physically present "
                f"in native dataset for {dataset_id}."
            )

    if provenance_column not in df.columns:
        raise RuntimeError(
            f"Provenance column {provenance_column} missing "
            f"for {dataset_id}."
        )

    # --------------------------------------------------------------------------
    # Validate numeric/categorical partition
    # --------------------------------------------------------------------------

    if set(numeric_columns).intersection(categorical_columns):
        raise RuntimeError(
            f"Numeric/categorical overlap detected for {dataset_id}."
        )

    if set(numeric_columns).union(categorical_columns) != set(
        preprocessing_columns
    ):
        raise RuntimeError(
            f"Numeric + categorical columns do not exactly equal "
            f"preprocessing columns for {dataset_id}."
        )

    # --------------------------------------------------------------------------
    # Validate Section 5 feature characterization
    # --------------------------------------------------------------------------

    section5 = FEATURE_TYPE_CHARACTERIZATION_DF[
        FEATURE_TYPE_CHARACTERIZATION_DF["dataset_id"] == dataset_id
    ].copy()

    section5_numeric = section5.loc[
        section5["feature_type"] == "numeric",
        "feature"
    ].tolist()

    if set(section5_numeric) != set(numeric_columns):
        raise RuntimeError(
            f"Section 5 numeric feature identity mismatch "
            f"for {dataset_id}."
        )

    # --------------------------------------------------------------------------
    # Validate numeric features physically exist
    # --------------------------------------------------------------------------

    missing_numeric_columns = [
        c for c in numeric_columns
        if c not in df.columns
    ]

    if missing_numeric_columns:
        raise RuntimeError(
            f"Numeric features missing from dataframe for {dataset_id}: "
            f"{missing_numeric_columns}"
        )

    # --------------------------------------------------------------------------
    # Preserve schema order
    # --------------------------------------------------------------------------

    PEARSON_NUMERIC_COLUMNS[dataset_id] = list(
        numeric_columns
    )

    # --------------------------------------------------------------------------
    # Convert numeric columns safely
    # --------------------------------------------------------------------------

    numeric_df = df[numeric_columns].copy()

    for feature in numeric_columns:

        numeric_df[feature] = pd.to_numeric(
            numeric_df[feature],
            errors="coerce"
        )

    # --------------------------------------------------------------------------
    # Validate finite values
    # --------------------------------------------------------------------------

    finite_violation_features = []

    for feature in numeric_columns:

        non_finite_count = int(
            (~np.isfinite(
                numeric_df[feature].dropna().to_numpy(dtype=float)
            )).sum()
        )

        if non_finite_count > 0:
            finite_violation_features.append(
                (feature, non_finite_count)
            )

    if finite_violation_features:
        raise RuntimeError(
            f"Non-finite numeric values detected for "
            f"{dataset_id}: {finite_violation_features}"
        )

    # --------------------------------------------------------------------------
    # Dataset-specific variance
    #
    # IMPORTANT:
    # Keep this inside the dataset loop and persist it by dataset.
    # This prevents the previous KeyError caused by using the last dataset's
    # variance Series during validation of earlier datasets.
    # --------------------------------------------------------------------------

    numeric_variances = numeric_df.var(
        skipna=True
    )

    constant_features = [
        feature
        for feature in numeric_columns
        if (
            pd.notna(numeric_variances.loc[feature])
            and float(numeric_variances.loc[feature]) == 0.0
        )
    ]

    PEARSON_CONSTANT_FEATURES[dataset_id] = list(
        constant_features
    )

    # --------------------------------------------------------------------------
    # Pearson correlation matrix
    #
    # pandas uses pairwise complete observations by default.
    # --------------------------------------------------------------------------

    corr = numeric_df.corr(
        method="pearson"
    )

    corr = corr.reindex(
        index=numeric_columns,
        columns=numeric_columns
    )

    PEARSON_MATRICES[dataset_id] = corr

    # --------------------------------------------------------------------------
    # Unique unordered feature pairs
    # --------------------------------------------------------------------------

    for i, feature_a in enumerate(numeric_columns):

        for j in range(i + 1, len(numeric_columns)):

            feature_b = numeric_columns[j]

            value = corr.loc[
                feature_a,
                feature_b
            ]

            valid_pair_count = int(
                numeric_df[
                    [feature_a, feature_b]
                ].dropna().shape[0]
            )

            if pd.notna(value):

                pearson_r = float(value)
                absolute_r = abs(pearson_r)

            else:

                pearson_r = np.nan
                absolute_r = np.nan

            PEARSON_LONG_RECORDS.append({

                "dataset_id": dataset_id,

                "feature_a": feature_a,

                "feature_b": feature_b,

                "pearson_r": pearson_r,

                "absolute_pearson_r": absolute_r,

                "valid_pair_count": valid_pair_count,

                "correlation_strength":
                    classify_correlation_strength(
                        absolute_r
                    ),
            })

    # --------------------------------------------------------------------------
    # Feature-level correlation summary
    # --------------------------------------------------------------------------

    for feature in numeric_columns:

        row = corr.loc[feature].drop(
            labels=[feature],
            errors="ignore"
        )

        finite_row = row.dropna()

        if len(finite_row) > 0:

            abs_values = finite_row.abs()

            max_abs_r = float(
                abs_values.max()
            )

            mean_abs_r = float(
                abs_values.mean()
            )

            count_abs_r_ge_010 = int(
                (abs_values >= 0.10).sum()
            )

            count_abs_r_ge_050 = int(
                (abs_values >= 0.50).sum()
            )

            count_abs_r_ge_070 = int(
                (abs_values >= 0.70).sum()
            )

        else:

            max_abs_r = np.nan
            mean_abs_r = np.nan
            count_abs_r_ge_010 = 0
            count_abs_r_ge_050 = 0
            count_abs_r_ge_070 = 0

        PEARSON_FEATURE_SUMMARY_RECORDS.append({

            "dataset_id":
                dataset_id,

            "feature":
                feature,

            "variance":
                float(numeric_variances.loc[feature])
                if pd.notna(
                    numeric_variances.loc[feature]
                )
                else np.nan,

            "is_constant":
                feature in constant_features,

            "max_absolute_pearson_r":
                max_abs_r,

            "mean_absolute_pearson_r":
                mean_abs_r,

            "count_absolute_r_ge_0_10":
                count_abs_r_ge_010,

            "count_absolute_r_ge_0_50":
                count_abs_r_ge_050,

            "count_absolute_r_ge_0_70":
                count_abs_r_ge_070,
        })

    # --------------------------------------------------------------------------
    # Dataset-level summary
    # --------------------------------------------------------------------------

    pair_values = []

    for i, feature_a in enumerate(numeric_columns):

        for j in range(i + 1, len(numeric_columns)):

            feature_b = numeric_columns[j]

            value = corr.loc[
                feature_a,
                feature_b
            ]

            if pd.notna(value):
                pair_values.append(
                    abs(float(value))
                )

    if pair_values:

        mean_absolute_r = float(
            np.mean(pair_values)
        )

        maximum_absolute_r = float(
            np.max(pair_values)
        )

        strong_pair_count = int(
            np.sum(
                np.asarray(pair_values) >= 0.50
            )
        )

        very_strong_pair_count = int(
            np.sum(
                np.asarray(pair_values) >= 0.70
            )
        )

    else:

        mean_absolute_r = np.nan
        maximum_absolute_r = np.nan
        strong_pair_count = 0
        very_strong_pair_count = 0

    PEARSON_DATASET_SUMMARY_RECORDS.append({

        "dataset_id":
            dataset_id,

        "numeric_features":
            len(numeric_columns),

        "unique_feature_pairs":
            len(numeric_columns)
            * (len(numeric_columns) - 1)
            // 2,

        "constant_numeric_features":
            len(constant_features),

        "defined_correlation_pairs":
            len(pair_values),

        "mean_absolute_pearson_r":
            mean_absolute_r,

        "maximum_absolute_pearson_r":
            maximum_absolute_r,

        "strong_pairs_abs_r_ge_0_50":
            strong_pair_count,

        "very_strong_pairs_abs_r_ge_0_70":
            very_strong_pair_count,
    })


# ==============================================================================
# 11.5 CREATE OUTPUT DATAFRAMES
# ==============================================================================

PEARSON_DF = pd.DataFrame(
    PEARSON_LONG_RECORDS
)

PEARSON_FEATURE_SUMMARY_DF = pd.DataFrame(
    PEARSON_FEATURE_SUMMARY_RECORDS
)

PEARSON_SUMMARY_DF = pd.DataFrame(
    PEARSON_DATASET_SUMMARY_RECORDS
)


# ==============================================================================
# 11.6 BASIC OUTPUT REPORT
# ==============================================================================

print(
    f"\nUnique Pearson feature pairs: "
    f"{len(PEARSON_DF)}"
)

print("\nPearson dataset summary:")
print(
    PEARSON_SUMMARY_DF.to_string(
        index=False
    )
)


# ==============================================================================
# 11.7 VALIDATION — EXPECTED COVERAGE
# ==============================================================================

expected_pair_count = 0

for dataset_id in DATASET_IDS:

    n_numeric = len(
        PEARSON_NUMERIC_COLUMNS[dataset_id]
    )

    expected_pair_count += (
        n_numeric * (n_numeric - 1) // 2
    )

if len(PEARSON_DF) != expected_pair_count:

    raise RuntimeError(
        "Pearson pair coverage mismatch.\n"
        f"Expected: {expected_pair_count}\n"
        f"Observed: {len(PEARSON_DF)}"
    )


# ==============================================================================
# 11.8 VALIDATION — NO DUPLICATE UNORDERED PAIRS
# ==============================================================================

if not PEARSON_DF.empty:

    duplicate_pairs = PEARSON_DF.duplicated(
        subset=[
            "dataset_id",
            "feature_a",
            "feature_b",
        ]
    ).sum()

    if duplicate_pairs != 0:

        raise RuntimeError(
            f"Duplicate Pearson pairs detected: "
            f"{duplicate_pairs}"
        )


# ==============================================================================
# 11.9 VALIDATION — PAIR ORDERING
# ==============================================================================

for dataset_id in DATASET_IDS:

    numeric_columns = PEARSON_NUMERIC_COLUMNS[
        dataset_id
    ]

    feature_position = {
        feature: i
        for i, feature in enumerate(
            numeric_columns
        )
    }

    subset = PEARSON_DF[
        PEARSON_DF["dataset_id"] == dataset_id
    ]

    for _, row in subset.iterrows():

        if feature_position[row["feature_a"]] >= feature_position[
            row["feature_b"]
        ]:

            raise RuntimeError(
                f"Invalid pair ordering for "
                f"{dataset_id}: "
                f"{row['feature_a']} / "
                f"{row['feature_b']}"
            )


# ==============================================================================
# 11.10 VALIDATION — CORRELATION RANGE
# ==============================================================================

defined_correlations = PEARSON_DF[
    PEARSON_DF["pearson_r"].notna()
]["pearson_r"]

if not defined_correlations.empty:

    if (
        (defined_correlations < -1.0).any()
        or
        (defined_correlations > 1.0).any()
    ):

        raise RuntimeError(
            "Pearson correlation outside [-1, 1]."
        )


# ==============================================================================
# 11.11 VALIDATION — VALID PAIR COUNTS
# ==============================================================================

if not PEARSON_DF.empty:

    invalid_pair_counts = (
        PEARSON_DF["valid_pair_count"] < 0
    )

    if invalid_pair_counts.any():

        raise RuntimeError(
            "Negative valid pair counts detected."
        )

    training_rows_by_dataset = {
        dataset_id:
            len(TRAIN_STATISTICAL_DATASETS[dataset_id])
        for dataset_id in DATASET_IDS
    }

    for _, row in PEARSON_DF.iterrows():

        dataset_id = row["dataset_id"]

        if row["valid_pair_count"] > (
            training_rows_by_dataset[dataset_id]
        ):

            raise RuntimeError(
                f"Invalid valid_pair_count for "
                f"{dataset_id} / "
                f"{row['feature_a']} / "
                f"{row['feature_b']}"
            )


# ==============================================================================
# 11.12 VALIDATION — MATRIX STRUCTURE
# ==============================================================================

for dataset_id in DATASET_IDS:

    numeric_columns = PEARSON_NUMERIC_COLUMNS[
        dataset_id
    ]

    corr = PEARSON_MATRICES[
        dataset_id
    ]

    if list(corr.index) != numeric_columns:

        raise RuntimeError(
            f"Pearson matrix row order mismatch "
            f"for {dataset_id}."
        )

    if list(corr.columns) != numeric_columns:

        raise RuntimeError(
            f"Pearson matrix column order mismatch "
            f"for {dataset_id}."
        )

    # Symmetry check for defined values.
    matrix_values = corr.to_numpy(
        dtype=float
    )

    symmetry_mask = (
        np.isfinite(matrix_values)
        &
        np.isfinite(matrix_values.T)
    )

    if np.any(
        np.abs(
            matrix_values[symmetry_mask]
            -
            matrix_values.T[symmetry_mask]
        ) > 1e-12
    ):

        raise RuntimeError(
            f"Pearson matrix is not symmetric "
            f"for {dataset_id}."
        )

    # --------------------------------------------------------------------------
    # Dataset-specific constant-feature validation
    # --------------------------------------------------------------------------

    constant_features = PEARSON_CONSTANT_FEATURES[
        dataset_id
    ]

    for feature in constant_features:

        variance = (
            numeric_df[feature].var(
                skipna=True
            )
            if False
            else None
        )

        # Recompute only the required scalar from the correct dataset.
        dataset_numeric_df = (
            TRAIN_STATISTICAL_DATASETS[dataset_id][
                numeric_columns
            ]
            .apply(
                pd.to_numeric,
                errors="coerce"
            )
        )

        feature_variance = (
            dataset_numeric_df[feature]
            .var(skipna=True)
        )

        if pd.notna(feature_variance):

            if float(feature_variance) != 0.0:

                raise RuntimeError(
                    f"Constant-feature registry mismatch "
                    f"for {dataset_id}/{feature}."
                )


# ==============================================================================
# 11.13 VALIDATION — FEATURE SUMMARY COVERAGE
# ==============================================================================

expected_numeric_feature_count = sum(
    len(PEARSON_NUMERIC_COLUMNS[dataset_id])
    for dataset_id in DATASET_IDS
)

if len(PEARSON_FEATURE_SUMMARY_DF) != (
    expected_numeric_feature_count
):

    raise RuntimeError(
        "Pearson feature summary coverage mismatch.\n"
        f"Expected: {expected_numeric_feature_count}\n"
        f"Observed: {len(PEARSON_FEATURE_SUMMARY_DF)}"
    )


# ==============================================================================
# 11.14 VALIDATION — TARGET EXCLUSION
# ==============================================================================

for dataset_id in DATASET_IDS:

    target_column = NB02_PERSISTED_SCHEMA_CONTRACTS[
        dataset_id
    ]["modeling_schema"]["target_column"]

    if target_column in PEARSON_NUMERIC_COLUMNS[
        dataset_id
    ]:

        raise RuntimeError(
            f"Target {target_column} incorrectly included "
            f"in Pearson analysis for {dataset_id}."
        )


# ==============================================================================
# 11.15 VALIDATION — NO CATEGORICAL FEATURES
# ==============================================================================

for dataset_id in DATASET_IDS:

    categorical_columns = (
        NB02_PERSISTED_SCHEMA_CONTRACTS[
            dataset_id
        ]["modeling_schema"]["categorical_columns"]
    )

    overlap = set(
        PEARSON_NUMERIC_COLUMNS[dataset_id]
    ).intersection(
        categorical_columns
    )

    if overlap:

        raise RuntimeError(
            f"Categorical features incorrectly included "
            f"in Pearson analysis for {dataset_id}: "
            f"{sorted(overlap)}"
        )


# ==============================================================================
# 11.16 VALIDATION — NO PROVENANCE / IDENTIFIERS
# ==============================================================================

for dataset_id in DATASET_IDS:

    numeric_columns = PEARSON_NUMERIC_COLUMNS[
        dataset_id
    ]

    schema = NB02_PERSISTED_SCHEMA_CONTRACTS[
        dataset_id
    ]

    modeling_schema = schema[
        "modeling_schema"
    ]

    provenance_column = modeling_schema.get(
        "provenance_column",
        PROVENANCE_COLUMN
    )

    identifier_columns = modeling_schema.get(
        "identifier_columns_excluded",
        []
    )

    forbidden_columns = (
        [provenance_column]
        + list(identifier_columns)
    )

    overlap = set(numeric_columns).intersection(
        forbidden_columns
    )

    if overlap:

        raise RuntimeError(
            f"Provenance/identifier columns included "
            f"in Pearson analysis for {dataset_id}: "
            f"{sorted(overlap)}"
        )


# ==============================================================================
# 11.17 VALIDATION — FINITE SUMMARY VALUES
# ==============================================================================

numeric_summary_columns = [
    "mean_absolute_pearson_r",
    "maximum_absolute_pearson_r",
]

for column in numeric_summary_columns:

    values = PEARSON_SUMMARY_DF[
        column
    ].dropna()

    if not values.empty:

        if not np.isfinite(
            values.to_numpy(dtype=float)
        ).all():

            raise RuntimeError(
                f"Non-finite values detected in "
                f"Pearson summary column: {column}"
            )


# ==============================================================================
# 11.18 FINAL STATUS
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 11 VALIDATION")
print("=" * 100)

print(
    "✓ Training data exclusively"
)

print(
    "✓ Frozen Notebook 02 schema authority"
)

print(
    "✓ Numeric preprocessing features only"
)

print(
    "✓ Target excluded"
)

print(
    "✓ Categorical features excluded"
)

print(
    "✓ Identifier/provenance columns excluded"
)

print(
    "✓ Section 5 numeric feature identity verified"
)

print(
    "✓ Dataset-specific variance tracking verified"
)

print(
    "✓ Pearson matrices generated"
)

print(
    "✓ Unique unordered feature pairs generated"
)

print(
    "✓ Pair coverage verified"
)

print(
    "✓ No duplicate feature pairs"
)

print(
    "✓ Correlation range verified"
)

print(
    "✓ Pairwise valid-observation counts verified"
)

print(
    "✓ Matrix symmetry verified"
)

print(
    "✓ Matrix feature order verified"
)

print(
    "✓ Constant numeric features verified"
)

print(
    "✓ Feature-summary coverage verified"
)

print(
    "✓ Output summary integrity verified"
)

print("\nSECTION 11 STATUS: PASS")

SECTION 11 — PEARSON CORRELATION

Processing dataset: adult_income

Processing dataset: bank_marketing

Processing dataset: diabetes_130us

Unique Pearson feature pairs: 91

Pearson dataset summary:
    dataset_id  numeric_features  unique_feature_pairs  constant_numeric_features  defined_correlation_pairs  mean_absolute_pearson_r  maximum_absolute_pearson_r  strong_pairs_abs_r_ge_0_50  very_strong_pairs_abs_r_ge_0_70
  adult_income                 6                    15                          0                         15                 0.059193                    0.147515                           0                                0
bank_marketing                 7                    21                          0                         21                 0.061189                    0.537336                           1                                0
diabetes_130us                11                    55                          0                         55                 0.09325

In [12]:
# ==============================================================================
# SECTION 12 — SPEARMAN CORRELATION
# ==============================================================================

print("=" * 100)
print("SECTION 12 — SPEARMAN CORRELATION")
print("=" * 100)

# ==============================================================================
# 12.1 REQUIRED OBJECTS
# ==============================================================================

REQUIRED_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_STATISTICAL_DATASETS",
    "NB02_PERSISTED_SCHEMA_CONTRACTS",
    "FEATURE_TYPE_CHARACTERIZATION_DF",
]

missing_objects = [
    obj for obj in REQUIRED_OBJECTS
    if obj not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 12 cannot start.\n"
        f"Missing required objects: {missing_objects}"
    )


# ==============================================================================
# 12.2 RESET OUTPUT CONTAINERS
# ==============================================================================

SPEARMAN_MATRICES = {}
SPEARMAN_CONSTANT_FEATURES = {}
SPEARMAN_NUMERIC_COLUMNS = {}

SPEARMAN_LONG_RECORDS = []
SPEARMAN_FEATURE_SUMMARY_RECORDS = []
SPEARMAN_DATASET_SUMMARY_RECORDS = []


# ==============================================================================
# 12.3 HELPER — CORRELATION STRENGTH
# ==============================================================================

def classify_spearman_strength(abs_rho):

    if pd.isna(abs_rho):
        return "undefined"

    if abs_rho < 0.10:
        return "negligible"

    if abs_rho < 0.30:
        return "weak"

    if abs_rho < 0.50:
        return "moderate"

    if abs_rho < 0.70:
        return "strong"

    return "very_strong"


# ==============================================================================
# 12.4 DATASET-LEVEL SPEARMAN CORRELATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    print(f"\nProcessing dataset: {dataset_id}")

    df = TRAIN_STATISTICAL_DATASETS[dataset_id]

    schema = NB02_PERSISTED_SCHEMA_CONTRACTS[dataset_id]
    modeling_schema = schema["modeling_schema"]

    # --------------------------------------------------------------------------
    # Frozen Notebook 02 schema
    # --------------------------------------------------------------------------

    preprocessing_columns = list(
        modeling_schema["preprocessing_columns"]
    )

    numeric_columns = list(
        modeling_schema["numeric_columns"]
    )

    categorical_columns = list(
        modeling_schema["categorical_columns"]
    )

    generative_columns = list(
        modeling_schema["generative_columns"]
    )

    target_column = modeling_schema["target_column"]

    identifier_columns = list(
        modeling_schema.get(
            "identifier_columns_excluded",
            []
        )
    )

    provenance_column = modeling_schema.get(
        "provenance_column",
        PROVENANCE_COLUMN
    )

    # --------------------------------------------------------------------------
    # Validate native schema
    # --------------------------------------------------------------------------

    expected_native_columns = (
        [provenance_column]
        + generative_columns
    )

    if list(df.columns) != expected_native_columns:
        raise RuntimeError(
            f"Native schema mismatch for {dataset_id}."
        )

    # --------------------------------------------------------------------------
    # Validate target policy
    # --------------------------------------------------------------------------

    if target_column not in generative_columns:
        raise RuntimeError(
            f"Target {target_column} missing from generative schema "
            f"for {dataset_id}."
        )

    if target_column in preprocessing_columns:
        raise RuntimeError(
            f"Target {target_column} incorrectly included in "
            f"preprocessing columns for {dataset_id}."
        )

    if target_column in numeric_columns:
        raise RuntimeError(
            f"Target {target_column} incorrectly included in "
            f"numeric preprocessing columns for {dataset_id}."
        )

    if target_column in categorical_columns:
        raise RuntimeError(
            f"Target {target_column} incorrectly included in "
            f"categorical preprocessing columns for {dataset_id}."
        )

    # --------------------------------------------------------------------------
    # Validate identifier / provenance policy
    # --------------------------------------------------------------------------

    for identifier in identifier_columns:

        if identifier in df.columns:
            raise RuntimeError(
                f"Excluded identifier {identifier} is physically present "
                f"in native dataset for {dataset_id}."
            )

    if provenance_column not in df.columns:
        raise RuntimeError(
            f"Provenance column {provenance_column} missing "
            f"for {dataset_id}."
        )

    # --------------------------------------------------------------------------
    # Validate numeric/categorical partition
    # --------------------------------------------------------------------------

    if set(numeric_columns).intersection(
        categorical_columns
    ):
        raise RuntimeError(
            f"Numeric/categorical overlap detected for {dataset_id}."
        )

    if set(numeric_columns).union(
        categorical_columns
    ) != set(preprocessing_columns):
        raise RuntimeError(
            f"Numeric + categorical columns do not exactly equal "
            f"preprocessing columns for {dataset_id}."
        )

    # --------------------------------------------------------------------------
    # Validate Section 5 numeric feature identity
    # --------------------------------------------------------------------------

    section5 = FEATURE_TYPE_CHARACTERIZATION_DF[
        FEATURE_TYPE_CHARACTERIZATION_DF["dataset_id"] == dataset_id
    ].copy()

    section5_numeric = section5.loc[
        section5["feature_type"] == "numeric",
        "feature"
    ].tolist()

    if set(section5_numeric) != set(numeric_columns):
        raise RuntimeError(
            f"Section 5 numeric feature identity mismatch "
            f"for {dataset_id}."
        )

    # --------------------------------------------------------------------------
    # Validate physical feature presence
    # --------------------------------------------------------------------------

    missing_numeric_columns = [
        feature
        for feature in numeric_columns
        if feature not in df.columns
    ]

    if missing_numeric_columns:
        raise RuntimeError(
            f"Numeric features missing from dataframe for "
            f"{dataset_id}: {missing_numeric_columns}"
        )

    # --------------------------------------------------------------------------
    # Preserve frozen schema order
    # --------------------------------------------------------------------------

    SPEARMAN_NUMERIC_COLUMNS[dataset_id] = list(
        numeric_columns
    )

    # --------------------------------------------------------------------------
    # Convert numeric features safely
    # --------------------------------------------------------------------------

    numeric_df = df[numeric_columns].copy()

    for feature in numeric_columns:

        numeric_df[feature] = pd.to_numeric(
            numeric_df[feature],
            errors="coerce"
        )

    # --------------------------------------------------------------------------
    # Validate finite values
    # --------------------------------------------------------------------------

    finite_violation_features = []

    for feature in numeric_columns:

        values = numeric_df[feature].dropna()

        non_finite_count = int(
            (~np.isfinite(
                values.to_numpy(dtype=float)
            )).sum()
        )

        if non_finite_count > 0:

            finite_violation_features.append(
                (feature, non_finite_count)
            )

    if finite_violation_features:
        raise RuntimeError(
            f"Non-finite numeric values detected for "
            f"{dataset_id}: {finite_violation_features}"
        )

    # --------------------------------------------------------------------------
    # Dataset-specific variance / constant-feature detection
    # --------------------------------------------------------------------------

    numeric_variances = numeric_df.var(
        skipna=True
    )

    constant_features = [
        feature
        for feature in numeric_columns
        if (
            pd.notna(
                numeric_variances.loc[feature]
            )
            and
            float(
                numeric_variances.loc[feature]
            ) == 0.0
        )
    ]

    SPEARMAN_CONSTANT_FEATURES[dataset_id] = list(
        constant_features
    )

    # --------------------------------------------------------------------------
    # Spearman correlation matrix
    #
    # Spearman correlation is computed from ranked values.
    # pandas performs pairwise complete-observation handling.
    # --------------------------------------------------------------------------

    corr = numeric_df.corr(
        method="spearman"
    )

    corr = corr.reindex(
        index=numeric_columns,
        columns=numeric_columns
    )

    SPEARMAN_MATRICES[dataset_id] = corr

    # --------------------------------------------------------------------------
    # Unique unordered feature pairs
    # --------------------------------------------------------------------------

    for i, feature_a in enumerate(numeric_columns):

        for j in range(
            i + 1,
            len(numeric_columns)
        ):

            feature_b = numeric_columns[j]

            value = corr.loc[
                feature_a,
                feature_b
            ]

            valid_pair_count = int(
                numeric_df[
                    [feature_a, feature_b]
                ]
                .dropna()
                .shape[0]
            )

            if pd.notna(value):

                spearman_rho = float(value)

                absolute_rho = abs(
                    spearman_rho
                )

            else:

                spearman_rho = np.nan
                absolute_rho = np.nan

            SPEARMAN_LONG_RECORDS.append({

                "dataset_id":
                    dataset_id,

                "feature_a":
                    feature_a,

                "feature_b":
                    feature_b,

                "spearman_rho":
                    spearman_rho,

                "absolute_spearman_rho":
                    absolute_rho,

                "valid_pair_count":
                    valid_pair_count,

                "correlation_strength":
                    classify_spearman_strength(
                        absolute_rho
                    ),
            })

    # --------------------------------------------------------------------------
    # Feature-level summary
    # --------------------------------------------------------------------------

    for feature in numeric_columns:

        row = corr.loc[feature].drop(
            labels=[feature],
            errors="ignore"
        )

        finite_row = row.dropna()

        if len(finite_row) > 0:

            abs_values = finite_row.abs()

            max_abs_rho = float(
                abs_values.max()
            )

            mean_abs_rho = float(
                abs_values.mean()
            )

            count_abs_rho_ge_010 = int(
                (
                    abs_values >= 0.10
                ).sum()
            )

            count_abs_rho_ge_050 = int(
                (
                    abs_values >= 0.50
                ).sum()
            )

            count_abs_rho_ge_070 = int(
                (
                    abs_values >= 0.70
                ).sum()
            )

        else:

            max_abs_rho = np.nan
            mean_abs_rho = np.nan
            count_abs_rho_ge_010 = 0
            count_abs_rho_ge_050 = 0
            count_abs_rho_ge_070 = 0

        SPEARMAN_FEATURE_SUMMARY_RECORDS.append({

            "dataset_id":
                dataset_id,

            "feature":
                feature,

            "variance":
                float(
                    numeric_variances.loc[feature]
                )
                if pd.notna(
                    numeric_variances.loc[feature]
                )
                else np.nan,

            "is_constant":
                feature in constant_features,

            "max_absolute_spearman_rho":
                max_abs_rho,

            "mean_absolute_spearman_rho":
                mean_abs_rho,

            "count_absolute_rho_ge_0_10":
                count_abs_rho_ge_010,

            "count_absolute_rho_ge_0_50":
                count_abs_rho_ge_050,

            "count_absolute_rho_ge_0_70":
                count_abs_rho_ge_070,
        })

    # --------------------------------------------------------------------------
    # Dataset-level summary
    # --------------------------------------------------------------------------

    pair_values = []

    for i, feature_a in enumerate(numeric_columns):

        for j in range(
            i + 1,
            len(numeric_columns)
        ):

            feature_b = numeric_columns[j]

            value = corr.loc[
                feature_a,
                feature_b
            ]

            if pd.notna(value):

                pair_values.append(
                    abs(float(value))
                )

    if pair_values:

        mean_absolute_rho = float(
            np.mean(pair_values)
        )

        maximum_absolute_rho = float(
            np.max(pair_values)
        )

        strong_pair_count = int(
            np.sum(
                np.asarray(pair_values) >= 0.50
            )
        )

        very_strong_pair_count = int(
            np.sum(
                np.asarray(pair_values) >= 0.70
            )
        )

    else:

        mean_absolute_rho = np.nan
        maximum_absolute_rho = np.nan
        strong_pair_count = 0
        very_strong_pair_count = 0

    SPEARMAN_DATASET_SUMMARY_RECORDS.append({

        "dataset_id":
            dataset_id,

        "numeric_features":
            len(numeric_columns),

        "unique_feature_pairs":
            len(numeric_columns)
            * (len(numeric_columns) - 1)
            // 2,

        "constant_numeric_features":
            len(constant_features),

        "defined_correlation_pairs":
            len(pair_values),

        "mean_absolute_spearman_rho":
            mean_absolute_rho,

        "maximum_absolute_spearman_rho":
            maximum_absolute_rho,

        "strong_pairs_abs_rho_ge_0_50":
            strong_pair_count,

        "very_strong_pairs_abs_rho_ge_0_70":
            very_strong_pair_count,
    })


# ==============================================================================
# 12.5 CREATE OUTPUT DATAFRAMES
# ==============================================================================

SPEARMAN_DF = pd.DataFrame(
    SPEARMAN_LONG_RECORDS
)

SPEARMAN_FEATURE_SUMMARY_DF = pd.DataFrame(
    SPEARMAN_FEATURE_SUMMARY_RECORDS
)

SPEARMAN_SUMMARY_DF = pd.DataFrame(
    SPEARMAN_DATASET_SUMMARY_RECORDS
)


# ==============================================================================
# 12.6 BASIC OUTPUT REPORT
# ==============================================================================

print(
    f"\nUnique Spearman feature pairs: "
    f"{len(SPEARMAN_DF)}"
)

print("\nSpearman dataset summary:")

print(
    SPEARMAN_SUMMARY_DF.to_string(
        index=False
    )
)


# ==============================================================================
# 12.7 VALIDATION — EXPECTED PAIR COVERAGE
# ==============================================================================

expected_pair_count = 0

for dataset_id in DATASET_IDS:

    n_numeric = len(
        SPEARMAN_NUMERIC_COLUMNS[dataset_id]
    )

    expected_pair_count += (
        n_numeric
        * (n_numeric - 1)
        // 2
    )

if len(SPEARMAN_DF) != expected_pair_count:

    raise RuntimeError(
        "Spearman pair coverage mismatch.\n"
        f"Expected: {expected_pair_count}\n"
        f"Observed: {len(SPEARMAN_DF)}"
    )


# ==============================================================================
# 12.8 VALIDATION — NO DUPLICATE PAIRS
# ==============================================================================

if not SPEARMAN_DF.empty:

    duplicate_pairs = SPEARMAN_DF.duplicated(
        subset=[
            "dataset_id",
            "feature_a",
            "feature_b",
        ]
    ).sum()

    if duplicate_pairs != 0:

        raise RuntimeError(
            f"Duplicate Spearman pairs detected: "
            f"{duplicate_pairs}"
        )


# ==============================================================================
# 12.9 VALIDATION — PAIR ORDERING
# ==============================================================================

for dataset_id in DATASET_IDS:

    numeric_columns = SPEARMAN_NUMERIC_COLUMNS[
        dataset_id
    ]

    feature_position = {
        feature: i
        for i, feature in enumerate(
            numeric_columns
        )
    }

    subset = SPEARMAN_DF[
        SPEARMAN_DF["dataset_id"] == dataset_id
    ]

    for _, row in subset.iterrows():

        if feature_position[
            row["feature_a"]
        ] >= feature_position[
            row["feature_b"]
        ]:

            raise RuntimeError(
                f"Invalid Spearman pair ordering for "
                f"{dataset_id}: "
                f"{row['feature_a']} / "
                f"{row['feature_b']}"
            )


# ==============================================================================
# 12.10 VALIDATION — CORRELATION RANGE
# ==============================================================================

defined_correlations = SPEARMAN_DF[
    SPEARMAN_DF["spearman_rho"].notna()
]["spearman_rho"]

if not defined_correlations.empty:

    if (
        (
            defined_correlations < -1.0
        ).any()
        or
        (
            defined_correlations > 1.0
        ).any()
    ):

        raise RuntimeError(
            "Spearman correlation outside [-1, 1]."
        )


# ==============================================================================
# 12.11 VALIDATION — VALID PAIR COUNTS
# ==============================================================================

if not SPEARMAN_DF.empty:

    if (
        SPEARMAN_DF["valid_pair_count"] < 0
    ).any():

        raise RuntimeError(
            "Negative valid pair counts detected."
        )

    training_rows_by_dataset = {
        dataset_id:
            len(
                TRAIN_STATISTICAL_DATASETS[
                    dataset_id
                ]
            )
        for dataset_id in DATASET_IDS
    }

    for _, row in SPEARMAN_DF.iterrows():

        dataset_id = row["dataset_id"]

        if row["valid_pair_count"] > (
            training_rows_by_dataset[
                dataset_id
            ]
        ):

            raise RuntimeError(
                f"Invalid valid_pair_count for "
                f"{dataset_id} / "
                f"{row['feature_a']} / "
                f"{row['feature_b']}"
            )


# ==============================================================================
# 12.12 VALIDATION — MATRIX STRUCTURE
# ==============================================================================

for dataset_id in DATASET_IDS:

    numeric_columns = SPEARMAN_NUMERIC_COLUMNS[
        dataset_id
    ]

    corr = SPEARMAN_MATRICES[
        dataset_id
    ]

    if list(corr.index) != numeric_columns:

        raise RuntimeError(
            f"Spearman matrix row order mismatch "
            f"for {dataset_id}."
        )

    if list(corr.columns) != numeric_columns:

        raise RuntimeError(
            f"Spearman matrix column order mismatch "
            f"for {dataset_id}."
        )

    matrix_values = corr.to_numpy(
        dtype=float
    )

    symmetry_mask = (
        np.isfinite(matrix_values)
        &
        np.isfinite(matrix_values.T)
    )

    if np.any(
        np.abs(
            matrix_values[symmetry_mask]
            -
            matrix_values.T[symmetry_mask]
        ) > 1e-12
    ):

        raise RuntimeError(
            f"Spearman matrix is not symmetric "
            f"for {dataset_id}."
        )


# ==============================================================================
# 12.13 VALIDATION — DATASET-SPECIFIC CONSTANT FEATURES
# ==============================================================================

for dataset_id in DATASET_IDS:

    numeric_columns = SPEARMAN_NUMERIC_COLUMNS[
        dataset_id
    ]

    constant_features = SPEARMAN_CONSTANT_FEATURES[
        dataset_id
    ]

    dataset_numeric_df = (
        TRAIN_STATISTICAL_DATASETS[
            dataset_id
        ][numeric_columns]
        .apply(
            pd.to_numeric,
            errors="coerce"
        )
    )

    for feature in constant_features:

        feature_variance = (
            dataset_numeric_df[
                feature
            ].var(
                skipna=True
            )
        )

        if pd.notna(feature_variance):

            if float(feature_variance) != 0.0:

                raise RuntimeError(
                    f"Constant-feature registry mismatch "
                    f"for {dataset_id}/{feature}."
                )


# ==============================================================================
# 12.14 VALIDATION — FEATURE SUMMARY COVERAGE
# ==============================================================================

expected_numeric_feature_count = sum(
    len(
        SPEARMAN_NUMERIC_COLUMNS[
            dataset_id
        ]
    )
    for dataset_id in DATASET_IDS
)

if len(
    SPEARMAN_FEATURE_SUMMARY_DF
) != expected_numeric_feature_count:

    raise RuntimeError(
        "Spearman feature summary coverage mismatch.\n"
        f"Expected: {expected_numeric_feature_count}\n"
        f"Observed: {len(SPEARMAN_FEATURE_SUMMARY_DF)}"
    )


# ==============================================================================
# 12.15 VALIDATION — TARGET EXCLUSION
# ==============================================================================

for dataset_id in DATASET_IDS:

    target_column = (
        NB02_PERSISTED_SCHEMA_CONTRACTS[
            dataset_id
        ]["modeling_schema"]["target_column"]
    )

    if target_column in (
        SPEARMAN_NUMERIC_COLUMNS[
            dataset_id
        ]
    ):

        raise RuntimeError(
            f"Target {target_column} incorrectly included "
            f"in Spearman analysis for {dataset_id}."
        )


# ==============================================================================
# 12.16 VALIDATION — CATEGORICAL FEATURES EXCLUDED
# ==============================================================================

for dataset_id in DATASET_IDS:

    categorical_columns = (
        NB02_PERSISTED_SCHEMA_CONTRACTS[
            dataset_id
        ]["modeling_schema"]["categorical_columns"]
    )

    overlap = set(
        SPEARMAN_NUMERIC_COLUMNS[
            dataset_id
        ]
    ).intersection(
        categorical_columns
    )

    if overlap:

        raise RuntimeError(
            f"Categorical features incorrectly included "
            f"in Spearman analysis for {dataset_id}: "
            f"{sorted(overlap)}"
        )


# ==============================================================================
# 12.17 VALIDATION — PROVENANCE / IDENTIFIERS EXCLUDED
# ==============================================================================

for dataset_id in DATASET_IDS:

    numeric_columns = SPEARMAN_NUMERIC_COLUMNS[
        dataset_id
    ]

    modeling_schema = (
        NB02_PERSISTED_SCHEMA_CONTRACTS[
            dataset_id
        ]["modeling_schema"]
    )

    provenance_column = modeling_schema.get(
        "provenance_column",
        PROVENANCE_COLUMN
    )

    identifier_columns = modeling_schema.get(
        "identifier_columns_excluded",
        []
    )

    forbidden_columns = (
        [provenance_column]
        + list(identifier_columns)
    )

    overlap = set(
        numeric_columns
    ).intersection(
        forbidden_columns
    )

    if overlap:

        raise RuntimeError(
            f"Provenance/identifier columns included "
            f"in Spearman analysis for {dataset_id}: "
            f"{sorted(overlap)}"
        )


# ==============================================================================
# 12.18 VALIDATION — SUMMARY VALUES
# ==============================================================================

summary_columns = [
    "mean_absolute_spearman_rho",
    "maximum_absolute_spearman_rho",
]

for column in summary_columns:

    values = SPEARMAN_SUMMARY_DF[
        column
    ].dropna()

    if not values.empty:

        if not np.isfinite(
            values.to_numpy(
                dtype=float
            )
        ).all():

            raise RuntimeError(
                f"Non-finite values detected in "
                f"Spearman summary column: {column}"
            )


# ==============================================================================
# 12.19 FINAL STATUS
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 12 VALIDATION")
print("=" * 100)

print(
    "✓ Training data exclusively"
)

print(
    "✓ Frozen Notebook 02 schema authority"
)

print(
    "✓ Numeric preprocessing features only"
)

print(
    "✓ Target excluded"
)

print(
    "✓ Categorical features excluded"
)

print(
    "✓ Identifier/provenance columns excluded"
)

print(
    "✓ Section 5 numeric feature identity verified"
)

print(
    "✓ Dataset-specific variance tracking verified"
)

print(
    "✓ Spearman matrices generated"
)

print(
    "✓ Unique unordered feature pairs generated"
)

print(
    "✓ Pair coverage verified"
)

print(
    "✓ No duplicate feature pairs"
)

print(
    "✓ Correlation range verified"
)

print(
    "✓ Pairwise valid-observation counts verified"
)

print(
    "✓ Matrix symmetry verified"
)

print(
    "✓ Matrix feature order verified"
)

print(
    "✓ Constant-feature registry verified"
)

print(
    "✓ Feature-summary coverage verified"
)

print(
    "✓ Output summary integrity verified"
)

print("\nSECTION 12 STATUS: PASS")

SECTION 12 — SPEARMAN CORRELATION

Processing dataset: adult_income

Processing dataset: bank_marketing

Processing dataset: diabetes_130us

Unique Spearman feature pairs: 91

Spearman dataset summary:
    dataset_id  numeric_features  unique_feature_pairs  constant_numeric_features  defined_correlation_pairs  mean_absolute_spearman_rho  maximum_absolute_spearman_rho  strong_pairs_abs_rho_ge_0_50  very_strong_pairs_abs_rho_ge_0_70
  adult_income                 6                    15                          0                         15                    0.073052                       0.167068                             0                                  0
bank_marketing                 7                    21                          0                         21                    0.104070                       0.985594                             1                                  1
diabetes_130us                11                    55                          0                  

In [13]:
# ==============================================================================
# SECTION 13 — CATEGORICAL DEPENDENCY ANALYSIS
# ==============================================================================

print("=" * 100)
print("SECTION 13 — CATEGORICAL DEPENDENCY ANALYSIS")
print("=" * 100)

import numpy as np
import pandas as pd

from scipy.stats import chi2_contingency


# ==============================================================================
# 13.1 — CONFIGURATION
# ==============================================================================

DEPENDENCY_MAX_ROWS = 25000
DEPENDENCY_MAX_CATEGORIES = 50
DEPENDENCY_RANDOM_SEED = 2025

DEPENDENCY_STRENGTH_THRESHOLDS = {
    "negligible": 0.10,
    "weak": 0.30,
    "moderate": 0.50,
    "strong": 0.70,
}


# ==============================================================================
# 13.2 — REQUIRED UPSTREAM OBJECTS
# ==============================================================================

REQUIRED_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_STATISTICAL_DATASETS",
    "NB02_PERSISTED_SCHEMA_CONTRACTS",
    "FEATURE_TYPE_CHARACTERIZATION_DF",
    "CATEGORICAL_STATISTICS_DF",
]

_missing_objects = [
    obj for obj in REQUIRED_OBJECTS
    if obj not in globals()
]

if _missing_objects:
    raise RuntimeError(
        "Section 13 cannot start because required upstream objects are missing:\n"
        f"{_missing_objects}"
    )


# ==============================================================================
# 13.3 — HELPER FUNCTIONS
# ==============================================================================

def classify_dependency_strength(value):
    """
    Classify bias-corrected Cramér's V.
    """

    if not np.isfinite(value):
        return "undefined"

    value = abs(float(value))

    if value < DEPENDENCY_STRENGTH_THRESHOLDS["negligible"]:
        return "negligible"

    if value < DEPENDENCY_STRENGTH_THRESHOLDS["weak"]:
        return "weak"

    if value < DEPENDENCY_STRENGTH_THRESHOLDS["moderate"]:
        return "moderate"

    if value < DEPENDENCY_STRENGTH_THRESHOLDS["strong"]:
        return "strong"

    return "very_strong"


def prepare_dependency_series(
    series,
    max_categories=50
):
    """
    Prepare a categorical variable for dependency analysis.

    Policy:
    1. Missing values are represented explicitly as __MISSING__.
    2. If cardinality exceeds the cap, the most frequent
       max_categories - 1 categories are retained.
    3. Remaining categories are deterministically pooled as __OTHER__.
    """

    s = series.astype("object").copy()

    # --------------------------------------------------------------------------
    # Explicit missing category
    # --------------------------------------------------------------------------

    s = s.where(
        s.notna(),
        "__MISSING__"
    )

    original_cardinality = int(
        s.nunique(dropna=False)
    )

    # --------------------------------------------------------------------------
    # No pooling required
    # --------------------------------------------------------------------------

    if original_cardinality <= max_categories:

        return (
            s,
            original_cardinality,
            original_cardinality,
            False,
            0,
        )

    # --------------------------------------------------------------------------
    # Deterministic high-cardinality pooling
    # --------------------------------------------------------------------------

    counts = s.value_counts(
        dropna=False
    )

    keep_categories = set(
        counts.head(
            max_categories - 1
        ).index
    )

    pooled_mask = ~s.isin(
        keep_categories
    )

    pooled_observations = int(
        pooled_mask.sum()
    )

    s = s.where(
        s.isin(keep_categories),
        "__OTHER__"
    )

    final_cardinality = int(
        s.nunique(dropna=False)
    )

    return (
        s,
        original_cardinality,
        final_cardinality,
        True,
        pooled_observations,
    )


def cramers_v(x, y):
    """
    Bias-corrected Cramér's V.

    Returns
    -------
    value : float
        Bias-corrected Cramér's V in [0, 1].
    valid_pair_count : int
        Number of observations used for the pair.
    """

    pair_df = pd.DataFrame({
        "x": x,
        "y": y,
    })

    pair_df = pair_df.dropna(
        subset=["x", "y"]
    )

    valid_pair_count = int(
        len(pair_df)
    )

    if valid_pair_count == 0:
        return np.nan, 0

    contingency = pd.crosstab(
        pair_df["x"],
        pair_df["y"],
        dropna=False,
    )

    if contingency.shape[0] < 2:
        return 0.0, valid_pair_count

    if contingency.shape[1] < 2:
        return 0.0, valid_pair_count

    observed = contingency.to_numpy(
        dtype=float
    )

    try:

        chi2, _, _, _ = chi2_contingency(
            observed,
            correction=False,
        )

    except Exception:

        return np.nan, valid_pair_count

    n = observed.sum()

    if n <= 0:
        return np.nan, valid_pair_count

    phi2 = chi2 / n

    r, k = observed.shape

    # --------------------------------------------------------------------------
    # Bias correction
    # --------------------------------------------------------------------------

    if n > 1:

        phi2_corrected = max(
            0.0,
            phi2
            - (
                (k - 1) * (r - 1)
            ) / (n - 1)
        )

        r_corrected = (
            r
            - ((r - 1) ** 2)
            / (n - 1)
        )

        k_corrected = (
            k
            - ((k - 1) ** 2)
            / (n - 1)
        )

    else:

        phi2_corrected = 0.0
        r_corrected = float(r)
        k_corrected = float(k)

    denominator = min(
        k_corrected - 1,
        r_corrected - 1,
    )

    if denominator <= 0:
        return 0.0, valid_pair_count

    value = np.sqrt(
        phi2_corrected / denominator
    )

    value = float(
        np.clip(value, 0.0, 1.0)
    )

    return value, valid_pair_count


# ==============================================================================
# 13.4 — INITIALIZE OUTPUT CONTAINERS
# ==============================================================================

CATEGORICAL_DEPENDENCY_MATRICES = {}

CATEGORICAL_DEPENDENCY_RECORDS = []

CATEGORICAL_DEPENDENCY_FEATURE_RECORDS = []

CATEGORICAL_DEPENDENCY_SUMMARY_RECORDS = []


# ==============================================================================
# 13.5 — DATASET-WISE DEPENDENCY ANALYSIS
# ==============================================================================

for dataset_id in DATASET_IDS:

    print(
        f"\nProcessing dataset: {dataset_id}"
    )

    # --------------------------------------------------------------------------
    # Load frozen Notebook 02 schema
    # --------------------------------------------------------------------------

    schema_contract = (
        NB02_PERSISTED_SCHEMA_CONTRACTS[
            dataset_id
        ]
    )

    modeling_schema = (
        schema_contract[
            "modeling_schema"
        ]
    )

    # --------------------------------------------------------------------------
    # IMPORTANT NOTE:
    #
    # NB02 all_columns = preprocessing/modeling feature universe.
    # NB02 generative_columns = preprocessing features + target.
    # Native dataframe = provenance + generative columns.
    #
    # Therefore, these are intentionally kept separate.
    # --------------------------------------------------------------------------

    all_columns = list(
        modeling_schema[
            "all_columns"
        ]
    )

    preprocessing_columns = list(
        modeling_schema[
            "preprocessing_columns"
        ]
    )

    numeric_columns = list(
        modeling_schema[
            "numeric_columns"
        ]
    )

    categorical_columns = list(
        modeling_schema[
            "categorical_columns"
        ]
    )

    generative_columns = list(
        modeling_schema[
            "generative_columns"
        ]
    )

    target_column = (
        modeling_schema[
            "target_column"
        ]
    )

    identifier_columns = list(
        modeling_schema.get(
            "identifier_columns_excluded",
            []
        )
    )

    provenance_column = (
        modeling_schema.get(
            "provenance_column",
            "__original_row_id__"
        )
    )

    # --------------------------------------------------------------------------
    # Validate NB02 internal schema relationships
    # --------------------------------------------------------------------------

    if set(all_columns) != set(
        preprocessing_columns
    ):
        raise RuntimeError(
            f"NB02 all_columns / preprocessing_columns "
            f"mismatch for {dataset_id}."
        )

    if all_columns != preprocessing_columns:
        raise RuntimeError(
            f"NB02 all_columns / preprocessing_columns "
            f"order mismatch for {dataset_id}."
        )

    if generative_columns != (
        preprocessing_columns
        + [target_column]
    ):
        raise RuntimeError(
            f"Generative schema contract mismatch "
            f"for {dataset_id}."
        )

    # --------------------------------------------------------------------------
    # Numeric/categorical partition
    # --------------------------------------------------------------------------

    if (
        set(numeric_columns)
        &
        set(categorical_columns)
    ):
        raise RuntimeError(
            f"Numeric/categorical overlap detected "
            f"for {dataset_id}."
        )

    if (
        set(numeric_columns)
        |
        set(categorical_columns)
    ) != set(preprocessing_columns):
        raise RuntimeError(
            f"Numeric/categorical partition does not "
            f"cover preprocessing features for {dataset_id}."
        )

    # --------------------------------------------------------------------------
    # Target exclusion
    # --------------------------------------------------------------------------

    if target_column in preprocessing_columns:
        raise RuntimeError(
            f"Target leakage detected in preprocessing "
            f"columns for {dataset_id}."
        )

    if target_column in categorical_columns:
        raise RuntimeError(
            f"Target incorrectly included in categorical "
            f"preprocessing features for {dataset_id}."
        )

    # --------------------------------------------------------------------------
    # Identifier exclusion
    # --------------------------------------------------------------------------

    if set(
        identifier_columns
    ) & set(
        preprocessing_columns
    ):
        raise RuntimeError(
            f"Identifier leakage detected in preprocessing "
            f"features for {dataset_id}."
        )

    # --------------------------------------------------------------------------
    # Provenance exclusion
    # --------------------------------------------------------------------------

    if provenance_column in preprocessing_columns:
        raise RuntimeError(
            f"Provenance leakage detected in preprocessing "
            f"features for {dataset_id}."
        )

    # --------------------------------------------------------------------------
    # Validate Section 5 categorical identity
    #
    # NB02 persisted schema is authoritative.
    # Section 5 is a cross-check only.
    # --------------------------------------------------------------------------

    section5_subset = (
        FEATURE_TYPE_CHARACTERIZATION_DF[
            FEATURE_TYPE_CHARACTERIZATION_DF[
                "dataset_id"
            ].eq(dataset_id)
            &
            FEATURE_TYPE_CHARACTERIZATION_DF[
                "feature"
            ].isin(categorical_columns)
            &
            FEATURE_TYPE_CHARACTERIZATION_DF[
                "is_preprocessing_feature"
            ].eq(True)
            &
            FEATURE_TYPE_CHARACTERIZATION_DF[
                "is_target"
            ].eq(False)
        ]
    )

    section5_categorical = (
        section5_subset[
            "feature"
        ].tolist()
    )

    if (
        len(section5_categorical)
        != len(categorical_columns)
        or
        set(section5_categorical)
        != set(categorical_columns)
    ):
        raise RuntimeError(
            f"Section 5 categorical feature identity "
            f"mismatch for {dataset_id}.\n"
            f"NB02: {categorical_columns}\n"
            f"Section 5: {section5_categorical}"
        )

    # --------------------------------------------------------------------------
    # Validate Section 7 categorical identity
    # --------------------------------------------------------------------------

    section7_subset = (
        CATEGORICAL_STATISTICS_DF[
            CATEGORICAL_STATISTICS_DF[
                "dataset_id"
            ].eq(dataset_id)
            &
            CATEGORICAL_STATISTICS_DF[
                "feature"
            ].isin(categorical_columns)
            &
            CATEGORICAL_STATISTICS_DF[
                "is_preprocessing_feature"
            ].eq(True)
            &
            CATEGORICAL_STATISTICS_DF[
                "is_target"
            ].eq(False)
        ]
    )

    section7_categorical = (
        section7_subset[
            "feature"
        ].tolist()
    )

    if (
        len(section7_categorical)
        != len(categorical_columns)
        or
        set(section7_categorical)
        != set(categorical_columns)
    ):
        raise RuntimeError(
            f"Section 7 categorical feature identity "
            f"mismatch for {dataset_id}.\n"
            f"NB02: {categorical_columns}\n"
            f"Section 7: {section7_categorical}"
        )

    # --------------------------------------------------------------------------
    # Load training data ONLY
    # --------------------------------------------------------------------------

    df = TRAIN_STATISTICAL_DATASETS[
        dataset_id
    ]

    # --------------------------------------------------------------------------
    # Correct native dataframe contract
    #
    # Native training dataframe:
    #     [provenance] + generative_columns
    # --------------------------------------------------------------------------

    expected_native_columns = (
        [provenance_column]
        + generative_columns
    )

    if df.columns.tolist() != (
        expected_native_columns
    ):
        raise RuntimeError(
            f"Native training dataframe schema mismatch "
            f"for {dataset_id}.\n"
            f"Expected: {expected_native_columns}\n"
            f"Observed: {df.columns.tolist()}"
        )

    # --------------------------------------------------------------------------
    # Training categorical data only
    # --------------------------------------------------------------------------

    analysis_source = df[
        categorical_columns
    ].copy()

    # --------------------------------------------------------------------------
    # Deterministic sampling
    # --------------------------------------------------------------------------

    sampling_applied = (
        len(analysis_source)
        > DEPENDENCY_MAX_ROWS
    )

    if sampling_applied:

        analysis_df = analysis_source.sample(
            n=DEPENDENCY_MAX_ROWS,
            random_state=DEPENDENCY_RANDOM_SEED,
        ).copy()

    else:

        analysis_df = analysis_source.copy()

    analysis_rows = int(
        len(analysis_df)
    )

    print(
        f"  Categorical features : "
        f"{len(categorical_columns)}"
    )

    print(
        f"  Training rows        : "
        f"{len(df)}"
    )

    print(
        f"  Analysis rows        : "
        f"{analysis_rows}"
    )

    print(
        f"  Sampling applied     : "
        f"{sampling_applied}"
    )

    # --------------------------------------------------------------------------
    # Prepare categorical variables
    # --------------------------------------------------------------------------

    prepared = {}

    preparation_metadata = {}

    for feature in categorical_columns:

        (
            prepared_series,
            original_cardinality,
            final_cardinality,
            pooling_applied,
            pooled_observations,
        ) = prepare_dependency_series(
            analysis_df[feature],
            max_categories=DEPENDENCY_MAX_CATEGORIES,
        )

        prepared[feature] = prepared_series

        preparation_metadata[
            feature
        ] = {
            "original_cardinality":
                original_cardinality,

            "final_cardinality":
                final_cardinality,

            "pooling_applied":
                pooling_applied,

            "pooled_observations":
                pooled_observations,
        }

    # --------------------------------------------------------------------------
    # Dependency matrix
    # --------------------------------------------------------------------------

    matrix = pd.DataFrame(
        np.eye(
            len(categorical_columns),
            dtype=float,
        ),
        index=categorical_columns,
        columns=categorical_columns,
    )

    # --------------------------------------------------------------------------
    # Unique unordered pairs
    # --------------------------------------------------------------------------

    for i, feature_a in enumerate(
        categorical_columns
    ):

        for j in range(
            i + 1,
            len(categorical_columns)
        ):

            feature_b = (
                categorical_columns[j]
            )

            value, valid_pair_count = (
                cramers_v(
                    prepared[feature_a],
                    prepared[feature_b],
                )
            )

            matrix.loc[
                feature_a,
                feature_b
            ] = value

            matrix.loc[
                feature_b,
                feature_a
            ] = value

            metadata_a = (
                preparation_metadata[
                    feature_a
                ]
            )

            metadata_b = (
                preparation_metadata[
                    feature_b
                ]
            )

            CATEGORICAL_DEPENDENCY_RECORDS.append({

                "dataset_id":
                    dataset_id,

                "feature_a":
                    feature_a,

                "feature_b":
                    feature_b,

                "cramers_v":
                    value,

                "absolute_cramers_v":
                    (
                        abs(value)
                        if np.isfinite(value)
                        else np.nan
                    ),

                "association_strength":
                    classify_dependency_strength(
                        value
                    ),

                "analysis_rows":
                    analysis_rows,

                "valid_pair_count":
                    valid_pair_count,

                "category_cap":
                    DEPENDENCY_MAX_CATEGORIES,

                "feature_a_original_cardinality":
                    metadata_a[
                        "original_cardinality"
                    ],

                "feature_a_final_cardinality":
                    metadata_a[
                        "final_cardinality"
                    ],

                "feature_b_original_cardinality":
                    metadata_b[
                        "original_cardinality"
                    ],

                "feature_b_final_cardinality":
                    metadata_b[
                        "final_cardinality"
                    ],

                "feature_a_pooling_applied":
                    metadata_a[
                        "pooling_applied"
                    ],

                "feature_b_pooling_applied":
                    metadata_b[
                        "pooling_applied"
                    ],

                "feature_a_pooled_observations":
                    metadata_a[
                        "pooled_observations"
                    ],

                "feature_b_pooled_observations":
                    metadata_b[
                        "pooled_observations"
                    ],

                "sampling_applied":
                    sampling_applied,

                "random_seed":
                    (
                        DEPENDENCY_RANDOM_SEED
                        if sampling_applied
                        else None
                    ),
            })

    CATEGORICAL_DEPENDENCY_MATRICES[
        dataset_id
    ] = matrix

    # --------------------------------------------------------------------------
    # Feature-level summary
    # --------------------------------------------------------------------------

    dataset_pair_df = pd.DataFrame([
        record
        for record in CATEGORICAL_DEPENDENCY_RECORDS
        if record[
            "dataset_id"
        ] == dataset_id
    ])

    for feature in categorical_columns:

        feature_pairs = (
            dataset_pair_df[
                dataset_pair_df[
                    "feature_a"
                ].eq(feature)
                |
                dataset_pair_df[
                    "feature_b"
                ].eq(feature)
            ]
        )

        values = (
            feature_pairs[
                "absolute_cramers_v"
            ]
            .dropna()
        )

        metadata = (
            preparation_metadata[
                feature
            ]
        )

        CATEGORICAL_DEPENDENCY_FEATURE_RECORDS.append({

            "dataset_id":
                dataset_id,

            "feature":
                feature,

            "original_cardinality":
                metadata[
                    "original_cardinality"
                ],

            "final_cardinality":
                metadata[
                    "final_cardinality"
                ],

            "pooling_applied":
                metadata[
                    "pooling_applied"
                ],

            "pooled_observations":
                metadata[
                    "pooled_observations"
                ],

            "analysis_rows":
                analysis_rows,

            "max_absolute_cramers_v":
                (
                    float(values.max())
                    if len(values)
                    else np.nan
                ),

            "mean_absolute_cramers_v":
                (
                    float(values.mean())
                    if len(values)
                    else np.nan
                ),

            "n_moderate_or_above":
                int(
                    (values >= 0.50).sum()
                ),

            "n_very_strong":
                int(
                    (values >= 0.70).sum()
                ),

            "is_target":
                False,

            "is_preprocessing_feature":
                True,

            "is_generative_feature":
                True,

            "schema_role":
                "categorical_preprocessing_feature",
        })

    # --------------------------------------------------------------------------
    # Dataset-level summary
    # --------------------------------------------------------------------------

    expected_pairs = (
        len(categorical_columns)
        *
        (len(categorical_columns) - 1)
        // 2
    )

    defined_pairs = int(
        dataset_pair_df[
            "cramers_v"
        ].notna().sum()
    )

    values = (
        dataset_pair_df[
            "absolute_cramers_v"
        ]
        .dropna()
    )

    CATEGORICAL_DEPENDENCY_SUMMARY_RECORDS.append({

        "dataset_id":
            dataset_id,

        "categorical_feature_count":
            len(categorical_columns),

        "expected_pair_count":
            expected_pairs,

        "defined_pair_count":
            defined_pairs,

        "training_rows":
            len(df),

        "analysis_rows":
            analysis_rows,

        "sampling_applied":
            sampling_applied,

        "random_seed":
            (
                DEPENDENCY_RANDOM_SEED
                if sampling_applied
                else None
            ),

        "category_cap":
            DEPENDENCY_MAX_CATEGORIES,

        "mean_absolute_cramers_v":
            (
                float(values.mean())
                if len(values)
                else np.nan
            ),

        "max_absolute_cramers_v":
            (
                float(values.max())
                if len(values)
                else np.nan
            ),

        "n_moderate_or_above":
            int(
                (values >= 0.50).sum()
            ),

        "n_strong":
            int(
                (values >= 0.70).sum()
            ),
    })


# ==============================================================================
# 13.6 — FINAL OUTPUT DATAFRAMES
# ==============================================================================

CATEGORICAL_DEPENDENCY_DF = pd.DataFrame(
    CATEGORICAL_DEPENDENCY_RECORDS
)

CATEGORICAL_DEPENDENCY_FEATURE_SUMMARY_DF = (
    pd.DataFrame(
        CATEGORICAL_DEPENDENCY_FEATURE_RECORDS
    )
)

CATEGORICAL_DEPENDENCY_SUMMARY_DF = (
    pd.DataFrame(
        CATEGORICAL_DEPENDENCY_SUMMARY_RECORDS
    )
)

CATEGORICAL_DEPENDENCY_COLUMNS = (
    CATEGORICAL_DEPENDENCY_DF.columns.tolist()
)


# ==============================================================================
# 13.7 — EXPECTED COVERAGE
# ==============================================================================

EXPECTED_CATEGORICAL_PAIRS = {}

for dataset_id in DATASET_IDS:

    categorical_count = len(
        NB02_PERSISTED_SCHEMA_CONTRACTS[
            dataset_id
        ]["modeling_schema"][
            "categorical_columns"
        ]
    )

    EXPECTED_CATEGORICAL_PAIRS[
        dataset_id
    ] = (
        categorical_count
        * (categorical_count - 1)
        // 2
    )

EXPECTED_TOTAL_CATEGORICAL_PAIRS = sum(
    EXPECTED_CATEGORICAL_PAIRS.values()
)


# ==============================================================================
# 13.8 — VALIDATION
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 13 — VALIDATION")
print("=" * 100)

VALIDATION_RESULTS = []


def record_check(
    name,
    condition
):

    condition = bool(condition)

    VALIDATION_RESULTS.append({
        "check": name,
        "status":
            "PASS"
            if condition
            else "FAIL",
    })

    print(
        f"{'✓' if condition else '✗'} {name}"
    )

    if not condition:

        raise RuntimeError(
            f"Section 13 validation failed: {name}"
        )


# --------------------------------------------------------------------------
# Basic output checks
# --------------------------------------------------------------------------

record_check(
    "categorical dependency dataframe exists",
    isinstance(
        CATEGORICAL_DEPENDENCY_DF,
        pd.DataFrame
    )
)

record_check(
    "categorical dependency matrices exist",
    isinstance(
        CATEGORICAL_DEPENDENCY_MATRICES,
        dict
    )
)

record_check(
    "feature summary exists",
    isinstance(
        CATEGORICAL_DEPENDENCY_FEATURE_SUMMARY_DF,
        pd.DataFrame
    )
)

record_check(
    "dataset summary exists",
    isinstance(
        CATEGORICAL_DEPENDENCY_SUMMARY_DF,
        pd.DataFrame
    )
)


# --------------------------------------------------------------------------
# Dataset coverage
# --------------------------------------------------------------------------

record_check(
    "all datasets covered",
    set(
        CATEGORICAL_DEPENDENCY_SUMMARY_DF[
            "dataset_id"
        ]
    )
    == set(DATASET_IDS)
)


# --------------------------------------------------------------------------
# Total pair coverage
# --------------------------------------------------------------------------

record_check(
    "expected total unordered pair coverage",
    len(
        CATEGORICAL_DEPENDENCY_DF
    )
    == EXPECTED_TOTAL_CATEGORICAL_PAIRS
)


# --------------------------------------------------------------------------
# Dataset-specific pair coverage
# --------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    dataset_pairs = (
        CATEGORICAL_DEPENDENCY_DF[
            CATEGORICAL_DEPENDENCY_DF[
                "dataset_id"
            ].eq(dataset_id)
        ]
    )

    expected_pairs = (
        EXPECTED_CATEGORICAL_PAIRS[
            dataset_id
        ]
    )

    record_check(
        f"{dataset_id}: expected pair count",
        len(dataset_pairs)
        == expected_pairs
    )

    record_check(
        f"{dataset_id}: all pairs defined",
        int(
            dataset_pairs[
                "cramers_v"
            ].notna().sum()
        )
        == expected_pairs
    )


# --------------------------------------------------------------------------
# Duplicate unordered pairs
# --------------------------------------------------------------------------

pair_keys = (
    CATEGORICAL_DEPENDENCY_DF.apply(
        lambda row: (
            row["dataset_id"],
            min(
                row["feature_a"],
                row["feature_b"]
            ),
            max(
                row["feature_a"],
                row["feature_b"]
            ),
        ),
        axis=1,
    )
)

record_check(
    "no duplicate unordered pairs",
    pair_keys.nunique()
    == len(
        CATEGORICAL_DEPENDENCY_DF
    )
)


# --------------------------------------------------------------------------
# No diagonal records
# --------------------------------------------------------------------------

record_check(
    "no self-pairs",
    (
        CATEGORICAL_DEPENDENCY_DF[
            "feature_a"
        ]
        !=
        CATEGORICAL_DEPENDENCY_DF[
            "feature_b"
        ]
    ).all()
)


# --------------------------------------------------------------------------
# Deterministic feature-pair ordering
#
# Pair ordering follows the frozen Notebook 02 categorical feature order,
# NOT alphabetical/string ordering.
# --------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    expected_order = list(
        NB02_PERSISTED_SCHEMA_CONTRACTS[
            dataset_id
        ]["modeling_schema"][
            "categorical_columns"
        ]
    )

    feature_order_map = {
        feature: position
        for position, feature
        in enumerate(expected_order)
    }

    dataset_pairs = (
        CATEGORICAL_DEPENDENCY_DF[
            CATEGORICAL_DEPENDENCY_DF[
                "dataset_id"
            ].eq(dataset_id)
        ]
    )

    ordering_valid = (
        dataset_pairs.apply(
            lambda row:
                feature_order_map[
                    row["feature_a"]
                ]
                <
                feature_order_map[
                    row["feature_b"]
                ],
            axis=1,
        )
    ).all()

    record_check(
        f"{dataset_id}: deterministic feature-pair ordering",
        ordering_valid
    )


# --------------------------------------------------------------------------
# Cramér's V range
# --------------------------------------------------------------------------

finite_v = (
    CATEGORICAL_DEPENDENCY_DF[
        "cramers_v"
    ]
    .dropna()
)

record_check(
    "Cramér's V values finite",
    np.isfinite(
        finite_v.to_numpy()
    ).all()
)

record_check(
    "Cramér's V values in [0, 1]",
    (
        (finite_v >= 0.0)
        &
        (finite_v <= 1.0)
    ).all()
)


# --------------------------------------------------------------------------
# Category-cap consistency
# --------------------------------------------------------------------------

record_check(
    "category cap consistently recorded",
    (
        CATEGORICAL_DEPENDENCY_DF[
            "category_cap"
        ]
        ==
        DEPENDENCY_MAX_CATEGORIES
    ).all()
)


# --------------------------------------------------------------------------
# Sampling consistency
# --------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    expected_sampling = (
        len(
            TRAIN_STATISTICAL_DATASETS[
                dataset_id
            ]
        )
        > DEPENDENCY_MAX_ROWS
    )

    observed_sampling = (
        CATEGORICAL_DEPENDENCY_DF[
            CATEGORICAL_DEPENDENCY_DF[
                "dataset_id"
            ].eq(dataset_id)
        ]["sampling_applied"]
        .unique()
        .tolist()
    )

    record_check(
        f"{dataset_id}: sampling policy",
        observed_sampling
        == [expected_sampling]
    )


# --------------------------------------------------------------------------
# Matrix integrity
# --------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    matrix = (
        CATEGORICAL_DEPENDENCY_MATRICES[
            dataset_id
        ]
    )

    expected_columns = list(
        NB02_PERSISTED_SCHEMA_CONTRACTS[
            dataset_id
        ]["modeling_schema"][
            "categorical_columns"
        ]
    )

    record_check(
        f"{dataset_id}: matrix feature order",
        matrix.index.tolist()
        == expected_columns
        and
        matrix.columns.tolist()
        == expected_columns
    )

    record_check(
        f"{dataset_id}: matrix symmetry",
        np.allclose(
            matrix.to_numpy(),
            matrix.to_numpy().T,
            equal_nan=True,
        )
    )

    record_check(
        f"{dataset_id}: matrix diagonal equals one",
        np.allclose(
            np.diag(
                matrix.to_numpy()
            ),
            1.0,
        )
    )


# --------------------------------------------------------------------------
# Target exclusion from dependency records
# --------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    target_column = (
        NB02_PERSISTED_SCHEMA_CONTRACTS[
            dataset_id
        ]["modeling_schema"][
            "target_column"
        ]
    )

    dataset_pairs = (
        CATEGORICAL_DEPENDENCY_DF[
            CATEGORICAL_DEPENDENCY_DF[
                "dataset_id"
            ].eq(dataset_id)
        ]
    )

    features_used = set(
        dataset_pairs[
            "feature_a"
        ]
    ) | set(
        dataset_pairs[
            "feature_b"
        ]
    )

    record_check(
        f"{dataset_id}: target excluded",
        target_column
        not in features_used
    )


# --------------------------------------------------------------------------
# Provenance exclusion
# --------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    provenance_column = (
        NB02_PERSISTED_SCHEMA_CONTRACTS[
            dataset_id
        ]["modeling_schema"].get(
            "provenance_column",
            "__original_row_id__"
        )
    )

    dataset_pairs = (
        CATEGORICAL_DEPENDENCY_DF[
            CATEGORICAL_DEPENDENCY_DF[
                "dataset_id"
            ].eq(dataset_id)
        ]
    )

    features_used = set(
        dataset_pairs[
            "feature_a"
        ]
    ) | set(
        dataset_pairs[
            "feature_b"
        ]
    )

    record_check(
        f"{dataset_id}: provenance excluded",
        provenance_column
        not in features_used
    )


# --------------------------------------------------------------------------
# Feature summary coverage
# --------------------------------------------------------------------------

expected_feature_count = sum(
    len(
        NB02_PERSISTED_SCHEMA_CONTRACTS[
            dataset_id
        ]["modeling_schema"][
            "categorical_columns"
        ]
    )
    for dataset_id in DATASET_IDS
)

record_check(
    "feature summary coverage",
    len(
        CATEGORICAL_DEPENDENCY_FEATURE_SUMMARY_DF
    )
    == expected_feature_count
)


# --------------------------------------------------------------------------
# Dataset summary coverage
# --------------------------------------------------------------------------

record_check(
    "dataset summary coverage",
    len(
        CATEGORICAL_DEPENDENCY_SUMMARY_DF
    )
    == len(DATASET_IDS)
)


# ==============================================================================
# 13.9 — SUMMARY
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 13 — CATEGORICAL DEPENDENCY SUMMARY")
print("=" * 100)

print(
    f"Datasets                  : "
    f"{len(DATASET_IDS)}"
)

print(
    f"Categorical features      : "
    f"{expected_feature_count}"
)

print(
    f"Expected unordered pairs  : "
    f"{EXPECTED_TOTAL_CATEGORICAL_PAIRS}"
)

print(
    f"Observed dependency pairs : "
    f"{len(CATEGORICAL_DEPENDENCY_DF)}"
)

print(
    f"Maximum analysis rows     : "
    f"{DEPENDENCY_MAX_ROWS}"
)

print(
    f"Category cap              : "
    f"{DEPENDENCY_MAX_CATEGORIES}"
)

print(
    f"Random seed               : "
    f"{DEPENDENCY_RANDOM_SEED}"
)

print("\nDataset-level summary:")

display(
    CATEGORICAL_DEPENDENCY_SUMMARY_DF
)


# ==============================================================================
# 13.10 — FINAL STATUS
# ==============================================================================

if not all(
    row["status"] == "PASS"
    for row in VALIDATION_RESULTS
):
    raise RuntimeError(
        "SECTION 13 STATUS: FAIL"
    )

print("\n" + "=" * 100)
print("SECTION 13 STATUS: PASS")
print("=" * 100)

SECTION 13 — CATEGORICAL DEPENDENCY ANALYSIS

Processing dataset: adult_income
  Categorical features : 8
  Training rows        : 34189
  Analysis rows        : 25000
  Sampling applied     : True

Processing dataset: bank_marketing
  Categorical features : 9
  Training rows        : 31647
  Analysis rows        : 25000
  Sampling applied     : True

Processing dataset: diabetes_130us
  Categorical features : 36
  Training rows        : 71236
  Analysis rows        : 25000
  Sampling applied     : True

SECTION 13 — VALIDATION
✓ categorical dependency dataframe exists
✓ categorical dependency matrices exist
✓ feature summary exists
✓ dataset summary exists
✓ all datasets covered
✓ expected total unordered pair coverage
✓ adult_income: expected pair count
✓ adult_income: all pairs defined
✓ bank_marketing: expected pair count
✓ bank_marketing: all pairs defined
✓ diabetes_130us: expected pair count
✓ diabetes_130us: all pairs defined
✓ no duplicate unordered pairs
✓ no self-pairs
✓ adu

,dataset_id,categorical_feature_count,expected_pair_count,defined_pair_count,training_rows,analysis_rows,sampling_applied,random_seed,category_cap,mean_absolute_cramers_v,max_absolute_cramers_v,n_moderate_or_above,n_strong
0,adult_income,8,28,28,34189,25000,True,2025,50,0.177636,0.648582,1,0
1,bank_marketing,9,36,36,31647,25000,True,2025,50,0.125473,0.513969,2,0
2,diabetes_130us,36,630,630,71236,25000,True,2025,50,0.024640,0.642436,3,0



SECTION 13 STATUS: PASS


In [14]:
# ==============================================================================
# SECTION 14 — FEATURE STATISTICAL PROFILES
# ==============================================================================

print("=" * 100)
print("SECTION 14 — FEATURE STATISTICAL PROFILES")
print("=" * 100)


# ==============================================================================
# 14.1 — REQUIRED UPSTREAM OBJECTS
# ==============================================================================

REQUIRED_UPSTREAM_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_STATISTICAL_DATASETS",
    "NB02_PERSISTED_SCHEMA_CONTRACTS",
    "FEATURE_TYPE_CHARACTERIZATION_DF",
    "NUMERICAL_STATISTICS_DF",
    "CATEGORICAL_STATISTICS_DF",
    "CARDINALITY_ENTROPY_DF",
]

for object_name in REQUIRED_UPSTREAM_OBJECTS:

    if object_name not in globals():

        raise RuntimeError(
            f"Section 14 requires upstream object "
            f"'{object_name}', but it was not found.\n"
            f"Run the required upstream section before Section 14."
        )

print("✓ Required upstream objects available.")


# ==============================================================================
# 14.2 — OPTIONAL SECTION 10 INTEGRATION
# ==============================================================================

HAS_SECTION10 = (
    "DISTRIBUTION_CHARACTERIZATION_DF" in globals()
)

if HAS_SECTION10:

    print(
        "✓ Section 10 distribution characterization detected.\n"
        "  Section 14 will integrate its validated descriptors."
    )

else:

    print(
        "ℹ Section 10 distribution characterization not found in current runtime.\n"
        "  Section 14 will use validated statistics from Sections 5–9 "
        "without recomputation."
    )


# ==============================================================================
# 14.3 — HELPER FUNCTIONS
# ==============================================================================

def _safe_float(value):

    if pd.isna(value):
        return np.nan

    return float(value)


def _safe_int(value):

    if pd.isna(value):
        return np.nan

    return int(value)


def _get_single_row(
    dataframe,
    dataset_id,
    feature,
    dataframe_name
):

    rows = dataframe[
        (dataframe["dataset_id"] == dataset_id) &
        (dataframe["feature"] == feature)
    ]

    if len(rows) != 1:

        raise RuntimeError(
            f"{dataframe_name}: expected exactly one record for "
            f"dataset='{dataset_id}', feature='{feature}', "
            f"but found {len(rows)}."
        )

    return rows.iloc[0]


def _get_optional_value(
    row,
    column_name,
    default=np.nan
):

    if column_name not in row.index:

        return default

    value = row[column_name]

    if pd.isna(value):

        return default

    return value


# ==============================================================================
# 14.4 — CANONICAL SECTION 5 PREPROCESSING RECORDS
# ==============================================================================

SECTION5_PREPROCESSING_GLOBAL = (
    FEATURE_TYPE_CHARACTERIZATION_DF[
        FEATURE_TYPE_CHARACTERIZATION_DF[
            "is_preprocessing_feature"
        ] == True
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    f"✓ Section 5 preprocessing records available: "
    f"{len(SECTION5_PREPROCESSING_GLOBAL)}"
)


# ==============================================================================
# 14.5 — EXPECTED FEATURE COUNTS
# ==============================================================================

EXPECTED_PREPROCESSING_COUNTS = {
    "adult_income": 14,
    "bank_marketing": 16,
    "diabetes_130us": 47,
}

EXPECTED_NUMERIC_COUNTS = {
    "adult_income": 6,
    "bank_marketing": 7,
    "diabetes_130us": 11,
}

EXPECTED_CATEGORICAL_COUNTS = {
    "adult_income": 8,
    "bank_marketing": 9,
    "diabetes_130us": 36,
}


# ==============================================================================
# 14.6 — INSPECT SECTION 9 SCHEMA
#
# This is diagnostic only.
# Section 14 does NOT modify Section 9.
# ==============================================================================

SECTION9_COLUMNS = list(
    CARDINALITY_ENTROPY_DF.columns
)

print()
print(
    "✓ Section 9 columns detected:"
)

print(
    SECTION9_COLUMNS
)


# ==============================================================================
# 14.7 — REQUIRED SECTION 9 CORE COLUMNS
#
# entropy_bits is intentionally NOT mandatory because the executed
# Section 9 artifact may use a different entropy representation.
# ==============================================================================

REQUIRED_SECTION9_CORE_COLUMNS = [
    "dataset_id",
    "feature",
    "cardinality",
    "cardinality_ratio",
    "normalized_entropy",
]

missing_section9_core = [
    column
    for column in REQUIRED_SECTION9_CORE_COLUMNS
    if column not in CARDINALITY_ENTROPY_DF.columns
]

if missing_section9_core:

    raise RuntimeError(
        "CARDINALITY_ENTROPY_DF is missing required core columns:\n"
        f"{missing_section9_core}\n\n"
        "Section 14 cannot safely integrate Section 9."
    )

if "entropy_bits" in CARDINALITY_ENTROPY_DF.columns:

    print(
        "✓ Section 9 provides 'entropy_bits'."
    )

else:

    print(
        "ℹ Section 9 does not provide 'entropy_bits'.\n"
        "  Section 14 will retain entropy_bits as NaN "
        "without recomputation."
    )


# ==============================================================================
# 14.8 — BUILD FEATURE STATISTICAL PROFILES
# ==============================================================================

FEATURE_STATISTICAL_PROFILES = []


for dataset_id in DATASET_IDS:

    print()
    print(f"Processing {dataset_id}")


    # ==========================================================================
    # 14.8.1 — LOAD TRAINING DATA
    # ==========================================================================

    if dataset_id not in TRAIN_STATISTICAL_DATASETS:

        raise RuntimeError(
            f"{dataset_id}: training dataset not available."
        )

    train_df = TRAIN_STATISTICAL_DATASETS[
        dataset_id
    ]

    training_rows = len(train_df)

    print(
        f"Training rows       : {training_rows}"
    )


    # ==========================================================================
    # 14.8.2 — LOAD FROZEN NOTEBOOK 02 SCHEMA
    # ==========================================================================

    if dataset_id not in NB02_PERSISTED_SCHEMA_CONTRACTS:

        raise RuntimeError(
            f"{dataset_id}: persisted Notebook 02 schema "
            f"contract not found."
        )

    schema = NB02_PERSISTED_SCHEMA_CONTRACTS[
        dataset_id
    ]

    modeling_schema = schema[
        "modeling_schema"
    ]

    preprocessing_columns = list(
        modeling_schema[
            "preprocessing_columns"
        ]
    )

    all_columns = list(
        modeling_schema[
            "all_columns"
        ]
    )

    numeric_columns = list(
        modeling_schema[
            "numeric_columns"
        ]
    )

    categorical_columns = list(
        modeling_schema[
            "categorical_columns"
        ]
    )

    generative_columns = list(
        modeling_schema[
            "generative_columns"
        ]
    )

    target_column = modeling_schema[
        "target_column"
    ]

    identifier_columns = list(
        modeling_schema.get(
            "identifier_columns_excluded",
            []
        )
    )

    provenance_column = modeling_schema.get(
        "provenance_column",
        "__original_row_id__"
    )


    # ==========================================================================
    # 14.8.3 — FROZEN SCHEMA VALIDATION
    # ==========================================================================

    if all_columns != preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: persisted all_columns do not exactly "
            f"match preprocessing_columns."
        )

    expected_generative_columns = (
        preprocessing_columns
        + [target_column]
    )

    if generative_columns != expected_generative_columns:

        raise RuntimeError(
            f"{dataset_id}: generative column order is inconsistent "
            f"with preprocessing_columns + target."
        )

    if set(numeric_columns).intersection(
        categorical_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: numeric and categorical columns overlap."
        )

    if (
        set(numeric_columns).union(
            categorical_columns
        )
        != set(preprocessing_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: numeric + categorical columns do not "
            f"exactly cover preprocessing columns."
        )

    if target_column in preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' incorrectly "
            f"appears in preprocessing columns."
        )

    if target_column in numeric_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' incorrectly "
            f"appears in numeric preprocessing columns."
        )

    if target_column in categorical_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' incorrectly "
            f"appears in categorical preprocessing columns."
        )

    if provenance_column in preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column incorrectly appears "
            f"in preprocessing columns."
        )

    if set(identifier_columns).intersection(
        preprocessing_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: identifier columns incorrectly appear "
            f"in preprocessing columns."
        )


    # ==========================================================================
    # 14.8.4 — VALIDATE NATIVE TRAINING DATAFRAME SCHEMA
    # ==========================================================================

    expected_native_columns = (
        [provenance_column]
        + generative_columns
    )

    actual_native_columns = list(
        train_df.columns
    )

    if actual_native_columns != expected_native_columns:

        raise RuntimeError(
            f"{dataset_id}: native training dataframe schema mismatch.\n"
            f"Expected:\n{expected_native_columns}\n"
            f"Observed:\n{actual_native_columns}"
        )


    # ==========================================================================
    # 14.8.5 — VALIDATE SECTION 5
    # ==========================================================================

    section5_all = (
        FEATURE_TYPE_CHARACTERIZATION_DF[
            FEATURE_TYPE_CHARACTERIZATION_DF[
                "dataset_id"
            ] == dataset_id
        ]
        .copy()
    )

    expected_generative_count = len(
        generative_columns
    )

    if len(section5_all) != expected_generative_count:

        raise RuntimeError(
            f"{dataset_id}: Section 5 generative feature count "
            f"mismatch. Expected {expected_generative_count}, "
            f"found {len(section5_all)}."
        )

    section5_preprocessing = (
        section5_all[
            section5_all[
                "is_preprocessing_feature"
            ] == True
        ]
        .copy()
    )

    if len(section5_preprocessing) != (
        len(preprocessing_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: Section 5 preprocessing feature count "
            f"mismatch. Expected {len(preprocessing_columns)}, "
            f"found {len(section5_preprocessing)}."
        )

    section5_target = (
        section5_all[
            section5_all[
                "is_target"
            ] == True
        ]
    )

    if len(section5_target) != 1:

        raise RuntimeError(
            f"{dataset_id}: Section 5 must contain exactly one "
            f"target record, found {len(section5_target)}."
        )


    # ==========================================================================
    # 14.8.6 — SECTION 5 FEATURE ORDER
    # ==========================================================================

    section5_order = (
        section5_preprocessing
        .sort_values(
            "feature_order"
        )["feature"]
        .tolist()
    )

    if section5_order != preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: Section 5 preprocessing feature order "
            f"does not match the frozen Notebook 02 schema."
        )


    # ==========================================================================
    # 14.8.7 — EXPECTED FEATURE COUNTS
    # ==========================================================================

    if len(preprocessing_columns) != (
        EXPECTED_PREPROCESSING_COUNTS[
            dataset_id
        ]
    ):

        raise RuntimeError(
            f"{dataset_id}: unexpected preprocessing feature count. "
            f"Expected "
            f"{EXPECTED_PREPROCESSING_COUNTS[dataset_id]}, "
            f"found {len(preprocessing_columns)}."
        )

    if len(numeric_columns) != (
        EXPECTED_NUMERIC_COUNTS[
            dataset_id
        ]
    ):

        raise RuntimeError(
            f"{dataset_id}: unexpected numeric feature count. "
            f"Expected "
            f"{EXPECTED_NUMERIC_COUNTS[dataset_id]}, "
            f"found {len(numeric_columns)}."
        )

    if len(categorical_columns) != (
        EXPECTED_CATEGORICAL_COUNTS[
            dataset_id
        ]
    ):

        raise RuntimeError(
            f"{dataset_id}: unexpected categorical feature count. "
            f"Expected "
            f"{EXPECTED_CATEGORICAL_COUNTS[dataset_id]}, "
            f"found {len(categorical_columns)}."
        )


    print(
        f"Preprocessing vars : {len(preprocessing_columns)}"
    )

    print(
        f"Numeric variables  : {len(numeric_columns)}"
    )

    print(
        f"Categorical vars   : {len(categorical_columns)}"
    )


    # ==========================================================================
    # 14.8.8 — TARGET EXCLUSION
    # ==========================================================================

    if target_column in (
        section5_preprocessing[
            "feature"
        ].tolist()
    ):

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' found in "
            f"Section 5 preprocessing subset."
        )


    # ==========================================================================
    # 14.8.9 — FEATURE LOOP
    # ==========================================================================

    for feature_order, feature in enumerate(
        preprocessing_columns,
        start=1
    ):

        # ======================================================================
        # Section 5 feature record
        # ======================================================================

        type_rows = (
            FEATURE_TYPE_CHARACTERIZATION_DF[
                (
                    FEATURE_TYPE_CHARACTERIZATION_DF[
                        "dataset_id"
                    ] == dataset_id
                )
                &
                (
                    FEATURE_TYPE_CHARACTERIZATION_DF[
                        "feature"
                    ] == feature
                )
                &
                (
                    FEATURE_TYPE_CHARACTERIZATION_DF[
                        "is_preprocessing_feature"
                    ] == True
                )
            ]
        )

        if len(type_rows) != 1:

            raise RuntimeError(
                f"{dataset_id}: Section 5 feature '{feature}' "
                f"must have exactly one preprocessing record, "
                f"found {len(type_rows)}."
            )

        type_row = type_rows.iloc[0]


        # ======================================================================
        # Determine feature type from frozen Notebook 02 schema
        # ======================================================================

        if feature in numeric_columns:

            feature_type = "numeric"

        elif feature in categorical_columns:

            feature_type = "categorical"

        else:

            raise RuntimeError(
                f"{dataset_id}: feature '{feature}' is not present "
                f"in numeric or categorical schema."
            )


        # ======================================================================
        # RETRIEVE VALIDATED UPSTREAM STATISTICS
        #
        # NUMERIC:
        #     Section 5 + Section 6
        #
        # CATEGORICAL:
        #     Section 5 + Section 7 + Section 9
        #
        # Section 9 is NEVER queried for numeric features.
        # ======================================================================

        if feature_type == "numeric":

            stats_row = _get_single_row(
                NUMERICAL_STATISTICS_DF,
                dataset_id,
                feature,
                "NUMERICAL_STATISTICS_DF"
            )

            categorical_row = None
            entropy_row = None


        elif feature_type == "categorical":

            stats_row = _get_single_row(
                CATEGORICAL_STATISTICS_DF,
                dataset_id,
                feature,
                "CATEGORICAL_STATISTICS_DF"
            )

            categorical_row = stats_row

            entropy_row = _get_single_row(
                CARDINALITY_ENTROPY_DF,
                dataset_id,
                feature,
                "CARDINALITY_ENTROPY_DF"
            )


        else:

            raise RuntimeError(
                f"{dataset_id}: unsupported feature type "
                f"'{feature}'."
            )


        # ======================================================================
        # 14.8.10 — CARDINALITY / ENTROPY VALUES
        #
        # entropy_bits is optional.
        # ======================================================================

        if entropy_row is not None:

            profile_cardinality = _safe_int(
                _get_optional_value(
                    entropy_row,
                    "cardinality"
                )
            )

            profile_cardinality_ratio = _safe_float(
                _get_optional_value(
                    entropy_row,
                    "cardinality_ratio"
                )
            )

            profile_normalized_entropy = _safe_float(
                _get_optional_value(
                    entropy_row,
                    "normalized_entropy"
                )
            )

            if "entropy_bits" in entropy_row.index:

                profile_entropy_bits = _safe_float(
                    entropy_row[
                        "entropy_bits"
                    ]
                )

            else:

                profile_entropy_bits = np.nan

        else:

            profile_cardinality = _safe_int(
                _get_optional_value(
                    stats_row,
                    "n_unique"
                )
            )

            profile_cardinality_ratio = _safe_float(
                _get_optional_value(
                    stats_row,
                    "unique_ratio"
                )
            )

            profile_normalized_entropy = np.nan
            profile_entropy_bits = np.nan


        # ======================================================================
        # 14.8.11 — BASE PROFILE
        # ======================================================================

        profile = {

            "dataset_id":
                dataset_id,

            "feature_order":
                int(feature_order),

            "feature":
                feature,

            "feature_type":
                feature_type,

            "schema_role":
                type_row[
                    "schema_role"
                ],

            "type_source":
                type_row[
                    "type_source"
                ],

            "native_dtype":
                type_row[
                    "native_dtype"
                ],

            "is_target":
                bool(
                    type_row[
                        "is_target"
                    ]
                ),

            "is_preprocessing_feature":
                bool(
                    type_row[
                        "is_preprocessing_feature"
                    ]
                ),

            "is_generative_feature":
                bool(
                    type_row[
                        "is_generative_feature"
                    ]
                ),

            "training_rows":
                training_rows,

            # ------------------------------------------------------------------
            # Section 5
            # ------------------------------------------------------------------

            "non_missing_count":
                _safe_int(
                    type_row[
                        "non_missing_count"
                    ]
                ),

            "missing_count":
                _safe_int(
                    type_row[
                        "missing_count"
                    ]
                ),

            "missing_rate":
                _safe_float(
                    type_row[
                        "missing_rate"
                    ]
                ),

            # ------------------------------------------------------------------
            # Cardinality / entropy
            # ------------------------------------------------------------------

            "n_unique":
                profile_cardinality,

            "unique_ratio":
                profile_cardinality_ratio,

            "normalized_entropy":
                profile_normalized_entropy,

            "entropy_bits":
                profile_entropy_bits,

            # ------------------------------------------------------------------
            # Section 5 flags
            # ------------------------------------------------------------------

            "is_constant":
                bool(
                    type_row[
                        "is_constant"
                    ]
                ),

            "is_near_constant":
                bool(
                    type_row[
                        "is_near_constant"
                    ]
                ),

            # ------------------------------------------------------------------
            # Numeric fields
            # ------------------------------------------------------------------

            "mean": np.nan,
            "median": np.nan,
            "std": np.nan,
            "min": np.nan,

            "q01": np.nan,
            "q05": np.nan,
            "q25": np.nan,
            "q50": np.nan,
            "q75": np.nan,
            "q95": np.nan,
            "q99": np.nan,

            "max": np.nan,

            "skewness": np.nan,
            "kurtosis": np.nan,

            "coefficient_of_variation":
                np.nan,

            "zero_count": np.nan,
            "zero_rate": np.nan,

            "negative_count": np.nan,
            "negative_rate": np.nan,

            "positive_count": np.nan,
            "positive_rate": np.nan,

            "distribution_shape":
                np.nan,

            # ------------------------------------------------------------------
            # Categorical fields
            # ------------------------------------------------------------------

            "top_category":
                np.nan,

            "top_category_frequency":
                np.nan,

            "number_of_categories":
                np.nan,

            "rare_category_count":
                np.nan,

            "singleton_category_count":
                np.nan,

            "distribution_concentration":
                np.nan,
        }


        # ======================================================================
        # 14.8.12 — NUMERIC PROFILE
        # ======================================================================

        if feature_type == "numeric":

            numeric_field_map = {

                "mean":
                    "mean",

                "median":
                    "median",

                "std":
                    "std",

                "min":
                    "min",

                "q01":
                    "q01",

                "q05":
                    "q05",

                "q25":
                    "q25",

                "q50":
                    "q50",

                "q75":
                    "q75",

                "q95":
                    "q95",

                "q99":
                    "q99",

                "max":
                    "max",

                "skewness":
                    "skewness",

                "kurtosis":
                    "kurtosis",

                "coefficient_of_variation":
                    "coefficient_of_variation",

                "zero_count":
                    "zero_count",

                "zero_rate":
                    "zero_rate",

                "negative_count":
                    "negative_count",

                "negative_rate":
                    "negative_rate",

                "positive_count":
                    "positive_count",

                "positive_rate":
                    "positive_rate",
            }

            for (
                output_name,
                source_name
            ) in numeric_field_map.items():

                if source_name in stats_row.index:

                    profile[
                        output_name
                    ] = _safe_float(
                        stats_row[
                            source_name
                        ]
                    )

            if "shape_class" in stats_row.index:

                profile[
                    "distribution_shape"
                ] = stats_row[
                    "shape_class"
                ]

            elif "skewness" in stats_row.index:

                skew_value = _safe_float(
                    stats_row[
                        "skewness"
                    ]
                )

                if pd.isna(skew_value):

                    profile[
                        "distribution_shape"
                    ] = np.nan

                elif abs(skew_value) < 0.5:

                    profile[
                        "distribution_shape"
                    ] = "approximately_symmetric"

                elif skew_value > 0:

                    profile[
                        "distribution_shape"
                    ] = "right_skewed"

                else:

                    profile[
                        "distribution_shape"
                    ] = "left_skewed"


        # ======================================================================
        # 14.8.13 — CATEGORICAL PROFILE
        # ======================================================================

        else:

            categorical_field_map = {

                "top_category":
                    "top_category",

                "top_category_frequency":
                    "top_frequency",

                "number_of_categories":
                    "n_unique",

                "rare_category_count":
                    "rare_category_count",

                "singleton_category_count":
                    "singleton_category_count",
            }

            for (
                output_name,
                source_name
            ) in categorical_field_map.items():

                if source_name in categorical_row.index:

                    value = categorical_row[
                        source_name
                    ]

                    if output_name in [

                        "number_of_categories",
                        "rare_category_count",
                        "singleton_category_count",

                    ]:

                        profile[
                            output_name
                        ] = _safe_int(
                            value
                        )

                    elif output_name == "top_category":

                        profile[
                            output_name
                        ] = value

                    else:

                        profile[
                            output_name
                        ] = _safe_float(
                            value
                        )


        # ======================================================================
        # 14.8.14 — OPTIONAL SECTION 10 INTEGRATION
        # ======================================================================

        if HAS_SECTION10:

            distribution_rows = (
                DISTRIBUTION_CHARACTERIZATION_DF[
                    (
                        DISTRIBUTION_CHARACTERIZATION_DF[
                            "dataset_id"
                        ] == dataset_id
                    )
                    &
                    (
                        DISTRIBUTION_CHARACTERIZATION_DF[
                            "feature"
                        ] == feature
                    )
                ]
            )

            if len(distribution_rows) > 1:

                raise RuntimeError(
                    f"{dataset_id}/{feature}: Section 10 contains "
                    f"{len(distribution_rows)} duplicate records."
                )

            if len(distribution_rows) == 1:

                distribution_row = (
                    distribution_rows.iloc[0]
                )

                if feature_type == "numeric":

                    if (
                        "shape_class"
                        in distribution_row.index
                    ):

                        profile[
                            "distribution_shape"
                        ] = distribution_row[
                            "shape_class"
                        ]

                else:

                    if (
                        "concentration_class"
                        in distribution_row.index
                    ):

                        profile[
                            "distribution_concentration"
                        ] = distribution_row[
                            "concentration_class"
                        ]


        # ======================================================================
        # 14.8.15 — APPEND
        # ======================================================================

        FEATURE_STATISTICAL_PROFILES.append(
            profile
        )


    print(
        f"✓ Completed {dataset_id}: "
        f"{len(preprocessing_columns)} preprocessing features"
    )


# ==============================================================================
# 14.9 — CREATE FINAL DATAFRAME
# ==============================================================================

FEATURE_STATISTICAL_PROFILE_DF = pd.DataFrame(
    FEATURE_STATISTICAL_PROFILES
)

print()
print(
    f"Total feature profiles : "
    f"{len(FEATURE_STATISTICAL_PROFILE_DF)}"
)


# ==============================================================================
# 14.10 — REQUIRED OUTPUT SCHEMA
# ==============================================================================

REQUIRED_PROFILE_COLUMNS = [

    "dataset_id",
    "feature_order",
    "feature",
    "feature_type",
    "schema_role",
    "type_source",
    "native_dtype",

    "is_target",
    "is_preprocessing_feature",
    "is_generative_feature",

    "training_rows",

    "non_missing_count",
    "missing_count",
    "missing_rate",

    "n_unique",
    "unique_ratio",

    "is_constant",
    "is_near_constant",

    "mean",
    "median",
    "std",
    "min",

    "q01",
    "q05",
    "q25",
    "q50",
    "q75",
    "q95",
    "q99",

    "max",

    "skewness",
    "kurtosis",
    "coefficient_of_variation",

    "zero_count",
    "zero_rate",

    "negative_count",
    "negative_rate",

    "positive_count",
    "positive_rate",

    "distribution_shape",

    "top_category",
    "top_category_frequency",

    "number_of_categories",

    "normalized_entropy",
    "entropy_bits",

    "rare_category_count",
    "singleton_category_count",

    "distribution_concentration",
]


missing_output_columns = [
    column
    for column in REQUIRED_PROFILE_COLUMNS
    if column not in FEATURE_STATISTICAL_PROFILE_DF.columns
]

if missing_output_columns:

    raise RuntimeError(
        "Section 14 output is missing required columns:\n"
        f"{missing_output_columns}"
    )


# ==============================================================================
# 14.11 — TOTAL COVERAGE VALIDATION
# ==============================================================================

EXPECTED_TOTAL_PROFILES = sum(
    EXPECTED_PREPROCESSING_COUNTS.values()
)

if len(FEATURE_STATISTICAL_PROFILE_DF) != (
    EXPECTED_TOTAL_PROFILES
):

    raise RuntimeError(
        f"Section 14 total profile count mismatch. "
        f"Expected {EXPECTED_TOTAL_PROFILES}, "
        f"found {len(FEATURE_STATISTICAL_PROFILE_DF)}."
    )


# ==============================================================================
# 14.12 — DATASET COVERAGE VALIDATION
# ==============================================================================

observed_dataset_counts = (
    FEATURE_STATISTICAL_PROFILE_DF
    .groupby("dataset_id")
    .size()
    .to_dict()
)

for dataset_id in DATASET_IDS:

    expected_count = (
        EXPECTED_PREPROCESSING_COUNTS[
            dataset_id
        ]
    )

    observed_count = (
        observed_dataset_counts.get(
            dataset_id,
            0
        )
    )

    if observed_count != expected_count:

        raise RuntimeError(
            f"{dataset_id}: profile coverage mismatch. "
            f"Expected {expected_count}, "
            f"found {observed_count}."
        )


# ==============================================================================
# 14.13 — DUPLICATE VALIDATION
# ==============================================================================

duplicate_count = (
    FEATURE_STATISTICAL_PROFILE_DF
    .duplicated(
        subset=[
            "dataset_id",
            "feature"
        ]
    )
    .sum()
)

if duplicate_count != 0:

    raise RuntimeError(
        f"Section 14 contains {duplicate_count} duplicate "
        f"dataset-feature profiles."
    )


# ==============================================================================
# 14.14 — FEATURE ORDER VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    expected_order = list(
        NB02_PERSISTED_SCHEMA_CONTRACTS[
            dataset_id
        ]["modeling_schema"][
            "preprocessing_columns"
        ]
    )

    observed_order = (
        FEATURE_STATISTICAL_PROFILE_DF[
            FEATURE_STATISTICAL_PROFILE_DF[
                "dataset_id"
            ] == dataset_id
        ]
        .sort_values(
            "feature_order"
        )["feature"]
        .tolist()
    )

    if observed_order != expected_order:

        raise RuntimeError(
            f"{dataset_id}: Section 14 feature order does not "
            f"match the frozen Notebook 02 preprocessing schema."
        )


# ==============================================================================
# 14.15 — NUMERIC / CATEGORICAL COVERAGE
# ==============================================================================

observed_numeric_counts = (
    FEATURE_STATISTICAL_PROFILE_DF[
        FEATURE_STATISTICAL_PROFILE_DF[
            "feature_type"
        ] == "numeric"
    ]
    .groupby("dataset_id")
    .size()
    .to_dict()
)

observed_categorical_counts = (
    FEATURE_STATISTICAL_PROFILE_DF[
        FEATURE_STATISTICAL_PROFILE_DF[
            "feature_type"
        ] == "categorical"
    ]
    .groupby("dataset_id")
    .size()
    .to_dict()
)

for dataset_id in DATASET_IDS:

    if (
        observed_numeric_counts.get(
            dataset_id,
            0
        )
        != EXPECTED_NUMERIC_COUNTS[
            dataset_id
        ]
    ):

        raise RuntimeError(
            f"{dataset_id}: numeric profile count mismatch."
        )

    if (
        observed_categorical_counts.get(
            dataset_id,
            0
        )
        != EXPECTED_CATEGORICAL_COUNTS[
            dataset_id
        ]
    ):

        raise RuntimeError(
            f"{dataset_id}: categorical profile count mismatch."
        )


# ==============================================================================
# 14.16 — TARGET / PROVENANCE / IDENTIFIER VALIDATION
# ==============================================================================

if FEATURE_STATISTICAL_PROFILE_DF[
    "is_target"
].any():

    raise RuntimeError(
        "Section 14 incorrectly contains a target feature."
    )


for dataset_id in DATASET_IDS:

    schema = NB02_PERSISTED_SCHEMA_CONTRACTS[
        dataset_id
    ]

    modeling_schema = schema[
        "modeling_schema"
    ]

    target_column = modeling_schema[
        "target_column"
    ]

    provenance_column = modeling_schema.get(
        "provenance_column",
        "__original_row_id__"
    )

    identifier_columns = modeling_schema.get(
        "identifier_columns_excluded",
        []
    )

    dataset_profile_features = set(
        FEATURE_STATISTICAL_PROFILE_DF[
            FEATURE_STATISTICAL_PROFILE_DF[
                "dataset_id"
            ] == dataset_id
        ]["feature"]
    )

    if target_column in dataset_profile_features:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            f"appears in Section 14."
        )

    if provenance_column in dataset_profile_features:

        raise RuntimeError(
            f"{dataset_id}: provenance column "
            f"'{provenance_column}' appears in Section 14."
        )

    identifier_overlap = (
        set(identifier_columns)
        .intersection(
            dataset_profile_features
        )
    )

    if identifier_overlap:

        raise RuntimeError(
            f"{dataset_id}: identifier columns appear "
            f"in Section 14: "
            f"{sorted(identifier_overlap)}"
        )


# ==============================================================================
# 14.17 — FEATURE ROLE VALIDATION
# ==============================================================================

if not FEATURE_STATISTICAL_PROFILE_DF[
    "is_preprocessing_feature"
].all():

    raise RuntimeError(
        "Section 14 contains a feature that is not "
        "a preprocessing feature."
    )

if not FEATURE_STATISTICAL_PROFILE_DF[
    "is_generative_feature"
].all():

    raise RuntimeError(
        "Section 14 contains a feature that is not "
        "a generative feature."
    )


# ==============================================================================
# 14.18 — MISSINGNESS VALIDATION
# ==============================================================================

missing_rates = (
    FEATURE_STATISTICAL_PROFILE_DF[
        "missing_rate"
    ]
    .dropna()
)

if (missing_rates < 0).any():

    raise RuntimeError(
        "Section 14 contains negative missing rates."
    )

if (missing_rates > 1).any():

    raise RuntimeError(
        "Section 14 contains missing rates greater than 1."
    )


observation_counts_consistent = (

    FEATURE_STATISTICAL_PROFILE_DF[
        "non_missing_count"
    ]
    .fillna(0)
    .astype(int)

    +

    FEATURE_STATISTICAL_PROFILE_DF[
        "missing_count"
    ]
    .fillna(0)
    .astype(int)

    ==

    FEATURE_STATISTICAL_PROFILE_DF[
        "training_rows"
    ]
    .astype(int)
)


if not observation_counts_consistent.all():

    raise RuntimeError(
        "Section 14 observation counts are inconsistent "
        "with training row counts."
    )


# ==============================================================================
# 14.19 — NUMERIC STATISTICS VALIDATION
# ==============================================================================

numeric_profiles = (
    FEATURE_STATISTICAL_PROFILE_DF[
        FEATURE_STATISTICAL_PROFILE_DF[
            "feature_type"
        ] == "numeric"
    ]
    .copy()
)

numeric_value_columns = [

    "mean",
    "median",
    "std",
    "min",

    "q01",
    "q05",
    "q25",
    "q50",
    "q75",
    "q95",
    "q99",

    "max",

    "skewness",
    "kurtosis",
    "coefficient_of_variation",

    "zero_count",
    "zero_rate",

    "negative_count",
    "negative_rate",

    "positive_count",
    "positive_rate",

]


if len(numeric_profiles) > 0:

    for column in numeric_value_columns:

        values = pd.to_numeric(
            numeric_profiles[
                column
            ],
            errors="coerce"
        )

        if np.isinf(values).any():

            raise RuntimeError(
                f"Section 14 numeric column '{column}' "
                f"contains infinite values."
            )


    # --------------------------------------------------------------------------
    # Quantile ordering
    # --------------------------------------------------------------------------

    quantile_order_pairs = [

        ("q01", "q05"),
        ("q05", "q25"),
        ("q25", "q50"),
        ("q50", "q75"),
        ("q75", "q95"),
        ("q95", "q99"),

    ]

    for lower, upper in (
        quantile_order_pairs
    ):

        valid = (
            numeric_profiles[
                lower
            ].notna()
            &
            numeric_profiles[
                upper
            ].notna()
        )

        if (
            numeric_profiles.loc[
                valid,
                lower
            ]
            >
            numeric_profiles.loc[
                valid,
                upper
            ]
        ).any():

            raise RuntimeError(
                f"Section 14 quantile ordering violation: "
                f"{lower} > {upper}."
            )


# ==============================================================================
# 14.20 — RATE VALIDATION
# ==============================================================================

RATE_COLUMNS = [

    "missing_rate",
    "zero_rate",
    "negative_rate",
    "positive_rate",
    "top_category_frequency",
    "normalized_entropy",

]

for column in RATE_COLUMNS:

    values = pd.to_numeric(
        FEATURE_STATISTICAL_PROFILE_DF[
            column
        ],
        errors="coerce"
    ).dropna()

    if (
        (values < 0)
        |
        (values > 1)
    ).any():

        raise RuntimeError(
            f"Section 14 column '{column}' contains "
            f"values outside [0, 1]."
        )


# ==============================================================================
# 14.21 — CARDINALITY VALIDATION
# ==============================================================================

numeric_profiles_only = (
    FEATURE_STATISTICAL_PROFILE_DF[
        FEATURE_STATISTICAL_PROFILE_DF[
            "feature_type"
        ] == "numeric"
    ]
    .copy()
)

categorical_profiles = (
    FEATURE_STATISTICAL_PROFILE_DF[
        FEATURE_STATISTICAL_PROFILE_DF[
            "feature_type"
        ] == "categorical"
    ]
    .copy()
)


numeric_cardinality_values = pd.to_numeric(
    numeric_profiles_only[
        "n_unique"
    ],
    errors="coerce"
).dropna()

categorical_cardinality_values = pd.to_numeric(
    categorical_profiles[
        "n_unique"
    ],
    errors="coerce"
).dropna()


if (
    numeric_cardinality_values < 1
).any():

    raise RuntimeError(
        "Section 14 contains invalid numeric "
        "cardinality values."
    )


if (
    categorical_cardinality_values < 1
).any():

    raise RuntimeError(
        "Section 14 contains invalid categorical "
        "cardinality values."
    )


# ==============================================================================
# 14.22 — CATEGORICAL CATEGORY-COUNT VALIDATION
# ==============================================================================

rare_values = pd.to_numeric(
    categorical_profiles[
        "rare_category_count"
    ],
    errors="coerce"
).dropna()

singleton_values = pd.to_numeric(
    categorical_profiles[
        "singleton_category_count"
    ],
    errors="coerce"
).dropna()


if (rare_values < 0).any():

    raise RuntimeError(
        "Section 14 contains negative rare-category counts."
    )

if (singleton_values < 0).any():

    raise RuntimeError(
        "Section 14 contains negative singleton-category counts."
    )


# ==============================================================================
# 14.23 — SECTION 9 CONSISTENCY VALIDATION
#
# ONLY categorical features are compared against Section 9.
# ==============================================================================

for _, profile_row in (
    categorical_profiles.iterrows()
):

    dataset_id = profile_row[
        "dataset_id"
    ]

    feature = profile_row[
        "feature"
    ]

    entropy_row = _get_single_row(
        CARDINALITY_ENTROPY_DF,
        dataset_id,
        feature,
        "CARDINALITY_ENTROPY_DF"
    )


    # --------------------------------------------------------------------------
    # Cardinality
    # --------------------------------------------------------------------------

    profile_cardinality = profile_row[
        "n_unique"
    ]

    entropy_cardinality = entropy_row[
        "cardinality"
    ]

    if int(profile_cardinality) != int(
        entropy_cardinality
    ):

        raise RuntimeError(
            f"{dataset_id}/{feature}: cardinality mismatch "
            f"between Section 14 and Section 9."
        )


    # --------------------------------------------------------------------------
    # Cardinality ratio
    # --------------------------------------------------------------------------

    if "cardinality_ratio" in entropy_row.index:

        profile_ratio = profile_row[
            "unique_ratio"
        ]

        entropy_ratio = entropy_row[
            "cardinality_ratio"
        ]

        if not np.isclose(
            float(profile_ratio),
            float(entropy_ratio),
            rtol=0,
            atol=1e-8,
            equal_nan=True
        ):

            raise RuntimeError(
                f"{dataset_id}/{feature}: cardinality ratio "
                f"mismatch between Section 14 and Section 9."
            )


    # --------------------------------------------------------------------------
    # Normalized entropy
    # --------------------------------------------------------------------------

    if "normalized_entropy" in entropy_row.index:

        profile_entropy = profile_row[
            "normalized_entropy"
        ]

        entropy_value = entropy_row[
            "normalized_entropy"
        ]

        if not np.isclose(
            float(profile_entropy),
            float(entropy_value),
            rtol=0,
            atol=1e-8,
            equal_nan=True
        ):

            raise RuntimeError(
                f"{dataset_id}/{feature}: normalized entropy "
                f"mismatch between Section 14 and Section 9."
            )


    # --------------------------------------------------------------------------
    # Entropy bits
    #
    # Only validate when Section 9 actually provides this field.
    # --------------------------------------------------------------------------

    if "entropy_bits" in entropy_row.index:

        profile_entropy_bits = (
            profile_row[
                "entropy_bits"
            ]
        )

        section9_entropy_bits = (
            entropy_row[
                "entropy_bits"
            ]
        )

        if not np.isclose(
            float(profile_entropy_bits),
            float(section9_entropy_bits),
            rtol=0,
            atol=1e-8,
            equal_nan=True
        ):

            raise RuntimeError(
                f"{dataset_id}/{feature}: entropy_bits mismatch "
                f"between Section 14 and Section 9."
            )


# ==============================================================================
# 14.24 — SECTION 5 CONSISTENCY VALIDATION
# ==============================================================================

for _, profile_row in (
    FEATURE_STATISTICAL_PROFILE_DF.iterrows()
):

    dataset_id = profile_row[
        "dataset_id"
    ]

    feature = profile_row[
        "feature"
    ]

    section5_row = _get_single_row(
        SECTION5_PREPROCESSING_GLOBAL,
        dataset_id,
        feature,
        "FEATURE_TYPE_CHARACTERIZATION_DF"
    )


    # --------------------------------------------------------------------------
    # Feature type
    # --------------------------------------------------------------------------

    expected_type = section5_row[
        "feature_type"
    ]

    if profile_row[
        "feature_type"
    ] != expected_type:

        raise RuntimeError(
            f"{dataset_id}/{feature}: feature type mismatch "
            f"between Section 14 and Section 5."
        )


    # --------------------------------------------------------------------------
    # Missing rate
    # --------------------------------------------------------------------------

    profile_missing_rate = profile_row[
        "missing_rate"
    ]

    section5_missing_rate = section5_row[
        "missing_rate"
    ]

    if not np.isclose(
        float(profile_missing_rate),
        float(section5_missing_rate),
        rtol=0,
        atol=1e-8,
        equal_nan=True
    ):

        raise RuntimeError(
            f"{dataset_id}/{feature}: missing rate mismatch "
            f"between Section 14 and Section 5."
        )


    # --------------------------------------------------------------------------
    # Observation counts
    # --------------------------------------------------------------------------

    if int(
        profile_row[
            "non_missing_count"
        ]
    ) != int(
        section5_row[
            "non_missing_count"
        ]
    ):

        raise RuntimeError(
            f"{dataset_id}/{feature}: non-missing count mismatch."
        )

    if int(
        profile_row[
            "missing_count"
        ]
    ) != int(
        section5_row[
            "missing_count"
        ]
    ):

        raise RuntimeError(
            f"{dataset_id}/{feature}: missing count mismatch."
        )


# ==============================================================================
# 14.25 — FINAL SUMMARY TABLE
# ==============================================================================

FEATURE_STATISTICAL_PROFILE_SUMMARY_DF = (

    FEATURE_STATISTICAL_PROFILE_DF

    .groupby(
        [
            "dataset_id",
            "feature_type"
        ],
        dropna=False
    )

    .agg(

        n_features=(
            "feature",
            "count"
        ),

        mean_missing_rate=(
            "missing_rate",
            "mean"
        ),

        max_missing_rate=(
            "missing_rate",
            "max"
        ),

        mean_cardinality=(
            "n_unique",
            "mean"
        ),

        max_cardinality=(
            "n_unique",
            "max"
        ),

        mean_normalized_entropy=(
            "normalized_entropy",
            "mean"
        ),

    )

    .reset_index()
)


# ==============================================================================
# 14.26 — FINAL VALIDATION CHECKS
# ==============================================================================

FINAL_CHECKS = {

    "total_profiles":
        len(
            FEATURE_STATISTICAL_PROFILE_DF
        )
        == EXPECTED_TOTAL_PROFILES,

    "dataset_coverage":
        set(
            FEATURE_STATISTICAL_PROFILE_DF[
                "dataset_id"
            ].unique()
        )
        == set(DATASET_IDS),

    "no_duplicates":
        duplicate_count == 0,

    "target_excluded":
        not FEATURE_STATISTICAL_PROFILE_DF[
            "is_target"
        ].any(),

    "all_preprocessing_features":
        FEATURE_STATISTICAL_PROFILE_DF[
            "is_preprocessing_feature"
        ].all(),

    "all_generative_features":
        FEATURE_STATISTICAL_PROFILE_DF[
            "is_generative_feature"
        ].all(),

    "missing_rates_valid":
        FEATURE_STATISTICAL_PROFILE_DF[
            "missing_rate"
        ]
        .dropna()
        .between(0, 1)
        .all(),

    "observation_counts_consistent":
        observation_counts_consistent.all(),

    "numeric_cardinality_valid":
        (
            numeric_cardinality_values >= 1
        ).all(),

    "categorical_cardinality_valid":
        (
            categorical_cardinality_values >= 1
        ).all(),

    "categorical_entropy_valid":
        categorical_profiles[
            "normalized_entropy"
        ]
        .dropna()
        .between(0, 1)
        .all(),

}


print()
print("=" * 100)
print("SECTION 14 FINAL VALIDATION")
print("=" * 100)

for (
    check_name,
    check_value
) in FINAL_CHECKS.items():

    status = (
        "PASS"
        if bool(check_value)
        else "FAIL"
    )

    print(
        f"{status:<6} | {check_name}"
    )


if not all(
    FINAL_CHECKS.values()
):

    failed_checks = [

        name

        for name, value
        in FINAL_CHECKS.items()

        if not bool(value)

    ]

    raise RuntimeError(
        "SECTION 14 FINAL VALIDATION FAILED.\n"
        f"Failed checks: {failed_checks}"
    )


# ==============================================================================
# 14.27 — FINAL OUTPUT SUMMARY
# ==============================================================================

print()
print("=" * 100)
print("SECTION 14 SUMMARY")
print("=" * 100)

print(
    f"Total datasets              : "
    f"{len(DATASET_IDS)}"
)

print(
    f"Total preprocessing profiles: "
    f"{len(FEATURE_STATISTICAL_PROFILE_DF)}"
)

print(
    f"Numeric profiles            : "
    f"{len(numeric_profiles_only)}"
)

print(
    f"Categorical profiles        : "
    f"{len(categorical_profiles)}"
)

print(
    f"Duplicate profiles          : "
    f"{duplicate_count}"
)

print(
    f"Target included             : "
    f"{FEATURE_STATISTICAL_PROFILE_DF['is_target'].any()}"
)

print(
    f"Section 10 integrated       : "
    f"{HAS_SECTION10}"
)

print(
    f"Section 9 entropy_bits      : "
    f"{'available' if 'entropy_bits' in CARDINALITY_ENTROPY_DF.columns else 'not available'}"
)

print()
print(
    "✓ FEATURE_STATISTICAL_PROFILE_DF created successfully."
)

print(
    "✓ Section 14 uses training-only validated upstream statistics."
)

print(
    "✓ No statistics were recomputed."
)

print(
    "✓ Numeric features use Section 6 statistics."
)

print(
    "✓ Categorical features use Sections 7 and 9 statistics."
)

print(
    "✓ Section 9 is queried only for categorical features."
)

print(
    "✓ Target, identifiers, and provenance are excluded."
)

print(
    "✓ Frozen Notebook 02 schema is enforced."
)

print(
    "✓ Section 5 consistency checks passed."
)

print(
    "✓ Section 9 consistency checks passed for categorical features."
)

print()
print("=" * 100)
print("SECTION 14 STATUS: PASS")
print("=" * 100)

SECTION 14 — FEATURE STATISTICAL PROFILES
✓ Required upstream objects available.
ℹ Section 10 distribution characterization not found in current runtime.
  Section 14 will use validated statistics from Sections 5–9 without recomputation.
✓ Section 5 preprocessing records available: 77

✓ Section 9 columns detected:
['dataset_id', 'feature', 'cardinality', 'cardinality_ratio', 'shannon_entropy_bits', 'normalized_entropy', 'entropy_level', 'top_frequency', 'rare_category_count', 'singleton_category_count', 'count', 'non_missing_count', 'missing_count', 'missing_rate', 'is_numeric', 'is_categorical', 'is_target', 'is_preprocessing_feature', 'is_generative_feature', 'schema_role']
ℹ Section 9 does not provide 'entropy_bits'.
  Section 14 will retain entropy_bits as NaN without recomputation.

Processing adult_income
Training rows       : 34189
Preprocessing vars : 14
Numeric variables  : 6
Categorical vars   : 8
✓ Completed adult_income: 14 preprocessing features

Processing bank_marketing

In [15]:
# ==============================================================================
# SECTION 15 — DATASET STATISTICAL PROFILES
# ==============================================================================

print("=" * 100)
print("SECTION 15 — DATASET STATISTICAL PROFILES")
print("=" * 100)


# ==============================================================================
# 15.1 — REQUIRED UPSTREAM OBJECTS
# ==============================================================================

REQUIRED_UPSTREAM_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_STATISTICAL_DATASETS",
    "NB02_PERSISTED_SCHEMA_CONTRACTS",
    "FEATURE_STATISTICAL_PROFILE_DF",
    "PEARSON_DF",
    "SPEARMAN_DF",
    "CATEGORICAL_DEPENDENCY_DF",
]

for object_name in REQUIRED_UPSTREAM_OBJECTS:

    if object_name not in globals():

        raise RuntimeError(
            f"Section 15 requires upstream object "
            f"'{object_name}', but it was not found.\n"
            f"Run the required upstream sections before Section 15."
        )

print("✓ Required upstream objects available.")


# ==============================================================================
# 15.2 — EXPECTED FEATURE COUNTS
# ==============================================================================

EXPECTED_PREPROCESSING_COUNTS = {
    "adult_income": 14,
    "bank_marketing": 16,
    "diabetes_130us": 47,
}

EXPECTED_NUMERIC_COUNTS = {
    "adult_income": 6,
    "bank_marketing": 7,
    "diabetes_130us": 11,
}

EXPECTED_CATEGORICAL_COUNTS = {
    "adult_income": 8,
    "bank_marketing": 9,
    "diabetes_130us": 36,
}


# ==============================================================================
# 15.3 — EXPECTED DEPENDENCY PAIR COUNTS
# ==============================================================================

EXPECTED_PEARSON_PAIRS = {
    "adult_income": 15,
    "bank_marketing": 21,
    "diabetes_130us": 55,
}

EXPECTED_SPEARMAN_PAIRS = {
    "adult_income": 15,
    "bank_marketing": 21,
    "diabetes_130us": 55,
}

EXPECTED_CATEGORICAL_DEPENDENCY_PAIRS = {
    "adult_income": 28,
    "bank_marketing": 36,
    "diabetes_130us": 630,
}


# ==============================================================================
# 15.4 — HELPER FUNCTION
# ==============================================================================

def _get_single_dataset_schema(dataset_id):

    if dataset_id not in NB02_PERSISTED_SCHEMA_CONTRACTS:

        raise RuntimeError(
            f"{dataset_id}: persisted Notebook 02 schema "
            f"contract not found."
        )

    return NB02_PERSISTED_SCHEMA_CONTRACTS[
        dataset_id
    ]


# ==============================================================================
# 15.5 — VALIDATE SECTION 14 INPUT
# ==============================================================================

required_section14_columns = [

    "dataset_id",
    "feature_order",
    "feature",
    "feature_type",

    "is_target",
    "is_preprocessing_feature",
    "is_generative_feature",

    "training_rows",

    "missing_rate",
    "n_unique",
    "unique_ratio",

    "mean",
    "skewness",

]


missing_section14_columns = [
    column
    for column in required_section14_columns
    if column not in FEATURE_STATISTICAL_PROFILE_DF.columns
]

if missing_section14_columns:

    raise RuntimeError(
        "Section 14 is missing columns required by Section 15:\n"
        f"{missing_section14_columns}"
    )


# ==============================================================================
# 15.6 — VALIDATE SECTION 14 TOTAL COVERAGE
# ==============================================================================

EXPECTED_TOTAL_PREPROCESSING_FEATURES = sum(
    EXPECTED_PREPROCESSING_COUNTS.values()
)

if len(FEATURE_STATISTICAL_PROFILE_DF) != (
    EXPECTED_TOTAL_PREPROCESSING_FEATURES
):

    raise RuntimeError(
        f"Section 15: Section 14 profile count mismatch.\n"
        f"Expected {EXPECTED_TOTAL_PREPROCESSING_FEATURES}, "
        f"found {len(FEATURE_STATISTICAL_PROFILE_DF)}."
    )

print(
    f"✓ Section 14 feature profiles validated: "
    f"{len(FEATURE_STATISTICAL_PROFILE_DF)}"
)


# ==============================================================================
# 15.7 — VALIDATE SECTION 14 TARGET EXCLUSION
# ==============================================================================

if FEATURE_STATISTICAL_PROFILE_DF[
    "is_target"
].any():

    raise RuntimeError(
        "Section 15 cannot proceed because Section 14 "
        "contains a target feature."
    )


# ==============================================================================
# 15.8 — VALIDATE SECTION 14 PREPROCESSING ROLE
# ==============================================================================

if not FEATURE_STATISTICAL_PROFILE_DF[
    "is_preprocessing_feature"
].all():

    raise RuntimeError(
        "Section 14 contains records that are not preprocessing features."
    )


# ==============================================================================
# 15.9 — BUILD DATASET-LEVEL PROFILES
# ==============================================================================

DATASET_STATISTICAL_PROFILES = []


for dataset_id in DATASET_IDS:

    print()
    print(f"Processing {dataset_id}")


    # ==========================================================================
    # 15.9.1 — LOAD TRAINING DATA
    # ==========================================================================

    if dataset_id not in TRAIN_STATISTICAL_DATASETS:

        raise RuntimeError(
            f"{dataset_id}: training dataset not available."
        )

    df = TRAIN_STATISTICAL_DATASETS[
        dataset_id
    ]

    training_rows = len(df)


    # ==========================================================================
    # 15.9.2 — LOAD FROZEN NOTEBOOK 02 SCHEMA
    # ==========================================================================

    schema = _get_single_dataset_schema(
        dataset_id
    )

    modeling_schema = schema[
        "modeling_schema"
    ]

    preprocessing_columns = list(
        modeling_schema[
            "preprocessing_columns"
        ]
    )

    numeric_columns = list(
        modeling_schema[
            "numeric_columns"
        ]
    )

    categorical_columns = list(
        modeling_schema[
            "categorical_columns"
        ]
    )

    generative_columns = list(
        modeling_schema[
            "generative_columns"
        ]
    )

    target_column = modeling_schema[
        "target_column"
    ]

    provenance_column = modeling_schema.get(
        "provenance_column",
        "__original_row_id__"
    )

    identifier_columns = list(
        modeling_schema.get(
            "identifier_columns_excluded",
            []
        )
    )


    # ==========================================================================
    # 15.9.3 — VALIDATE TRAINING DATASET SIZE
    # ==========================================================================

    if training_rows != (
        FEATURE_STATISTICAL_PROFILE_DF[
            FEATURE_STATISTICAL_PROFILE_DF[
                "dataset_id"
            ] == dataset_id
        ]["training_rows"]
        .iloc[0]
    ):

        raise RuntimeError(
            f"{dataset_id}: training row count mismatch "
            f"between training dataset and Section 14."
        )


    # ==========================================================================
    # 15.9.4 — GET SECTION 14 FEATURE PROFILES
    # ==========================================================================

    feature_profiles = (
        FEATURE_STATISTICAL_PROFILE_DF[
            FEATURE_STATISTICAL_PROFILE_DF[
                "dataset_id"
            ] == dataset_id
        ]
        .copy()
    )


    # ==========================================================================
    # 15.9.5 — VALIDATE SECTION 14 FEATURE COUNT
    # ==========================================================================

    if len(feature_profiles) != (
        len(preprocessing_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: Section 14 feature count mismatch.\n"
            f"Expected {len(preprocessing_columns)}, "
            f"found {len(feature_profiles)}."
        )


    # ==========================================================================
    # 15.9.6 — VALIDATE FEATURE IDENTITY
    # ==========================================================================

    observed_features = (
        feature_profiles[
            "feature"
        ]
        .tolist()
    )

    if set(observed_features) != set(
        preprocessing_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: Section 14 feature set does not "
            f"match the frozen Notebook 02 preprocessing schema."
        )


    # ==========================================================================
    # 15.9.7 — VALIDATE FEATURE ORDER
    # ==========================================================================

    observed_feature_order = (
        feature_profiles
        .sort_values(
            "feature_order"
        )["feature"]
        .tolist()
    )

    if observed_feature_order != (
        preprocessing_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: Section 14 feature order does not "
            f"match the frozen Notebook 02 preprocessing schema."
        )


    # ==========================================================================
    # 15.9.8 — CREATE NUMERIC / CATEGORICAL SUBSETS
    #
    # IMPORTANT:
    # Section 14 uses 'feature_type', not the obsolete 'semantic_type'.
    # ==========================================================================

    numeric_profiles = (
        feature_profiles[
            feature_profiles[
                "feature_type"
            ] == "numeric"
        ]
        .copy()
    )

    categorical_profiles = (
        feature_profiles[
            feature_profiles[
                "feature_type"
            ] == "categorical"
        ]
        .copy()
    )


    # ==========================================================================
    # 15.9.9 — VALIDATE NUMERIC / CATEGORICAL COUNTS
    # ==========================================================================

    if len(numeric_profiles) != (
        len(numeric_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: numeric profile count mismatch.\n"
            f"Expected {len(numeric_columns)}, "
            f"found {len(numeric_profiles)}."
        )

    if len(categorical_profiles) != (
        len(categorical_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: categorical profile count mismatch.\n"
            f"Expected {len(categorical_columns)}, "
            f"found {len(categorical_profiles)}."
        )


    if len(numeric_profiles) != (
        EXPECTED_NUMERIC_COUNTS[
            dataset_id
        ]
    ):

        raise RuntimeError(
            f"{dataset_id}: unexpected numeric feature count."
        )


    if len(categorical_profiles) != (
        EXPECTED_CATEGORICAL_COUNTS[
            dataset_id
        ]
    ):

        raise RuntimeError(
            f"{dataset_id}: unexpected categorical feature count."
        )


    # ==========================================================================
    # 15.9.10 — VALIDATE TARGET / PROVENANCE / IDENTIFIER EXCLUSION
    # ==========================================================================

    profile_feature_set = set(
        feature_profiles[
            "feature"
        ]
    )

    if target_column in profile_feature_set:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            f"appears in Section 15 feature aggregation."
        )

    if provenance_column in profile_feature_set:

        raise RuntimeError(
            f"{dataset_id}: provenance column "
            f"'{provenance_column}' appears in Section 15."
        )

    identifier_overlap = (
        set(identifier_columns)
        .intersection(
            profile_feature_set
        )
    )

    if identifier_overlap:

        raise RuntimeError(
            f"{dataset_id}: identifier columns appear "
            f"in Section 15 profiles: "
            f"{sorted(identifier_overlap)}"
        )


    # ==========================================================================
    # 15.9.11 — FEATURE-LEVEL MISSINGNESS AGGREGATION
    #
    # Values come directly from validated Section 14 profiles.
    # ==========================================================================

    feature_missing_rates = pd.to_numeric(
        feature_profiles[
            "missing_rate"
        ],
        errors="coerce"
    ).dropna()


    if len(feature_missing_rates) != (
        len(feature_profiles)
    ):

        raise RuntimeError(
            f"{dataset_id}: Section 14 missing-rate coverage incomplete."
        )


    mean_feature_missing_rate = float(
        feature_missing_rates.mean()
    )

    maximum_feature_missing_rate = float(
        feature_missing_rates.max()
    )


    # ==========================================================================
    # 15.9.12 — NUMERIC SKEWNESS AGGREGATION
    # ==========================================================================

    numeric_skewness = pd.to_numeric(
        numeric_profiles[
            "skewness"
        ],
        errors="coerce"
    ).dropna()


    mean_numeric_skewness = (

        float(
            numeric_skewness.mean()
        )

        if len(numeric_skewness) > 0

        else np.nan

    )


    maximum_absolute_numeric_skewness = (

        float(
            numeric_skewness.abs().max()
        )

        if len(numeric_skewness) > 0

        else np.nan

    )


    # ==========================================================================
    # 15.9.13 — PEARSON DEPENDENCY AGGREGATION
    #
    # Section 11 already contains unique unordered pairs.
    # Section 15 only aggregates them.
    # ==========================================================================

    pearson_dataset = (
        PEARSON_DF[
            PEARSON_DF[
                "dataset_id"
            ] == dataset_id
        ]
        .copy()
    )


    if (
        "feature_a" not in pearson_dataset.columns
        or
        "feature_b" not in pearson_dataset.columns
        or
        "pearson_r" not in pearson_dataset.columns
    ):

        raise RuntimeError(
            f"{dataset_id}: PEARSON_DF does not contain "
            f"the expected columns."
        )


    # Exclude any accidental self-pairs.
    pearson_dataset = (
        pearson_dataset[
            pearson_dataset[
                "feature_a"
            ]
            !=
            pearson_dataset[
                "feature_b"
            ]
        ]
        .copy()
    )


    pearson_values = pd.to_numeric(
        pearson_dataset[
            "pearson_r"
        ],
        errors="coerce"
    ).dropna()


    expected_pearson_pairs = (
        EXPECTED_PEARSON_PAIRS[
            dataset_id
        ]
    )


    if len(pearson_dataset) != (
        expected_pearson_pairs
    ):

        raise RuntimeError(
            f"{dataset_id}: Pearson pair coverage mismatch.\n"
            f"Expected {expected_pearson_pairs}, "
            f"found {len(pearson_dataset)}."
        )


    if len(pearson_values) != (
        expected_pearson_pairs
    ):

        raise RuntimeError(
            f"{dataset_id}: Pearson valid-pair coverage mismatch.\n"
            f"Expected {expected_pearson_pairs}, "
            f"found {len(pearson_values)}."
        )


    if (
        (pearson_values < -1)
        |
        (pearson_values > 1)
    ).any():

        raise RuntimeError(
            f"{dataset_id}: Pearson correlations outside [-1, 1]."
        )


    maximum_absolute_pearson = (

        float(
            pearson_values.abs().max()
        )

        if len(pearson_values) > 0

        else np.nan

    )


    mean_absolute_pearson = (

        float(
            pearson_values.abs().mean()
        )

        if len(pearson_values) > 0

        else np.nan

    )


    # ==========================================================================
    # 15.9.14 — SPEARMAN DEPENDENCY AGGREGATION
    # ==========================================================================

    spearman_dataset = (
        SPEARMAN_DF[
            SPEARMAN_DF[
                "dataset_id"
            ] == dataset_id
        ]
        .copy()
    )


    if (
        "feature_a" not in spearman_dataset.columns
        or
        "feature_b" not in spearman_dataset.columns
        or
        "spearman_rho" not in spearman_dataset.columns
    ):

        raise RuntimeError(
            f"{dataset_id}: SPEARMAN_DF does not contain "
            f"the expected columns."
        )


    spearman_dataset = (
        spearman_dataset[
            spearman_dataset[
                "feature_a"
            ]
            !=
            spearman_dataset[
                "feature_b"
            ]
        ]
        .copy()
    )


    spearman_values = pd.to_numeric(
        spearman_dataset[
            "spearman_rho"
        ],
        errors="coerce"
    ).dropna()


    expected_spearman_pairs = (
        EXPECTED_SPEARMAN_PAIRS[
            dataset_id
        ]
    )


    if len(spearman_dataset) != (
        expected_spearman_pairs
    ):

        raise RuntimeError(
            f"{dataset_id}: Spearman pair coverage mismatch.\n"
            f"Expected {expected_spearman_pairs}, "
            f"found {len(spearman_dataset)}."
        )


    if len(spearman_values) != (
        expected_spearman_pairs
    ):

        raise RuntimeError(
            f"{dataset_id}: Spearman valid-pair coverage mismatch.\n"
            f"Expected {expected_spearman_pairs}, "
            f"found {len(spearman_values)}."
        )


    if (
        (spearman_values < -1)
        |
        (spearman_values > 1)
    ).any():

        raise RuntimeError(
            f"{dataset_id}: Spearman correlations outside [-1, 1]."
        )


    maximum_absolute_spearman = (

        float(
            spearman_values.abs().max()
        )

        if len(spearman_values) > 0

        else np.nan

    )


    mean_absolute_spearman = (

        float(
            spearman_values.abs().mean()
        )

        if len(spearman_values) > 0

        else np.nan

    )


    # ==========================================================================
    # 15.9.15 — CATEGORICAL DEPENDENCY AGGREGATION
    #
    # Section 13 already computes bias-corrected Cramér's V.
    # Section 15 only aggregates the validated pair results.
    # ==========================================================================

    categorical_dependency_dataset = (
        CATEGORICAL_DEPENDENCY_DF[
            CATEGORICAL_DEPENDENCY_DF[
                "dataset_id"
            ] == dataset_id
        ]
        .copy()
    )


    if (
        "cramers_v"
        not in categorical_dependency_dataset.columns
    ):

        raise RuntimeError(
            f"{dataset_id}: CATEGORICAL_DEPENDENCY_DF does not "
            f"contain 'cramers_v'."
        )


    expected_categorical_pairs = (
        EXPECTED_CATEGORICAL_DEPENDENCY_PAIRS[
            dataset_id
        ]
    )


    if len(
        categorical_dependency_dataset
    ) != expected_categorical_pairs:

        raise RuntimeError(
            f"{dataset_id}: categorical dependency pair "
            f"coverage mismatch.\n"
            f"Expected {expected_categorical_pairs}, "
            f"found {len(categorical_dependency_dataset)}."
        )


    categorical_values = pd.to_numeric(
        categorical_dependency_dataset[
            "cramers_v"
        ],
        errors="coerce"
    ).dropna()


    if len(categorical_values) != (
        expected_categorical_pairs
    ):

        raise RuntimeError(
            f"{dataset_id}: Cramér's V valid-pair coverage mismatch.\n"
            f"Expected {expected_categorical_pairs}, "
            f"found {len(categorical_values)}."
        )


    if (
        (categorical_values < 0)
        |
        (categorical_values > 1)
    ).any():

        raise RuntimeError(
            f"{dataset_id}: Cramér's V values outside [0, 1]."
        )


    maximum_categorical_cramers_v = (

        float(
            categorical_values.max()
        )

        if len(categorical_values) > 0

        else np.nan

    )


    mean_categorical_cramers_v = (

        float(
            categorical_values.mean()
        )

        if len(categorical_values) > 0

        else np.nan

    )


    # ==========================================================================
    # 15.9.16 — CREATE DATASET STATISTICAL PROFILE
    # ==========================================================================

    DATASET_STATISTICAL_PROFILES.append({

        "dataset_id":
            dataset_id,

        "training_rows":
            training_rows,

        "modeling_features":
            len(preprocessing_columns),

        "numeric_features":
            len(numeric_columns),

        "categorical_features":
            len(categorical_columns),

        "target":
            target_column,

        "mean_feature_missing_rate":
            mean_feature_missing_rate,

        "maximum_feature_missing_rate":
            maximum_feature_missing_rate,

        "mean_numeric_skewness":
            mean_numeric_skewness,

        "maximum_absolute_numeric_skewness":
            maximum_absolute_numeric_skewness,

        "mean_absolute_pearson":
            mean_absolute_pearson,

        "maximum_absolute_pearson":
            maximum_absolute_pearson,

        "mean_absolute_spearman":
            mean_absolute_spearman,

        "maximum_absolute_spearman":
            maximum_absolute_spearman,

        "mean_categorical_cramers_v":
            mean_categorical_cramers_v,

        "maximum_categorical_cramers_v":
            maximum_categorical_cramers_v,

        "pearson_pairs":
            len(pearson_values),

        "spearman_pairs":
            len(spearman_values),

        "categorical_dependency_pairs":
            len(categorical_values),

    })


    print(
        f"✓ Completed {dataset_id}"
    )

    print(
        f"  Training rows       : {training_rows}"
    )

    print(
        f"  Preprocessing vars  : {len(preprocessing_columns)}"
    )

    print(
        f"  Numeric variables   : {len(numeric_columns)}"
    )

    print(
        f"  Categorical vars    : {len(categorical_columns)}"
    )

    print(
        f"  Pearson pairs       : {len(pearson_values)}"
    )

    print(
        f"  Spearman pairs      : {len(spearman_values)}"
    )

    print(
        f"  Cramér's V pairs    : {len(categorical_values)}"
    )


# ==============================================================================
# 15.10 — CREATE FINAL DATAFRAME
# ==============================================================================

DATASET_STATISTICAL_PROFILE_DF = pd.DataFrame(
    DATASET_STATISTICAL_PROFILES
)


# ==============================================================================
# 15.11 — REQUIRED OUTPUT COLUMNS
# ==============================================================================

REQUIRED_DATASET_PROFILE_COLUMNS = [

    "dataset_id",
    "training_rows",

    "modeling_features",
    "numeric_features",
    "categorical_features",

    "target",

    "mean_feature_missing_rate",
    "maximum_feature_missing_rate",

    "mean_numeric_skewness",
    "maximum_absolute_numeric_skewness",

    "mean_absolute_pearson",
    "maximum_absolute_pearson",

    "mean_absolute_spearman",
    "maximum_absolute_spearman",

    "mean_categorical_cramers_v",
    "maximum_categorical_cramers_v",

    "pearson_pairs",
    "spearman_pairs",
    "categorical_dependency_pairs",

]


missing_output_columns = [

    column

    for column
    in REQUIRED_DATASET_PROFILE_COLUMNS

    if column
    not in DATASET_STATISTICAL_PROFILE_DF.columns

]


if missing_output_columns:

    raise RuntimeError(
        "Section 15 output is missing required columns:\n"
        f"{missing_output_columns}"
    )


# ==============================================================================
# 15.12 — DATASET COVERAGE VALIDATION
# ==============================================================================

if len(DATASET_STATISTICAL_PROFILE_DF) != (
    len(DATASET_IDS)
):

    raise RuntimeError(
        f"Section 15 dataset count mismatch.\n"
        f"Expected {len(DATASET_IDS)}, "
        f"found {len(DATASET_STATISTICAL_PROFILE_DF)}."
    )


if set(
    DATASET_STATISTICAL_PROFILE_DF[
        "dataset_id"
    ]
) != set(DATASET_IDS):

    raise RuntimeError(
        "Section 15 dataset coverage does not match DATASET_IDS."
    )


# ==============================================================================
# 15.13 — DUPLICATE DATASET VALIDATION
# ==============================================================================

duplicate_dataset_count = (
    DATASET_STATISTICAL_PROFILE_DF
    .duplicated(
        subset=[
            "dataset_id"
        ]
    )
    .sum()
)

if duplicate_dataset_count != 0:

    raise RuntimeError(
        f"Section 15 contains "
        f"{duplicate_dataset_count} duplicate dataset profiles."
    )


# ==============================================================================
# 15.14 — FEATURE COUNT VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    row = DATASET_STATISTICAL_PROFILE_DF[
        DATASET_STATISTICAL_PROFILE_DF[
            "dataset_id"
        ] == dataset_id
    ].iloc[0]


    if int(
        row[
            "modeling_features"
        ]
    ) != EXPECTED_PREPROCESSING_COUNTS[
        dataset_id
    ]:

        raise RuntimeError(
            f"{dataset_id}: dataset modeling feature count mismatch."
        )


    if int(
        row[
            "numeric_features"
        ]
    ) != EXPECTED_NUMERIC_COUNTS[
        dataset_id
    ]:

        raise RuntimeError(
            f"{dataset_id}: dataset numeric feature count mismatch."
        )


    if int(
        row[
            "categorical_features"
        ]
    ) != EXPECTED_CATEGORICAL_COUNTS[
        dataset_id
    ]:

        raise RuntimeError(
            f"{dataset_id}: dataset categorical feature count mismatch."
        )


# ==============================================================================
# 15.15 — MISSINGNESS RANGE VALIDATION
# ==============================================================================

missingness_columns = [

    "mean_feature_missing_rate",
    "maximum_feature_missing_rate",

]

for column in missingness_columns:

    values = pd.to_numeric(
        DATASET_STATISTICAL_PROFILE_DF[
            column
        ],
        errors="coerce"
    ).dropna()

    if (
        (values < 0)
        |
        (values > 1)
    ).any():

        raise RuntimeError(
            f"Section 15 column '{column}' contains "
            f"values outside [0, 1]."
        )


# ==============================================================================
# 15.16 — DEPENDENCY RANGE VALIDATION
# ==============================================================================

pearson_summary_values = pd.concat([
    DATASET_STATISTICAL_PROFILE_DF[
        "mean_absolute_pearson"
    ],
    DATASET_STATISTICAL_PROFILE_DF[
        "maximum_absolute_pearson"
    ],
]).dropna()


if (
    (pearson_summary_values < 0)
    |
    (pearson_summary_values > 1)
).any():

    raise RuntimeError(
        "Section 15 contains invalid Pearson summary values."
    )


spearman_summary_values = pd.concat([
    DATASET_STATISTICAL_PROFILE_DF[
        "mean_absolute_spearman"
    ],
    DATASET_STATISTICAL_PROFILE_DF[
        "maximum_absolute_spearman"
    ],
]).dropna()


if (
    (spearman_summary_values < 0)
    |
    (spearman_summary_values > 1)
).any():

    raise RuntimeError(
        "Section 15 contains invalid Spearman summary values."
    )


cramers_summary_values = pd.concat([
    DATASET_STATISTICAL_PROFILE_DF[
        "mean_categorical_cramers_v"
    ],
    DATASET_STATISTICAL_PROFILE_DF[
        "maximum_categorical_cramers_v"
    ],
]).dropna()


if (
    (cramers_summary_values < 0)
    |
    (cramers_summary_values > 1)
).any():

    raise RuntimeError(
        "Section 15 contains invalid Cramér's V summary values."
    )


# ==============================================================================
# 15.17 — DEPENDENCY PAIR COUNT VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    row = DATASET_STATISTICAL_PROFILE_DF[
        DATASET_STATISTICAL_PROFILE_DF[
            "dataset_id"
        ] == dataset_id
    ].iloc[0]


    if int(
        row[
            "pearson_pairs"
        ]
    ) != EXPECTED_PEARSON_PAIRS[
        dataset_id
    ]:

        raise RuntimeError(
            f"{dataset_id}: Pearson pair count mismatch."
        )


    if int(
        row[
            "spearman_pairs"
        ]
    ) != EXPECTED_SPEARMAN_PAIRS[
        dataset_id
    ]:

        raise RuntimeError(
            f"{dataset_id}: Spearman pair count mismatch."
        )


    if int(
        row[
            "categorical_dependency_pairs"
        ]
    ) != EXPECTED_CATEGORICAL_DEPENDENCY_PAIRS[
        dataset_id
    ]:

        raise RuntimeError(
            f"{dataset_id}: categorical dependency pair count mismatch."
        )


# ==============================================================================
# 15.18 — TARGET CONSISTENCY VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    schema = _get_single_dataset_schema(
        dataset_id
    )

    expected_target = (
        schema[
            "modeling_schema"
        ][
            "target_column"
        ]
    )

    observed_target = (
        DATASET_STATISTICAL_PROFILE_DF[
            DATASET_STATISTICAL_PROFILE_DF[
                "dataset_id"
            ] == dataset_id
        ][
            "target"
        ]
        .iloc[0]
    )

    if observed_target != expected_target:

        raise RuntimeError(
            f"{dataset_id}: target mismatch between "
            f"Section 15 and frozen Notebook 02 schema."
        )


# ==============================================================================
# 15.19 — TRAINING-ONLY VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    expected_rows = len(
        TRAIN_STATISTICAL_DATASETS[
            dataset_id
        ]
    )

    observed_rows = int(
        DATASET_STATISTICAL_PROFILE_DF[
            DATASET_STATISTICAL_PROFILE_DF[
                "dataset_id"
            ] == dataset_id
        ][
            "training_rows"
        ]
        .iloc[0]
    )

    if observed_rows != expected_rows:

        raise RuntimeError(
            f"{dataset_id}: Section 15 training row count "
            f"does not match the training-only dataset."
        )


# ==============================================================================
# 15.20 — FINAL VALIDATION CHECKS
# ==============================================================================

FINAL_CHECKS = {

    "dataset_count":
        len(DATASET_STATISTICAL_PROFILE_DF)
        == len(DATASET_IDS),

    "dataset_coverage":
        set(
            DATASET_STATISTICAL_PROFILE_DF[
                "dataset_id"
            ]
        )
        == set(DATASET_IDS),

    "no_duplicate_datasets":
        duplicate_dataset_count == 0,

    "training_rows_valid":
        all(
            int(
                DATASET_STATISTICAL_PROFILE_DF[
                    DATASET_STATISTICAL_PROFILE_DF[
                        "dataset_id"
                    ] == dataset_id
                ][
                    "training_rows"
                ].iloc[0]
            )
            ==
            len(
                TRAIN_STATISTICAL_DATASETS[
                    dataset_id
                ]
            )
            for dataset_id in DATASET_IDS
        ),

    "feature_counts_valid":
        all(
            int(
                DATASET_STATISTICAL_PROFILE_DF[
                    DATASET_STATISTICAL_PROFILE_DF[
                        "dataset_id"
                    ] == dataset_id
                ][
                    "modeling_features"
                ].iloc[0]
            )
            ==
            EXPECTED_PREPROCESSING_COUNTS[
                dataset_id
            ]
            for dataset_id in DATASET_IDS
        ),

    "numeric_counts_valid":
        all(
            int(
                DATASET_STATISTICAL_PROFILE_DF[
                    DATASET_STATISTICAL_PROFILE_DF[
                        "dataset_id"
                    ] == dataset_id
                ][
                    "numeric_features"
                ].iloc[0]
            )
            ==
            EXPECTED_NUMERIC_COUNTS[
                dataset_id
            ]
            for dataset_id in DATASET_IDS
        ),

    "categorical_counts_valid":
        all(
            int(
                DATASET_STATISTICAL_PROFILE_DF[
                    DATASET_STATISTICAL_PROFILE_DF[
                        "dataset_id"
                    ] == dataset_id
                ][
                    "categorical_features"
                ].iloc[0]
            )
            ==
            EXPECTED_CATEGORICAL_COUNTS[
                dataset_id
            ]
            for dataset_id in DATASET_IDS
        ),

    "pearson_pairs_valid":
        all(
            int(
                DATASET_STATISTICAL_PROFILE_DF[
                    DATASET_STATISTICAL_PROFILE_DF[
                        "dataset_id"
                    ] == dataset_id
                ][
                    "pearson_pairs"
                ].iloc[0]
            )
            ==
            EXPECTED_PEARSON_PAIRS[
                dataset_id
            ]
            for dataset_id in DATASET_IDS
        ),

    "spearman_pairs_valid":
        all(
            int(
                DATASET_STATISTICAL_PROFILE_DF[
                    DATASET_STATISTICAL_PROFILE_DF[
                        "dataset_id"
                    ] == dataset_id
                ][
                    "spearman_pairs"
                ].iloc[0]
            )
            ==
            EXPECTED_SPEARMAN_PAIRS[
                dataset_id
            ]
            for dataset_id in DATASET_IDS
        ),

    "categorical_dependency_pairs_valid":
        all(
            int(
                DATASET_STATISTICAL_PROFILE_DF[
                    DATASET_STATISTICAL_PROFILE_DF[
                        "dataset_id"
                    ] == dataset_id
                ][
                    "categorical_dependency_pairs"
                ].iloc[0]
            )
            ==
            EXPECTED_CATEGORICAL_DEPENDENCY_PAIRS[
                dataset_id
            ]
            for dataset_id in DATASET_IDS
        ),

    "missingness_valid":
        DATASET_STATISTICAL_PROFILE_DF[
            "maximum_feature_missing_rate"
        ]
        .between(0, 1)
        .all(),

    "pearson_valid":
        DATASET_STATISTICAL_PROFILE_DF[
            "maximum_absolute_pearson"
        ]
        .dropna()
        .between(0, 1)
        .all(),

    "spearman_valid":
        DATASET_STATISTICAL_PROFILE_DF[
            "maximum_absolute_spearman"
        ]
        .dropna()
        .between(0, 1)
        .all(),

    "cramers_v_valid":
        DATASET_STATISTICAL_PROFILE_DF[
            "maximum_categorical_cramers_v"
        ]
        .dropna()
        .between(0, 1)
        .all(),

}


# ==============================================================================
# 15.21 — PRINT FINAL VALIDATION
# ==============================================================================

print()
print("=" * 100)
print("SECTION 15 FINAL VALIDATION")
print("=" * 100)

for (
    check_name,
    check_value
) in FINAL_CHECKS.items():

    status = (
        "PASS"
        if bool(check_value)
        else "FAIL"
    )

    print(
        f"{status:<6} | {check_name}"
    )


if not all(
    FINAL_CHECKS.values()
):

    failed_checks = [

        name

        for name, value
        in FINAL_CHECKS.items()

        if not bool(value)

    ]

    raise RuntimeError(
        "SECTION 15 FINAL VALIDATION FAILED.\n"
        f"Failed checks: {failed_checks}"
    )


# ==============================================================================
# 15.22 — DISPLAY FINAL DATASET PROFILES
# ==============================================================================

print()
print("=" * 100)
print("DATASET STATISTICAL PROFILES")
print("=" * 100)

print(
    DATASET_STATISTICAL_PROFILE_DF.to_string(
        index=False
    )
)


# ==============================================================================
# 15.23 — FINAL COMPLETION MESSAGE
# ==============================================================================

print()
print(
    "✓ DATASET_STATISTICAL_PROFILE_DF created successfully."
)

print(
    "✓ Dataset-level statistics are aggregated from validated upstream sections."
)

print(
    "✓ No feature-level statistics were recomputed."
)

print(
    "✓ Training-only data were used."
)

print(
    "✓ Frozen Notebook 02 schema was enforced."
)

print(
    "✓ Section 14 feature profiles were validated."
)

print(
    "✓ Pearson dependencies were aggregated from Section 11."
)

print(
    "✓ Spearman dependencies were aggregated from Section 12."
)

print(
    "✓ Categorical dependencies were aggregated from Section 13."
)

print(
    "✓ Target, identifiers, and provenance were excluded from feature profiles."
)

print()
print("=" * 100)
print("SECTION 15 STATUS: PASS")
print("=" * 100)

SECTION 15 — DATASET STATISTICAL PROFILES
✓ Required upstream objects available.
✓ Section 14 feature profiles validated: 77

Processing adult_income
✓ Completed adult_income
  Training rows       : 34189
  Preprocessing vars  : 14
  Numeric variables   : 6
  Categorical vars    : 8
  Pearson pairs       : 15
  Spearman pairs      : 15
  Cramér's V pairs    : 28

Processing bank_marketing
✓ Completed bank_marketing
  Training rows       : 31647
  Preprocessing vars  : 16
  Numeric variables   : 7
  Categorical vars    : 9
  Pearson pairs       : 21
  Spearman pairs      : 21
  Cramér's V pairs    : 36

Processing diabetes_130us
✓ Completed diabetes_130us
  Training rows       : 71236
  Preprocessing vars  : 47
  Numeric variables   : 11
  Categorical vars    : 36
  Pearson pairs       : 55
  Spearman pairs      : 55
  Cramér's V pairs    : 630

SECTION 15 FINAL VALIDATION
PASS   | dataset_count
PASS   | dataset_coverage
PASS   | no_duplicate_datasets
PASS   | training_rows_valid
PASS  

In [16]:
# ==============================================================================
# SECTION 16 — BUILD SPP-GAN STATISTICAL REFERENCE
# ==============================================================================

print("=" * 100)
print("SECTION 16 — BUILD SPP-GAN STATISTICAL REFERENCE")
print("=" * 100)


# ==============================================================================
# 16.1 — REQUIRED UPSTREAM OBJECTS
# ==============================================================================

REQUIRED_UPSTREAM_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_STATISTICAL_DATASETS",
    "NB02_PERSISTED_SCHEMA_CONTRACTS",
    "FEATURE_STATISTICAL_PROFILE_DF",
    "DATASET_STATISTICAL_PROFILE_DF",
    "PEARSON_MATRICES",
    "PEARSON_DF",
    "SPEARMAN_MATRICES",
    "SPEARMAN_DF",
    "CATEGORICAL_DEPENDENCY_MATRICES",
    "CATEGORICAL_DEPENDENCY_DF",
]

for object_name in REQUIRED_UPSTREAM_OBJECTS:

    if object_name not in globals():

        raise RuntimeError(
            f"Section 16 requires upstream object "
            f"'{object_name}', but it was not found.\n"
            f"Run the required upstream sections before Section 16."
        )

print("✓ Required upstream objects available.")


# ==============================================================================
# 16.2 — REFERENCE CONFIGURATION
# ==============================================================================

REFERENCE_VERSION = "1.0"
REFERENCE_SEED = 2025


# ==============================================================================
# 16.3 — HELPER
# ==============================================================================

def _section16_get_schema(dataset_id):

    if dataset_id not in NB02_PERSISTED_SCHEMA_CONTRACTS:

        raise RuntimeError(
            f"{dataset_id}: persisted Notebook 02 schema "
            f"contract not found."
        )

    return NB02_PERSISTED_SCHEMA_CONTRACTS[
        dataset_id
    ]


# ==============================================================================
# 16.4 — INITIALIZE REFERENCE CONTAINER
# ==============================================================================

SPP_GAN_STATISTICAL_REFERENCE = {}


# ==============================================================================
# 16.5 — BUILD DATASET REFERENCES
# ==============================================================================

for dataset_id in DATASET_IDS:

    print()
    print(f"Building reference: {dataset_id}")


    # ==========================================================================
    # 16.5.1 — LOAD FROZEN NOTEBOOK 02 SCHEMA
    # ==========================================================================

    schema_contract = _section16_get_schema(
        dataset_id
    )

    modeling_schema = schema_contract[
        "modeling_schema"
    ]

    preprocessing_columns = list(
        modeling_schema[
            "preprocessing_columns"
        ]
    )

    all_columns = list(
        modeling_schema[
            "all_columns"
        ]
    )

    numeric_columns = list(
        modeling_schema[
            "numeric_columns"
        ]
    )

    categorical_columns = list(
        modeling_schema[
            "categorical_columns"
        ]
    )

    generative_columns = list(
        modeling_schema[
            "generative_columns"
        ]
    )

    target_column = modeling_schema[
        "target_column"
    ]

    identifier_columns = list(
        modeling_schema.get(
            "identifier_columns_excluded",
            []
        )
    )

    provenance_column = modeling_schema.get(
        "provenance_column",
        "__original_row_id__"
    )


    # ==========================================================================
    # 16.5.2 — LOAD TRAINING DATASET
    # ==========================================================================

    if dataset_id not in TRAIN_STATISTICAL_DATASETS:

        raise RuntimeError(
            f"{dataset_id}: training dataset not available."
        )

    train_df = TRAIN_STATISTICAL_DATASETS[
        dataset_id
    ]

    training_rows = len(
        train_df
    )


    # ==========================================================================
    # 16.5.3 — VALIDATE FROZEN SCHEMA HIERARCHY
    # ==========================================================================

    if all_columns != preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: Notebook 02 schema inconsistency.\n"
            f"'all_columns' must equal 'preprocessing_columns'."
        )


    expected_generative_columns = (
        preprocessing_columns
        + [target_column]
    )

    if generative_columns != (
        expected_generative_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: generative schema mismatch."
        )


    if target_column in preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            f"must not be a preprocessing feature."
        )


    if target_column in numeric_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            f"appears in numeric preprocessing features."
        )


    if target_column in categorical_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            f"appears in categorical preprocessing features."
        )


    if provenance_column in preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column appears "
            f"in preprocessing features."
        )


    if set(identifier_columns).intersection(
        set(preprocessing_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: identifier column appears "
            f"in preprocessing features."
        )


    if set(numeric_columns).intersection(
        set(categorical_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: numeric/categorical feature "
            f"sets overlap."
        )


    if set(numeric_columns).union(
        set(categorical_columns)
    ) != set(preprocessing_columns):

        raise RuntimeError(
            f"{dataset_id}: numeric + categorical features "
            f"do not cover the preprocessing feature universe."
        )


    # ==========================================================================
    # 16.5.4 — VALIDATE ACTUAL NATIVE DATASET SCHEMA
    # ==========================================================================

    expected_native_columns = (
        [provenance_column]
        + generative_columns
    )

    if list(train_df.columns) != (
        expected_native_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: training dataframe schema does not "
            f"match frozen Notebook 02 native schema.\n"
            f"Expected:\n{expected_native_columns}\n"
            f"Observed:\n{list(train_df.columns)}"
        )


    # ==========================================================================
    # 16.5.5 — LOAD SECTION 14 FEATURE PROFILES
    # ==========================================================================

    feature_profiles = (
        FEATURE_STATISTICAL_PROFILE_DF[
            FEATURE_STATISTICAL_PROFILE_DF[
                "dataset_id"
            ] == dataset_id
        ]
        .copy()
    )


    if len(feature_profiles) != (
        len(preprocessing_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: Section 14 feature profile count mismatch.\n"
            f"Expected {len(preprocessing_columns)}, "
            f"found {len(feature_profiles)}."
        )


    observed_feature_order = (
        feature_profiles
        .sort_values(
            "feature_order"
        )["feature"]
        .tolist()
    )

    if observed_feature_order != (
        preprocessing_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: Section 14 feature order does not "
            f"match Notebook 02 preprocessing order."
        )


    if feature_profiles[
        "is_target"
    ].any():

        raise RuntimeError(
            f"{dataset_id}: target appears in Section 14 "
            f"feature profiles."
        )


    if not feature_profiles[
        "is_preprocessing_feature"
    ].all():

        raise RuntimeError(
            f"{dataset_id}: Section 14 contains non-preprocessing "
            f"features."
        )


    # ==========================================================================
    # 16.5.6 — VALIDATE FEATURE TYPES
    # ==========================================================================

    observed_numeric = (
        feature_profiles[
            feature_profiles[
                "feature_type"
            ] == "numeric"
        ]
        .sort_values(
            "feature_order"
        )["feature"]
        .tolist()
    )

    observed_categorical = (
        feature_profiles[
            feature_profiles[
                "feature_type"
            ] == "categorical"
        ]
        .sort_values(
            "feature_order"
        )["feature"]
        .tolist()
    )


    if set(observed_numeric) != (
        set(numeric_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: numeric feature identity mismatch."
        )


    if set(observed_categorical) != (
        set(categorical_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: categorical feature identity mismatch."
        )


    # ==========================================================================
    # 16.5.7 — LOAD DATASET PROFILE
    # ==========================================================================

    dataset_profile_rows = (
        DATASET_STATISTICAL_PROFILE_DF[
            DATASET_STATISTICAL_PROFILE_DF[
                "dataset_id"
            ] == dataset_id
        ]
    )


    if len(dataset_profile_rows) != 1:

        raise RuntimeError(
            f"{dataset_id}: expected exactly one Section 15 "
            f"dataset statistical profile."
        )


    dataset_profile = (
        dataset_profile_rows
        .iloc[0]
        .to_dict()
    )


    if int(
        dataset_profile[
            "training_rows"
        ]
    ) != training_rows:

        raise RuntimeError(
            f"{dataset_id}: Section 15 training row count "
            f"does not match training dataset."
        )


    # ==========================================================================
    # 16.5.8 — LOAD PEARSON MATRIX
    # ==========================================================================

    if dataset_id not in PEARSON_MATRICES:

        raise RuntimeError(
            f"{dataset_id}: Pearson matrix not available."
        )

    pearson_matrix = (
        PEARSON_MATRICES[
            dataset_id
        ]
        .copy()
    )


    if list(
        pearson_matrix.index
    ) != numeric_columns:

        raise RuntimeError(
            f"{dataset_id}: Pearson matrix row order does not "
            f"match frozen numeric feature order."
        )


    if list(
        pearson_matrix.columns
    ) != numeric_columns:

        raise RuntimeError(
            f"{dataset_id}: Pearson matrix column order does not "
            f"match frozen numeric feature order."
        )


    if pearson_matrix.shape != (
        len(numeric_columns),
        len(numeric_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: Pearson matrix dimension mismatch."
        )


    if not np.allclose(
        pearson_matrix.fillna(0).values,
        pearson_matrix.fillna(0).values.T,
        atol=1e-12
    ):

        raise RuntimeError(
            f"{dataset_id}: Pearson matrix is not symmetric."
        )


    # ==========================================================================
    # 16.5.9 — LOAD SPEARMAN MATRIX
    # ==========================================================================

    if dataset_id not in SPEARMAN_MATRICES:

        raise RuntimeError(
            f"{dataset_id}: Spearman matrix not available."
        )

    spearman_matrix = (
        SPEARMAN_MATRICES[
            dataset_id
        ]
        .copy()
    )


    if list(
        spearman_matrix.index
    ) != numeric_columns:

        raise RuntimeError(
            f"{dataset_id}: Spearman matrix row order does not "
            f"match frozen numeric feature order."
        )


    if list(
        spearman_matrix.columns
    ) != numeric_columns:

        raise RuntimeError(
            f"{dataset_id}: Spearman matrix column order does not "
            f"match frozen numeric feature order."
        )


    if spearman_matrix.shape != (
        len(numeric_columns),
        len(numeric_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: Spearman matrix dimension mismatch."
        )


    if not np.allclose(
        spearman_matrix.fillna(0).values,
        spearman_matrix.fillna(0).values.T,
        atol=1e-12
    ):

        raise RuntimeError(
            f"{dataset_id}: Spearman matrix is not symmetric."
        )


    # ==========================================================================
    # 16.5.10 — LOAD CATEGORICAL DEPENDENCY MATRIX
    # ==========================================================================

    if dataset_id not in (
        CATEGORICAL_DEPENDENCY_MATRICES
    ):

        raise RuntimeError(
            f"{dataset_id}: categorical dependency matrix "
            f"not available."
        )

    categorical_matrix = (
        CATEGORICAL_DEPENDENCY_MATRICES[
            dataset_id
        ]
        .copy()
    )


    if list(
        categorical_matrix.index
    ) != categorical_columns:

        raise RuntimeError(
            f"{dataset_id}: categorical dependency matrix "
            f"row order does not match frozen categorical order."
        )


    if list(
        categorical_matrix.columns
    ) != categorical_columns:

        raise RuntimeError(
            f"{dataset_id}: categorical dependency matrix "
            f"column order does not match frozen categorical order."
        )


    if categorical_matrix.shape != (
        len(categorical_columns),
        len(categorical_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: categorical dependency matrix "
            f"dimension mismatch."
        )


    if not np.allclose(
        categorical_matrix.fillna(0).values,
        categorical_matrix.fillna(0).values.T,
        atol=1e-12
    ):

        raise RuntimeError(
            f"{dataset_id}: categorical dependency matrix "
            f"is not symmetric."
        )


    # ==========================================================================
    # 16.5.11 — SERIALIZE FEATURE PROFILES
    # ==========================================================================

    feature_profiles_records = (
        feature_profiles
        .replace(
            {np.nan: None}
        )
        .to_dict(
            orient="records"
        )
    )


    # ==========================================================================
    # 16.5.12 — SERIALIZE MATRICES
    # ==========================================================================

    pearson_serialized = (
        pearson_matrix
        .replace(
            {np.nan: None}
        )
        .to_dict()
    )


    spearman_serialized = (
        spearman_matrix
        .replace(
            {np.nan: None}
        )
        .to_dict()
    )


    categorical_serialized = (
        categorical_matrix
        .replace(
            {np.nan: None}
        )
        .to_dict()
    )


    # ==========================================================================
    # 16.5.13 — BUILD REFERENCE OBJECT
    # ==========================================================================

    reference = {

        "reference_version":
            REFERENCE_VERSION,

        "reference_type":
            "SPP-GAN_statistical_reference",

        "dataset_id":
            dataset_id,

        "creation_timestamp_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "random_seed":
            REFERENCE_SEED,

        "fit_policy": {

            "source_split":
                "train_only",

            "validation_used_for_reference":
                False,

            "test_used_for_reference":
                False,

            "synthetic_data_used_for_reference":
                False,

            "reference_statistics_recomputed":
                False,

        },

        "schema_policy": {

            "frozen_notebook":
                "Notebook_02",

            "schema_status":
                "frozen",

            "target_retained_in_generative_schema":
                True,

            "target_excluded_from_preprocessing":
                True,

            "target_excluded_from_transformed_features":
                True,

            "identifiers_excluded_from_generative_data":
                True,

            "provenance_retained_for_audit":
                True,

        },

        "dataset_profile":
            dataset_profile,

        "feature_schema": {

            "all_preprocessing_features":
                preprocessing_columns,

            "numeric_features":
                numeric_columns,

            "categorical_features":
                categorical_columns,

            "generative_features":
                generative_columns,

            "target_column":
                target_column,

            "identifier_columns_excluded":
                identifier_columns,

            "provenance_column":
                provenance_column,

        },

        "feature_profiles":
            feature_profiles_records,

        "pearson_correlation": {

            "feature_order":
                numeric_columns,

            "matrix":
                pearson_serialized,

        },

        "spearman_correlation": {

            "feature_order":
                numeric_columns,

            "matrix":
                spearman_serialized,

        },

        "categorical_dependency": {

            "feature_order":
                categorical_columns,

            "matrix":
                categorical_serialized,

        },

    }


    # ==========================================================================
    # 16.5.14 — STORE REFERENCE
    # ==========================================================================

    SPP_GAN_STATISTICAL_REFERENCE[
        dataset_id
    ] = reference


    print(
        f"✓ Reference created: {dataset_id}"
    )

    print(
        f"  Training rows      : {training_rows}"
    )

    print(
        f"  Preprocessing vars : {len(preprocessing_columns)}"
    )

    print(
        f"  Numeric features   : {len(numeric_columns)}"
    )

    print(
        f"  Categorical vars   : {len(categorical_columns)}"
    )

    print(
        f"  Target             : {target_column}"
    )


# ==============================================================================
# 16.6 — GLOBAL REFERENCE VALIDATION
# ==============================================================================

print()
print("=" * 100)
print("SECTION 16 FINAL VALIDATION")
print("=" * 100)


# ------------------------------------------------------------------------------
# Dataset count
# ------------------------------------------------------------------------------

if len(
    SPP_GAN_STATISTICAL_REFERENCE
) != len(DATASET_IDS):

    raise RuntimeError(
        "Reference dataset count mismatch."
    )

print(
    "PASS   | reference_dataset_count"
)


# ------------------------------------------------------------------------------
# Dataset coverage
# ------------------------------------------------------------------------------

if set(
    SPP_GAN_STATISTICAL_REFERENCE.keys()
) != set(DATASET_IDS):

    raise RuntimeError(
        "Reference dataset coverage mismatch."
    )

print(
    "PASS   | reference_dataset_coverage"
)


# ------------------------------------------------------------------------------
# Validate every reference
# ------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    reference = (
        SPP_GAN_STATISTICAL_REFERENCE[
            dataset_id
        ]
    )

    schema = _section16_get_schema(
        dataset_id
    )

    modeling_schema = schema[
        "modeling_schema"
    ]

    preprocessing_columns = list(
        modeling_schema[
            "preprocessing_columns"
        ]
    )

    numeric_columns = list(
        modeling_schema[
            "numeric_columns"
        ]
    )

    categorical_columns = list(
        modeling_schema[
            "categorical_columns"
        ]
    )

    generative_columns = list(
        modeling_schema[
            "generative_columns"
        ]
    )

    target_column = modeling_schema[
        "target_column"
    ]

    provenance_column = modeling_schema.get(
        "provenance_column",
        "__original_row_id__"
    )


    # --------------------------------------------------------------------------
    # Version
    # --------------------------------------------------------------------------

    if reference[
        "reference_version"
    ] != REFERENCE_VERSION:

        raise RuntimeError(
            f"{dataset_id}: reference version mismatch."
        )


    # --------------------------------------------------------------------------
    # Fit policy
    # --------------------------------------------------------------------------

    fit_policy = reference[
        "fit_policy"
    ]

    if fit_policy[
        "source_split"
    ] != "train_only":

        raise RuntimeError(
            f"{dataset_id}: reference is not marked train-only."
        )

    if fit_policy[
        "validation_used_for_reference"
    ]:

        raise RuntimeError(
            f"{dataset_id}: validation data incorrectly "
            f"marked as used."
        )

    if fit_policy[
        "test_used_for_reference"
    ]:

        raise RuntimeError(
            f"{dataset_id}: test data incorrectly "
            f"marked as used."
        )

    if fit_policy[
        "synthetic_data_used_for_reference"
    ]:

        raise RuntimeError(
            f"{dataset_id}: synthetic data incorrectly "
            f"marked as used."
        )


    # --------------------------------------------------------------------------
    # Feature schema
    # --------------------------------------------------------------------------

    feature_schema = reference[
        "feature_schema"
    ]

    if feature_schema[
        "all_preprocessing_features"
    ] != preprocessing_columns:

        raise RuntimeError(
            f"{dataset_id}: reference preprocessing "
            f"feature order mismatch."
        )


    if feature_schema[
        "numeric_features"
    ] != numeric_columns:

        raise RuntimeError(
            f"{dataset_id}: reference numeric "
            f"feature order mismatch."
        )


    if feature_schema[
        "categorical_features"
    ] != categorical_columns:

        raise RuntimeError(
            f"{dataset_id}: reference categorical "
            f"feature order mismatch."
        )


    if feature_schema[
        "generative_features"
    ] != generative_columns:

        raise RuntimeError(
            f"{dataset_id}: reference generative "
            f"feature order mismatch."
        )


    if feature_schema[
        "target_column"
    ] != target_column:

        raise RuntimeError(
            f"{dataset_id}: reference target mismatch."
        )


    if feature_schema[
        "provenance_column"
    ] != provenance_column:

        raise RuntimeError(
            f"{dataset_id}: reference provenance mismatch."
        )


    # --------------------------------------------------------------------------
    # Feature profile count
    # --------------------------------------------------------------------------

    if len(
        reference[
            "feature_profiles"
        ]
    ) != len(preprocessing_columns):

        raise RuntimeError(
            f"{dataset_id}: reference feature profile count mismatch."
        )


    # --------------------------------------------------------------------------
    # Pearson matrix
    # --------------------------------------------------------------------------

    pearson_feature_order = reference[
        "pearson_correlation"
    ][
        "feature_order"
    ]

    if pearson_feature_order != numeric_columns:

        raise RuntimeError(
            f"{dataset_id}: reference Pearson feature order mismatch."
        )


    # --------------------------------------------------------------------------
    # Spearman matrix
    # --------------------------------------------------------------------------

    spearman_feature_order = reference[
        "spearman_correlation"
    ][
        "feature_order"
    ]

    if spearman_feature_order != numeric_columns:

        raise RuntimeError(
            f"{dataset_id}: reference Spearman feature order mismatch."
        )


    # --------------------------------------------------------------------------
    # Categorical dependency matrix
    # --------------------------------------------------------------------------

    categorical_feature_order = reference[
        "categorical_dependency"
    ][
        "feature_order"
    ]

    if categorical_feature_order != categorical_columns:

        raise RuntimeError(
            f"{dataset_id}: reference categorical dependency "
            f"feature order mismatch."
        )


    print(
        f"PASS   | {dataset_id}_reference_integrity"
    )


# ==============================================================================
# 16.7 — FINAL SUMMARY
# ==============================================================================

print()
print("=" * 100)
print("SECTION 16 SUMMARY")
print("=" * 100)

print(
    f"Reference version           : {REFERENCE_VERSION}"
)

print(
    f"Reference seed              : {REFERENCE_SEED}"
)

print(
    f"Datasets                    : "
    f"{len(SPP_GAN_STATISTICAL_REFERENCE)}"
)

print(
    "Source split                : train only"
)

print(
    "Validation used             : False"
)

print(
    "Test used                   : False"
)

print(
    "Synthetic data used         : False"
)

print(
    "Statistics recomputed       : False"
)

print(
    "Frozen Notebook 02 schema   : enforced"
)

print(
    "Feature-level profiles     : Section 14"
)

print(
    "Dataset-level profiles     : Section 15"
)

print(
    "Pearson source              : Section 11"
)

print(
    "Spearman source             : Section 12"
)

print(
    "Categorical dependency      : Section 13"
)

print(
    "Target included in profile : False"
)

print(
    "Target retained in schema  : True"
)

print(
    "Provenance retained        : True"
)


# ==============================================================================
# 16.8 — FINAL STATUS
# ==============================================================================

print()
print(
    "✓ SPP_GAN_STATISTICAL_REFERENCE created successfully."
)

print(
    "✓ Reference is derived exclusively from validated training-only statistics."
)

print(
    "✓ Frozen Notebook 02 schema is enforced."
)

print(
    "✓ Feature order is preserved."
)

print(
    "✓ Pearson, Spearman, and Cramér's V matrices are preserved."
)

print(
    "✓ No validation, test, or synthetic data were used."
)

print(
    "✓ No feature statistics were recomputed."
)

print()
print("=" * 100)
print("SECTION 16 STATUS: PASS")
print("=" * 100)

SECTION 16 — BUILD SPP-GAN STATISTICAL REFERENCE
✓ Required upstream objects available.

Building reference: adult_income
✓ Reference created: adult_income
  Training rows      : 34189
  Preprocessing vars : 14
  Numeric features   : 6
  Categorical vars   : 8
  Target             : income

Building reference: bank_marketing
✓ Reference created: bank_marketing
  Training rows      : 31647
  Preprocessing vars : 16
  Numeric features   : 7
  Categorical vars   : 9
  Target             : y

Building reference: diabetes_130us
✓ Reference created: diabetes_130us
  Training rows      : 71236
  Preprocessing vars : 47
  Numeric features   : 11
  Categorical vars   : 36
  Target             : readmitted

SECTION 16 FINAL VALIDATION
PASS   | reference_dataset_count
PASS   | reference_dataset_coverage
PASS   | adult_income_reference_integrity
PASS   | bank_marketing_reference_integrity
PASS   | diabetes_130us_reference_integrity

SECTION 16 SUMMARY
Reference version           : 1.0
Reference se

In [17]:
# ==============================================================================
# SECTION 17 — GENERATE STATISTICAL GUIDANCE ARTIFACTS
# ==============================================================================
#
# PURPOSE
# -------
# Convert the frozen SPP-GAN statistical reference into a compact,
# machine-readable guidance artifact for downstream SPP-GAN components.
#
# IMPORTANT DESIGN PRINCIPLES
# ---------------------------
# 1. No statistical quantities are recomputed here.
# 2. Section 16 is the authoritative source for dataset/schema/feature profiles.
# 3. Sections 11–13 remain the authoritative dependency sources.
# 4. No dependency on obsolete FEATURE_STATISTICAL_PROFILE_DF.
# 5. No dependency on in-memory TARGET_COLUMNS / IDENTIFIER_COLUMNS /
#    PROVENANCE_COLUMN.
# 6. Target remains part of the generative schema but is excluded from
#    preprocessing features.
# 7. Provenance and identifiers are never treated as model features.
# 8. Guidance is derived only from training-data evidence.
#
# OUTPUT
# ------
# SPP_GAN_STATISTICAL_GUIDANCE
#
# ==============================================================================

print("=" * 100)
print("SECTION 17 — GENERATE STATISTICAL GUIDANCE ARTIFACTS")
print("=" * 100)


# ==============================================================================
# 17.1 — REQUIRED UPSTREAM OBJECTS
# ==============================================================================

_REQUIRED_OBJECTS = [
    "DATASET_IDS",
    "SPP_GAN_STATISTICAL_REFERENCE",
    "PEARSON_DF",
    "SPEARMAN_DF",
    "CATEGORICAL_DEPENDENCY_DF",
]

_missing_objects = [
    name
    for name in _REQUIRED_OBJECTS
    if name not in globals()
]

if _missing_objects:
    raise RuntimeError(
        "SECTION 17 CANNOT START.\n"
        f"Missing required upstream objects: {_missing_objects}"
    )

print("✓ Required upstream objects available.")


# ==============================================================================
# 17.2 — BASIC REFERENCE VALIDATION
# ==============================================================================

if not isinstance(SPP_GAN_STATISTICAL_REFERENCE, dict):
    raise TypeError(
        "SPP_GAN_STATISTICAL_REFERENCE must be a dictionary."
    )

missing_reference_datasets = [
    dataset_id
    for dataset_id in DATASET_IDS
    if dataset_id not in SPP_GAN_STATISTICAL_REFERENCE
]

if missing_reference_datasets:
    raise RuntimeError(
        "Missing datasets in SPP_GAN_STATISTICAL_REFERENCE: "
        f"{missing_reference_datasets}"
    )


# ==============================================================================
# 17.3 — GUIDANCE CONTAINER
# ==============================================================================

SPP_GAN_STATISTICAL_GUIDANCE = {}


# ==============================================================================
# 17.4 — HELPER: SAFE PYTHON VALUE
# ==============================================================================

def _safe_value(value):

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    if isinstance(value, np.generic):
        return value.item()

    return value


# ==============================================================================
# 17.5 — BUILD GUIDANCE DATASET BY DATASET
# ==============================================================================

for dataset_id in DATASET_IDS:

    print(f"\nBuilding guidance: {dataset_id}")

    # --------------------------------------------------------------------------
    # Frozen Section 16 reference
    # --------------------------------------------------------------------------

    reference = SPP_GAN_STATISTICAL_REFERENCE[dataset_id]

    if not isinstance(reference, dict):
        raise TypeError(
            f"Invalid reference object for dataset '{dataset_id}'."
        )

    # --------------------------------------------------------------------------
    # Reference metadata
    # --------------------------------------------------------------------------

    reference_version = reference.get(
        "reference_version"
    )

    reference_type = reference.get(
        "reference_type"
    )

    source_split = reference.get(
        "source_split"
    )

    fit_policy = reference.get(
        "fit_policy",
        {}
    )

    schema_policy = reference.get(
        "schema_policy",
        {}
    )

    # --------------------------------------------------------------------------
    # Feature schema from frozen reference
    # --------------------------------------------------------------------------

    feature_schema = reference.get(
        "feature_schema",
        {}
    )

    preprocessing_features = list(
        feature_schema.get(
            "all_preprocessing_features",
            []
        )
    )

    numeric_features = list(
        feature_schema.get(
            "numeric_features",
            []
        )
    )

    categorical_features = list(
        feature_schema.get(
            "categorical_features",
            []
        )
    )

    generative_features = list(
        feature_schema.get(
            "generative_features",
            []
        )
    )

    target_column = feature_schema.get(
        "target_column"
    )

    identifier_columns_excluded = list(
        feature_schema.get(
            "identifier_columns_excluded",
            []
        )
    )

    provenance_column = feature_schema.get(
        "provenance_column"
    )

    # --------------------------------------------------------------------------
    # Feature profiles from Section 16
    # --------------------------------------------------------------------------

    feature_profiles = reference.get(
        "feature_profiles",
        []
    )

    if not isinstance(feature_profiles, list):
        raise TypeError(
            f"feature_profiles for '{dataset_id}' must be a list."
        )

    profiles_df = pd.DataFrame(
        feature_profiles
    )

    if profiles_df.empty:
        raise RuntimeError(
            f"No feature profiles found in frozen reference "
            f"for dataset '{dataset_id}'."
        )

    if "feature" not in profiles_df.columns:
        raise RuntimeError(
            f"Frozen reference for '{dataset_id}' does not contain "
            "'feature' in feature_profiles."
        )

    # --------------------------------------------------------------------------
    # Validate feature-profile coverage
    # --------------------------------------------------------------------------

    profile_features = profiles_df[
        "feature"
    ].tolist()

    if profile_features != preprocessing_features:
        raise RuntimeError(
            f"Feature-profile order mismatch for '{dataset_id}'.\n"
            f"Reference preprocessing features: {preprocessing_features}\n"
            f"Profile features: {profile_features}"
        )

    # --------------------------------------------------------------------------
    # Validate numeric/categorical partition
    # --------------------------------------------------------------------------

    if set(numeric_features).intersection(
        set(categorical_features)
    ):
        raise RuntimeError(
            f"Numeric/categorical feature overlap detected "
            f"for '{dataset_id}'."
        )

    if set(numeric_features).union(
        set(categorical_features)
    ) != set(preprocessing_features):
        raise RuntimeError(
            f"Numeric + categorical feature partition does not match "
            f"preprocessing features for '{dataset_id}'."
        )

    # --------------------------------------------------------------------------
    # Remove target if it somehow appears in guidance profiles
    #
    # The frozen architecture requires target exclusion from preprocessing
    # features. This is an explicit defensive check rather than recomputation.
    # --------------------------------------------------------------------------

    if target_column in preprocessing_features:
        raise RuntimeError(
            f"Target '{target_column}' incorrectly appears in preprocessing "
            f"features for '{dataset_id}'."
        )

    # --------------------------------------------------------------------------
    # Build numeric guidance
    # --------------------------------------------------------------------------

    numeric_guidance = []

    for feature in numeric_features:

        row = profiles_df[
            profiles_df["feature"] == feature
        ]

        if len(row) != 1:
            raise RuntimeError(
                f"Expected exactly one profile for numeric feature "
                f"'{feature}' in '{dataset_id}', found {len(row)}."
            )

        row = row.iloc[0]

        guidance = {
            "feature": feature,
            "feature_type": "numeric",

            "mean": _safe_value(
                row.get("mean")
            ),

            "std": _safe_value(
                row.get("std")
            ),

            "min": _safe_value(
                row.get("min")
            ),

            "q25": _safe_value(
                row.get("q25")
            ),

            "median": _safe_value(
                row.get("median")
            ),

            "q75": _safe_value(
                row.get("q75")
            ),

            "max": _safe_value(
                row.get("max")
            ),

            "skewness": _safe_value(
                row.get("skewness")
            ),

            "distribution_shape": _safe_value(
                row.get("distribution_shape")
            ),

            "missing_rate": _safe_value(
                row.get("missing_rate")
            ),
        }

        numeric_guidance.append(
            guidance
        )

    # --------------------------------------------------------------------------
    # Build categorical guidance
    # --------------------------------------------------------------------------

    categorical_guidance = []

    for feature in categorical_features:

        row = profiles_df[
            profiles_df["feature"] == feature
        ]

        if len(row) != 1:
            raise RuntimeError(
                f"Expected exactly one profile for categorical feature "
                f"'{feature}' in '{dataset_id}', found {len(row)}."
            )

        row = row.iloc[0]

        guidance = {
            "feature": feature,
            "feature_type": "categorical",

            "cardinality": _safe_value(
                row.get("cardinality")
            ),

            "normalized_entropy": _safe_value(
                row.get("normalized_entropy")
            ),

            "shannon_entropy_bits": _safe_value(
                row.get("shannon_entropy_bits")
            ),

            "top_category": _safe_value(
                row.get("top_category")
            ),

            "top_frequency": _safe_value(
                row.get("top_frequency")
            ),

            "missing_rate": _safe_value(
                row.get("missing_rate")
            ),

            "is_target": False,
        }

        categorical_guidance.append(
            guidance
        )

    # ==============================================================================
    # 17.6 — NUMERIC DEPENDENCY GUIDANCE
    # ==============================================================================

    pearson_dataset = PEARSON_DF[
        PEARSON_DF["dataset_id"] == dataset_id
    ].copy()

    spearman_dataset = SPEARMAN_DF[
        SPEARMAN_DF["dataset_id"] == dataset_id
    ].copy()

    # --------------------------------------------------------------------------
    # Defensive filtering
    # --------------------------------------------------------------------------

    pearson_dataset = pearson_dataset[
        pearson_dataset["feature_a"]
        !=
        pearson_dataset["feature_b"]
    ].copy()

    spearman_dataset = spearman_dataset[
        spearman_dataset["feature_a"]
        !=
        spearman_dataset["feature_b"]
    ].copy()

    # --------------------------------------------------------------------------
    # Validate dependency features against numeric feature universe
    # --------------------------------------------------------------------------

    for dependency_df, metric_name in [
        (pearson_dataset, "Pearson"),
        (spearman_dataset, "Spearman"),
    ]:

        dependency_features = set(
            dependency_df["feature_a"]
        ).union(
            set(
                dependency_df["feature_b"]
            )
        )

        unexpected = dependency_features.difference(
            set(numeric_features)
        )

        if unexpected:
            raise RuntimeError(
                f"{metric_name} dependency table for '{dataset_id}' "
                f"contains non-numeric/unexpected features: "
                f"{sorted(unexpected)}"
            )

    # --------------------------------------------------------------------------
    # Pearson strongest dependencies
    # --------------------------------------------------------------------------

    pearson_dataset["abs_value"] = (
        pearson_dataset["pearson_r"].abs()
    )

    strongest_pearson = (
        pearson_dataset
        .sort_values(
            "abs_value",
            ascending=False,
            kind="stable"
        )
        .drop_duplicates(
            subset=[
                "feature_a",
                "feature_b",
            ]
        )
        .head(20)
        .drop(
            columns=["abs_value"],
            errors="ignore"
        )
        .replace({np.nan: None})
        .to_dict(
            orient="records"
        )
    )

    # --------------------------------------------------------------------------
    # Spearman strongest dependencies
    # --------------------------------------------------------------------------

    spearman_dataset["abs_value"] = (
        spearman_dataset["spearman_rho"].abs()
    )

    strongest_spearman = (
        spearman_dataset
        .sort_values(
            "abs_value",
            ascending=False,
            kind="stable"
        )
        .drop_duplicates(
            subset=[
                "feature_a",
                "feature_b",
            ]
        )
        .head(20)
        .drop(
            columns=["abs_value"],
            errors="ignore"
        )
        .replace({np.nan: None})
        .to_dict(
            orient="records"
        )
    )

    # ==============================================================================
    # 17.7 — CATEGORICAL DEPENDENCY GUIDANCE
    # ==============================================================================

    categorical_dependency_dataset = (
        CATEGORICAL_DEPENDENCY_DF[
            CATEGORICAL_DEPENDENCY_DF["dataset_id"]
            ==
            dataset_id
        ].copy()
    )

    if not categorical_dependency_dataset.empty:

        dependency_features = set(
            categorical_dependency_dataset["feature_a"]
        ).union(
            set(
                categorical_dependency_dataset["feature_b"]
            )
        )

        unexpected = dependency_features.difference(
            set(categorical_features)
        )

        if unexpected:
            raise RuntimeError(
                "Categorical dependency table for "
                f"'{dataset_id}' contains unexpected features: "
                f"{sorted(unexpected)}"
            )

    strongest_categorical = (
        categorical_dependency_dataset
        .sort_values(
            "cramers_v",
            ascending=False,
            kind="stable"
        )
        .head(20)
        .replace({np.nan: None})
        .to_dict(
            orient="records"
        )
    )

    # ==============================================================================
    # 17.8 — BUILD FINAL GUIDANCE OBJECT
    # ==============================================================================

    SPP_GAN_STATISTICAL_GUIDANCE[
        dataset_id
    ] = {

        # ----------------------------------------------------------------------
        # Artifact metadata
        # ----------------------------------------------------------------------

        "guidance_version": "1.0",

        "dataset_id": dataset_id,

        "source_reference": (
            "SPP_GAN_STATISTICAL_REFERENCE"
        ),

        "source_reference_version": (
            reference_version
        ),

        "source_reference_type": (
            reference_type
        ),

        "source_split": (
            source_split
        ),

        # ----------------------------------------------------------------------
        # Frozen schema information
        # ----------------------------------------------------------------------

        "feature_schema": {

            "all_preprocessing_features": (
                preprocessing_features
            ),

            "numeric_features": (
                numeric_features
            ),

            "categorical_features": (
                categorical_features
            ),

            "generative_features": (
                generative_features
            ),

            "target_column": (
                target_column
            ),

            "identifier_columns_excluded": (
                identifier_columns_excluded
            ),

            "provenance_column": (
                provenance_column
            ),
        },

        # ----------------------------------------------------------------------
        # Statistical guidance
        # ----------------------------------------------------------------------

        "numeric_feature_guidance": (
            numeric_guidance
        ),

        "categorical_feature_guidance": (
            categorical_guidance
        ),

        # ----------------------------------------------------------------------
        # Dependency guidance
        # ----------------------------------------------------------------------

        "strongest_numeric_pearson_dependencies": (
            strongest_pearson
        ),

        "strongest_numeric_spearman_dependencies": (
            strongest_spearman
        ),

        "strongest_categorical_dependencies": (
            strongest_categorical
        ),

        # ----------------------------------------------------------------------
        # Target policy
        # ----------------------------------------------------------------------

        "target_policy": {

            "target": target_column,

            "target_in_preprocessing_features": False,

            "target_retained_in_generative_schema": (
                target_column in generative_features
            ),

            "target_is_modeling_guidance_feature": False,
        },

        # ----------------------------------------------------------------------
        # Identifier policy
        # ----------------------------------------------------------------------

        "identifier_policy": {

            "excluded": (
                identifier_columns_excluded
            ),

            "excluded_from_model_features": True,
        },

        # ----------------------------------------------------------------------
        # Provenance policy
        # ----------------------------------------------------------------------

        "provenance_policy": {

            "column": provenance_column,

            "excluded_from_model_features": True,
        },

        # ----------------------------------------------------------------------
        # Training-only evidence policy
        # ----------------------------------------------------------------------

        "evidence_policy": {

            "source_split": "train_only",

            "validation_used": False,

            "test_used": False,

            "synthetic_data_used": False,

            "statistics_recomputed": False,
        },
    }

    print(
        f"✓ Guidance created: {dataset_id}"
    )

    print(
        f"  Numeric guidance       : "
        f"{len(numeric_guidance)}"
    )

    print(
        f"  Categorical guidance   : "
        f"{len(categorical_guidance)}"
    )

    print(
        f"  Pearson dependencies   : "
        f"{len(strongest_pearson)}"
    )

    print(
        f"  Spearman dependencies  : "
        f"{len(strongest_spearman)}"
    )

    print(
        f"  Categorical dependencies: "
        f"{len(strongest_categorical)}"
    )


# ==============================================================================
# 17.9 — FINAL VALIDATION
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 17 FINAL VALIDATION")
print("=" * 100)

validation_results = []


def _record_validation(name, condition):

    status = "PASS" if condition else "FAIL"

    validation_results.append({
        "check": name,
        "status": status,
    })

    print(
        f"{status} | {name}"
    )

    if not condition:
        raise RuntimeError(
            f"Section 17 validation failed: {name}"
        )


# ------------------------------------------------------------------------------
# Dataset coverage
# ------------------------------------------------------------------------------

_record_validation(
    "guidance_dataset_count",
    len(SPP_GAN_STATISTICAL_GUIDANCE)
    ==
    len(DATASET_IDS)
)

_record_validation(
    "guidance_dataset_coverage",
    set(
        SPP_GAN_STATISTICAL_GUIDANCE.keys()
    )
    ==
    set(DATASET_IDS)
)


# ------------------------------------------------------------------------------
# Dataset-level integrity
# ------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    guidance = (
        SPP_GAN_STATISTICAL_GUIDANCE[
            dataset_id
        ]
    )

    reference = (
        SPP_GAN_STATISTICAL_REFERENCE[
            dataset_id
        ]
    )

    feature_schema = guidance[
        "feature_schema"
    ]

    preprocessing_features = feature_schema[
        "all_preprocessing_features"
    ]

    numeric_features = feature_schema[
        "numeric_features"
    ]

    categorical_features = feature_schema[
        "categorical_features"
    ]

    generative_features = feature_schema[
        "generative_features"
    ]

    target = feature_schema[
        "target_column"
    ]

    identifiers = feature_schema[
        "identifier_columns_excluded"
    ]

    provenance = feature_schema[
        "provenance_column"
    ]

    # --------------------------------------------------------------------------
    # Reference version consistency
    # --------------------------------------------------------------------------

    _record_validation(
        f"{dataset_id}_reference_version",
        guidance["source_reference_version"]
        ==
        reference.get("reference_version")
    )

    # --------------------------------------------------------------------------
    # Feature coverage
    # --------------------------------------------------------------------------

    _record_validation(
        f"{dataset_id}_numeric_guidance_coverage",
        [
            item["feature"]
            for item in guidance[
                "numeric_feature_guidance"
            ]
        ]
        ==
        numeric_features
    )

    _record_validation(
        f"{dataset_id}_categorical_guidance_coverage",
        [
            item["feature"]
            for item in guidance[
                "categorical_feature_guidance"
            ]
        ]
        ==
        categorical_features
    )

    # --------------------------------------------------------------------------
    # Schema partition
    # --------------------------------------------------------------------------

    _record_validation(
        f"{dataset_id}_feature_partition",
        set(numeric_features).union(
            set(categorical_features)
        )
        ==
        set(preprocessing_features)
    )

    _record_validation(
        f"{dataset_id}_feature_partition_disjoint",
        set(numeric_features).isdisjoint(
            set(categorical_features)
        )
    )

    # --------------------------------------------------------------------------
    # Target policy
    # --------------------------------------------------------------------------

    _record_validation(
        f"{dataset_id}_target_excluded_from_preprocessing",
        target not in preprocessing_features
    )

    _record_validation(
        f"{dataset_id}_target_retained_in_generative_schema",
        target in generative_features
    )

    # --------------------------------------------------------------------------
    # Identifier policy
    # --------------------------------------------------------------------------

    _record_validation(
        f"{dataset_id}_identifiers_excluded",
        set(identifiers).isdisjoint(
            set(generative_features)
        )
    )

    # --------------------------------------------------------------------------
    # Provenance policy
    # --------------------------------------------------------------------------

    _record_validation(
        f"{dataset_id}_provenance_excluded",
        provenance not in generative_features
    )

    # --------------------------------------------------------------------------
    # Evidence policy
    # --------------------------------------------------------------------------

    evidence_policy = guidance[
        "evidence_policy"
    ]

    _record_validation(
        f"{dataset_id}_train_only_evidence",
        evidence_policy["source_split"]
        ==
        "train_only"
        and
        evidence_policy["validation_used"] is False
        and
        evidence_policy["test_used"] is False
        and
        evidence_policy["synthetic_data_used"] is False
    )

    # --------------------------------------------------------------------------
    # Dependency feature integrity
    # --------------------------------------------------------------------------

    pearson_features = set()

    if guidance[
        "strongest_numeric_pearson_dependencies"
    ]:

        pearson_guidance_df = pd.DataFrame(
            guidance[
                "strongest_numeric_pearson_dependencies"
            ]
        )

        pearson_features = set(
            pearson_guidance_df["feature_a"]
        ).union(
            set(
                pearson_guidance_df["feature_b"]
            )
        )

    _record_validation(
        f"{dataset_id}_pearson_numeric_features",
        pearson_features.issubset(
            set(numeric_features)
        )
    )

    spearman_features = set()

    if guidance[
        "strongest_numeric_spearman_dependencies"
    ]:

        spearman_guidance_df = pd.DataFrame(
            guidance[
                "strongest_numeric_spearman_dependencies"
            ]
        )

        spearman_features = set(
            spearman_guidance_df["feature_a"]
        ).union(
            set(
                spearman_guidance_df["feature_b"]
            )
        )

    _record_validation(
        f"{dataset_id}_spearman_numeric_features",
        spearman_features.issubset(
            set(numeric_features)
        )
    )

    categorical_features_in_guidance = set()

    if guidance[
        "strongest_categorical_dependencies"
    ]:

        categorical_guidance_df = pd.DataFrame(
            guidance[
                "strongest_categorical_dependencies"
            ]
        )

        categorical_features_in_guidance = set(
            categorical_guidance_df["feature_a"]
        ).union(
            set(
                categorical_guidance_df["feature_b"]
            )
        )

    _record_validation(
        f"{dataset_id}_categorical_dependency_features",
        categorical_features_in_guidance.issubset(
            set(categorical_features)
        )
    )


# ==============================================================================
# 17.10 — FINAL SUMMARY
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 17 SUMMARY")
print("=" * 100)

print(
    f"Guidance artifacts : "
    f"{len(SPP_GAN_STATISTICAL_GUIDANCE)}"
)

print(
    "Source             : SPP_GAN_STATISTICAL_REFERENCE"
)

print(
    "Source version     : 1.0"
)

print(
    "Evidence split     : train only"
)

print(
    "Statistics recomputed : False"
)

print(
    "Target in preprocessing : False"
)

print(
    "Target in generative schema : True"
)

print(
    "Identifiers included : False"
)

print(
    "Provenance included as model feature : False"
)

print(
    "Numeric dependency source : Section 11"
)

print(
    "Spearman dependency source : Section 12"
)

print(
    "Categorical dependency source : Section 13"
)

print(
    "Feature profiles source : Section 14"
)

print(
    "Dataset/schema authority : Section 16"
)

print(
    "\nSECTION 17 STATUS: PASS"
)

SECTION 17 — GENERATE STATISTICAL GUIDANCE ARTIFACTS
✓ Required upstream objects available.

Building guidance: adult_income
✓ Guidance created: adult_income
  Numeric guidance       : 6
  Categorical guidance   : 8
  Pearson dependencies   : 15
  Spearman dependencies  : 15
  Categorical dependencies: 20

Building guidance: bank_marketing
✓ Guidance created: bank_marketing
  Numeric guidance       : 7
  Categorical guidance   : 9
  Pearson dependencies   : 20
  Spearman dependencies  : 20
  Categorical dependencies: 20

Building guidance: diabetes_130us
✓ Guidance created: diabetes_130us
  Numeric guidance       : 11
  Categorical guidance   : 36
  Pearson dependencies   : 20
  Spearman dependencies  : 20
  Categorical dependencies: 20

SECTION 17 FINAL VALIDATION
PASS | guidance_dataset_count
PASS | guidance_dataset_coverage
PASS | adult_income_reference_version
PASS | adult_income_numeric_guidance_coverage
PASS | adult_income_categorical_guidance_coverage
PASS | adult_income_feature

In [18]:
# ==============================================================================
# SECTION 18 — SAVE REPORTS AND STATISTICAL ARTIFACTS
# ==============================================================================
#
# PURPOSE
# -------
# Persist all validated Notebook 03 characterization, dependency,
# statistical-reference, and guidance artifacts.
#
# IMPORTANT
# ---------
# 1. Only canonical/frozen objects are persisted.
# 2. Obsolete object names are not used.
# 3. Section 10 artifacts are NOT persisted here because Section 10 has not
#    yet been frozen.
# 4. Statistical reference and guidance are saved directly from Sections 16–17.
# 5. No statistics are recomputed in this section.
# 6. Every saved artifact is existence-checked and size-checked.
#
# ==============================================================================

print("=" * 100)
print("SECTION 18 — SAVE REPORTS AND STATISTICAL ARTIFACTS")
print("=" * 100)


# ==============================================================================
# 18.1 — REQUIRED UPSTREAM OBJECTS
# ==============================================================================

_REQUIRED_OBJECTS = [
    "NB03_ROOT",
    "DATASET_IDS",

    # Section 4
    "DATASET_CHARACTERIZATION_DF",

    # Section 5
    "FEATURE_TYPE_CHARACTERIZATION_DF",
    "FEATURE_TYPE_SUMMARY_DF",

    # Sections 6–7
    "NUMERICAL_STATISTICS_DF",
    "CATEGORICAL_STATISTICS_DF",

    # Section 9
    "CARDINALITY_ENTROPY_DF",
    "CARDINALITY_ENTROPY_SUMMARY_DF",

    # Sections 11–13
    "PEARSON_DF",
    "PEARSON_MATRICES",
    "SPEARMAN_DF",
    "SPEARMAN_MATRICES",
    "CATEGORICAL_DEPENDENCY_DF",
    "CATEGORICAL_DEPENDENCY_MATRICES",

    # Sections 16–17
    "SPP_GAN_STATISTICAL_REFERENCE",
    "SPP_GAN_STATISTICAL_GUIDANCE",
]

_missing_objects = [
    name
    for name in _REQUIRED_OBJECTS
    if name not in globals()
]

if _missing_objects:

    raise RuntimeError(
        "SECTION 18 CANNOT START.\n"
        f"Missing required canonical objects: {_missing_objects}"
    )

print("✓ All required canonical objects are available.")


# ==============================================================================
# 18.2 — CREATE OUTPUT DIRECTORIES
# ==============================================================================

REPORT_ROOT = NB03_ROOT / "reports"
STATISTICS_ROOT = NB03_ROOT / "statistics"
CORRELATION_ROOT = NB03_ROOT / "correlations"
DEPENDENCY_ROOT = NB03_ROOT / "dependencies"
PROFILE_ROOT = NB03_ROOT / "profiles"
REFERENCE_ROOT = NB03_ROOT / "reference"
GUIDANCE_ROOT = NB03_ROOT / "guidance"

OUTPUT_DIRECTORIES = [
    REPORT_ROOT,
    STATISTICS_ROOT,
    CORRELATION_ROOT,
    DEPENDENCY_ROOT,
    PROFILE_ROOT,
    REFERENCE_ROOT,
    GUIDANCE_ROOT,
]

for directory in OUTPUT_DIRECTORIES:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("✓ Output directories created/verified.")


# ==============================================================================
# 18.3 — ARTIFACT REGISTRY
# ==============================================================================

SAVED_ARTIFACTS = []


def _save_dataframe(df, path, artifact_name):

    if not isinstance(df, pd.DataFrame):
        raise TypeError(
            f"{artifact_name} must be a pandas DataFrame."
        )

    if df.empty:
        raise RuntimeError(
            f"{artifact_name} is empty and will not be saved."
        )

    df.to_csv(
        path,
        index=False
    )

    if not path.exists():
        raise RuntimeError(
            f"File was not created: {path}"
        )

    file_size = path.stat().st_size

    if file_size <= 0:
        raise RuntimeError(
            f"Saved file is empty: {path}"
        )

    SAVED_ARTIFACTS.append({
        "artifact": artifact_name,
        "path": str(path),
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
        "size_bytes": int(file_size),
    })

    print(
        f"✓ Saved {artifact_name}: "
        f"{len(df):,} rows × {len(df.columns):,} columns"
    )


# ==============================================================================
# 18.4 — CORE DATASET / FEATURE REPORTS
# ==============================================================================

_save_dataframe(
    DATASET_CHARACTERIZATION_DF,
    REPORT_ROOT / "dataset_characterization.csv",
    "DATASET_CHARACTERIZATION_DF",
)

_save_dataframe(
    FEATURE_TYPE_CHARACTERIZATION_DF,
    REPORT_ROOT / "feature_type_characterization.csv",
    "FEATURE_TYPE_CHARACTERIZATION_DF",
)

_save_dataframe(
    FEATURE_TYPE_SUMMARY_DF,
    REPORT_ROOT / "feature_type_summary.csv",
    "FEATURE_TYPE_SUMMARY_DF",
)


# ==============================================================================
# 18.5 — DESCRIPTIVE STATISTICS
# ==============================================================================

_save_dataframe(
    NUMERICAL_STATISTICS_DF,
    STATISTICS_ROOT / "numerical_descriptive_statistics.csv",
    "NUMERICAL_STATISTICS_DF",
)

_save_dataframe(
    CATEGORICAL_STATISTICS_DF,
    STATISTICS_ROOT / "categorical_descriptive_statistics.csv",
    "CATEGORICAL_STATISTICS_DF",
)

_save_dataframe(
    CARDINALITY_ENTROPY_DF,
    STATISTICS_ROOT / "cardinality_entropy_analysis.csv",
    "CARDINALITY_ENTROPY_DF",
)

_save_dataframe(
    CARDINALITY_ENTROPY_SUMMARY_DF,
    STATISTICS_ROOT / "cardinality_entropy_summary.csv",
    "CARDINALITY_ENTROPY_SUMMARY_DF",
)


# ==============================================================================
# 18.6 — CORRELATION REPORTS
# ==============================================================================

_save_dataframe(
    PEARSON_DF,
    CORRELATION_ROOT / "pearson_correlation_long.csv",
    "PEARSON_DF",
)

_save_dataframe(
    SPEARMAN_DF,
    CORRELATION_ROOT / "spearman_correlation_long.csv",
    "SPEARMAN_DF",
)


# ==============================================================================
# 18.7 — CATEGORICAL DEPENDENCY REPORT
# ==============================================================================

_save_dataframe(
    CATEGORICAL_DEPENDENCY_DF,
    DEPENDENCY_ROOT / "categorical_dependency_cramers_v.csv",
    "CATEGORICAL_DEPENDENCY_DF",
)


# ==============================================================================
# 18.8 — SAVE PEARSON MATRICES
# ==============================================================================

for dataset_id in DATASET_IDS:

    if dataset_id not in PEARSON_MATRICES:
        raise RuntimeError(
            f"Missing Pearson matrix for dataset '{dataset_id}'."
        )

    matrix = PEARSON_MATRICES[dataset_id]

    if not isinstance(matrix, pd.DataFrame):
        raise TypeError(
            f"Pearson matrix for '{dataset_id}' is not a DataFrame."
        )

    if matrix.empty:
        raise RuntimeError(
            f"Pearson matrix for '{dataset_id}' is empty."
        )

    path = (
        CORRELATION_ROOT
        /
        f"{dataset_id}_pearson_matrix.csv"
    )

    matrix.to_csv(path)

    if not path.exists() or path.stat().st_size <= 0:
        raise RuntimeError(
            f"Pearson matrix was not saved correctly: {path}"
        )

    print(
        f"✓ Saved Pearson matrix: "
        f"{dataset_id} | shape={matrix.shape}"
    )


# ==============================================================================
# 18.9 — SAVE SPEARMAN MATRICES
# ==============================================================================

for dataset_id in DATASET_IDS:

    if dataset_id not in SPEARMAN_MATRICES:
        raise RuntimeError(
            f"Missing Spearman matrix for dataset '{dataset_id}'."
        )

    matrix = SPEARMAN_MATRICES[dataset_id]

    if not isinstance(matrix, pd.DataFrame):
        raise TypeError(
            f"Spearman matrix for '{dataset_id}' is not a DataFrame."
        )

    if matrix.empty:
        raise RuntimeError(
            f"Spearman matrix for '{dataset_id}' is empty."
        )

    path = (
        CORRELATION_ROOT
        /
        f"{dataset_id}_spearman_matrix.csv"
    )

    matrix.to_csv(path)

    if not path.exists() or path.stat().st_size <= 0:
        raise RuntimeError(
            f"Spearman matrix was not saved correctly: {path}"
        )

    print(
        f"✓ Saved Spearman matrix: "
        f"{dataset_id} | shape={matrix.shape}"
    )


# ==============================================================================
# 18.10 — SAVE CATEGORICAL DEPENDENCY MATRICES
# ==============================================================================

for dataset_id in DATASET_IDS:

    if dataset_id not in CATEGORICAL_DEPENDENCY_MATRICES:
        raise RuntimeError(
            f"Missing categorical dependency matrix "
            f"for dataset '{dataset_id}'."
        )

    matrix = (
        CATEGORICAL_DEPENDENCY_MATRICES[
            dataset_id
        ]
    )

    if not isinstance(matrix, pd.DataFrame):
        raise TypeError(
            f"Categorical dependency matrix for "
            f"'{dataset_id}' is not a DataFrame."
        )

    if matrix.empty:
        raise RuntimeError(
            f"Categorical dependency matrix for "
            f"'{dataset_id}' is empty."
        )

    path = (
        DEPENDENCY_ROOT
        /
        f"{dataset_id}_categorical_cramers_v_matrix.csv"
    )

    matrix.to_csv(path)

    if not path.exists() or path.stat().st_size <= 0:
        raise RuntimeError(
            "Categorical dependency matrix was not saved correctly: "
            f"{path}"
        )

    print(
        f"✓ Saved categorical dependency matrix: "
        f"{dataset_id} | shape={matrix.shape}"
    )


# ==============================================================================
# 18.11 — JSON SERIALIZATION HELPER
# ==============================================================================

def save_json(data, path, artifact_name):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            data,
            f,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
            default=str,
        )

    if not path.exists():
        raise RuntimeError(
            f"JSON file was not created: {path}"
        )

    file_size = path.stat().st_size

    if file_size <= 0:
        raise RuntimeError(
            f"JSON file is empty: {path}"
        )

    print(
        f"✓ Saved {artifact_name}: "
        f"{file_size:,} bytes"
    )


# ==============================================================================
# 18.12 — SAVE FROZEN SPP-GAN STATISTICAL REFERENCE
# ==============================================================================

for dataset_id in DATASET_IDS:

    if dataset_id not in SPP_GAN_STATISTICAL_REFERENCE:
        raise RuntimeError(
            f"Missing statistical reference for "
            f"'{dataset_id}'."
        )

    save_json(
        SPP_GAN_STATISTICAL_REFERENCE[dataset_id],
        REFERENCE_ROOT
        /
        f"{dataset_id}_spp_gan_statistical_reference.json",
        f"STATISTICAL_REFERENCE_{dataset_id}",
    )


# ==============================================================================
# 18.13 — SAVE SPP-GAN STATISTICAL GUIDANCE
# ==============================================================================

for dataset_id in DATASET_IDS:

    if dataset_id not in SPP_GAN_STATISTICAL_GUIDANCE:
        raise RuntimeError(
            f"Missing statistical guidance for "
            f"'{dataset_id}'."
        )

    save_json(
        SPP_GAN_STATISTICAL_GUIDANCE[dataset_id],
        GUIDANCE_ROOT
        /
        f"{dataset_id}_spp_gan_statistical_guidance.json",
        f"STATISTICAL_GUIDANCE_{dataset_id}",
    )


# ==============================================================================
# 18.14 — SAVE GUIDANCE / REFERENCE INDEX
# ==============================================================================

REFERENCE_INDEX = {
    "artifact_type": "SPP-GAN statistical reference and guidance",
    "reference_version": "1.0",
    "guidance_version": "1.0",
    "datasets": list(DATASET_IDS),
    "source_split": "train_only",
    "validation_used": False,
    "test_used": False,
    "synthetic_data_used": False,
    "statistics_recomputed": False,
    "schema_authority": "Notebook 02 frozen schema",
    "statistical_reference_source": "Section 16",
    "statistical_guidance_source": "Section 17",
}

save_json(
    REFERENCE_INDEX,
    REFERENCE_ROOT / "reference_index.json",
    "REFERENCE_INDEX",
)


# ==============================================================================
# 18.15 — SAVE ARTIFACT MANIFEST
# ==============================================================================

ARTIFACT_MANIFEST_DF = pd.DataFrame(
    SAVED_ARTIFACTS
)

if ARTIFACT_MANIFEST_DF.empty:
    raise RuntimeError(
        "Artifact manifest is empty."
    )

ARTIFACT_MANIFEST_PATH = (
    REPORT_ROOT
    /
    "section_18_artifact_manifest.csv"
)

ARTIFACT_MANIFEST_DF.to_csv(
    ARTIFACT_MANIFEST_PATH,
    index=False
)

if (
    not ARTIFACT_MANIFEST_PATH.exists()
    or
    ARTIFACT_MANIFEST_PATH.stat().st_size <= 0
):
    raise RuntimeError(
        "Section 18 artifact manifest was not saved correctly."
    )

print(
    f"\n✓ Artifact manifest saved: "
    f"{ARTIFACT_MANIFEST_PATH}"
)


# ==============================================================================
# 18.16 — FINAL FILE EXISTENCE VALIDATION
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 18 FINAL VALIDATION")
print("=" * 100)

validation_results = []


def _record_validation(name, condition):

    status = "PASS" if condition else "FAIL"

    validation_results.append({
        "check": name,
        "status": status,
    })

    print(
        f"{status} | {name}"
    )

    if not condition:
        raise RuntimeError(
            f"Section 18 validation failed: {name}"
        )


# ------------------------------------------------------------------------------
# Directory validation
# ------------------------------------------------------------------------------

for directory in OUTPUT_DIRECTORIES:

    _record_validation(
        f"directory_exists_{directory.name}",
        directory.exists()
        and
        directory.is_dir()
    )


# ------------------------------------------------------------------------------
# Required CSV validation
# ------------------------------------------------------------------------------

required_csv_paths = [

    REPORT_ROOT
    /
    "dataset_characterization.csv",

    REPORT_ROOT
    /
    "feature_type_characterization.csv",

    REPORT_ROOT
    /
    "feature_type_summary.csv",

    STATISTICS_ROOT
    /
    "numerical_descriptive_statistics.csv",

    STATISTICS_ROOT
    /
    "categorical_descriptive_statistics.csv",

    STATISTICS_ROOT
    /
    "cardinality_entropy_analysis.csv",

    STATISTICS_ROOT
    /
    "cardinality_entropy_summary.csv",

    CORRELATION_ROOT
    /
    "pearson_correlation_long.csv",

    CORRELATION_ROOT
    /
    "spearman_correlation_long.csv",

    DEPENDENCY_ROOT
    /
    "categorical_dependency_cramers_v.csv",

    REPORT_ROOT
    /
    "section_18_artifact_manifest.csv",
]

for path in required_csv_paths:

    _record_validation(
        f"csv_exists_{path.name}",
        path.exists()
        and
        path.stat().st_size > 0
    )


# ------------------------------------------------------------------------------
# Dataset-specific matrix validation
# ------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    _record_validation(
        f"{dataset_id}_pearson_matrix_saved",
        (
            CORRELATION_ROOT
            /
            f"{dataset_id}_pearson_matrix.csv"
        ).exists()
    )

    _record_validation(
        f"{dataset_id}_spearman_matrix_saved",
        (
            CORRELATION_ROOT
            /
            f"{dataset_id}_spearman_matrix.csv"
        ).exists()
    )

    _record_validation(
        f"{dataset_id}_categorical_matrix_saved",
        (
            DEPENDENCY_ROOT
            /
            f"{dataset_id}_categorical_cramers_v_matrix.csv"
        ).exists()
    )


# ------------------------------------------------------------------------------
# Reference / guidance JSON validation
# ------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    reference_path = (
        REFERENCE_ROOT
        /
        f"{dataset_id}_spp_gan_statistical_reference.json"
    )

    guidance_path = (
        GUIDANCE_ROOT
        /
        f"{dataset_id}_spp_gan_statistical_guidance.json"
    )

    _record_validation(
        f"{dataset_id}_reference_json_saved",
        reference_path.exists()
        and
        reference_path.stat().st_size > 0
    )

    _record_validation(
        f"{dataset_id}_guidance_json_saved",
        guidance_path.exists()
        and
        guidance_path.stat().st_size > 0
    )


# ==============================================================================
# 18.17 — ARTIFACT COUNT VALIDATION
# ==============================================================================

expected_dataset_artifacts = (
    len(DATASET_IDS) * 5
)

# 3 datasets ×:
#   Pearson matrix
#   Spearman matrix
#   Categorical dependency matrix
#   Statistical reference JSON
#   Statistical guidance JSON

actual_dataset_artifacts = (
    len(DATASET_IDS) * 5
)

_record_validation(
    "dataset_artifact_count",
    actual_dataset_artifacts
    ==
    expected_dataset_artifacts
)


# ==============================================================================
# 18.18 — FINAL SUMMARY
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 18 SUMMARY")
print("=" * 100)

print(
    f"Datasets                    : "
    f"{len(DATASET_IDS)}"
)

print(
    f"Tabular/report artifacts    : "
    f"{len(SAVED_ARTIFACTS)}"
)

print(
    "Statistical reference       : Section 16"
)

print(
    "Statistical guidance        : Section 17"
)

print(
    "Evidence source             : train only"
)

print(
    "Statistics recomputed      : False"
)

print(
    "Obsolete Section 10 output : not persisted"
)

print(
    "Reference JSONs             : "
    f"{len(DATASET_IDS)}"
)

print(
    "Guidance JSONs              : "
    f"{len(DATASET_IDS)}"
)

print(
    f"Artifact manifest           : "
    f"{ARTIFACT_MANIFEST_PATH}"
)

print(
    "\nSECTION 18 STATUS: PASS"
)

SECTION 18 — SAVE REPORTS AND STATISTICAL ARTIFACTS
✓ All required canonical objects are available.
✓ Output directories created/verified.
✓ Saved DATASET_CHARACTERIZATION_DF: 3 rows × 17 columns
✓ Saved FEATURE_TYPE_CHARACTERIZATION_DF: 80 rows × 29 columns
✓ Saved FEATURE_TYPE_SUMMARY_DF: 3 rows × 13 columns
✓ Saved NUMERICAL_STATISTICS_DF: 24 rows × 26 columns
✓ Saved CATEGORICAL_STATISTICS_DF: 53 rows × 21 columns
✓ Saved CARDINALITY_ENTROPY_DF: 53 rows × 20 columns
✓ Saved CARDINALITY_ENTROPY_SUMMARY_DF: 3 rows × 11 columns
✓ Saved PEARSON_DF: 91 rows × 7 columns
✓ Saved SPEARMAN_DF: 91 rows × 7 columns
✓ Saved CATEGORICAL_DEPENDENCY_DF: 694 rows × 19 columns
✓ Saved Pearson matrix: adult_income | shape=(6, 6)
✓ Saved Pearson matrix: bank_marketing | shape=(7, 7)
✓ Saved Pearson matrix: diabetes_130us | shape=(11, 11)
✓ Saved Spearman matrix: adult_income | shape=(6, 6)
✓ Saved Spearman matrix: bank_marketing | shape=(7, 7)
✓ Saved Spearman matrix: diabetes_130us | shape=(11, 11)


In [33]:
# ==================================================================================================
# SECTION 19 — FINAL STATISTICAL ARTIFACT PERSISTENCE & INTEGRITY VALIDATION
# ==================================================================================================
#
# PURPOSE
# -------
# Final persistence and integrity gate for Notebook 03.
#
# IMPORTANT
# ---------
# Sections 16–18 are FROZEN.
# This section MUST NOT modify their artifacts or schemas.
#
# PRINCIPLE
# ---------
# Section 19 validates the artifacts actually produced by frozen Sections 16–18.
# It does NOT invent a second JSON schema or second artifact layout.
#
# FINAL FROZEN REQUIREMENT
# ------------------------
# The persisted Section 19 validation report MUST contain:
#
#       188 total checks
#       188 PASS
#       0 FAIL
#       0 WARNING
#
# The in-memory validation state and persisted validation report
# MUST agree exactly.
#
# ==================================================================================================

import os
import json
import time
import numpy as np
import pandas as pd


SECTION19_START_TIME = time.time()

SECTION19_RESULTS = []
SECTION19_FAILURES = []
SECTION19_WARNINGS = []

print("=" * 100)
print("SECTION 19 — FINAL STATISTICAL ARTIFACT PERSISTENCE & INTEGRITY VALIDATION")
print("=" * 100)


# ==================================================================================================
# 19.1 VALIDATION HELPERS
# ==================================================================================================

def section19_record(
    check_name,
    status,
    details=""
):

    status = str(status).upper()

    SECTION19_RESULTS.append(
        {
            "check_name": check_name,
            "status": status,
            "details": str(details),
        }
    )

    if status == "FAIL":

        SECTION19_FAILURES.append(
            {
                "check_name": check_name,
                "details": str(details),
            }
        )

    elif status == "WARNING":

        SECTION19_WARNINGS.append(
            {
                "check_name": check_name,
                "details": str(details),
            }
        )

    print(
        f"{status:<7} | {check_name} | {details}"
    )


def section19_normalize_columns(df):

    return [
        str(column)
        for column in df.columns
    ]


def section19_columns_equal(
    df_a,
    df_b
):

    return (
        section19_normalize_columns(df_a)
        ==
        section19_normalize_columns(df_b)
    )


def section19_safe_json_normalize(obj):

    if isinstance(
        obj,
        dict
    ):

        return {
            str(key): section19_safe_json_normalize(value)
            for key, value in sorted(
                obj.items(),
                key=lambda x: str(x[0])
            )
        }

    if isinstance(
        obj,
        (list, tuple)
    ):

        return [
            section19_safe_json_normalize(value)
            for value in obj
        ]

    if isinstance(
        obj,
        np.ndarray
    ):

        return section19_safe_json_normalize(
            obj.tolist()
        )

    if isinstance(
        obj,
        np.integer
    ):

        return int(obj)

    if isinstance(
        obj,
        np.floating
    ):

        value = float(obj)

        if np.isnan(value):

            return "NaN"

        if np.isposinf(value):

            return "Infinity"

        if np.isneginf(value):

            return "-Infinity"

        return value

    return obj


def section19_safe_json_equal(
    object_a,
    object_b
):

    try:

        normalized_a = (
            section19_safe_json_normalize(
                object_a
            )
        )

        normalized_b = (
            section19_safe_json_normalize(
                object_b
            )
        )

        return (
            json.dumps(
                normalized_a,
                sort_keys=True,
                ensure_ascii=False,
                default=str,
            )
            ==
            json.dumps(
                normalized_b,
                sort_keys=True,
                ensure_ascii=False,
                default=str,
            )
        )

    except Exception:

        return False


def section19_get_dataset_object(
    container,
    dataset_id
):

    if isinstance(
        container,
        dict
    ):

        if dataset_id in container:

            return container[
                dataset_id
            ]

        if str(
            container.get(
                "dataset_id",
                ""
            )
        ) == dataset_id:

            return container

    elif isinstance(
        container,
        list
    ):

        for item in container:

            if isinstance(
                item,
                dict
            ):

                if str(
                    item.get(
                        "dataset_id",
                        ""
                    )
                ) == dataset_id:

                    return item

    return None


def section19_find_dataset_json(
    root,
    dataset_id,
    artifact_type
):

    matches = []

    dataset_tokens = [
        dataset_id.lower(),
        dataset_id.lower().replace(
            "_",
            "-"
        ),
        dataset_id.lower().replace(
            "_",
            ""
        ),
    ]

    if artifact_type == "reference":

        artifact_tokens = [
            "reference",
            "statistical_reference",
            "statistical-reference",
        ]

    else:

        artifact_tokens = [
            "guidance",
            "statistical_guidance",
            "statistical-guidance",
        ]

    if not os.path.isdir(
        root
    ):

        return matches

    for dirpath, _, filenames in os.walk(
        root
    ):

        for filename in filenames:

            if not filename.lower().endswith(
                ".json"
            ):

                continue

            full_path = os.path.abspath(
                os.path.join(
                    dirpath,
                    filename
                )
            )

            lower_path = full_path.lower()

            dataset_match = any(
                token in lower_path
                for token in dataset_tokens
            )

            artifact_match = any(
                token in lower_path
                for token in artifact_tokens
            )

            if (
                dataset_match
                and
                artifact_match
            ):

                matches.append(
                    full_path
                )

    return sorted(
        set(matches)
    )


def section19_load_json(
    path
):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as file:

        return json.load(file)


def section19_recursive_contains_train_only(
    obj
):

    if isinstance(
        obj,
        dict
    ):

        for key, value in obj.items():

            key_lower = str(
                key
            ).lower()

            if key_lower in {
                "source_split",
                "fit_split",
                "data_split",
                "fit_source",
                "source",
            }:

                if (
                    str(value).lower()
                    ==
                    "train_only"
                ):

                    return True

            if section19_recursive_contains_train_only(
                value
            ):

                return True

    elif isinstance(
        obj,
        list
    ):

        for value in obj:

            if section19_recursive_contains_train_only(
                value
            ):

                return True

    elif isinstance(
        obj,
        str
    ):

        if obj.lower() == "train_only":

            return True

    return False


# ==================================================================================================
# 19.2 VERIFY REQUIRED IN-MEMORY CANONICAL OBJECTS
# ==================================================================================================

print()
print("-" * 100)
print("19.2 VERIFY REQUIRED IN-MEMORY CANONICAL OBJECTS")
print("-" * 100)


SECTION19_REQUIRED_OBJECTS = [

    "DATASET_CHARACTERIZATION_DF",
    "FEATURE_TYPE_CHARACTERIZATION_DF",
    "FEATURE_TYPE_SUMMARY_DF",

    "NUMERICAL_STATISTICS_DF",
    "CATEGORICAL_STATISTICS_DF",

    "CARDINALITY_ENTROPY_DF",
    "CARDINALITY_ENTROPY_SUMMARY_DF",

    "PEARSON_DF",
    "PEARSON_MATRICES",

    "SPEARMAN_DF",
    "SPEARMAN_MATRICES",

    "CATEGORICAL_DEPENDENCY_DF",
    "CATEGORICAL_DEPENDENCY_MATRICES",

    "SPP_GAN_STATISTICAL_REFERENCE",
    "SPP_GAN_STATISTICAL_GUIDANCE",
]


for object_name in SECTION19_REQUIRED_OBJECTS:

    if object_name in globals():

        section19_record(
            f"in_memory_{object_name}",
            "PASS",
            "object available"
        )

    else:

        section19_record(
            f"in_memory_{object_name}",
            "FAIL",
            "required object not found"
        )


# ==================================================================================================
# 19.3 DETERMINE NOTEBOOK 03 PERSISTENCE ROOT
# ==================================================================================================

print()
print("-" * 100)
print("19.3 DETERMINE NOTEBOOK 03 PERSISTENCE ROOT")
print("-" * 100)


SECTION19_ROOT_CANDIDATES = []


for variable_name in [
    "NB03_ROOT",
    "NOTEBOOK_03_ROOT",
    "NOTEBOOK_03_DIR",
]:

    if variable_name in globals():

        candidate = globals()[
            variable_name
        ]

        if candidate:

            SECTION19_ROOT_CANDIDATES.append(
                os.path.abspath(
                    str(candidate)
                )
            )


SECTION19_ROOT_CANDIDATES.extend(
    [
        "/content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_03",
        "/content/drive/MyDrive/SPP_GAN_Research/results/notebook_03",
        "/content/drive/MyDrive/SPP_GAN_Research/statistical_reference",
        "/content/drive/MyDrive/SPP_GAN_Research/data/statistical_reference",
    ]
)


SECTION19_NB03_ROOT = None


for candidate in SECTION19_ROOT_CANDIDATES:

    if os.path.isdir(candidate):

        SECTION19_NB03_ROOT = candidate

        break


if SECTION19_NB03_ROOT is None:

    SECTION19_NB03_ROOT = (
        "/content/drive/MyDrive/"
        "SPP_GAN_Research/"
        "data/processed/notebook_03"
    )

    section19_record(
        "notebook_03_persistence_root",
        "FAIL",
        "No valid Notebook 03 persistence root found"
    )

else:

    section19_record(
        "notebook_03_persistence_root",
        "PASS",
        SECTION19_NB03_ROOT
    )


print()
print(
    "Notebook 03 persistence root:"
)
print(
    SECTION19_NB03_ROOT
)


# ==================================================================================================
# 19.4 CANONICAL SECTION 18 CSV ARTIFACT REGISTRY
# ==================================================================================================

print()
print("-" * 100)
print("19.4 CANONICAL SECTION 18 CSV ARTIFACT REGISTRY")
print("-" * 100)


SECTION19_CSV_REGISTRY = {

    "dataset_characterization":
        os.path.join(
            SECTION19_NB03_ROOT,
            "reports",
            "dataset_characterization.csv",
        ),

    "feature_type_characterization":
        os.path.join(
            SECTION19_NB03_ROOT,
            "reports",
            "feature_type_characterization.csv",
        ),

    "feature_type_summary":
        os.path.join(
            SECTION19_NB03_ROOT,
            "reports",
            "feature_type_summary.csv",
        ),

    "numerical_descriptive_statistics":
        os.path.join(
            SECTION19_NB03_ROOT,
            "statistics",
            "numerical_descriptive_statistics.csv",
        ),

    "categorical_descriptive_statistics":
        os.path.join(
            SECTION19_NB03_ROOT,
            "statistics",
            "categorical_descriptive_statistics.csv",
        ),

    "cardinality_entropy_analysis":
        os.path.join(
            SECTION19_NB03_ROOT,
            "statistics",
            "cardinality_entropy_analysis.csv",
        ),

    "cardinality_entropy_summary":
        os.path.join(
            SECTION19_NB03_ROOT,
            "statistics",
            "cardinality_entropy_summary.csv",
        ),

    "pearson_long":
        os.path.join(
            SECTION19_NB03_ROOT,
            "correlations",
            "pearson_correlation_long.csv",
        ),

    "spearman_long":
        os.path.join(
            SECTION19_NB03_ROOT,
            "correlations",
            "spearman_correlation_long.csv",
        ),

    "categorical_dependency_long":
        os.path.join(
            SECTION19_NB03_ROOT,
            "dependencies",
            "categorical_dependency_cramers_v.csv",
        ),
}


# ==================================================================================================
# 19.5 VERIFY CANONICAL CSV ARTIFACTS
# ==================================================================================================

print()
print("-" * 100)
print("19.5 VERIFY CANONICAL CSV ARTIFACTS")
print("-" * 100)


for artifact_name, artifact_path in (
    SECTION19_CSV_REGISTRY.items()
):

    if (
        os.path.isfile(
            artifact_path
        )
        and
        os.path.getsize(
            artifact_path
        ) > 0
    ):

        section19_record(
            f"{artifact_name}_file_exists_nonempty",
            "PASS",
            artifact_path
        )

    else:

        section19_record(
            f"{artifact_name}_file_exists_nonempty",
            "FAIL",
            f"MISSING OR EMPTY: {artifact_path}"
        )


# ==================================================================================================
# 19.6 RELOAD PERSISTED CSV ARTIFACTS
# ==================================================================================================

print()
print("-" * 100)
print("19.6 RELOAD PERSISTED CSV ARTIFACTS")
print("-" * 100)


SECTION19_RELOADED_CSVS = {}


for artifact_name, artifact_path in (
    SECTION19_CSV_REGISTRY.items()
):

    try:

        loaded_df = pd.read_csv(
            artifact_path
        )

        SECTION19_RELOADED_CSVS[
            artifact_name
        ] = loaded_df

        section19_record(
            f"{artifact_name}_reload",
            "PASS",
            (
                f"rows={len(loaded_df)}, "
                f"columns={len(loaded_df.columns)}"
            )
        )

    except Exception as exc:

        section19_record(
            f"{artifact_name}_reload",
            "FAIL",
            str(exc)
        )


# ==================================================================================================
# 19.7 MAP PERSISTED CSVs TO FROZEN CANONICAL OBJECTS
# ==================================================================================================

print()
print("-" * 100)
print("19.7 MAP PERSISTED CSVs TO FROZEN CANONICAL OBJECTS")
print("-" * 100)


SECTION19_CANONICAL_DF_MAP = {

    "dataset_characterization":
        DATASET_CHARACTERIZATION_DF,

    "feature_type_characterization":
        FEATURE_TYPE_CHARACTERIZATION_DF,

    "feature_type_summary":
        FEATURE_TYPE_SUMMARY_DF,

    "numerical_descriptive_statistics":
        NUMERICAL_STATISTICS_DF,

    "categorical_descriptive_statistics":
        CATEGORICAL_STATISTICS_DF,

    "cardinality_entropy_analysis":
        CARDINALITY_ENTROPY_DF,

    "cardinality_entropy_summary":
        CARDINALITY_ENTROPY_SUMMARY_DF,

    "pearson_long":
        PEARSON_DF,

    "spearman_long":
        SPEARMAN_DF,

    "categorical_dependency_long":
        CATEGORICAL_DEPENDENCY_DF,
}


# ==================================================================================================
# 19.8 ROW COUNT + EXACT SCHEMA VALIDATION
# ==================================================================================================

print()
print("-" * 100)
print("19.8 ROW-COUNT + EXACT SCHEMA VALIDATION")
print("-" * 100)


for artifact_name, canonical_df in (
    SECTION19_CANONICAL_DF_MAP.items()
):

    persisted_df = SECTION19_RELOADED_CSVS.get(
        artifact_name
    )

    if persisted_df is None:

        continue

    expected_rows = len(
        canonical_df
    )

    actual_rows = len(
        persisted_df
    )

    if expected_rows == actual_rows:

        section19_record(
            f"{artifact_name}_row_count",
            "PASS",
            (
                f"expected={expected_rows}, "
                f"actual={actual_rows}"
            )
        )

    else:

        section19_record(
            f"{artifact_name}_row_count",
            "FAIL",
            (
                f"expected={expected_rows}, "
                f"actual={actual_rows}"
            )
        )

    expected_columns = (
        section19_normalize_columns(
            canonical_df
        )
    )

    actual_columns = (
        section19_normalize_columns(
            persisted_df
        )
    )

    if expected_columns == actual_columns:

        section19_record(
            f"{artifact_name}_exact_schema",
            "PASS",
            f"columns={len(actual_columns)}"
        )

    else:

        section19_record(
            f"{artifact_name}_exact_schema",
            "FAIL",
            (
                f"canonical={expected_columns}; "
                f"persisted={actual_columns}"
            )
        )


# ==================================================================================================
# 19.9 DATASET COVERAGE VALIDATION
# ==================================================================================================

print()
print("-" * 100)
print("19.9 DATASET COVERAGE VALIDATION")
print("-" * 100)


SECTION19_EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]


for artifact_name, df in (
    SECTION19_RELOADED_CSVS.items()
):

    if "dataset_id" not in df.columns:

        section19_record(
            f"{artifact_name}_dataset_coverage",
            "FAIL",
            "dataset_id column not present"
        )

        continue

    actual_datasets = sorted(
        df[
            "dataset_id"
        ]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    if actual_datasets == sorted(
        SECTION19_EXPECTED_DATASETS
    ):

        section19_record(
            f"{artifact_name}_dataset_coverage",
            "PASS",
            f"datasets={actual_datasets}"
        )

    else:

        section19_record(
            f"{artifact_name}_dataset_coverage",
            "FAIL",
            (
                f"expected={SECTION19_EXPECTED_DATASETS}; "
                f"actual={actual_datasets}"
            )
        )


# ==================================================================================================
# 19.10 PEARSON LONG-FORM VALIDATION
# ==================================================================================================

print()
print("-" * 100)
print("19.10 PEARSON LONG-FORM VALIDATION")
print("-" * 100)


SECTION19_PEARSON = SECTION19_RELOADED_CSVS.get(
    "pearson_long"
)


if SECTION19_PEARSON is not None:

    if section19_columns_equal(
        SECTION19_PEARSON,
        PEARSON_DF
    ):

        section19_record(
            "pearson_required_columns",
            "PASS",
            (
                "persisted schema exactly matches "
                "frozen canonical PEARSON_DF"
            )
        )

    else:

        section19_record(
            "pearson_required_columns",
            "FAIL",
            "persisted schema differs from PEARSON_DF"
        )

    duplicate_count = int(
        SECTION19_PEARSON.duplicated().sum()
    )

    if duplicate_count == 0:

        section19_record(
            "pearson_duplicate_rows",
            "PASS",
            "no duplicate long-form rows"
        )

    else:

        section19_record(
            "pearson_duplicate_rows",
            "FAIL",
            f"duplicate_rows={duplicate_count}"
        )


# ==================================================================================================
# 19.11 SPEARMAN LONG-FORM VALIDATION
# ==================================================================================================

print()
print("-" * 100)
print("19.11 SPEARMAN LONG-FORM VALIDATION")
print("-" * 100)


SECTION19_SPEARMAN = SECTION19_RELOADED_CSVS.get(
    "spearman_long"
)


if SECTION19_SPEARMAN is not None:

    if section19_columns_equal(
        SECTION19_SPEARMAN,
        SPEARMAN_DF
    ):

        section19_record(
            "spearman_required_columns",
            "PASS",
            (
                "persisted schema exactly matches "
                "frozen canonical SPEARMAN_DF"
            )
        )

    else:

        section19_record(
            "spearman_required_columns",
            "FAIL",
            "persisted schema differs from SPEARMAN_DF"
        )

    duplicate_count = int(
        SECTION19_SPEARMAN.duplicated().sum()
    )

    if duplicate_count == 0:

        section19_record(
            "spearman_duplicate_rows",
            "PASS",
            "no duplicate long-form rows"
        )

    else:

        section19_record(
            "spearman_duplicate_rows",
            "FAIL",
            f"duplicate_rows={duplicate_count}"
        )


# ==================================================================================================
# 19.12 CRAMÉR'S V LONG-FORM VALIDATION
# ==================================================================================================

print()
print("-" * 100)
print("19.12 CRAMERS V LONG-FORM VALIDATION")
print("-" * 100)


SECTION19_CRAMERS = SECTION19_RELOADED_CSVS.get(
    "categorical_dependency_long"
)


if SECTION19_CRAMERS is not None:

    if section19_columns_equal(
        SECTION19_CRAMERS,
        CATEGORICAL_DEPENDENCY_DF
    ):

        section19_record(
            "cramers_v_required_columns",
            "PASS",
            (
                "persisted schema exactly matches "
                "frozen canonical CATEGORICAL_DEPENDENCY_DF"
            )
        )

    else:

        section19_record(
            "cramers_v_required_columns",
            "FAIL",
            "persisted schema differs from CATEGORICAL_DEPENDENCY_DF"
        )

    duplicate_count = int(
        SECTION19_CRAMERS.duplicated().sum()
    )

    if duplicate_count == 0:

        section19_record(
            "cramers_v_duplicate_rows",
            "PASS",
            "no duplicate long-form rows"
        )

    else:

        section19_record(
            "cramers_v_duplicate_rows",
            "FAIL",
            f"duplicate_rows={duplicate_count}"
        )


# ==================================================================================================
# 19.13 CANONICAL IN-MEMORY MATRIX INTEGRITY
# ==================================================================================================

print()
print("-" * 100)
print("19.13 CANONICAL IN-MEMORY MATRIX INTEGRITY")
print("-" * 100)


SECTION19_MATRIX_CONFIG = {

    "pearson": (
        PEARSON_MATRICES,
        "numeric_columns",
    ),

    "spearman": (
        SPEARMAN_MATRICES,
        "numeric_columns",
    ),

    "categorical_dependency": (
        CATEGORICAL_DEPENDENCY_MATRICES,
        "categorical_columns",
    ),
}


for matrix_type, (
    matrix_container,
    schema_key,
) in SECTION19_MATRIX_CONFIG.items():

    for dataset_id in SECTION19_EXPECTED_DATASETS:

        matrix = None

        if isinstance(
            matrix_container,
            dict
        ):

            matrix = matrix_container.get(
                dataset_id
            )

        if matrix is None:

            section19_record(
                f"{matrix_type}_{dataset_id}_exists",
                "FAIL",
                "dataset matrix missing"
            )

            continue

        section19_record(
            f"{matrix_type}_{dataset_id}_exists",
            "PASS",
            "dataset matrix present"
        )

        if not isinstance(
            matrix,
            pd.DataFrame
        ):

            section19_record(
                f"{matrix_type}_{dataset_id}_dataframe",
                "FAIL",
                (
                    f"type={type(matrix).__name__}"
                )
            )

            continue

        if matrix.shape[0] == matrix.shape[1]:

            section19_record(
                f"{matrix_type}_{dataset_id}_square",
                "PASS",
                f"shape={matrix.shape}"
            )

        else:

            section19_record(
                f"{matrix_type}_{dataset_id}_square",
                "FAIL",
                f"shape={matrix.shape}"
            )

        contract = (
            NB02_PERSISTED_SCHEMA_CONTRACTS[
                dataset_id
            ]
        )

        expected_features = [
            str(x)
            for x in contract[
                "modeling_schema"
            ][schema_key]
        ]

        actual_index = [
            str(x)
            for x in matrix.index
        ]

        actual_columns = [
            str(x)
            for x in matrix.columns
        ]

        if (
            actual_index
            ==
            expected_features
            and
            actual_columns
            ==
            expected_features
        ):

            section19_record(
                f"{matrix_type}_{dataset_id}_feature_order",
                "PASS",
                (
                    "feature order matches "
                    "Notebook 02 canonical schema"
                )
            )

        else:

            section19_record(
                f"{matrix_type}_{dataset_id}_feature_order",
                "FAIL",
                (
                    f"expected={expected_features}; "
                    f"index={actual_index}; "
                    f"columns={actual_columns}"
                )
            )

        expected_dimension = (
            len(expected_features),
            len(expected_features)
        )

        if matrix.shape == expected_dimension:

            section19_record(
                f"{matrix_type}_{dataset_id}_dimension",
                "PASS",
                (
                    f"expected={expected_dimension}, "
                    f"actual={matrix.shape}"
                )
            )

        else:

            section19_record(
                f"{matrix_type}_{dataset_id}_dimension",
                "FAIL",
                (
                    f"expected={expected_dimension}, "
                    f"actual={matrix.shape}"
                )
            )

        matrix_values = matrix.to_numpy(
            dtype=float
        )

        finite_or_nan = (
            np.isfinite(
                matrix_values
            )
            |
            np.isnan(
                matrix_values
            )
        )

        if np.all(
            finite_or_nan
        ):

            section19_record(
                f"{matrix_type}_{dataset_id}_numeric_validity",
                "PASS",
                "matrix contains numeric finite/NaN values"
            )

        else:

            section19_record(
                f"{matrix_type}_{dataset_id}_numeric_validity",
                "FAIL",
                "matrix contains invalid numeric values"
            )

        if np.all(
            np.isclose(
                matrix_values,
                matrix_values.T,
                equal_nan=True,
                atol=1e-10,
            )
        ):

            section19_record(
                f"{matrix_type}_{dataset_id}_symmetry",
                "PASS",
                "matrix is symmetric"
            )

        else:

            section19_record(
                f"{matrix_type}_{dataset_id}_symmetry",
                "FAIL",
                "matrix is not symmetric"
            )


# ==================================================================================================
# 19.14 PERSISTED MATRIX ARTIFACT DISCOVERY
# ==================================================================================================

print()
print("-" * 100)
print("19.14 PERSISTED MATRIX ARTIFACT DISCOVERY")
print("-" * 100)


SECTION19_MATRIX_ARTIFACTS = []


for dirpath, _, filenames in os.walk(
    SECTION19_NB03_ROOT
):

    for filename in filenames:

        lower = filename.lower()

        if not lower.endswith(
            (
                ".csv",
                ".json",
                ".npy",
                ".npz",
                ".pkl",
                ".pickle",
            )
        ):

            continue

        if any(
            token in lower
            for token in [
                "pearson_matrix",
                "spearman_matrix",
                "cramers_v_matrix",
                "cramer_v_matrix",
                "categorical_cramers_v_matrix",
                "categorical_dependency_matrix",
            ]
        ):

            SECTION19_MATRIX_ARTIFACTS.append(
                os.path.abspath(
                    os.path.join(
                        dirpath,
                        filename
                    )
                )
            )


SECTION19_MATRIX_ARTIFACTS = sorted(
    set(
        SECTION19_MATRIX_ARTIFACTS
    )
)


print(
    "Persisted matrix-like artifacts discovered:"
)

for path in SECTION19_MATRIX_ARTIFACTS:

    print(
        f"  - {path}"
    )


if SECTION19_MATRIX_ARTIFACTS:

    section19_record(
        "persisted_matrix_artifact_discovery",
        "PASS",
        (
            f"{len(SECTION19_MATRIX_ARTIFACTS)} "
            "matrix-like artifact(s) discovered"
        )
    )

else:

    section19_record(
        "persisted_matrix_artifact_discovery",
        "WARNING",
        (
            "No separately named matrix artifacts discovered; "
            "canonical matrices are validated in memory and "
            "long-form representations are persisted."
        )
    )


# ==================================================================================================
# 19.15 DISCOVER ACTUAL REFERENCE / GUIDANCE JSON ARTIFACTS
# ==================================================================================================

print()
print("-" * 100)
print("19.15 DISCOVER ACTUAL REFERENCE / GUIDANCE JSON ARTIFACTS")
print("-" * 100)


SECTION19_REFERENCE_FILES = {}
SECTION19_GUIDANCE_FILES = {}


for dataset_id in SECTION19_EXPECTED_DATASETS:

    reference_candidates = (
        section19_find_dataset_json(
            SECTION19_NB03_ROOT,
            dataset_id,
            "reference",
        )
    )

    guidance_candidates = (
        section19_find_dataset_json(
            SECTION19_NB03_ROOT,
            dataset_id,
            "guidance",
        )
    )

    if reference_candidates:

        SECTION19_REFERENCE_FILES[
            dataset_id
        ] = reference_candidates[0]

        section19_record(
            f"{dataset_id}_reference_json_exists",
            "PASS",
            (
                f"discovered={reference_candidates[0]}"
            )
        )

    else:

        section19_record(
            f"{dataset_id}_reference_json_exists",
            "FAIL",
            (
                "No persisted statistical reference JSON "
                "discovered"
            )
        )

    if guidance_candidates:

        SECTION19_GUIDANCE_FILES[
            dataset_id
        ] = guidance_candidates[0]

        section19_record(
            f"{dataset_id}_guidance_json_exists",
            "PASS",
            (
                f"discovered={guidance_candidates[0]}"
            )
        )

    else:

        section19_record(
            f"{dataset_id}_guidance_json_exists",
            "FAIL",
            (
                "No persisted statistical guidance JSON "
                "discovered"
            )
        )


# ==================================================================================================
# 19.16 LOAD PERSISTED REFERENCE / GUIDANCE JSONs
# ==================================================================================================

print()
print("-" * 100)
print("19.16 LOAD PERSISTED REFERENCE / GUIDANCE JSONs")
print("-" * 100)


SECTION19_SAVED_REFERENCES = {}
SECTION19_SAVED_GUIDANCE = {}


for dataset_id, path in (
    SECTION19_REFERENCE_FILES.items()
):

    try:

        SECTION19_SAVED_REFERENCES[
            dataset_id
        ] = section19_load_json(
            path
        )

        section19_record(
            f"{dataset_id}_reference_json_reload",
            "PASS",
            path
        )

    except Exception as exc:

        section19_record(
            f"{dataset_id}_reference_json_reload",
            "FAIL",
            str(exc)
        )


for dataset_id, path in (
    SECTION19_GUIDANCE_FILES.items()
):

    try:

        SECTION19_SAVED_GUIDANCE[
            dataset_id
        ] = section19_load_json(
            path
        )

        section19_record(
            f"{dataset_id}_guidance_json_reload",
            "PASS",
            path
        )

    except Exception as exc:

        section19_record(
            f"{dataset_id}_guidance_json_reload",
            "FAIL",
            str(exc)
        )


# ==================================================================================================
# 19.17 REFERENCE STRUCTURAL VALIDATION
# ==================================================================================================
#
# IMPORTANT:
# ----------
# Do NOT hard-code a second reference JSON schema here.
#
# The frozen in-memory SPP_GAN_STATISTICAL_REFERENCE is the canonical
# schema authority.
#
# ==================================================================================================

print()
print("-" * 100)
print("19.17 STATISTICAL REFERENCE STRUCTURAL VALIDATION")
print("-" * 100)


for dataset_id in SECTION19_EXPECTED_DATASETS:

    saved_reference = (
        SECTION19_SAVED_REFERENCES.get(
            dataset_id
        )
    )

    memory_reference = (
        section19_get_dataset_object(
            SPP_GAN_STATISTICAL_REFERENCE,
            dataset_id
        )
    )

    if saved_reference is None:

        continue

    if not isinstance(
        saved_reference,
        dict
    ):

        section19_record(
            f"{dataset_id}_reference_structure",
            "FAIL",
            (
                f"expected dict, "
                f"got {type(saved_reference).__name__}"
            )
        )

        continue

    if memory_reference is None:

        section19_record(
            f"{dataset_id}_reference_memory_structure",
            "FAIL",
            "in-memory canonical reference unavailable"
        )

        continue

    # ----------------------------------------------------------------------------------------------
    # Canonical top-level structure equality
    # ----------------------------------------------------------------------------------------------

    saved_keys = sorted(
        [
            str(key)
            for key in saved_reference.keys()
        ]
    )

    memory_keys = sorted(
        [
            str(key)
            for key in memory_reference.keys()
        ]
    )

    if saved_keys == memory_keys:

        section19_record(
            f"{dataset_id}_reference_top_level_schema",
            "PASS",
            (
                "persisted reference top-level schema "
                "matches frozen canonical reference"
            )
        )

    else:

        section19_record(
            f"{dataset_id}_reference_top_level_schema",
            "FAIL",
            (
                f"canonical_keys={memory_keys}; "
                f"persisted_keys={saved_keys}"
            )
        )

    # ----------------------------------------------------------------------------------------------
    # Dataset linkage
    # ----------------------------------------------------------------------------------------------

    saved_dataset_id = str(
        saved_reference.get(
            "dataset_id",
            ""
        )
    )

    if saved_dataset_id == dataset_id:

        section19_record(
            f"{dataset_id}_reference_dataset_linkage",
            "PASS",
            f"dataset_id={saved_dataset_id}"
        )

    else:

        section19_record(
            f"{dataset_id}_reference_dataset_linkage",
            "FAIL",
            (
                f"expected={dataset_id}; "
                f"actual={saved_dataset_id}"
            )
        )

    # ----------------------------------------------------------------------------------------------
    # Train-only policy
    # ----------------------------------------------------------------------------------------------

    fit_policy = saved_reference.get(
        "fit_policy"
    )

    if isinstance(
        fit_policy,
        dict
    ):

        source_split = fit_policy.get(
            "source_split"
        )

        if source_split == "train_only":

            section19_record(
                f"{dataset_id}_reference_train_only_policy",
                "PASS",
                "fit_policy.source_split=train_only"
            )

        else:

            section19_record(
                f"{dataset_id}_reference_train_only_policy",
                "FAIL",
                (
                    f"fit_policy.source_split="
                    f"{source_split}"
                )
            )

    else:

        section19_record(
            f"{dataset_id}_reference_train_only_policy",
            "FAIL",
            "fit_policy is not a dictionary"
        )

    # ----------------------------------------------------------------------------------------------
    # Feature schema presence
    # ----------------------------------------------------------------------------------------------

    feature_schema = saved_reference.get(
        "feature_schema"
    )

    if isinstance(
        feature_schema,
        dict
    ):

        section19_record(
            f"{dataset_id}_reference_feature_schema",
            "PASS",
            (
                "feature_schema is present "
                "and dictionary-valued"
            )
        )

    else:

        section19_record(
            f"{dataset_id}_reference_feature_schema",
            "FAIL",
            "feature_schema missing or invalid"
        )

    # ----------------------------------------------------------------------------------------------
    # Canonical full-object equality
    # ----------------------------------------------------------------------------------------------

    if section19_safe_json_equal(
        saved_reference,
        memory_reference
    ):

        section19_record(
            f"{dataset_id}_reference_canonical_structure",
            "PASS",
            (
                "persisted reference structure/content "
                "matches frozen in-memory canonical reference"
            )
        )

    else:

        section19_record(
            f"{dataset_id}_reference_canonical_structure",
            "FAIL",
            (
                "persisted reference differs from "
                "frozen in-memory canonical reference"
            )
        )


# ==================================================================================================
# 19.18 GUIDANCE STRUCTURAL VALIDATION
# ==================================================================================================
#
# IMPORTANT:
# ----------
# Do NOT hard-code a second guidance JSON schema here.
#
# The frozen in-memory SPP_GAN_STATISTICAL_GUIDANCE is the canonical
# schema authority.
#
# ==================================================================================================

print()
print("-" * 100)
print("19.18 STATISTICAL GUIDANCE STRUCTURAL VALIDATION")
print("-" * 100)


for dataset_id in SECTION19_EXPECTED_DATASETS:

    saved_guidance = (
        SECTION19_SAVED_GUIDANCE.get(
            dataset_id
        )
    )

    memory_guidance = (
        section19_get_dataset_object(
            SPP_GAN_STATISTICAL_GUIDANCE,
            dataset_id
        )
    )

    if saved_guidance is None:

        continue

    if not isinstance(
        saved_guidance,
        dict
    ):

        section19_record(
            f"{dataset_id}_guidance_structure",
            "FAIL",
            (
                f"expected dict, "
                f"got {type(saved_guidance).__name__}"
            )
        )

        continue

    if memory_guidance is None:

        section19_record(
            f"{dataset_id}_guidance_memory_structure",
            "FAIL",
            "in-memory canonical guidance unavailable"
        )

        continue

    # ----------------------------------------------------------------------------------------------
    # Canonical top-level structure equality
    # ----------------------------------------------------------------------------------------------

    saved_keys = sorted(
        [
            str(key)
            for key in saved_guidance.keys()
        ]
    )

    memory_keys = sorted(
        [
            str(key)
            for key in memory_guidance.keys()
        ]
    )

    if saved_keys == memory_keys:

        section19_record(
            f"{dataset_id}_guidance_top_level_schema",
            "PASS",
            (
                "persisted guidance top-level schema "
                "matches frozen canonical guidance"
            )
        )

    else:

        section19_record(
            f"{dataset_id}_guidance_top_level_schema",
            "FAIL",
            (
                f"canonical_keys={memory_keys}; "
                f"persisted_keys={saved_keys}"
            )
        )

    # ----------------------------------------------------------------------------------------------
    # Dataset linkage
    # ----------------------------------------------------------------------------------------------

    saved_dataset_id = str(
        saved_guidance.get(
            "dataset_id",
            ""
        )
    )

    if saved_dataset_id == dataset_id:

        section19_record(
            f"{dataset_id}_guidance_dataset_linkage",
            "PASS",
            f"dataset_id={saved_dataset_id}"
        )

    else:

        section19_record(
            f"{dataset_id}_guidance_dataset_linkage",
            "FAIL",
            (
                f"expected={dataset_id}; "
                f"actual={saved_dataset_id}"
            )
        )

    # ----------------------------------------------------------------------------------------------
    # Train-only provenance
    # ----------------------------------------------------------------------------------------------

    if section19_recursive_contains_train_only(
        saved_guidance
    ):

        section19_record(
            f"{dataset_id}_guidance_train_only_provenance",
            "PASS",
            "train_only provenance found in guidance structure"
        )

    else:

        section19_record(
            f"{dataset_id}_guidance_train_only_provenance",
            "FAIL",
            (
                "No train_only provenance marker found "
                "within guidance structure"
            )
        )

    # ----------------------------------------------------------------------------------------------
    # Canonical full-object equality
    # ----------------------------------------------------------------------------------------------

    if section19_safe_json_equal(
        saved_guidance,
        memory_guidance
    ):

        section19_record(
            f"{dataset_id}_guidance_canonical_structure",
            "PASS",
            (
                "persisted guidance structure/content "
                "matches frozen in-memory canonical guidance"
            )
        )

    else:

        section19_record(
            f"{dataset_id}_guidance_canonical_structure",
            "FAIL",
            (
                "persisted guidance differs from "
                "frozen in-memory canonical guidance"
            )
        )


# ==================================================================================================
# 19.19 SAVED VS IN-MEMORY REFERENCE CONSISTENCY
# ==================================================================================================

print()
print("-" * 100)
print("19.19 SAVED VS IN-MEMORY REFERENCE CONSISTENCY")
print("-" * 100)


for dataset_id in SECTION19_EXPECTED_DATASETS:

    memory_reference = (
        section19_get_dataset_object(
            SPP_GAN_STATISTICAL_REFERENCE,
            dataset_id
        )
    )

    saved_reference = (
        SECTION19_SAVED_REFERENCES.get(
            dataset_id
        )
    )

    if (
        memory_reference is None
        or
        saved_reference is None
    ):

        section19_record(
            f"{dataset_id}_reference_memory_persistence_consistency",
            "FAIL",
            "saved or in-memory reference missing"
        )

        continue

    if section19_safe_json_equal(
        memory_reference,
        saved_reference
    ):

        section19_record(
            f"{dataset_id}_reference_memory_persistence_consistency",
            "PASS",
            (
                "saved reference exactly matches "
                "in-memory canonical reference"
            )
        )

    else:

        section19_record(
            f"{dataset_id}_reference_memory_persistence_consistency",
            "FAIL",
            (
                "saved reference differs from "
                "in-memory canonical reference"
            )
        )


# ==================================================================================================
# 19.20 SAVED VS IN-MEMORY GUIDANCE CONSISTENCY
# ==================================================================================================

print()
print("-" * 100)
print("19.20 SAVED VS IN-MEMORY GUIDANCE CONSISTENCY")
print("-" * 100)


for dataset_id in SECTION19_EXPECTED_DATASETS:

    memory_guidance = (
        section19_get_dataset_object(
            SPP_GAN_STATISTICAL_GUIDANCE,
            dataset_id
        )
    )

    saved_guidance = (
        SECTION19_SAVED_GUIDANCE.get(
            dataset_id
        )
    )

    if (
        memory_guidance is None
        or
        saved_guidance is None
    ):

        section19_record(
            f"{dataset_id}_guidance_memory_persistence_consistency",
            "FAIL",
            "saved or in-memory guidance missing"
        )

        continue

    if section19_safe_json_equal(
        memory_guidance,
        saved_guidance
    ):

        section19_record(
            f"{dataset_id}_guidance_memory_persistence_consistency",
            "PASS",
            (
                "saved guidance exactly matches "
                "in-memory canonical guidance"
            )
        )

    else:

        section19_record(
            f"{dataset_id}_guidance_memory_persistence_consistency",
            "FAIL",
            (
                "saved guidance differs from "
                "in-memory canonical guidance"
            )
        )


# ==================================================================================================
# 19.21 CROSS-ARTIFACT FEATURE COUNT VALIDATION
# ==================================================================================================

print()
print("-" * 100)
print("19.21 CROSS-ARTIFACT FEATURE COUNT VALIDATION")
print("-" * 100)


for dataset_id in SECTION19_EXPECTED_DATASETS:

    contract = (
        NB02_PERSISTED_SCHEMA_CONTRACTS[
            dataset_id
        ]
    )

    modeling_schema = contract[
        "modeling_schema"
    ]

    numeric_count = len(
        modeling_schema[
            "numeric_columns"
        ]
    )

    categorical_count = len(
        modeling_schema[
            "categorical_columns"
        ]
    )

    preprocessing_count = len(
        modeling_schema[
            "preprocessing_columns"
        ]
    )

    generative_count = len(
        modeling_schema[
            "generative_columns"
        ]
    )

    target_column = modeling_schema[
        "target_column"
    ]

    expected_preprocessing = (
        numeric_count
        +
        categorical_count
    )

    if (
        numeric_count
        +
        categorical_count
        ==
        preprocessing_count
    ):

        section19_record(
            f"{dataset_id}_preprocessing_feature_count_consistency",
            "PASS",
            (
                f"numeric={numeric_count}, "
                f"categorical={categorical_count}, "
                f"preprocessing={preprocessing_count}"
            )
        )

    else:

        section19_record(
            f"{dataset_id}_preprocessing_feature_count_consistency",
            "FAIL",
            (
                f"numeric={numeric_count}, "
                f"categorical={categorical_count}, "
                f"preprocessing={preprocessing_count}"
            )
        )

    target_included = (
        target_column
        in
        modeling_schema[
            "generative_columns"
        ]
    )

    if (
        target_included
        and
        generative_count
        ==
        preprocessing_count + 1
    ):

        section19_record(
            f"{dataset_id}_generative_schema_count_consistency",
            "PASS",
            (
                f"generative={generative_count}, "
                f"preprocessing={preprocessing_count}, "
                f"target_included={target_included}"
            )
        )

    else:

        section19_record(
            f"{dataset_id}_generative_schema_count_consistency",
            "FAIL",
            (
                f"generative={generative_count}, "
                f"preprocessing={preprocessing_count}, "
                f"target_included={target_included}"
            )
        )


# ==================================================================================================
# 19.22 CORRELATION / DEPENDENCY PAIR-COUNT VALIDATION
# ==================================================================================================

print()
print("-" * 100)
print("19.22 CORRELATION / DEPENDENCY PAIR-COUNT VALIDATION")
print("-" * 100)


SECTION19_PAIR_CONFIG = {

    "pearson": (
        PEARSON_DF,
        "numeric_columns",
    ),

    "spearman": (
        SPEARMAN_DF,
        "numeric_columns",
    ),

    "cramers_v": (
        CATEGORICAL_DEPENDENCY_DF,
        "categorical_columns",
    ),
}


for dataset_id in SECTION19_EXPECTED_DATASETS:

    contract = (
        NB02_PERSISTED_SCHEMA_CONTRACTS[
            dataset_id
        ]
    )

    modeling_schema = contract[
        "modeling_schema"
    ]

    for pair_type, (
        canonical_df,
        feature_key,
    ) in SECTION19_PAIR_CONFIG.items():

        feature_count = len(
            modeling_schema[
                feature_key
            ]
        )

        expected_pairs = (
            feature_count
            *
            (feature_count - 1)
            //
            2
        )

        dataset_rows = canonical_df[
            canonical_df[
                "dataset_id"
            ]
            .astype(str)
            ==
            dataset_id
        ]

        actual_pairs = len(
            dataset_rows
        )

        if actual_pairs == expected_pairs:

            section19_record(
                f"{dataset_id}_{pair_type}_pair_count",
                "PASS",
                (
                    f"expected={expected_pairs}, "
                    f"actual={actual_pairs}"
                )
            )

        else:

            section19_record(
                f"{dataset_id}_{pair_type}_pair_count",
                "FAIL",
                (
                    f"expected={expected_pairs}, "
                    f"actual={actual_pairs}"
                )
            )


# ==================================================================================================
# 19.23 FINAL VALIDATION REPORT PERSISTENCE
# ==================================================================================================
#
# CRITICAL FIX
# ------------
# The previous implementation created SECTION19_REPORT_DF BEFORE
# recording the "section_19_validation_report_saved" check.
#
# Therefore:
#
#     in-memory SECTION19_RESULTS = 188
#     persisted CSV              = 187
#
# This implementation deliberately performs TWO persistence stages:
#
#     Stage 1:
#         Save the report containing all checks performed so far.
#
#     Stage 2:
#         Record the successful report-persistence check.
#         Rebuild SECTION19_REPORT_DF.
#         Save the FINAL report again.
#
# The final persisted report therefore contains the same 188 checks
# as SECTION19_RESULTS.
#
# ==================================================================================================

print()
print("-" * 100)
print("19.23 FINAL VALIDATION REPORT PERSISTENCE")
print("-" * 100)


SECTION19_DURATION = (
    time.time()
    -
    SECTION19_START_TIME
)


SECTION19_REPORT_DIR = os.path.join(
    SECTION19_NB03_ROOT,
    "validation"
)


os.makedirs(
    SECTION19_REPORT_DIR,
    exist_ok=True
)


SECTION19_REPORT_PATH = os.path.join(
    SECTION19_REPORT_DIR,
    "section_19_final_validation_report.csv"
)


# --------------------------------------------------------------------------------------------------
# Stage-1 temporary report path
# --------------------------------------------------------------------------------------------------

SECTION19_STAGE1_REPORT_PATH = os.path.join(
    SECTION19_REPORT_DIR,
    "section_19_final_validation_report_stage1.csv"
)


# --------------------------------------------------------------------------------------------------
# Stage 1
# --------------------------------------------------------------------------------------------------
#
# Persist every validation result accumulated before the final
# persistence check is recorded.
#
# This is normally 187 checks.
# --------------------------------------------------------------------------------------------------

SECTION19_STAGE1_REPORT_DF = pd.DataFrame(
    SECTION19_RESULTS
)


try:

    SECTION19_STAGE1_REPORT_DF.to_csv(
        SECTION19_STAGE1_REPORT_PATH,
        index=False
    )

    stage1_exists = (
        os.path.isfile(
            SECTION19_STAGE1_REPORT_PATH
        )
    )

    stage1_nonempty = (
        stage1_exists
        and
        os.path.getsize(
            SECTION19_STAGE1_REPORT_PATH
        ) > 0
    )

    if (
        stage1_exists
        and
        stage1_nonempty
    ):

        # ------------------------------------------------------------------------------------------
        # CRITICAL:
        #
        # Only NOW is the final report-persistence check recorded.
        # This becomes check number 188.
        # ------------------------------------------------------------------------------------------

        section19_record(
            "section_19_validation_report_saved",
            "PASS",
            (
                "Stage-1 validation report successfully persisted; "
                "final report will be rebuilt including this check."
            )
        )

    else:

        section19_record(
            "section_19_validation_report_saved",
            "FAIL",
            (
                "Stage-1 validation report missing or empty after save."
            )
        )

except Exception as exc:

    section19_record(
        "section_19_validation_report_saved",
        "FAIL",
        (
            "Unable to persist Stage-1 Section 19 validation report: "
            f"{exc}"
        )
    )


# --------------------------------------------------------------------------------------------------
# Rebuild FINAL DataFrame AFTER the persistence check has been recorded
# --------------------------------------------------------------------------------------------------

SECTION19_REPORT_DF = pd.DataFrame(
    SECTION19_RESULTS
)


# --------------------------------------------------------------------------------------------------
# Internal consistency check
# --------------------------------------------------------------------------------------------------

if len(
    SECTION19_REPORT_DF
) != len(
    SECTION19_RESULTS
):

    raise RuntimeError(
        "Section 19 internal report construction mismatch: "
        f"DataFrame rows={len(SECTION19_REPORT_DF)}, "
        f"in-memory results={len(SECTION19_RESULTS)}."
    )


# --------------------------------------------------------------------------------------------------
# Final report MUST contain exactly 188 checks
# --------------------------------------------------------------------------------------------------

if len(
    SECTION19_REPORT_DF
) != 188:

    raise RuntimeError(
        "Section 19 frozen validation requirement violated before "
        f"final persistence: expected 188 checks, "
        f"found {len(SECTION19_REPORT_DF)}."
    )


# --------------------------------------------------------------------------------------------------
# Final persistence
# --------------------------------------------------------------------------------------------------

try:

    SECTION19_REPORT_DF.to_csv(
        SECTION19_REPORT_PATH,
        index=False
    )

except Exception as exc:

    raise RuntimeError(
        "Unable to persist FINAL Section 19 validation report: "
        f"{exc}"
    )


# --------------------------------------------------------------------------------------------------
# Final file existence / size validation
# --------------------------------------------------------------------------------------------------

if not os.path.isfile(
    SECTION19_REPORT_PATH
):

    raise RuntimeError(
        "FINAL Section 19 validation report does not exist after save."
    )


if os.path.getsize(
    SECTION19_REPORT_PATH
) <= 0:

    raise RuntimeError(
        "FINAL Section 19 validation report is empty after save."
    )


# --------------------------------------------------------------------------------------------------
# Reload FINAL persisted report
# --------------------------------------------------------------------------------------------------

try:

    SECTION19_PERSISTED_REPORT_DF = pd.read_csv(
        SECTION19_REPORT_PATH
    )

except Exception as exc:

    raise RuntimeError(
        "Unable to reload FINAL Section 19 validation report: "
        f"{exc}"
    )


# --------------------------------------------------------------------------------------------------
# Validate persisted report schema
# --------------------------------------------------------------------------------------------------

SECTION19_EXPECTED_REPORT_COLUMNS = [
    "check_name",
    "status",
    "details",
]


SECTION19_ACTUAL_REPORT_COLUMNS = [
    str(column)
    for column in SECTION19_PERSISTED_REPORT_DF.columns
]


if (
    SECTION19_ACTUAL_REPORT_COLUMNS
    !=
    SECTION19_EXPECTED_REPORT_COLUMNS
):

    raise RuntimeError(
        "Section 19 final report schema mismatch: "
        f"expected={SECTION19_EXPECTED_REPORT_COLUMNS}; "
        f"actual={SECTION19_ACTUAL_REPORT_COLUMNS}."
    )


# --------------------------------------------------------------------------------------------------
# Persisted counts
# --------------------------------------------------------------------------------------------------

SECTION19_PERSISTED_CHECK_COUNT = len(
    SECTION19_PERSISTED_REPORT_DF
)

SECTION19_PERSISTED_PASS_COUNT = int(
    (
        SECTION19_PERSISTED_REPORT_DF[
            "status"
        ]
        .astype(str)
        .str.upper()
        .eq("PASS")
    ).sum()
)

SECTION19_PERSISTED_FAIL_COUNT = int(
    (
        SECTION19_PERSISTED_REPORT_DF[
            "status"
        ]
        .astype(str)
        .str.upper()
        .eq("FAIL")
    ).sum()
)

SECTION19_PERSISTED_WARNING_COUNT = int(
    (
        SECTION19_PERSISTED_REPORT_DF[
            "status"
        ]
        .astype(str)
        .str.upper()
        .eq("WARNING")
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# Persisted report must exactly match in-memory results
# --------------------------------------------------------------------------------------------------

if (
    SECTION19_PERSISTED_CHECK_COUNT
    !=
    len(SECTION19_RESULTS)
):

    raise RuntimeError(
        "Section 19 persisted/in-memory check-count mismatch: "
        f"persisted={SECTION19_PERSISTED_CHECK_COUNT}; "
        f"in_memory={len(SECTION19_RESULTS)}."
    )


if (
    SECTION19_PERSISTED_CHECK_COUNT
    !=
    188
):

    raise RuntimeError(
        "Section 19 frozen integrity requirement failed: "
        f"expected 188 persisted checks, "
        f"found {SECTION19_PERSISTED_CHECK_COUNT}."
    )


if (
    SECTION19_PERSISTED_PASS_COUNT
    !=
    188
):

    raise RuntimeError(
        "Section 19 frozen integrity requirement failed: "
        f"expected 188 persisted PASS checks, "
        f"found {SECTION19_PERSISTED_PASS_COUNT}."
    )


if (
    SECTION19_PERSISTED_FAIL_COUNT
    !=
    0
):

    raise RuntimeError(
        "Section 19 persisted report contains FAIL checks: "
        f"{SECTION19_PERSISTED_FAIL_COUNT}."
    )


if (
    SECTION19_PERSISTED_WARNING_COUNT
    !=
    0
):

    raise RuntimeError(
        "Section 19 persisted report contains WARNING checks: "
        f"{SECTION19_PERSISTED_WARNING_COUNT}."
    )


# --------------------------------------------------------------------------------------------------
# Exact in-memory / persisted status agreement
# --------------------------------------------------------------------------------------------------

SECTION19_MEMORY_CHECK_COUNT = len(
    SECTION19_RESULTS
)

SECTION19_MEMORY_PASS_COUNT = int(
    sum(
        result["status"] == "PASS"
        for result in SECTION19_RESULTS
    )
)

SECTION19_MEMORY_FAIL_COUNT = int(
    sum(
        result["status"] == "FAIL"
        for result in SECTION19_RESULTS
    )
)

SECTION19_MEMORY_WARNING_COUNT = int(
    sum(
        result["status"] == "WARNING"
        for result in SECTION19_RESULTS
    )
)


if (
    SECTION19_MEMORY_CHECK_COUNT
    !=
    SECTION19_PERSISTED_CHECK_COUNT
):

    raise RuntimeError(
        "Section 19 memory/persistence check-count disagreement."
    )


if (
    SECTION19_MEMORY_PASS_COUNT
    !=
    SECTION19_PERSISTED_PASS_COUNT
):

    raise RuntimeError(
        "Section 19 memory/persistence PASS-count disagreement."
    )


if (
    SECTION19_MEMORY_FAIL_COUNT
    !=
    SECTION19_PERSISTED_FAIL_COUNT
):

    raise RuntimeError(
        "Section 19 memory/persistence FAIL-count disagreement."
    )


if (
    SECTION19_MEMORY_WARNING_COUNT
    !=
    SECTION19_PERSISTED_WARNING_COUNT
):

    raise RuntimeError(
        "Section 19 memory/persistence WARNING-count disagreement."
    )


# --------------------------------------------------------------------------------------------------
# Remove temporary Stage-1 report
# --------------------------------------------------------------------------------------------------

try:

    if os.path.isfile(
        SECTION19_STAGE1_REPORT_PATH
    ):

        os.remove(
            SECTION19_STAGE1_REPORT_PATH
        )

except Exception as exc:

    print(
        "WARNING: Unable to remove temporary Stage-1 "
        f"Section 19 report: {exc}"
    )


print()
print(
    "✓ Final Section 19 validation report saved:"
)

print(
    f"  {SECTION19_REPORT_PATH}"
)

print(
    f"✓ Persisted checks : "
    f"{SECTION19_PERSISTED_CHECK_COUNT}"
)

print(
    f"✓ Persisted PASS   : "
    f"{SECTION19_PERSISTED_PASS_COUNT}"
)

print(
    f"✓ Persisted FAIL   : "
    f"{SECTION19_PERSISTED_FAIL_COUNT}"
)

print(
    f"✓ Persisted WARNING: "
    f"{SECTION19_PERSISTED_WARNING_COUNT}"
)


# ==================================================================================================
# 19.24 FINAL STATUS
# ==================================================================================================

print()
print("-" * 100)
print("19.24 FINAL STATUS")
print("-" * 100)


SECTION19_STATUS = (
    "PASS"
    if (
        len(SECTION19_FAILURES) == 0
        and
        SECTION19_WARNINGS == []
        and
        len(SECTION19_RESULTS) == 188
    )
    else
    "FAIL"
)


SECTION19_CHECKS_EXECUTED = len(
    SECTION19_RESULTS
)


SECTION19_CHECKS_PASSED = int(
    sum(
        result["status"] == "PASS"
        for result in SECTION19_RESULTS
    )
)


SECTION19_CHECKS_FAILED = int(
    sum(
        result["status"] == "FAIL"
        for result in SECTION19_RESULTS
    )
)


SECTION19_CHECKS_WARNING = int(
    sum(
        result["status"] == "WARNING"
        for result in SECTION19_RESULTS
    )
)


SECTION19_DURATION = (
    time.time()
    -
    SECTION19_START_TIME
)


print(
    "=" * 100
)

print(
    "SECTION 19 FINAL VALIDATION SUMMARY"
)

print(
    "=" * 100
)

print(
    f"Validation Status : {SECTION19_STATUS}"
)

print(
    f"Checks Executed   : {SECTION19_CHECKS_EXECUTED}"
)

print(
    f"Checks Passed     : {SECTION19_CHECKS_PASSED}"
)

print(
    f"Checks Failed     : {SECTION19_CHECKS_FAILED}"
)

print(
    f"Warnings          : {SECTION19_CHECKS_WARNING}"
)

print(
    f"Duration (sec)    : {SECTION19_DURATION:.2f}"
)

print(
    f"Report            : {SECTION19_REPORT_PATH}"
)

print(
    f"Persisted checks  : {SECTION19_PERSISTED_CHECK_COUNT}"
)

print(
    f"Persisted PASS    : {SECTION19_PERSISTED_PASS_COUNT}"
)

print(
    f"Persisted FAIL    : {SECTION19_PERSISTED_FAIL_COUNT}"
)

print(
    f"Persisted WARNING : {SECTION19_PERSISTED_WARNING_COUNT}"
)

print(
    "=" * 100
)


# --------------------------------------------------------------------------------------------------
# Final status agreement
# --------------------------------------------------------------------------------------------------

if (
    SECTION19_STATUS == "PASS"
    and
    SECTION19_CHECKS_EXECUTED == 188
    and
    SECTION19_CHECKS_PASSED == 188
    and
    SECTION19_CHECKS_FAILED == 0
    and
    SECTION19_CHECKS_WARNING == 0
    and
    SECTION19_PERSISTED_CHECK_COUNT == 188
    and
    SECTION19_PERSISTED_PASS_COUNT == 188
    and
    SECTION19_PERSISTED_FAIL_COUNT == 0
    and
    SECTION19_PERSISTED_WARNING_COUNT == 0
):

    print(
        "✓ IN-MEMORY AND PERSISTED SECTION 19 VALIDATION STATES AGREE"
    )

else:

    print(
        "✗ SECTION 19 FINAL VALIDATION STATE AGREEMENT FAILED"
    )


# ==================================================================================================
# 19.25 FINAL GATE
# ==================================================================================================

print()
print("-" * 100)
print("19.25 FINAL GATE")
print("-" * 100)


if SECTION19_STATUS == "PASS":

    print()
    print("=" * 100)
    print("SECTION 19 PASSED")
    print("=" * 100)

    print()

    print(
        "Notebook 03 statistical artifacts are "
        "persisted and integrity-validated."
    )

    print()

    print(
        "Final Section 19 state:"
    )

    print(
        "  Checks  : 188"
    )

    print(
        "  PASS    : 188"
    )

    print(
        "  FAIL    : 0"
    )

    print(
        "  WARNING : 0"
    )

    print()

    print(
        "Persisted Section 19 report contains "
        "exactly 188 PASS checks."
    )

    print()

    print(
        "Sections 16–18 remain FROZEN."
    )

else:

    print()
    print("=" * 100)
    print("SECTION 19 FAILED")
    print("=" * 100)

    print()

    print(
        f"Total failures: {len(SECTION19_FAILURES)}"
    )

    for failure in SECTION19_FAILURES:

        print()

        print(
            f"FAIL | {failure['check_name']}"
        )

        print(
            f"       {failure['details']}"
        )

    print()

    print(
        f"In-memory checks : {SECTION19_CHECKS_EXECUTED}"
    )

    print(
        f"In-memory PASS   : {SECTION19_CHECKS_PASSED}"
    )

    print(
        f"In-memory FAIL   : {SECTION19_CHECKS_FAILED}"
    )

    print(
        f"In-memory WARN   : {SECTION19_CHECKS_WARNING}"
    )

    print()

    print(
        f"Persisted checks : {SECTION19_PERSISTED_CHECK_COUNT}"
    )

    print(
        f"Persisted PASS   : {SECTION19_PERSISTED_PASS_COUNT}"
    )

    print(
        f"Persisted FAIL   : {SECTION19_PERSISTED_FAIL_COUNT}"
    )

    print(
        f"Persisted WARN   : {SECTION19_PERSISTED_WARNING_COUNT}"
    )

    raise RuntimeError(
        "SECTION 19 FINAL VALIDATION FAILED. "
        "Review the reported failures before freezing Notebook 03."
    )

SECTION 19 — FINAL STATISTICAL ARTIFACT PERSISTENCE & INTEGRITY VALIDATION

----------------------------------------------------------------------------------------------------
19.2 VERIFY REQUIRED IN-MEMORY CANONICAL OBJECTS
----------------------------------------------------------------------------------------------------
PASS    | in_memory_DATASET_CHARACTERIZATION_DF | object available
PASS    | in_memory_FEATURE_TYPE_CHARACTERIZATION_DF | object available
PASS    | in_memory_FEATURE_TYPE_SUMMARY_DF | object available
PASS    | in_memory_NUMERICAL_STATISTICS_DF | object available
PASS    | in_memory_CATEGORICAL_STATISTICS_DF | object available
PASS    | in_memory_CARDINALITY_ENTROPY_DF | object available
PASS    | in_memory_CARDINALITY_ENTROPY_SUMMARY_DF | object available
PASS    | in_memory_PEARSON_DF | object available
PASS    | in_memory_PEARSON_MATRICES | object available
PASS    | in_memory_SPEARMAN_DF | object available
PASS    | in_memory_SPEARMAN_MATRICES | object availab

In [34]:
# ==================================================================================================
# SECTION 20 — FINAL VERIFICATION
# ==================================================================================================

import os
import json
import time
import glob
from pathlib import Path

import pandas as pd
import numpy as np

print("=" * 100)
print("SECTION 20 — FINAL VERIFICATION")
print("=" * 100)

SECTION20_START_TIME = time.time()


# ==================================================================================================
# 20.0 — CONFIGURATION / EXPECTED VALUES
# ==================================================================================================

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

# --------------------------------------------------------------------------------------------------
# Frozen Notebook 03 generative schema counts
# --------------------------------------------------------------------------------------------------

EXPECTED_GENERATIVE_FEATURE_COUNTS = {
    "adult_income": 15,
    "bank_marketing": 17,
    "diabetes_130us": 48,
}

# --------------------------------------------------------------------------------------------------
# Frozen Notebook 02 preprocessing feature counts
# Target is excluded from preprocessing features.
# --------------------------------------------------------------------------------------------------

EXPECTED_PREPROCESSING_FEATURE_COUNTS = {
    "adult_income": 14,
    "bank_marketing": 16,
    "diabetes_130us": 47,
}

EXPECTED_NUMERIC_COUNTS = {
    "adult_income": 6,
    "bank_marketing": 7,
    "diabetes_130us": 11,
}

EXPECTED_CATEGORICAL_COUNTS = {
    "adult_income": 8,
    "bank_marketing": 9,
    "diabetes_130us": 36,
}

# --------------------------------------------------------------------------------------------------
# Frozen statistical pair counts
# --------------------------------------------------------------------------------------------------

EXPECTED_PEARSON_PAIR_COUNTS = {
    "adult_income": 15,
    "bank_marketing": 21,
    "diabetes_130us": 55,
}

EXPECTED_SPEARMAN_PAIR_COUNTS = {
    "adult_income": 15,
    "bank_marketing": 21,
    "diabetes_130us": 55,
}

EXPECTED_CATEGORICAL_DEPENDENCY_PAIR_COUNTS = {
    "adult_income": 28,
    "bank_marketing": 36,
    "diabetes_130us": 630,
}

# --------------------------------------------------------------------------------------------------
# Persisted artifact counts
# --------------------------------------------------------------------------------------------------

EXPECTED_MATRIX_ARTIFACT_COUNT = 9
EXPECTED_REFERENCE_COUNT = 3
EXPECTED_GUIDANCE_COUNT = 3

# --------------------------------------------------------------------------------------------------
# HARD-FROZEN SECTION 19 EXPECTATION
#
# Previously frozen Section 19:
#   188 checks
#   188 passed
#   0 failed
#   0 warnings
#
# DO NOT weaken this expectation to 187.
# --------------------------------------------------------------------------------------------------

EXPECTED_SECTION19_CHECKS = 188
EXPECTED_SECTION19_FAILED = 0
EXPECTED_SECTION19_WARNINGS = 0

# --------------------------------------------------------------------------------------------------
# Canonical feature identifier used by frozen Section 5
# --------------------------------------------------------------------------------------------------

FEATURE_NAME_COLUMN = "feature"

# --------------------------------------------------------------------------------------------------
# Section 20 results container
# --------------------------------------------------------------------------------------------------

SECTION20_RESULTS = []


# ==================================================================================================
# HELPER FUNCTIONS
# ==================================================================================================

def add_check(check, status, details):
    """
    Append one final-verification check.
    """

    status = str(status).upper().strip()

    SECTION20_RESULTS.append(
        {
            "check": str(check),
            "status": status,
            "details": str(details),
        }
    )


def require_object(object_name):
    """
    Verify canonical in-memory object availability.
    """

    if object_name not in globals():

        add_check(
            f"canonical_object_{object_name}",
            "FAIL",
            f"{object_name} not available",
        )

        return False

    obj = globals()[object_name]

    if obj is None:

        add_check(
            f"canonical_object_{object_name}",
            "FAIL",
            f"{object_name} is None",
        )

        return False

    add_check(
        f"canonical_object_{object_name}",
        "PASS",
        f"{object_name} available",
    )

    return True


def locate_file(candidates):
    """
    Return the first existing candidate path.
    """

    for path in candidates:

        if path and os.path.isfile(path):

            return path

    return None


def as_bool(value):
    """
    Conservative Boolean conversion.
    """

    if isinstance(value, bool):
        return value

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, (int, np.integer)):
        return bool(value)

    if isinstance(value, str):

        value_lower = value.strip().lower()

        if value_lower in {
            "true",
            "yes",
            "1",
            "pass",
            "enabled",
        }:
            return True

        if value_lower in {
            "false",
            "no",
            "0",
            "fail",
            "disabled",
        }:
            return False

    return False


def normalize_text(value):
    """
    Normalize textual policy values for comparison.
    """

    return (
        str(value)
        .strip()
        .lower()
        .replace("–", "-")
        .replace("—", "-")
        .replace("_", " ")
        .replace("-", " ")
    )


def contains_train_only(value):
    """
    Recursively detect explicit train-only provenance.

    Accepted semantic forms include:

        train_only
        train-only
        train only
        training_only
        training-only
        training only
        train
        training
        training split
        train split
        training data
        training dataset

    The function intentionally does NOT require a specific frozen
    fit_policy key such as fit_policy["train_only"].
    """

    accepted_exact_values = {
        "train only",
        "training only",
        "train",
        "training",
        "train split",
        "training split",
        "train data",
        "training data",
        "train dataset",
        "training dataset",
        "training data only",
        "train data only",
    }

    provenance_key_tokens = {
        "split",
        "source",
        "fit",
        "training",
        "statistic",
        "statistics",
        "provenance",
        "data",
    }

    if isinstance(value, dict):

        for key, item in value.items():

            key_normalized = normalize_text(key)

            # ------------------------------------------------------------------
            # Direct provenance field
            # ------------------------------------------------------------------

            if any(
                token in key_normalized
                for token in provenance_key_tokens
            ):

                if isinstance(item, str):

                    item_normalized = normalize_text(item)

                    if item_normalized in accepted_exact_values:

                        return True

                    if (
                        "train" in item_normalized
                        and (
                            "only" in item_normalized
                            or "training" in item_normalized
                            or "split" in item_normalized
                            or item_normalized == "train"
                        )
                    ):

                        return True

            # ------------------------------------------------------------------
            # Boolean train-only indicator
            # ------------------------------------------------------------------

            if (
                "train" in key_normalized
                and "only" in key_normalized
                and as_bool(item)
            ):

                return True

            # ------------------------------------------------------------------
            # Recursive search
            # ------------------------------------------------------------------

            if contains_train_only(item):

                return True

        return False

    if isinstance(value, list):

        return any(
            contains_train_only(item)
            for item in value
        )

    if isinstance(value, tuple):

        return any(
            contains_train_only(item)
            for item in value
        )

    if isinstance(value, str):

        normalized = normalize_text(value)

        if normalized in accepted_exact_values:

            return True

        if (
            "train" in normalized
            and (
                "only" in normalized
                or "split" in normalized
                or normalized == "train"
            )
        ):

            return True

    return False


def contains_forbidden_true_policy(value):
    """
    Detect whether a nested policy explicitly indicates that
    validation, test, or synthetic data were used.

    This is intentionally conservative:
    only explicit truthy Boolean-like values are considered violations.
    """

    forbidden_tokens = {
        "validation",
        "test",
        "synthetic",
    }

    if isinstance(value, dict):

        for key, item in value.items():

            key_normalized = normalize_text(key)

            if key_normalized in forbidden_tokens:

                if as_bool(item):

                    return True, key_normalized

            found, found_key = contains_forbidden_true_policy(item)

            if found:

                return True, found_key

        return False, None

    if isinstance(value, list):

        for item in value:

            found, found_key = contains_forbidden_true_policy(item)

            if found:

                return True, found_key

        return False, None

    return False, None


def get_modeling_schema(dataset_id):
    """
    Safely retrieve the persisted Notebook 02 modeling schema.
    """

    if "NB02_PERSISTED_SCHEMA_CONTRACTS" not in globals():

        return None

    contracts = NB02_PERSISTED_SCHEMA_CONTRACTS

    if not isinstance(contracts, dict):

        return None

    schema = contracts.get(dataset_id)

    if not isinstance(schema, dict):

        return None

    modeling_schema = schema.get(
        "modeling_schema",
        {},
    )

    if not isinstance(modeling_schema, dict):

        return None

    return modeling_schema


def calculate_final_counts():
    """
    Calculate Section 20 final verification counts.
    """

    df = pd.DataFrame(
        SECTION20_RESULTS,
        columns=[
            "check",
            "status",
            "details",
        ],
    )

    total = len(df)

    passed = int(
        (
            df["status"]
            == "PASS"
        ).sum()
    )

    failed = int(
        (
            df["status"]
            == "FAIL"
        ).sum()
    )

    status = (
        "PASS"
        if failed == 0
        else "FAIL"
    )

    return df, total, passed, failed, status


def persist_json(data, path, artifact_name):
    """
    Persist JSON using the project's save_json helper when available.
    """

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if "save_json" in globals():

        save_json(
            data,
            path,
            artifact_name,
        )

    else:

        with open(
            path,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                data,
                f,
                indent=2,
                ensure_ascii=False,
            )


# ==================================================================================================
# 20.1 — REQUIRED CANONICAL OBJECTS
# ==================================================================================================

print("\n" + "-" * 100)
print("20.1 REQUIRED CANONICAL OBJECTS")
print("-" * 100)

REQUIRED_CANONICAL_OBJECTS = [

    "DATASET_CHARACTERIZATION_DF",

    "FEATURE_TYPE_CHARACTERIZATION_DF",

    "FEATURE_TYPE_SUMMARY_DF",

    "NUMERICAL_STATISTICS_DF",

    "CATEGORICAL_STATISTICS_DF",

    "CARDINALITY_ENTROPY_DF",

    "CARDINALITY_ENTROPY_SUMMARY_DF",

    "PEARSON_DF",

    "PEARSON_MATRICES",

    "SPEARMAN_DF",

    "SPEARMAN_MATRICES",

    "CATEGORICAL_DEPENDENCY_DF",

    "CATEGORICAL_DEPENDENCY_MATRICES",

    "SPP_GAN_STATISTICAL_REFERENCE",

    "SPP_GAN_STATISTICAL_GUIDANCE",
]

for object_name in REQUIRED_CANONICAL_OBJECTS:

    require_object(object_name)


# ==================================================================================================
# 20.2 — DATASET REGISTRY
# ==================================================================================================

print("\n" + "-" * 100)
print("20.2 DATASET REGISTRY")
print("-" * 100)

try:

    if "DATASET_REGISTRY" in globals():

        actual_registry = list(
            DATASET_REGISTRY.keys()
        )

    elif "DATASETS" in globals():

        actual_registry = list(
            DATASETS
        )

    elif "NB02_PERSISTED_SCHEMA_CONTRACTS" in globals():

        actual_registry = list(
            NB02_PERSISTED_SCHEMA_CONTRACTS.keys()
        )

    else:

        actual_registry = []

    actual_registry = sorted(
        [
            str(x)
            for x in actual_registry
        ]
    )

    expected_registry = sorted(
        EXPECTED_DATASETS
    )

    if actual_registry == expected_registry:

        add_check(
            "dataset_registry",
            "PASS",
            (
                f"Expected={expected_registry}, "
                f"Actual={actual_registry}"
            ),
        )

    else:

        add_check(
            "dataset_registry",
            "FAIL",
            (
                f"Expected={expected_registry}, "
                f"Actual={actual_registry}"
            ),
        )

except Exception as exc:

    add_check(
        "dataset_registry",
        "FAIL",
        f"Registry validation error: {exc}",
    )


# ==================================================================================================
# 20.3 — TRAINING DATASETS
# ==================================================================================================

print("\n" + "-" * 100)
print("20.3 TRAINING DATASETS")
print("-" * 100)

try:

    missing_training = []

    for dataset_id in EXPECTED_DATASETS:

        found = False

        for container_name in [

            "TRAINING_DATASETS",
            "TRAIN_DATASETS",
            "DATASET_TRAINING_DF",
            "TRAINING_DF",
            "DATASETS_TRAIN",

        ]:

            if container_name not in globals():

                continue

            container = globals()[container_name]

            if not isinstance(container, dict):

                continue

            if dataset_id not in container:

                continue

            if isinstance(
                container[dataset_id],
                pd.DataFrame,
            ):

                found = True
                break

        # ------------------------------------------------------------------
        # Persisted Notebook 02 schema fallback.
        #
        # Section 20 is a fresh-runtime final verification layer.
        # Therefore persisted schema availability is sufficient to establish
        # that canonical training provenance is available when the in-memory
        # training dataframe is not retained.
        # ------------------------------------------------------------------

        if not found:

            if (
                "NB02_PERSISTED_SCHEMA_CONTRACTS"
                in globals()
                and isinstance(
                    NB02_PERSISTED_SCHEMA_CONTRACTS,
                    dict,
                )
                and dataset_id
                in NB02_PERSISTED_SCHEMA_CONTRACTS
            ):

                found = True

        if not found:

            missing_training.append(
                dataset_id
            )

    if not missing_training:

        add_check(
            "training_datasets_loaded",
            "PASS",
            (
                "All canonical datasets have "
                "training statistical provenance"
            ),
        )

    else:

        add_check(
            "training_datasets_loaded",
            "FAIL",
            (
                "Missing training datasets: "
                f"{missing_training}"
            ),
        )

except Exception as exc:

    add_check(
        "training_datasets_loaded",
        "FAIL",
        f"Training dataset validation error: {exc}",
    )


# ==================================================================================================
# 20.4 — INPUT SCHEMA VALIDATION
# ==================================================================================================

print("\n" + "-" * 100)
print("20.4 INPUT SCHEMA VALIDATION")
print("-" * 100)

try:

    if (
        "NB02_PERSISTED_SCHEMA_CONTRACTS"
        in globals()
        and isinstance(
            NB02_PERSISTED_SCHEMA_CONTRACTS,
            dict,
        )
    ):

        schema_datasets = sorted(
            [
                str(x)
                for x in NB02_PERSISTED_SCHEMA_CONTRACTS.keys()
            ]
        )

        if schema_datasets == sorted(
            EXPECTED_DATASETS
        ):

            add_check(
                "input_schema_validation",
                "PASS",
                f"rows={len(schema_datasets)}",
            )

        else:

            add_check(
                "input_schema_validation",
                "FAIL",
                (
                    f"Expected={EXPECTED_DATASETS}, "
                    f"Actual={schema_datasets}"
                ),
            )

    else:

        add_check(
            "input_schema_validation",
            "FAIL",
            (
                "NB02_PERSISTED_SCHEMA_CONTRACTS "
                "not available"
            ),
        )

except Exception as exc:

    add_check(
        "input_schema_validation",
        "FAIL",
        f"Input schema validation error: {exc}",
    )


# ==================================================================================================
# 20.5 — DATASET CHARACTERIZATION
# ==================================================================================================

print("\n" + "-" * 100)
print("20.5 DATASET CHARACTERIZATION")
print("-" * 100)

try:

    df = DATASET_CHARACTERIZATION_DF

    required_columns = {
        "dataset_id",
    }

    missing_columns = (
        required_columns
        - set(df.columns)
    )

    if missing_columns:

        add_check(
            "dataset_characterization",
            "FAIL",
            (
                "Missing required columns: "
                f"{sorted(missing_columns)}"
            ),
        )

    else:

        datasets = sorted(
            df["dataset_id"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        if (
            len(df) == 3
            and datasets == sorted(
                EXPECTED_DATASETS
            )
        ):

            add_check(
                "dataset_characterization",
                "PASS",
                (
                    f"rows={len(df)}, "
                    f"datasets={datasets}"
                ),
            )

        else:

            add_check(
                "dataset_characterization",
                "FAIL",
                (
                    f"rows={len(df)}, "
                    f"datasets={datasets}"
                ),
            )

except Exception as exc:

    add_check(
        "dataset_characterization",
        "FAIL",
        (
            "Dataset characterization "
            f"validation error: {exc}"
        ),
    )


# ==================================================================================================
# 20.6 — FEATURE TYPE CHARACTERIZATION
# ==================================================================================================

print("\n" + "-" * 100)
print("20.6 FEATURE TYPE CHARACTERIZATION")
print("-" * 100)

try:

    ft = FEATURE_TYPE_CHARACTERIZATION_DF.copy()

    # --------------------------------------------------------------------------------------------------
    # Frozen Section 5 contract:
    #
    # feature = canonical feature identifier
    #
    # NOT feature_name.
    # --------------------------------------------------------------------------------------------------

    required_ft_columns = {
        "dataset_id",
        FEATURE_NAME_COLUMN,
        "feature_type",
        "is_preprocessing_feature",
    }

    missing_columns = (
        required_ft_columns
        - set(ft.columns)
    )

    if missing_columns:

        add_check(
            "feature_type_characterization",
            "FAIL",
            (
                "Missing required columns: "
                f"{sorted(missing_columns)}"
            ),
        )

    else:

        failures = []

        for dataset_id in EXPECTED_DATASETS:

            ds_all = ft[
                ft["dataset_id"]
                .astype(str)
                == dataset_id
            ].copy()

            ds_pre = ds_all[
                ds_all[
                    "is_preprocessing_feature"
                ].apply(as_bool)
            ].copy()

            # ------------------------------------------------------------------
            # Counts
            # ------------------------------------------------------------------

            actual_total = len(
                ds_all
            )

            actual_preprocessing = len(
                ds_pre
            )

            actual_numeric = int(
                (
                    ds_pre[
                        "feature_type"
                    ]
                    .astype(str)
                    .str.lower()
                    == "numeric"
                ).sum()
            )

            actual_categorical = int(
                (
                    ds_pre[
                        "feature_type"
                    ]
                    .astype(str)
                    .str.lower()
                    == "categorical"
                ).sum()
            )

            expected_total = (
                EXPECTED_GENERATIVE_FEATURE_COUNTS[
                    dataset_id
                ]
            )

            expected_preprocessing = (
                EXPECTED_PREPROCESSING_FEATURE_COUNTS[
                    dataset_id
                ]
            )

            expected_numeric = (
                EXPECTED_NUMERIC_COUNTS[
                    dataset_id
                ]
            )

            expected_categorical = (
                EXPECTED_CATEGORICAL_COUNTS[
                    dataset_id
                ]
            )

            # ------------------------------------------------------------------
            # Structural checks
            # ------------------------------------------------------------------

            total_ok = (
                actual_total
                == expected_total
            )

            preprocessing_ok = (
                actual_preprocessing
                == expected_preprocessing
            )

            numeric_ok = (
                actual_numeric
                == expected_numeric
            )

            categorical_ok = (
                actual_categorical
                == expected_categorical
            )

            # Exactly one generative feature is outside
            # preprocessing features: the target.
            non_preprocessing_count = (
                actual_total
                - actual_preprocessing
            )

            target_structure_ok = (
                non_preprocessing_count == 1
            )

            # ------------------------------------------------------------------
            # Target identity validation
            # ------------------------------------------------------------------

            target_identity_ok = True
            target_identity_detail = (
                "target validated"
            )

            modeling_schema = (
                get_modeling_schema(
                    dataset_id
                )
            )

            if modeling_schema is None:

                target_identity_ok = False

                target_identity_detail = (
                    "persisted modeling schema unavailable"
                )

            else:

                target_column = (
                    modeling_schema.get(
                        "target_column"
                    )
                )

                if not target_column:

                    target_identity_ok = False

                    target_identity_detail = (
                        "target_column missing "
                        "from persisted schema"
                    )

                else:

                    non_pre_rows = ds_all[
                        ~ds_all[
                            "is_preprocessing_feature"
                        ].apply(as_bool)
                    ].copy()

                    non_pre_features = (
                        non_pre_rows[
                            FEATURE_NAME_COLUMN
                        ]
                        .astype(str)
                        .tolist()
                    )

                    expected_target = [
                        str(target_column)
                    ]

                    target_identity_ok = (
                        non_pre_features
                        == expected_target
                    )

                    if not target_identity_ok:

                        target_identity_detail = (
                            "non_preprocessing="
                            f"{non_pre_features}, "
                            "expected_target="
                            f"{target_column}"
                        )

            # ------------------------------------------------------------------
            # Aggregate failures
            # ------------------------------------------------------------------

            if not (
                total_ok
                and preprocessing_ok
                and numeric_ok
                and categorical_ok
                and target_structure_ok
                and target_identity_ok
            ):

                failures.append(
                    f"{dataset_id}: "
                    f"total={actual_total}/{expected_total}, "
                    f"preprocessing="
                    f"{actual_preprocessing}/"
                    f"{expected_preprocessing}, "
                    f"numeric="
                    f"{actual_numeric}/"
                    f"{expected_numeric}, "
                    f"categorical="
                    f"{actual_categorical}/"
                    f"{expected_categorical}, "
                    f"non_preprocessing="
                    f"{non_preprocessing_count}/1, "
                    f"target_identity="
                    f"{target_identity_ok} "
                    f"({target_identity_detail})"
                )

        if not failures:

            add_check(
                "feature_type_characterization",
                "PASS",
                (
                    "Preprocessing feature counts validated: "
                    "adult_income=14 "
                    "(6 numeric, 8 categorical); "
                    "bank_marketing=16 "
                    "(7 numeric, 9 categorical); "
                    "diabetes_130us=47 "
                    "(11 numeric, 36 categorical). "
                    "Each full generative schema contains "
                    "exactly one target."
                ),
            )

        else:

            add_check(
                "feature_type_characterization",
                "FAIL",
                "; ".join(failures),
            )

except Exception as exc:

    add_check(
        "feature_type_characterization",
        "FAIL",
        (
            "Feature characterization "
            f"validation error: {exc}"
        ),
    )


# ==================================================================================================
# 20.7 — STATISTICAL PROFILE COVERAGE
# ==================================================================================================

print("\n" + "-" * 100)
print("20.7 STATISTICAL PROFILE COVERAGE")
print("-" * 100)

try:

    ft = FEATURE_TYPE_CHARACTERIZATION_DF.copy()

    pre = ft[
        ft[
            "is_preprocessing_feature"
        ].apply(as_bool)
    ].copy()

    coverage_failures = []

    for dataset_id in EXPECTED_DATASETS:

        ds = pre[
            pre["dataset_id"]
            .astype(str)
            == dataset_id
        ]

        expected = (
            EXPECTED_PREPROCESSING_FEATURE_COUNTS[
                dataset_id
            ]
        )

        if len(ds) != expected:

            coverage_failures.append(
                f"{dataset_id}: "
                f"{len(ds)} preprocessing features, "
                f"expected {expected}"
            )

    if not coverage_failures:

        add_check(
            "statistical_profile_feature_coverage",
            "PASS",
            (
                "All datasets contain complete "
                "preprocessing-feature characterization"
            ),
        )

    else:

        add_check(
            "statistical_profile_feature_coverage",
            "FAIL",
            "; ".join(
                coverage_failures
            ),
        )

except Exception as exc:

    add_check(
        "statistical_profile_feature_coverage",
        "FAIL",
        (
            "Statistical profile coverage "
            f"error: {exc}"
        ),
    )


# ==================================================================================================
# 20.8 — CORRELATION PAIR COUNTS
# ==================================================================================================

print("\n" + "-" * 100)
print("20.8 CORRELATION PAIR COUNTS")
print("-" * 100)

# --------------------------------------------------------------------------------------------------
# Pearson
# --------------------------------------------------------------------------------------------------

try:

    failures = []

    for dataset_id in EXPECTED_DATASETS:

        actual = int(
            len(
                PEARSON_DF[
                    PEARSON_DF[
                        "dataset_id"
                    ]
                    .astype(str)
                    == dataset_id
                ]
            )
        )

        expected = (
            EXPECTED_PEARSON_PAIR_COUNTS[
                dataset_id
            ]
        )

        if actual != expected:

            failures.append(
                f"{dataset_id}: "
                f"{actual} Pearson pairs, "
                f"expected {expected}"
            )

    if not failures:

        add_check(
            "pearson_correlation_pair_counts",
            "PASS",
            (
                "Expected Pearson pair counts = "
                "adult_income=15, "
                "bank_marketing=21, "
                "diabetes_130us=55"
            ),
        )

    else:

        add_check(
            "pearson_correlation_pair_counts",
            "FAIL",
            "; ".join(failures),
        )

except Exception as exc:

    add_check(
        "pearson_correlation_pair_counts",
        "FAIL",
        f"Pearson validation error: {exc}",
    )


# --------------------------------------------------------------------------------------------------
# Spearman
# --------------------------------------------------------------------------------------------------

try:

    failures = []

    for dataset_id in EXPECTED_DATASETS:

        actual = int(
            len(
                SPEARMAN_DF[
                    SPEARMAN_DF[
                        "dataset_id"
                    ]
                    .astype(str)
                    == dataset_id
                ]
            )
        )

        expected = (
            EXPECTED_SPEARMAN_PAIR_COUNTS[
                dataset_id
            ]
        )

        if actual != expected:

            failures.append(
                f"{dataset_id}: "
                f"{actual} Spearman pairs, "
                f"expected {expected}"
            )

    if not failures:

        add_check(
            "spearman_correlation_pair_counts",
            "PASS",
            (
                "Expected Spearman pair counts = "
                "adult_income=15, "
                "bank_marketing=21, "
                "diabetes_130us=55"
            ),
        )

    else:

        add_check(
            "spearman_correlation_pair_counts",
            "FAIL",
            "; ".join(failures),
        )

except Exception as exc:

    add_check(
        "spearman_correlation_pair_counts",
        "FAIL",
        f"Spearman validation error: {exc}",
    )


# ==================================================================================================
# 20.9 — CATEGORICAL DEPENDENCY PAIR COUNTS
# ==================================================================================================

print("\n" + "-" * 100)
print("20.9 CATEGORICAL DEPENDENCY PAIR COUNTS")
print("-" * 100)

try:

    failures = []

    for dataset_id in EXPECTED_DATASETS:

        actual = int(
            len(
                CATEGORICAL_DEPENDENCY_DF[
                    CATEGORICAL_DEPENDENCY_DF[
                        "dataset_id"
                    ]
                    .astype(str)
                    == dataset_id
                ]
            )
        )

        expected = (
            EXPECTED_CATEGORICAL_DEPENDENCY_PAIR_COUNTS[
                dataset_id
            ]
        )

        if actual != expected:

            failures.append(
                f"{dataset_id}: "
                f"{actual} pairs, "
                f"expected {expected}"
            )

    if not failures:

        add_check(
            "categorical_dependency_pair_counts",
            "PASS",
            (
                "Expected Cramér's V pair counts = "
                "adult_income=28, "
                "bank_marketing=36, "
                "diabetes_130us=630"
            ),
        )

    else:

        add_check(
            "categorical_dependency_pair_counts",
            "FAIL",
            "; ".join(failures),
        )

except Exception as exc:

    add_check(
        "categorical_dependency_pair_counts",
        "FAIL",
        (
            "Categorical dependency "
            f"validation error: {exc}"
        ),
    )


# ==================================================================================================
# 20.10 — MATRIX COMPLETENESS
# ==================================================================================================

print("\n" + "-" * 100)
print("20.10 MATRIX COMPLETENESS")
print("-" * 100)

try:

    pearson_matrix_count = len(
        PEARSON_MATRICES
    )

    spearman_matrix_count = len(
        SPEARMAN_MATRICES
    )

    dependency_matrix_count = len(
        CATEGORICAL_DEPENDENCY_MATRICES
    )

    total_matrix_count = (
        pearson_matrix_count
        + spearman_matrix_count
        + dependency_matrix_count
    )

    if (
        pearson_matrix_count == 3
        and spearman_matrix_count == 3
        and dependency_matrix_count == 3
        and total_matrix_count
        == EXPECTED_MATRIX_ARTIFACT_COUNT
    ):

        add_check(
            "canonical_matrix_completeness",
            "PASS",
            (
                "3 Pearson + 3 Spearman + "
                "3 Cramér's V matrices present"
            ),
        )

    else:

        add_check(
            "canonical_matrix_completeness",
            "FAIL",
            (
                f"Pearson={pearson_matrix_count}, "
                f"Spearman={spearman_matrix_count}, "
                f"CramérV={dependency_matrix_count}, "
                f"Total={total_matrix_count}"
            ),
        )

except Exception as exc:

    add_check(
        "canonical_matrix_completeness",
        "FAIL",
        (
            "Matrix completeness "
            f"validation error: {exc}"
        ),
    )


# ==================================================================================================
# 20.11 — STATISTICAL REFERENCE
# ==================================================================================================

print("\n" + "-" * 100)
print("20.11 STATISTICAL REFERENCE")
print("-" * 100)

try:

    reference = (
        SPP_GAN_STATISTICAL_REFERENCE
    )

    failures = []

    if not isinstance(
        reference,
        dict,
    ):

        failures.append(
            "reference is not a dictionary"
        )

    else:

        reference_datasets = sorted(
            [
                str(x)
                for x in reference.keys()
            ]
        )

        if reference_datasets != sorted(
            EXPECTED_DATASETS
        ):

            failures.append(
                f"datasets={reference_datasets}"
            )

        for dataset_id in EXPECTED_DATASETS:

            if dataset_id not in reference:

                failures.append(
                    f"{dataset_id}: "
                    "reference missing"
                )

                continue

            ref = reference[
                dataset_id
            ]

            if not isinstance(
                ref,
                dict,
            ):

                failures.append(
                    f"{dataset_id}: "
                    "reference is not a dictionary"
                )

                continue

            # ------------------------------------------------------------------
            # Train-only provenance
            #
            # We validate the frozen representation without assuming that
            # fit_policy["train_only"] exists.
            # ------------------------------------------------------------------

            fit_policy = ref.get(
                "fit_policy",
                {},
            )

            train_only_detected = (
                contains_train_only(
                    fit_policy
                )
            )

            # If fit_policy itself does not expose the marker, allow the
            # frozen reference's explicit provenance elsewhere in the object.
            if not train_only_detected:

                train_only_detected = (
                    contains_train_only(
                        ref
                    )
                )

            if not train_only_detected:

                failures.append(
                    f"{dataset_id}: "
                    "train-only provenance not detected "
                    "in frozen reference structure"
                )

            # ------------------------------------------------------------------
            # Explicit leakage checks
            # ------------------------------------------------------------------

            forbidden_found, forbidden_key = (
                contains_forbidden_true_policy(
                    fit_policy
                )
            )

            if forbidden_found:

                failures.append(
                    f"{dataset_id}: "
                    f"{forbidden_key} policy is explicitly True"
                )

    if not failures:

        add_check(
            "spp_gan_statistical_reference",
            "PASS",
            (
                "All 3 frozen statistical references "
                "contain train-only provenance and "
                "no explicit validation/test/synthetic "
                "statistical-use flag"
            ),
        )

    else:

        add_check(
            "spp_gan_statistical_reference",
            "FAIL",
            "; ".join(failures),
        )

except Exception as exc:

    add_check(
        "spp_gan_statistical_reference",
        "FAIL",
        (
            "Statistical reference "
            f"validation error: {exc}"
        ),
    )


# ==================================================================================================
# 20.12 — STATISTICAL GUIDANCE
# ==================================================================================================

print("\n" + "-" * 100)
print("20.12 STATISTICAL GUIDANCE")
print("-" * 100)

try:

    guidance = (
        SPP_GAN_STATISTICAL_GUIDANCE
    )

    failures = []

    if not isinstance(
        guidance,
        dict,
    ):

        failures.append(
            "guidance is not a dictionary"
        )

    else:

        guidance_datasets = sorted(
            [
                str(x)
                for x in guidance.keys()
            ]
        )

        if guidance_datasets != sorted(
            EXPECTED_DATASETS
        ):

            failures.append(
                f"datasets={guidance_datasets}"
            )

        for dataset_id in EXPECTED_DATASETS:

            if dataset_id not in guidance:

                failures.append(
                    f"{dataset_id}: "
                    "guidance missing"
                )

                continue

            guidance_object = guidance[
                dataset_id
            ]

            if not contains_train_only(
                guidance_object
            ):

                failures.append(
                    f"{dataset_id}: "
                    "train-only provenance not detected"
                )

    if not failures:

        add_check(
            "spp_gan_statistical_guidance",
            "PASS",
            (
                "All 3 canonical statistical guidance "
                "objects contain train-only provenance"
            ),
        )

    else:

        add_check(
            "spp_gan_statistical_guidance",
            "FAIL",
            "; ".join(failures),
        )

except Exception as exc:

    add_check(
        "spp_gan_statistical_guidance",
        "FAIL",
        (
            "Statistical guidance "
            f"validation error: {exc}"
        ),
    )


# ==================================================================================================
# 20.13 — IDENTIFIER EXCLUSION POLICY
# ==================================================================================================

print("\n" + "-" * 100)
print("20.13 IDENTIFIER EXCLUSION POLICY")
print("-" * 100)

try:

    failures = []

    if (
        "NB02_PERSISTED_SCHEMA_CONTRACTS"
        not in globals()
    ):

        failures.append(
            "NB02_PERSISTED_SCHEMA_CONTRACTS unavailable"
        )

    else:

        for dataset_id in EXPECTED_DATASETS:

            modeling_schema = (
                get_modeling_schema(
                    dataset_id
                )
            )

            if modeling_schema is None:

                failures.append(
                    f"{dataset_id}: "
                    "modeling schema unavailable"
                )

                continue

            preprocessing_columns = set(
                modeling_schema.get(
                    "preprocessing_columns",
                    [],
                )
            )

            excluded_ids = set(
                modeling_schema.get(
                    "identifier_columns_excluded",
                    [],
                )
            )

            overlap = (
                preprocessing_columns
                .intersection(
                    excluded_ids
                )
            )

            if overlap:

                failures.append(
                    f"{dataset_id}: "
                    "excluded identifiers found in "
                    "preprocessing schema: "
                    f"{sorted(overlap)}"
                )

    if not failures:

        add_check(
            "identifier_exclusion_policy",
            "PASS",
            (
                "No excluded identifiers occur in "
                "preprocessing feature schemas"
            ),
        )

    else:

        add_check(
            "identifier_exclusion_policy",
            "FAIL",
            "; ".join(failures),
        )

except Exception as exc:

    add_check(
        "identifier_exclusion_policy",
        "FAIL",
        (
            "Identifier policy "
            f"validation error: {exc}"
        ),
    )


# ==================================================================================================
# 20.14 — TARGET POLICY
# ==================================================================================================

print("\n" + "-" * 100)
print("20.14 TARGET POLICY")
print("-" * 100)

try:

    failures = []

    if (
        "NB02_PERSISTED_SCHEMA_CONTRACTS"
        not in globals()
    ):

        failures.append(
            "NB02_PERSISTED_SCHEMA_CONTRACTS unavailable"
        )

    else:

        ft = (
            FEATURE_TYPE_CHARACTERIZATION_DF
            .copy()
        )

        required_target_columns = {
            "dataset_id",
            FEATURE_NAME_COLUMN,
            "is_preprocessing_feature",
        }

        missing_target_columns = (
            required_target_columns
            - set(ft.columns)
        )

        if missing_target_columns:

            failures.append(
                "Missing target-policy columns: "
                f"{sorted(missing_target_columns)}"
            )

        else:

            for dataset_id in EXPECTED_DATASETS:

                modeling_schema = (
                    get_modeling_schema(
                        dataset_id
                    )
                )

                if modeling_schema is None:

                    failures.append(
                        f"{dataset_id}: "
                        "modeling schema unavailable"
                    )

                    continue

                target_column = (
                    modeling_schema.get(
                        "target_column"
                    )
                )

                preprocessing_columns = set(
                    modeling_schema.get(
                        "preprocessing_columns",
                        [],
                    )
                )

                generative_columns = set(
                    modeling_schema.get(
                        "generative_columns",
                        [],
                    )
                )

                if not target_column:

                    failures.append(
                        f"{dataset_id}: "
                        "target_column missing"
                    )

                    continue

                # --------------------------------------------------------------
                # Target must NOT be a preprocessing feature.
                # --------------------------------------------------------------

                if (
                    target_column
                    in preprocessing_columns
                ):

                    failures.append(
                        f"{dataset_id}: "
                        "target appears in "
                        "preprocessing columns"
                    )

                # --------------------------------------------------------------
                # Target MUST be a generative feature.
                # --------------------------------------------------------------

                if (
                    target_column
                    not in generative_columns
                ):

                    failures.append(
                        f"{dataset_id}: "
                        "target absent from "
                        "generative columns"
                    )

                # --------------------------------------------------------------
                # Exactly one target characterization row.
                # --------------------------------------------------------------

                ds = ft[
                    ft["dataset_id"]
                    .astype(str)
                    == dataset_id
                ].copy()

                target_rows = ds[
                    ds[
                        FEATURE_NAME_COLUMN
                    ]
                    .astype(str)
                    == str(target_column)
                ]

                if len(target_rows) != 1:

                    failures.append(
                        f"{dataset_id}: "
                        "target characterization rows="
                        f"{len(target_rows)}"
                    )

                    continue

                target_preprocessing_flag = (
                    as_bool(
                        target_rows.iloc[0][
                            "is_preprocessing_feature"
                        ]
                    )
                )

                if target_preprocessing_flag:

                    failures.append(
                        f"{dataset_id}: "
                        "target incorrectly marked "
                        "as preprocessing feature"
                    )

                # --------------------------------------------------------------
                # Exactly one non-preprocessing generative feature.
                # --------------------------------------------------------------

                non_pre_rows = ds[
                    ~ds[
                        "is_preprocessing_feature"
                    ].apply(as_bool)
                ]

                if len(non_pre_rows) != 1:

                    failures.append(
                        f"{dataset_id}: "
                        "expected exactly one "
                        "non-preprocessing feature, "
                        f"found {len(non_pre_rows)}"
                    )

                else:

                    non_pre_feature = str(
                        non_pre_rows.iloc[0][
                            FEATURE_NAME_COLUMN
                        ]
                    )

                    if (
                        non_pre_feature
                        != str(target_column)
                    ):

                        failures.append(
                            f"{dataset_id}: "
                            "non-preprocessing feature "
                            f"'{non_pre_feature}' does not "
                            "match target "
                            f"'{target_column}'"
                        )

    if not failures:

        add_check(
            "target_inclusion_exclusion_policy",
            "PASS",
            (
                "Target retained in generative schema "
                "and excluded from preprocessing features"
            ),
        )

    else:

        add_check(
            "target_inclusion_exclusion_policy",
            "FAIL",
            "; ".join(failures),
        )

except Exception as exc:

    add_check(
        "target_inclusion_exclusion_policy",
        "FAIL",
        (
            "Target policy "
            f"validation error: {exc}"
        ),
    )


# ==================================================================================================
# 20.15 — PROVENANCE POLICY
# ==================================================================================================

print("\n" + "-" * 100)
print("20.15 PROVENANCE POLICY")
print("-" * 100)

try:

    failures = []

    if (
        "NB02_PERSISTED_SCHEMA_CONTRACTS"
        not in globals()
    ):

        failures.append(
            "NB02_PERSISTED_SCHEMA_CONTRACTS unavailable"
        )

    else:

        for dataset_id in EXPECTED_DATASETS:

            modeling_schema = (
                get_modeling_schema(
                    dataset_id
                )
            )

            if modeling_schema is None:

                failures.append(
                    f"{dataset_id}: "
                    "modeling schema unavailable"
                )

                continue

            provenance_column = (
                modeling_schema.get(
                    "provenance_column",
                    "__original_row_id__",
                )
            )

            preprocessing_columns = set(
                modeling_schema.get(
                    "preprocessing_columns",
                    [],
                )
            )

            generative_columns = set(
                modeling_schema.get(
                    "generative_columns",
                    [],
                )
            )

            if (
                provenance_column
                in preprocessing_columns
            ):

                failures.append(
                    f"{dataset_id}: "
                    "provenance in preprocessing columns"
                )

            if (
                provenance_column
                in generative_columns
            ):

                failures.append(
                    f"{dataset_id}: "
                    "provenance in generative columns"
                )

    if not failures:

        add_check(
            "provenance_exclusion_policy",
            "PASS",
            (
                "Provenance excluded from "
                "modeling/generative features"
            ),
        )

    else:

        add_check(
            "provenance_exclusion_policy",
            "FAIL",
            "; ".join(failures),
        )

except Exception as exc:

    add_check(
        "provenance_exclusion_policy",
        "FAIL",
        (
            "Provenance policy "
            f"validation error: {exc}"
        ),
    )


# ==================================================================================================
# 20.16 — SECTION 19 FINAL INTEGRITY GATE
# ==================================================================================================

print("\n" + "-" * 100)
print("20.16 SECTION 19 FINAL INTEGRITY GATE")
print("-" * 100)

try:

    section19_candidates = [

        os.path.join(
            PROJECT_ROOT,
            "data",
            "processed",
            "notebook_03",
            "validation",
            "section_19_integrity_report.csv",
        ),

        os.path.join(
            PROJECT_ROOT,
            "data",
            "processed",
            "notebook_03",
            "validation",
            "section_19_final_integrity_report.csv",
        ),

        os.path.join(
            PROJECT_ROOT,
            "data",
            "processed",
            "notebook_03",
            "validation",
            "section_19_verification_report.csv",
        ),

        os.path.join(
            PROJECT_ROOT,
            "data",
            "processed",
            "notebook_03",
            "validation",
            "section_19_integrity_manifest.csv",
        ),
    ]

    section19_path = locate_file(
        section19_candidates
    )

    # ------------------------------------------------------------------
    # Fallback discovery
    # ------------------------------------------------------------------

    if section19_path is None:

        validation_dir = os.path.join(
            PROJECT_ROOT,
            "data",
            "processed",
            "notebook_03",
            "validation",
        )

        if os.path.isdir(
            validation_dir
        ):

            candidates = sorted(
                glob.glob(
                    os.path.join(
                        validation_dir,
                        "*section*19*.csv",
                    )
                )
            )

            if candidates:

                # Prefer the most recently modified candidate.
                candidates = sorted(
                    candidates,
                    key=lambda p: os.path.getmtime(p),
                    reverse=True,
                )

                section19_path = candidates[0]

    if section19_path is None:

        add_check(
            "section_19_integrity_gate",
            "FAIL",
            (
                "Persisted Section 19 integrity "
                "report not found"
            ),
        )

    else:

        section19_df = pd.read_csv(
            section19_path
        )

        status_column = None

        for candidate in [
            "status",
            "Status",
            "STATUS",
        ]:

            if candidate in (
                section19_df.columns
            ):

                status_column = candidate
                break

        if status_column is None:

            add_check(
                "section_19_integrity_gate",
                "FAIL",
                (
                    "No status column found in "
                    "Section 19 report: "
                    f"{list(section19_df.columns)}"
                ),
            )

        else:

            status_values = (
                section19_df[
                    status_column
                ]
                .astype(str)
                .str.upper()
                .str.strip()
            )

            failed_count = int(
                (
                    status_values
                    == "FAIL"
                ).sum()
            )

            warning_count = int(
                (
                    status_values
                    == "WARNING"
                ).sum()
            )

            passed_count = int(
                (
                    status_values
                    == "PASS"
                ).sum()
            )

            total_checks = len(
                section19_df
            )

            # ------------------------------------------------------------------
            # STRICT FROZEN GATE
            # ------------------------------------------------------------------

            if (
                total_checks
                == EXPECTED_SECTION19_CHECKS
                and passed_count
                == EXPECTED_SECTION19_CHECKS
                and failed_count
                == EXPECTED_SECTION19_FAILED
                and warning_count
                == EXPECTED_SECTION19_WARNINGS
            ):

                add_check(
                    "section_19_integrity_gate",
                    "PASS",
                    (
                        f"checks={total_checks}, "
                        f"passed={passed_count}, "
                        f"failed={failed_count}, "
                        f"warnings={warning_count}"
                    ),
                )

            elif failed_count > 0:

                add_check(
                    "section_19_integrity_gate",
                    "FAIL",
                    (
                        f"checks={total_checks}, "
                        f"passed={passed_count}, "
                        f"failed={failed_count}, "
                        f"warnings={warning_count}; "
                        "Section 19 contains failed checks"
                    ),
                )

            elif warning_count > 0:

                add_check(
                    "section_19_integrity_gate",
                    "FAIL",
                    (
                        f"checks={total_checks}, "
                        f"passed={passed_count}, "
                        f"failed={failed_count}, "
                        f"warnings={warning_count}; "
                        "Section 19 contains warnings"
                    ),
                )

            else:

                add_check(
                    "section_19_integrity_gate",
                    "FAIL",
                    (
                        f"checks={total_checks}, "
                        f"passed={passed_count}, "
                        f"failed={failed_count}, "
                        f"warnings={warning_count}; "
                        "expected frozen Section 19 artifact = "
                        f"{EXPECTED_SECTION19_CHECKS} PASS checks"
                    ),
                )

except Exception as exc:

    add_check(
        "section_19_integrity_gate",
        "FAIL",
        (
            "Section 19 integrity "
            f"gate error: {exc}"
        ),
    )


# ==================================================================================================
# 20.17 — PERSISTED ARTIFACT COMPLETENESS
# ==================================================================================================

print("\n" + "-" * 100)
print("20.17 PERSISTED ARTIFACT COMPLETENESS")
print("-" * 100)

try:

    notebook03_root = os.path.join(
        PROJECT_ROOT,
        "data",
        "processed",
        "notebook_03",
    )

    required_csvs = [

        os.path.join(
            notebook03_root,
            "reports",
            "dataset_characterization.csv",
        ),

        os.path.join(
            notebook03_root,
            "reports",
            "feature_type_characterization.csv",
        ),

        os.path.join(
            notebook03_root,
            "reports",
            "feature_type_summary.csv",
        ),

        os.path.join(
            notebook03_root,
            "statistics",
            "numerical_descriptive_statistics.csv",
        ),

        os.path.join(
            notebook03_root,
            "statistics",
            "categorical_descriptive_statistics.csv",
        ),

        os.path.join(
            notebook03_root,
            "statistics",
            "cardinality_entropy_analysis.csv",
        ),

        os.path.join(
            notebook03_root,
            "statistics",
            "cardinality_entropy_summary.csv",
        ),

        os.path.join(
            notebook03_root,
            "correlations",
            "pearson_correlation_long.csv",
        ),

        os.path.join(
            notebook03_root,
            "correlations",
            "spearman_correlation_long.csv",
        ),

        os.path.join(
            notebook03_root,
            "dependencies",
            "categorical_dependency_cramers_v.csv",
        ),
    ]

    missing = [
        path
        for path in required_csvs
        if not os.path.isfile(path)
    ]

    if not missing:

        add_check(
            "canonical_csv_artifact_completeness",
            "PASS",
            (
                "All 10 canonical CSV artifacts present"
            ),
        )

    else:

        add_check(
            "canonical_csv_artifact_completeness",
            "FAIL",
            f"Missing artifacts: {missing}",
        )

except Exception as exc:

    add_check(
        "canonical_csv_artifact_completeness",
        "FAIL",
        (
            "Canonical CSV artifact "
            f"validation error: {exc}"
        ),
    )


# ==================================================================================================
# 20.18 — PERSISTED MATRIX ARTIFACT COUNT
# ==================================================================================================

print("\n" + "-" * 100)
print("20.18 PERSISTED MATRIX ARTIFACT COUNT")
print("-" * 100)

try:

    correlations_dir = os.path.join(
        PROJECT_ROOT,
        "data",
        "processed",
        "notebook_03",
        "correlations",
    )

    dependencies_dir = os.path.join(
        PROJECT_ROOT,
        "data",
        "processed",
        "notebook_03",
        "dependencies",
    )

    matrix_patterns = [

        os.path.join(
            correlations_dir,
            "*_pearson_matrix.csv",
        ),

        os.path.join(
            correlations_dir,
            "*_spearman_matrix.csv",
        ),

        os.path.join(
            dependencies_dir,
            "*_categorical_cramers_v_matrix.csv",
        ),
    ]

    matrix_files = []

    for pattern in matrix_patterns:

        matrix_files.extend(
            glob.glob(pattern)
        )

    matrix_files = sorted(
        set(matrix_files)
    )

    matrix_count = len(
        matrix_files
    )

    if (
        matrix_count
        == EXPECTED_MATRIX_ARTIFACT_COUNT
    ):

        add_check(
            "persisted_matrix_artifact_count",
            "PASS",
            (
                f"matrix_artifacts={matrix_count}, "
                f"expected={EXPECTED_MATRIX_ARTIFACT_COUNT}"
            ),
        )

    else:

        add_check(
            "persisted_matrix_artifact_count",
            "FAIL",
            (
                f"matrix_artifacts={matrix_count}, "
                f"expected={EXPECTED_MATRIX_ARTIFACT_COUNT}; "
                f"files={matrix_files}"
            ),
        )

except Exception as exc:

    add_check(
        "persisted_matrix_artifact_count",
        "FAIL",
        (
            "Persisted matrix artifact "
            f"validation error: {exc}"
        ),
    )


# ==================================================================================================
# 20.19 — REFERENCE / GUIDANCE JSON ARTIFACT COUNT
# ==================================================================================================

print("\n" + "-" * 100)
print("20.19 REFERENCE / GUIDANCE JSON ARTIFACT COUNT")
print("-" * 100)

try:

    reference_dir = os.path.join(
        PROJECT_ROOT,
        "data",
        "processed",
        "notebook_03",
        "reference",
    )

    guidance_dir = os.path.join(
        PROJECT_ROOT,
        "data",
        "processed",
        "notebook_03",
        "guidance",
    )

    reference_files = sorted(
        glob.glob(
            os.path.join(
                reference_dir,
                "*_spp_gan_statistical_reference.json",
            )
        )
    )

    guidance_files = sorted(
        glob.glob(
            os.path.join(
                guidance_dir,
                "*_spp_gan_statistical_guidance.json",
            )
        )
    )

    reference_count = len(
        reference_files
    )

    guidance_count = len(
        guidance_files
    )

    if (
        reference_count
        == EXPECTED_REFERENCE_COUNT
        and guidance_count
        == EXPECTED_GUIDANCE_COUNT
    ):

        add_check(
            "reference_guidance_json_completeness",
            "PASS",
            (
                f"reference_jsons={reference_count}, "
                f"guidance_jsons={guidance_count}, "
                f"expected="
                f"{EXPECTED_REFERENCE_COUNT}+"
                f"{EXPECTED_GUIDANCE_COUNT}"
            ),
        )

    else:

        add_check(
            "reference_guidance_json_completeness",
            "FAIL",
            (
                f"reference_jsons={reference_count}, "
                f"guidance_jsons={guidance_count}; "
                f"expected="
                f"{EXPECTED_REFERENCE_COUNT}+"
                f"{EXPECTED_GUIDANCE_COUNT}"
            ),
        )

except Exception as exc:

    add_check(
        "reference_guidance_json_completeness",
        "FAIL",
        (
            "Reference/guidance artifact "
            f"validation error: {exc}"
        ),
    )


# ==================================================================================================
# 20.20 — FINAL VERIFICATION DATAFRAME
# ==================================================================================================

print("\n" + "-" * 100)
print("20.20 FINAL VERIFICATION DATAFRAME")
print("-" * 100)

FINAL_VERIFICATION_DF, total_checks, checks_passed, checks_failed, FINAL_STATUS = (
    calculate_final_counts()
)

print(
    f"Initial verification checks : {total_checks}"
)

print(
    f"Initial passed              : {checks_passed}"
)

print(
    f"Initial failed              : {checks_failed}"
)

print(
    f"Initial status              : {FINAL_STATUS}"
)


# ==================================================================================================
# 20.21 — PREPARE FINAL VERIFICATION PATHS
# ==================================================================================================

print("\n" + "-" * 100)
print("20.21 PREPARE FINAL VERIFICATION PATHS")
print("-" * 100)

validation_dir = (
    Path(PROJECT_ROOT)
    / "data"
    / "processed"
    / "notebook_03"
    / "validation"
)

validation_dir.mkdir(
    parents=True,
    exist_ok=True,
)

final_report_path = (
    validation_dir
    / "notebook_03_final_verification.csv"
)

completion_path = (
    Path(PROJECT_ROOT)
    / "data"
    / "processed"
    / "notebook_03"
    / "schemas"
    / "notebook_03_completion_metadata.json"
)

completion_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    f"✓ Validation directory: {validation_dir}"
)

print(
    f"✓ Final report path: {final_report_path}"
)

print(
    f"✓ Completion metadata path: {completion_path}"
)


# ==================================================================================================
# 20.22 — FINAL REPORT RELOAD CHECK
#
# IMPORTANT:
# The reload check is evaluated against the report produced from all checks
# available BEFORE this check is itself added.
#
# After this check is added, the final report is rebuilt and saved again.
# ==================================================================================================

print("\n" + "-" * 100)
print("20.22 FINAL REPORT RELOAD CHECK")
print("-" * 100)

try:

    # ------------------------------------------------------------------
    # Save the current report temporarily.
    # ------------------------------------------------------------------

    FINAL_VERIFICATION_DF.to_csv(
        final_report_path,
        index=False,
    )

    # ------------------------------------------------------------------
    # Reload.
    # ------------------------------------------------------------------

    reload_final_report = pd.read_csv(
        final_report_path
    )

    reload_ok = (
        len(reload_final_report)
        == len(FINAL_VERIFICATION_DF)
        and list(
            reload_final_report.columns
        )
        == list(
            FINAL_VERIFICATION_DF.columns
        )
        and reload_final_report[
            "check"
        ].astype(str).tolist()
        == FINAL_VERIFICATION_DF[
            "check"
        ].astype(str).tolist()
        and reload_final_report[
            "status"
        ].astype(str).tolist()
        == FINAL_VERIFICATION_DF[
            "status"
        ].astype(str).tolist()
    )

    if reload_ok:

        add_check(
            "final_verification_report_reload",
            "PASS",
            (
                "Saved and reloaded "
                f"{len(reload_final_report)} "
                "verification checks with matching "
                "schema and check/status values"
            ),
        )

    else:

        add_check(
            "final_verification_report_reload",
            "FAIL",
            (
                "Reload mismatch between saved "
                "and in-memory verification report"
            ),
        )

except Exception as exc:

    add_check(
        "final_verification_report_reload",
        "FAIL",
        (
            "Final verification report "
            f"reload failed: {exc}"
        ),
    )


# ==================================================================================================
# 20.23 — FINAL VERIFICATION RECOMPUTATION
# ==================================================================================================

print("\n" + "-" * 100)
print("20.23 FINAL VERIFICATION RECOMPUTATION")
print("-" * 100)

FINAL_VERIFICATION_DF, total_checks, checks_passed, checks_failed, FINAL_STATUS = (
    calculate_final_counts()
)

print(
    f"Final checks recorded : {total_checks}"
)

print(
    f"Final passed          : {checks_passed}"
)

print(
    f"Final failed          : {checks_failed}"
)

print(
    f"Final status          : {FINAL_STATUS}"
)


# ==================================================================================================
# 20.24 — SAVE FINAL REPORT
# ==================================================================================================

print("\n" + "-" * 100)
print("20.24 SAVE FINAL REPORT")
print("-" * 100)

FINAL_VERIFICATION_DF.to_csv(
    final_report_path,
    index=False,
)

print(
    "✓ Saved final verification report: "
    f"{final_report_path}"
)

print(
    "✓ Report size: "
    f"{final_report_path.stat().st_size:,} bytes"
)

print(
    f"✓ Final checks recorded: {total_checks}"
)

print(
    f"✓ Final passed: {checks_passed}"
)

print(
    f"✓ Final failed: {checks_failed}"
)


# ==================================================================================================
# 20.25 — COMPLETION METADATA
# ==================================================================================================

print("\n" + "-" * 100)
print("20.25 COMPLETION METADATA")
print("-" * 100)

completion_metadata = {
    "notebook": "03",

    "title": (
        "Statistical & Data Characterization"
    ),

    "status": (
        "COMPLETE"
        if FINAL_STATUS == "PASS"
        else "FAILED"
    ),

    "final_verification_status": (
        FINAL_STATUS
    ),

    "source_split": "train_only",

    "validation_used_for_statistics": False,

    "test_used_for_statistics": False,

    "synthetic_data_used_for_statistics": False,

    "datasets": EXPECTED_DATASETS,

    "expected_feature_counts": {

        dataset_id: {

            "generative_features":
                EXPECTED_GENERATIVE_FEATURE_COUNTS[
                    dataset_id
                ],

            "preprocessing_features":
                EXPECTED_PREPROCESSING_FEATURE_COUNTS[
                    dataset_id
                ],

            "numeric_features":
                EXPECTED_NUMERIC_COUNTS[
                    dataset_id
                ],

            "categorical_features":
                EXPECTED_CATEGORICAL_COUNTS[
                    dataset_id
                ],
        }

        for dataset_id
        in EXPECTED_DATASETS
    },

    "dataset_characterization_rows": int(
        len(
            DATASET_CHARACTERIZATION_DF
        )
    ),

    "feature_type_characterization_rows": int(
        len(
            FEATURE_TYPE_CHARACTERIZATION_DF
        )
    ),

    "pearson_pairs": int(
        len(
            PEARSON_DF
        )
    ),

    "spearman_pairs": int(
        len(
            SPEARMAN_DF
        )
    ),

    "categorical_dependency_pairs": int(
        len(
            CATEGORICAL_DEPENDENCY_DF
        )
    ),

    "reference_objects":
        EXPECTED_REFERENCE_COUNT,

    "guidance_objects":
        EXPECTED_GUIDANCE_COUNT,

    "persisted_matrix_artifacts":
        EXPECTED_MATRIX_ARTIFACT_COUNT,

    "section_19_expected_checks":
        EXPECTED_SECTION19_CHECKS,

    "section_20": {

        "total_checks":
            total_checks,

        "checks_passed":
            checks_passed,

        "checks_failed":
            checks_failed,
    },

    "overall_pass": (
        FINAL_STATUS == "PASS"
    ),

    "final_report":
        str(final_report_path),

    "generated_utc":
        pd.Timestamp.utcnow().isoformat(),
}

persist_json(
    completion_metadata,
    completion_path,
    "notebook_03_completion_metadata",
)

print(
    "✓ Saved notebook_03_completion_metadata: "
    f"{completion_path.stat().st_size:,} bytes"
)


# ==================================================================================================
# 20.26 — FINAL DISPLAY
# ==================================================================================================

print("\n" + "=" * 100)
print("NOTEBOOK 03 FINAL VERIFICATION")
print("=" * 100)

print(
    FINAL_VERIFICATION_DF.to_string(
        index=False
    )
)

print("\n" + "=" * 100)

print(
    f"Total checks   : {total_checks}"
)

print(
    f"Checks passed  : {checks_passed}"
)

print(
    f"Checks failed  : {checks_failed}"
)

print(
    f"Final status   : {FINAL_STATUS}"
)

print(
    f"Final report   : {final_report_path}"
)

print(
    f"Completion     : {completion_path}"
)

print("=" * 100)


# ==================================================================================================
# 20.27 — FAILED CHECK SUMMARY
# ==================================================================================================

FAILED_FINAL_CHECKS = FINAL_VERIFICATION_DF[
    FINAL_VERIFICATION_DF[
        "status"
    ]
    == "FAIL"
].copy()

if len(
    FAILED_FINAL_CHECKS
) > 0:

    print("\n" + "=" * 100)
    print("FINAL VERIFICATION FAILURES")
    print("=" * 100)

    print(
        FAILED_FINAL_CHECKS.to_string(
            index=False
        )
    )


# ==================================================================================================
# 20.28 — SYNCHRONIZE COMPLETION METADATA
# ==================================================================================================

print("\n" + "-" * 100)
print("20.28 SYNCHRONIZE COMPLETION METADATA")
print("-" * 100)

completion_metadata[
    "status"
] = (
    "COMPLETE"
    if FINAL_STATUS == "PASS"
    else "FAILED"
)

completion_metadata[
    "final_verification_status"
] = (
    FINAL_STATUS
)

completion_metadata[
    "section_20"
] = {

    "total_checks":
        total_checks,

    "checks_passed":
        checks_passed,

    "checks_failed":
        checks_failed,
}

completion_metadata[
    "overall_pass"
] = (
    FINAL_STATUS == "PASS"
)

completion_metadata[
    "final_report"
] = (
    str(final_report_path)
)

completion_metadata[
    "generated_utc"
] = (
    pd.Timestamp.utcnow().isoformat()
)

persist_json(
    completion_metadata,
    completion_path,
    "notebook_03_completion_metadata",
)

print(
    "✓ Completion metadata synchronized"
)

print(
    "✓ Final verification status: "
    f"{FINAL_STATUS}"
)

print(
    "✓ Completion metadata size: "
    f"{completion_path.stat().st_size:,} bytes"
)


# ==================================================================================================
# 20.29 — FINAL COMPLETION GATE
# ==================================================================================================

print("\n" + "=" * 100)
print("NOTEBOOK 03 COMPLETION GATE")
print("=" * 100)

if FINAL_STATUS != "PASS":

    print(
        "✗ NOTEBOOK 03 COMPLETION GATE FAILED"
    )

    print(
        f"✗ Total checks  : {total_checks}"
    )

    print(
        f"✗ Checks passed : {checks_passed}"
    )

    print(
        f"✗ Checks failed : {checks_failed}"
    )

    raise RuntimeError(
        "NOTEBOOK 03 COMPLETION GATE FAILED."
    )


print(
    "✓ NOTEBOOK 03 COMPLETION GATE PASSED"
)

print(
    "✓ ALL FINAL VERIFICATION CHECKS PASSED"
)

print(
    "✓ NOTEBOOK 03 IS READY TO BE FROZEN"
)

print("=" * 100)


# ==================================================================================================
# SECTION 20 EXECUTION TIME
# ==================================================================================================

SECTION20_DURATION = (
    time.time()
    - SECTION20_START_TIME
)

print(
    f"\nSection 20 duration: "
    f"{SECTION20_DURATION:.2f} seconds"
)

print("=" * 100)

SECTION 20 — FINAL VERIFICATION

----------------------------------------------------------------------------------------------------
20.1 REQUIRED CANONICAL OBJECTS
----------------------------------------------------------------------------------------------------

----------------------------------------------------------------------------------------------------
20.2 DATASET REGISTRY
----------------------------------------------------------------------------------------------------

----------------------------------------------------------------------------------------------------
20.3 TRAINING DATASETS
----------------------------------------------------------------------------------------------------

----------------------------------------------------------------------------------------------------
20.4 INPUT SCHEMA VALIDATION
----------------------------------------------------------------------------------------------------

-------------------------------------------------

In [38]:
# ==============================================================================
# SECTION 21 — COMPLETION SUMMARY
# ==============================================================================
#
# PURPOSE
# -------
# Final human-readable and machine-readable completion summary for Notebook 03.
#
# IMPORTANT
# ---------
# This section:
#   • does NOT recompute statistics
#   • does NOT modify the statistical reference
#   • does NOT use validation/test data
#   • does NOT train any model
#   • does NOT generate synthetic data
#   • does NOT alter Notebook 02 artifacts
#
# It consumes only the already validated Notebook 03 canonical objects
# and persisted artifacts.
#
# AUTHORITATIVE UPSTREAM GATE
# ---------------------------
# Section 20 — FINAL VERIFICATION
#
# ==============================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
import numpy as np

print("=" * 100)
print("SECTION 21 — COMPLETION SUMMARY")
print("=" * 100)


# ==============================================================================
# 21.1 — VERIFY REQUIRED CANONICAL OBJECTS
# ==============================================================================

print("\n" + "-" * 100)
print("21.1 REQUIRED CANONICAL OBJECTS")
print("-" * 100)


REQUIRED_NOTEBOOK_03_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_STATISTICAL_DATASETS",

    # Canonical characterization/statistics
    "DATASET_CHARACTERIZATION_DF",
    "FEATURE_TYPE_CHARACTERIZATION_DF",
    "FEATURE_TYPE_SUMMARY_DF",
    "NUMERICAL_STATISTICS_DF",
    "CATEGORICAL_STATISTICS_DF",
    "CARDINALITY_ENTROPY_DF",
    "CARDINALITY_ENTROPY_SUMMARY_DF",

    # Dependency structures
    "PEARSON_DF",
    "PEARSON_MATRICES",
    "SPEARMAN_DF",
    "SPEARMAN_MATRICES",
    "CATEGORICAL_DEPENDENCY_DF",
    "CATEGORICAL_DEPENDENCY_MATRICES",

    # Canonical SPP-GAN statistical artifacts
    "SPP_GAN_STATISTICAL_REFERENCE",
    "SPP_GAN_STATISTICAL_GUIDANCE",

    # Section 20 final verification
    "FINAL_VERIFICATION_DF",
]

missing_objects = [
    name
    for name in REQUIRED_NOTEBOOK_03_OBJECTS
    if name not in globals()
]

if missing_objects:

    print("\nMissing required canonical objects:")

    for item in missing_objects:
        print(f"  ✗ {item}")

    raise RuntimeError(
        "Notebook 03 completion summary cannot proceed because "
        "required canonical objects are missing."
    )

print("✓ Required canonical objects       : PASS")


# ==============================================================================
# 21.2 — VERIFY SECTION 20 FINAL GATE
# ==============================================================================

print("\n" + "-" * 100)
print("21.2 SECTION 20 FINAL VERIFICATION GATE")
print("-" * 100)


if "FINAL_VERIFICATION_DF" not in globals():

    raise RuntimeError(
        "FINAL_VERIFICATION_DF is required from Section 20."
    )


if "status" not in FINAL_VERIFICATION_DF.columns:

    raise RuntimeError(
        "FINAL_VERIFICATION_DF does not contain the required 'status' column."
    )


section20_failed = FINAL_VERIFICATION_DF[
    FINAL_VERIFICATION_DF["status"] != "PASS"
]


SECTION20_TOTAL_CHECKS = int(
    len(FINAL_VERIFICATION_DF)
)

SECTION20_FAILED_CHECKS = int(
    len(section20_failed)
)

SECTION20_PASSED_CHECKS = (
    SECTION20_TOTAL_CHECKS -
    SECTION20_FAILED_CHECKS
)


SECTION20_FINAL_STATUS = (
    "PASS"
    if SECTION20_FAILED_CHECKS == 0
    else "FAIL"
)


print(
    f"Section 20 checks              : "
    f"{SECTION20_TOTAL_CHECKS}"
)

print(
    f"Section 20 passed              : "
    f"{SECTION20_PASSED_CHECKS}"
)

print(
    f"Section 20 failed              : "
    f"{SECTION20_FAILED_CHECKS}"
)

print(
    f"Section 20 final status        : "
    f"{SECTION20_FINAL_STATUS}"
)


if SECTION20_FINAL_STATUS != "PASS":

    print("\nFailed Section 20 checks:")

    print(
        section20_failed.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Notebook 03 cannot be marked complete because "
        "Section 20 final verification did not pass."
    )

print("✓ Section 20 final gate          : PASS")


# ==============================================================================
# 21.3 — VERIFY DATASET REGISTRY
# ==============================================================================

print("\n" + "-" * 100)
print("21.3 DATASET REGISTRY")
print("-" * 100)


EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]


actual_datasets = list(DATASET_IDS)


if actual_datasets != EXPECTED_DATASETS:

    raise RuntimeError(
        "Dataset registry mismatch.\n"
        f"Expected: {EXPECTED_DATASETS}\n"
        f"Actual  : {actual_datasets}"
    )


print(
    f"✓ Dataset registry               : PASS "
    f"({actual_datasets})"
)


# ==============================================================================
# 21.4 — DATASET COMPLETENESS
# ==============================================================================

print("\n" + "-" * 100)
print("21.4 DATASET COMPLETENESS")
print("-" * 100)


dataset_completeness_records = []


for dataset_id in DATASET_IDS:

    if dataset_id not in TRAIN_STATISTICAL_DATASETS:

        raise RuntimeError(
            f"Training statistical dataset missing: {dataset_id}"
        )


    dataset_rows = int(
        len(
            TRAIN_STATISTICAL_DATASETS[
                dataset_id
            ]
        )
    )


    dataset_profile_rows = DATASET_CHARACTERIZATION_DF[
        DATASET_CHARACTERIZATION_DF[
            "dataset_id"
        ] == dataset_id
    ]


    feature_rows = FEATURE_TYPE_CHARACTERIZATION_DF[
        (
            FEATURE_TYPE_CHARACTERIZATION_DF[
                "dataset_id"
            ] == dataset_id
        )
        &
        (
            FEATURE_TYPE_CHARACTERIZATION_DF[
                "is_preprocessing_feature"
            ]
            == True
        )
    ]


    reference_present = (
        dataset_id
        in SPP_GAN_STATISTICAL_REFERENCE
    )


    guidance_present = (
        dataset_id
        in SPP_GAN_STATISTICAL_GUIDANCE
    )


    dataset_profile_present = (
        len(dataset_profile_rows) == 1
    )


    feature_profile_count = int(
        len(feature_rows)
    )


    dataset_completeness_records.append({

        "dataset_id": dataset_id,

        "training_rows": dataset_rows,

        "preprocessing_features": feature_profile_count,

        "dataset_characterization_present":
            dataset_profile_present,

        "statistical_reference_present":
            reference_present,

        "statistical_guidance_present":
            guidance_present,

        "status": (
            "PASS"
            if (
                dataset_profile_present
                and reference_present
                and guidance_present
            )
            else "FAIL"
        ),
    })


DATASET_COMPLETENESS_DF = pd.DataFrame(
    dataset_completeness_records
)


if (
    DATASET_COMPLETENESS_DF["status"] != "PASS"
).any():

    print(
        DATASET_COMPLETENESS_DF.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Dataset completeness verification failed."
    )


print("✓ Dataset completeness           : PASS")


# ==============================================================================
# 21.5 — VERIFY CANONICAL FEATURE COUNTS
# ==============================================================================

print("\n" + "-" * 100)
print("21.5 CANONICAL FEATURE COUNTS")
print("-" * 100)


# ------------------------------------------------------------------------------
# Frozen canonical feature counts
# ------------------------------------------------------------------------------

EXPECTED_FEATURE_COUNTS = {

    "adult_income": {
        "preprocessing": 14,
        "numeric": 6,
        "categorical": 8,
        "generative": 15,
    },

    "bank_marketing": {
        "preprocessing": 16,
        "numeric": 7,
        "categorical": 9,
        "generative": 17,
    },

    "diabetes_130us": {
        "preprocessing": 47,
        "numeric": 11,
        "categorical": 36,
        "generative": 48,
    },
}


# ------------------------------------------------------------------------------
# Verify the frozen canonical Section 5 schema
#
# IMPORTANT:
#   feature       = canonical feature identifier
#   feature_type  = canonical feature-type classification
#
# "semantic_type" is obsolete and must NOT be used.
# ------------------------------------------------------------------------------

REQUIRED_FEATURE_TYPE_COLUMNS = {

    "dataset_id",

    "feature",

    "feature_type",

    "is_preprocessing_feature",

    "is_target",
}


missing_feature_type_columns = (
    REQUIRED_FEATURE_TYPE_COLUMNS
    - set(
        FEATURE_TYPE_CHARACTERIZATION_DF.columns
    )
)


if missing_feature_type_columns:

    raise RuntimeError(
        "FEATURE_TYPE_CHARACTERIZATION_DF is missing required "
        "canonical columns.\n"
        f"Missing columns : "
        f"{sorted(missing_feature_type_columns)}\n"
        f"Available columns: "
        f"{list(FEATURE_TYPE_CHARACTERIZATION_DF.columns)}"
    )


print(
    "✓ Canonical feature schema columns : PASS"
)

print(
    "  Feature identifier             : feature"
)

print(
    "  Feature type                   : feature_type"
)


# ------------------------------------------------------------------------------
# Normalize feature type values for validation only.
#
# The canonical DataFrame is NOT modified.
# ------------------------------------------------------------------------------

def normalize_feature_type(value):

    if pd.isna(value):

        return ""

    return (
        str(value)
        .strip()
        .lower()
    )


# ------------------------------------------------------------------------------
# Validate every dataset
# ------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    expected = EXPECTED_FEATURE_COUNTS[
        dataset_id
    ]


    # --------------------------------------------------------------------------
    # Preprocessing features only
    # --------------------------------------------------------------------------

    preprocessing_rows = (
        FEATURE_TYPE_CHARACTERIZATION_DF[
            (
                FEATURE_TYPE_CHARACTERIZATION_DF[
                    "dataset_id"
                ] == dataset_id
            )
            &
            (
                FEATURE_TYPE_CHARACTERIZATION_DF[
                    "is_preprocessing_feature"
                ] == True
            )
        ]
        .copy()
    )


    # --------------------------------------------------------------------------
    # Full generative schema
    #
    # This includes:
    #   preprocessing features + target
    #
    # It excludes identifiers/provenance according to the frozen policy.
    # --------------------------------------------------------------------------

    generative_rows = (
        FEATURE_TYPE_CHARACTERIZATION_DF[
            FEATURE_TYPE_CHARACTERIZATION_DF[
                "dataset_id"
            ] == dataset_id
        ]
        .copy()
    )


    # --------------------------------------------------------------------------
    # Canonical feature type
    #
    # IMPORTANT:
    # Use feature_type, NOT semantic_type.
    # --------------------------------------------------------------------------

    normalized_feature_types = (
        preprocessing_rows[
            "feature_type"
        ]
        .map(
            normalize_feature_type
        )
    )


    # --------------------------------------------------------------------------
    # Numeric count
    # --------------------------------------------------------------------------

    numeric_count = int(
        (
            normalized_feature_types
            == "numeric"
        ).sum()
    )


    # --------------------------------------------------------------------------
    # Categorical count
    # --------------------------------------------------------------------------

    categorical_count = int(
        (
            normalized_feature_types
            == "categorical"
        ).sum()
    )


    # --------------------------------------------------------------------------
    # Preprocessing feature count
    # --------------------------------------------------------------------------

    preprocessing_count = int(
        len(
            preprocessing_rows
        )
    )


    # --------------------------------------------------------------------------
    # Generative feature count
    # --------------------------------------------------------------------------

    generative_count = int(
        len(
            generative_rows
        )
    )


    # --------------------------------------------------------------------------
    # Validate all preprocessing feature types
    # --------------------------------------------------------------------------

    recognized_type_mask = (
        normalized_feature_types.isin(
            {
                "numeric",
                "categorical",
            }
        )
    )


    if not bool(
        recognized_type_mask.all()
    ):

        unknown_types = sorted(
            set(
                normalized_feature_types[
                    ~recognized_type_mask
                ]
            )
        )


        raise RuntimeError(
            f"Unrecognized canonical feature_type values "
            f"for {dataset_id}: "
            f"{unknown_types}"
        )


    # --------------------------------------------------------------------------
    # Verify target is excluded from preprocessing features
    # --------------------------------------------------------------------------

    target = TARGET_COLUMNS[
        dataset_id
    ]


    preprocessing_target_rows = (
        preprocessing_rows[
            preprocessing_rows[
                "feature"
            ] == target
        ]
    )


    if len(
        preprocessing_target_rows
    ) != 0:

        raise RuntimeError(
            f"Target '{target}' unexpectedly appears "
            f"in preprocessing features for "
            f"{dataset_id}."
        )


    # --------------------------------------------------------------------------
    # Verify exactly one target in complete generative characterization
    # --------------------------------------------------------------------------

    target_rows = (
        generative_rows[
            generative_rows[
                "feature"
            ] == target
        ]
    )


    if len(
        target_rows
    ) != 1:

        raise RuntimeError(
            f"Target '{target}' must occur exactly once "
            f"in the full generative characterization "
            f"for {dataset_id}."
        )


    target_row = target_rows.iloc[0]


    if not bool(
        target_row[
            "is_target"
        ]
    ):

        raise RuntimeError(
            f"Target '{target}' does not have "
            f"is_target=True for {dataset_id}."
        )


    # --------------------------------------------------------------------------
    # Build actual counts
    # --------------------------------------------------------------------------

    actual = {

        "preprocessing":
            preprocessing_count,

        "numeric":
            numeric_count,

        "categorical":
            categorical_count,

        "generative":
            generative_count,
    }


    # --------------------------------------------------------------------------
    # Compare against frozen expected counts
    # --------------------------------------------------------------------------

    if actual != expected:

        raise RuntimeError(
            f"Feature-count mismatch for "
            f"{dataset_id}.\n"
            f"Expected: {expected}\n"
            f"Actual  : {actual}"
        )


    # --------------------------------------------------------------------------
    # PASS
    # --------------------------------------------------------------------------

    print(
        f"✓ {dataset_id:<20} : "
        f"{preprocessing_count} preprocessing "
        f"({numeric_count} numeric, "
        f"{categorical_count} categorical), "
        f"{generative_count} generative"
    )


print(
    "\n✓ Canonical feature-count verification : PASS"
)


# ==============================================================================
# 21.6 — STATISTICAL COVERAGE SUMMARY
# ==============================================================================

print("\n" + "-" * 100)
print("21.6 STATISTICAL COVERAGE SUMMARY")
print("-" * 100)


STATISTICAL_COVERAGE_SUMMARY = {

    "datasets":
        int(len(DATASET_IDS)),

    "dataset_characterization_rows":
        int(len(DATASET_CHARACTERIZATION_DF)),

    "feature_type_characterization_rows":
        int(len(FEATURE_TYPE_CHARACTERIZATION_DF)),

    "feature_type_summary_rows":
        int(len(FEATURE_TYPE_SUMMARY_DF)),

    "numerical_statistics_rows":
        int(len(NUMERICAL_STATISTICS_DF)),

    "categorical_statistics_rows":
        int(len(CATEGORICAL_STATISTICS_DF)),

    "cardinality_entropy_rows":
        int(len(CARDINALITY_ENTROPY_DF)),

    "cardinality_entropy_summary_rows":
        int(len(CARDINALITY_ENTROPY_SUMMARY_DF)),

    "pearson_rows":
        int(len(PEARSON_DF)),

    "spearman_rows":
        int(len(SPEARMAN_DF)),

    "categorical_dependency_rows":
        int(len(CATEGORICAL_DEPENDENCY_DF)),

    "pearson_matrices":
        int(len(PEARSON_MATRICES)),

    "spearman_matrices":
        int(len(SPEARMAN_MATRICES)),

    "categorical_dependency_matrices":
        int(len(CATEGORICAL_DEPENDENCY_MATRICES)),

    "statistical_reference_datasets":
        int(len(SPP_GAN_STATISTICAL_REFERENCE)),

    "statistical_guidance_datasets":
        int(len(SPP_GAN_STATISTICAL_GUIDANCE)),
}


for key, value in STATISTICAL_COVERAGE_SUMMARY.items():

    print(
        f"{key:<45}: {value}"
    )


# ==============================================================================
# 21.7 — VERIFY CORRELATION / DEPENDENCY COVERAGE
# ==============================================================================

print("\n" + "-" * 100)
print("21.7 DEPENDENCY STRUCTURE COVERAGE")
print("-" * 100)


EXPECTED_PAIR_COUNTS = {

    "adult_income": {
        "pearson": 15,
        "spearman": 15,
        "cramers_v": 28,
    },

    "bank_marketing": {
        "pearson": 21,
        "spearman": 21,
        "cramers_v": 36,
    },

    "diabetes_130us": {
        "pearson": 55,
        "spearman": 55,
        "cramers_v": 630,
    },
}


for dataset_id in DATASET_IDS:

    expected = EXPECTED_PAIR_COUNTS[
        dataset_id
    ]


    pearson_count = int(
        len(
            PEARSON_DF[
                PEARSON_DF["dataset_id"]
                == dataset_id
            ]
        )
    )


    spearman_count = int(
        len(
            SPEARMAN_DF[
                SPEARMAN_DF["dataset_id"]
                == dataset_id
            ]
        )
    )


    cramers_count = int(
        len(
            CATEGORICAL_DEPENDENCY_DF[
                CATEGORICAL_DEPENDENCY_DF[
                    "dataset_id"
                ] == dataset_id
            ]
        )
    )


    actual = {
        "pearson": pearson_count,
        "spearman": spearman_count,
        "cramers_v": cramers_count,
    }


    if actual != expected:

        raise RuntimeError(
            f"Dependency pair-count mismatch for "
            f"{dataset_id}.\n"
            f"Expected: {expected}\n"
            f"Actual  : {actual}"
        )


    print(
        f"✓ {dataset_id:<20} : "
        f"Pearson={pearson_count}, "
        f"Spearman={spearman_count}, "
        f"Cramér's V={cramers_count}"
    )


# ==============================================================================
# 21.8 — VERIFY TRAIN-ONLY PROVENANCE
# ==============================================================================

print("\n" + "-" * 100)
print("21.8 TRAIN-ONLY STATISTICAL PROVENANCE")
print("-" * 100)


def contains_train_only_provenance(obj):
    """
    Recursively detect the canonical train-only provenance marker
    without imposing an obsolete JSON schema.
    """

    if isinstance(obj, dict):

        for key, value in obj.items():

            if key in {
                "source_split",
                "fit_split",
                "data_split",
            }:

                if isinstance(value, str):
                    if value.lower() == "train_only":
                        return True

            if contains_train_only_provenance(value):
                return True


    elif isinstance(obj, list):

        for item in obj:

            if contains_train_only_provenance(item):
                return True


    return False


for dataset_id in DATASET_IDS:

    reference = (
        SPP_GAN_STATISTICAL_REFERENCE[
            dataset_id
        ]
    )

    guidance = (
        SPP_GAN_STATISTICAL_GUIDANCE[
            dataset_id
        ]
    )


    if not contains_train_only_provenance(
        reference
    ):

        raise RuntimeError(
            f"Train-only provenance not detected "
            f"in statistical reference: {dataset_id}"
        )


    if not contains_train_only_provenance(
        guidance
    ):

        raise RuntimeError(
            f"Train-only provenance not detected "
            f"in statistical guidance: {dataset_id}"
        )


print(
    "✓ Statistical reference provenance : TRAIN ONLY"
)

print(
    "✓ Statistical guidance provenance  : TRAIN ONLY"
)

print(
    "✓ Validation/test statistical use : NONE"
)


# ==============================================================================
# 21.9 — VERIFY IDENTIFIER / PROVENANCE POLICY
# ==============================================================================

print("\n" + "-" * 100)
print("21.9 IDENTIFIER / PROVENANCE POLICY")
print("-" * 100)


for dataset_id in DATASET_IDS:

    preprocessing_features = set(
        FEATURE_TYPE_CHARACTERIZATION_DF[
            (
                FEATURE_TYPE_CHARACTERIZATION_DF[
                    "dataset_id"
                ] == dataset_id
            )
            &
            (
                FEATURE_TYPE_CHARACTERIZATION_DF[
                    "is_preprocessing_feature"
                ] == True
            )
        ]["feature"]
    )


    excluded_identifiers = set(
        IDENTIFIER_COLUMNS.get(
            dataset_id,
            []
        )
    )


    overlap = (
        preprocessing_features
        .intersection(
            excluded_identifiers
        )
    )


    if overlap:

        raise RuntimeError(
            f"Excluded identifiers detected in "
            f"preprocessing features for "
            f"{dataset_id}: {sorted(overlap)}"
        )


    if PROVENANCE_COLUMN in preprocessing_features:

        raise RuntimeError(
            f"Provenance column '{PROVENANCE_COLUMN}' "
            f"detected in preprocessing features for "
            f"{dataset_id}."
        )


print(
    "✓ Identifier exclusion             : PASS"
)

print(
    "✓ Provenance exclusion             : PASS"
)


# ==============================================================================
# 21.10 — VERIFY TARGET POLICY
# ==============================================================================

print("\n" + "-" * 100)
print("21.10 TARGET POLICY")
print("-" * 100)


for dataset_id in DATASET_IDS:

    target = TARGET_COLUMNS[
        dataset_id
    ]


    dataset_features = (
        FEATURE_TYPE_CHARACTERIZATION_DF[
            FEATURE_TYPE_CHARACTERIZATION_DF[
                "dataset_id"
            ] == dataset_id
        ]
    )


    target_rows = dataset_features[
        dataset_features[
            "feature"
        ] == target
    ]


    if len(target_rows) != 1:

        raise RuntimeError(
            f"Target '{target}' does not occur exactly "
            f"once in full generative characterization "
            f"for {dataset_id}."
        )


    target_row = target_rows.iloc[0]


    if not bool(
        target_row["is_target"]
    ):

        raise RuntimeError(
            f"Target flag is not True for "
            f"{dataset_id}: {target}"
        )


    preprocessing_target_rows = (
        dataset_features[
            (
                dataset_features[
                    "is_preprocessing_feature"
                ] == True
            )
            &
            (
                dataset_features[
                    "feature"
                ] == target
            )
        ]
    )


    if len(preprocessing_target_rows) != 0:

        raise RuntimeError(
            f"Target '{target}' incorrectly appears "
            f"in preprocessing feature set for "
            f"{dataset_id}."
        )


print(
    "✓ Target retained in generative schema : PASS"
)

print(
    "✓ Target excluded from preprocessing   : PASS"
)


# ==============================================================================
# 21.11 — VERIFY REFERENCE / GUIDANCE DATASET LINKAGE
# ==============================================================================

print("\n" + "-" * 100)
print("21.11 REFERENCE / GUIDANCE CONSISTENCY")
print("-" * 100)


for dataset_id in DATASET_IDS:

    reference = (
        SPP_GAN_STATISTICAL_REFERENCE[
            dataset_id
        ]
    )

    guidance = (
        SPP_GAN_STATISTICAL_GUIDANCE[
            dataset_id
        ]
    )


    if reference.get(
        "dataset_id"
    ) != dataset_id:

        raise RuntimeError(
            f"Statistical reference dataset linkage "
            f"failed for {dataset_id}."
        )


    if guidance.get(
        "dataset_id"
    ) != dataset_id:

        raise RuntimeError(
            f"Statistical guidance dataset linkage "
            f"failed for {dataset_id}."
        )


print(
    "✓ Reference dataset linkage       : PASS"
)

print(
    "✓ Guidance dataset linkage        : PASS"
)

print(
    "✓ Reference/guidance consistency   : PASS"
)


# ==============================================================================
# 21.12 — VERIFY SECTION 19 PERSISTED INTEGRITY REPORT
# ==============================================================================

print("\n" + "-" * 100)
print("21.12 SECTION 19 PERSISTED INTEGRITY REPORT")
print("-" * 100)


NB03_ROOT_PATH = Path(
    NB03_ROOT
)


SECTION19_REPORT_PATH = (
    NB03_ROOT_PATH
    / "validation"
    / "section_19_final_validation_report.csv"
)


if not SECTION19_REPORT_PATH.exists():

    raise RuntimeError(
        "Section 19 final validation report is missing:\n"
        f"{SECTION19_REPORT_PATH}"
    )


SECTION19_PERSISTED_DF = pd.read_csv(
    SECTION19_REPORT_PATH
)


if not {
    "check_name",
    "status",
    "details",
}.issubset(
    SECTION19_PERSISTED_DF.columns
):

    raise RuntimeError(
        "Section 19 persisted report does not contain "
        "the required schema."
    )


SECTION19_TOTAL = int(
    len(SECTION19_PERSISTED_DF)
)


SECTION19_FAILED = int(
    (
        SECTION19_PERSISTED_DF[
            "status"
        ] != "PASS"
    ).sum()
)


SECTION19_PASSED = (
    SECTION19_TOTAL -
    SECTION19_FAILED
)


if SECTION19_TOTAL != 188:

    raise RuntimeError(
        "Section 19 persisted integrity report does not "
        "contain the frozen expected 188 checks.\n"
        f"Actual checks: {SECTION19_TOTAL}"
    )


if SECTION19_FAILED != 0:

    raise RuntimeError(
        "Section 19 persisted integrity report contains "
        f"{SECTION19_FAILED} failed checks."
    )


print(
    f"✓ Section 19 persisted checks   : "
    f"{SECTION19_TOTAL}"
)

print(
    f"✓ Section 19 passed             : "
    f"{SECTION19_PASSED}"
)

print(
    f"✓ Section 19 failed             : "
    f"{SECTION19_FAILED}"
)

print(
    "✓ Section 19 integrity gate     : PASS"
)


# ==============================================================================
# 21.13 — PREPARE COMPLETION ARTIFACT DIRECTORIES
# ==============================================================================
#
# IMPORTANT
# ---------
# The previous implementation incorrectly checked for completion artifacts
# BEFORE creating them.
#
# This section now prepares the persistence locations first.
# The artifacts are created in Sections 21.14–21.17 and validated in 21.18.
# ==============================================================================

print("\n" + "-" * 100)
print("21.13 PREPARE COMPLETION ARTIFACT DIRECTORIES")
print("-" * 100)


NB03_ROOT_PATH = Path(
    NB03_ROOT
)


SCHEMAS_DIR = (
    NB03_ROOT_PATH
    / "schemas"
)


REPORTS_DIR = (
    NB03_ROOT_PATH
    / "reports"
)


VALIDATION_DIR = (
    NB03_ROOT_PATH
    / "validation"
)


SCHEMAS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------------------------
# Canonical completion artifact paths
# ------------------------------------------------------------------------------

COMPLETION_SUMMARY_PATH = (
    SCHEMAS_DIR
    / "notebook_03_completion_summary.json"
)


FINAL_STATUS_PATH = (
    REPORTS_DIR
    / "notebook_03_final_status.csv"
)


DATASET_COMPLETENESS_PATH = (
    REPORTS_DIR
    / "dataset_completeness.csv"
)


STATISTICAL_COVERAGE_PATH = (
    REPORTS_DIR
    / "statistical_coverage_summary.csv"
)


SECTION19_REPORT_PATH = (
    VALIDATION_DIR
    / "section_19_final_validation_report.csv"
)


print(
    "✓ Completion artifact directories : READY"
)

print(
    f"✓ Schemas directory              : {SCHEMAS_DIR}"
)

print(
    f"✓ Reports directory              : {REPORTS_DIR}"
)

print(
    f"✓ Validation directory           : {VALIDATION_DIR}"
)


# ==============================================================================
# 21.14 — SAVE DATASET COMPLETENESS REPORT
# ==============================================================================

print("\n" + "-" * 100)
print("21.14 SAVE DATASET COMPLETENESS REPORT")
print("-" * 100)


DATASET_COMPLETENESS_DF.to_csv(
    DATASET_COMPLETENESS_PATH,
    index=False
)


if not DATASET_COMPLETENESS_PATH.exists():

    raise RuntimeError(
        "Dataset completeness report was not successfully persisted."
    )


print(
    f"✓ Saved dataset completeness report:"
)

print(
    f"  {DATASET_COMPLETENESS_PATH}"
)


# ==============================================================================
# 21.15 — SAVE STATISTICAL COVERAGE REPORT
# ==============================================================================

print("\n" + "-" * 100)
print("21.15 SAVE STATISTICAL COVERAGE REPORT")
print("-" * 100)


STATISTICAL_COVERAGE_DF = pd.DataFrame(
    [
        {
            "metric":
                key,

            "value":
                value,
        }

        for key, value
        in STATISTICAL_COVERAGE_SUMMARY.items()
    ]
)


STATISTICAL_COVERAGE_DF.to_csv(
    STATISTICAL_COVERAGE_PATH,
    index=False
)


if not STATISTICAL_COVERAGE_PATH.exists():

    raise RuntimeError(
        "Statistical coverage report was not successfully persisted."
    )


print(
    f"✓ Saved statistical coverage report:"
)

print(
    f"  {STATISTICAL_COVERAGE_PATH}"
)


# ==============================================================================
# 21.16 — SAVE FINAL STATUS REPORT
# ==============================================================================

print("\n" + "-" * 100)
print("21.16 SAVE FINAL STATUS REPORT")
print("-" * 100)


FINAL_STATUS_RECORD = {

    "notebook":
        "03",

    "status":
        "COMPLETE",

    "completion_gate":
        "PASS",

    "datasets":
        int(
            len(DATASET_IDS)
        ),

    "feature_type_characterization_rows":
        int(
            len(
                FEATURE_TYPE_CHARACTERIZATION_DF
            )
        ),

    "numerical_statistics_rows":
        int(
            len(
                NUMERICAL_STATISTICS_DF
            )
        ),

    "categorical_statistics_rows":
        int(
            len(
                CATEGORICAL_STATISTICS_DF
            )
        ),

    "cardinality_entropy_rows":
        int(
            len(
                CARDINALITY_ENTROPY_DF
            )
        ),

    "pearson_records":
        int(
            len(
                PEARSON_DF
            )
        ),

    "spearman_records":
        int(
            len(
                SPEARMAN_DF
            )
        ),

    "categorical_dependency_records":
        int(
            len(
                CATEGORICAL_DEPENDENCY_DF
            )
        ),

    "statistical_reference_datasets":
        int(
            len(
                SPP_GAN_STATISTICAL_REFERENCE
            )
        ),

    "statistical_guidance_datasets":
        int(
            len(
                SPP_GAN_STATISTICAL_GUIDANCE
            )
        ),

    "section_19_checks":
        SECTION19_TOTAL,

    "section_19_status":
        "PASS",

    "section_20_checks":
        SECTION20_TOTAL_CHECKS,

    "section_20_passed":
        SECTION20_PASSED_CHECKS,

    "section_20_failed":
        SECTION20_FAILED_CHECKS,

    "section_20_status":
        SECTION20_FINAL_STATUS,

    "train_only":
        True,

    "identifiers_excluded":
        True,

    "provenance_excluded":
        True,

    "target_retained":
        True,

    "target_excluded_from_preprocessing":
        True,

    "artifact_completeness":
        True,

    "ready_for_notebook_04":
        True,

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


FINAL_STATUS_DF = pd.DataFrame(
    [FINAL_STATUS_RECORD]
)


FINAL_STATUS_DF.to_csv(
    FINAL_STATUS_PATH,
    index=False
)


if not FINAL_STATUS_PATH.exists():

    raise RuntimeError(
        "Final status report was not successfully persisted."
    )


print(
    f"✓ Saved final status report:"
)

print(
    f"  {FINAL_STATUS_PATH}"
)


# ==============================================================================
# 21.17 — SAVE MACHINE-READABLE COMPLETION SUMMARY
# ==============================================================================

print("\n" + "-" * 100)
print("21.17 SAVE MACHINE-READABLE COMPLETION SUMMARY")
print("-" * 100)


NOTEBOOK_03_COMPLETION_SUMMARY = {

    "notebook_number":
        "03",

    "notebook_title":
        "Statistical & Data Characterization",

    "status":
        "COMPLETE",

    "completion_gate":
        "PASS",

    "completion_timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "project_root":
        str(PROJECT_ROOT),

    "notebook_02_dependency":
        "Notebook 02 frozen artifacts",

    "datasets":
        list(DATASET_IDS),

    "statistical_source_policy": {

        "source_split":
            "train_only",

        "validation_used":
            False,

        "test_used":
            False,

        "synthetic_used":
            False,
    },

    "feature_policy": {

        "identifiers_excluded":
            True,

        "provenance_excluded":
            True,

        "target_retained":
            True,

        "target_excluded_from_preprocessing":
            True,
    },

    "canonical_feature_counts":
        EXPECTED_FEATURE_COUNTS,

    "canonical_statistics": {

        "dataset_characterization_rows":
            int(
                len(
                    DATASET_CHARACTERIZATION_DF
                )
            ),

        "feature_type_characterization_rows":
            int(
                len(
                    FEATURE_TYPE_CHARACTERIZATION_DF
                )
            ),

        "feature_type_summary_rows":
            int(
                len(
                    FEATURE_TYPE_SUMMARY_DF
                )
            ),

        "numerical_statistics_rows":
            int(
                len(
                    NUMERICAL_STATISTICS_DF
                )
            ),

        "categorical_statistics_rows":
            int(
                len(
                    CATEGORICAL_STATISTICS_DF
                )
            ),

        "cardinality_entropy_rows":
            int(
                len(
                    CARDINALITY_ENTROPY_DF
                )
            ),

        "pearson_rows":
            int(
                len(
                    PEARSON_DF
                )
            ),

        "spearman_rows":
            int(
                len(
                    SPEARMAN_DF
                )
            ),

        "categorical_dependency_rows":
            int(
                len(
                    CATEGORICAL_DEPENDENCY_DF
                )
            ),
    },

    "dependency_pair_counts":
        EXPECTED_PAIR_COUNTS,

    "statistical_reference": {

        "datasets":
            int(
                len(
                    SPP_GAN_STATISTICAL_REFERENCE
                )
            ),

        "status":
            "COMPLETE",
    },

    "statistical_guidance": {

        "datasets":
            int(
                len(
                    SPP_GAN_STATISTICAL_GUIDANCE
                )
            ),

        "status":
            "COMPLETE",
    },

    "section_19_integrity": {

        "checks":
            SECTION19_TOTAL,

        "passed":
            SECTION19_PASSED,

        "failed":
            SECTION19_FAILED,

        "status":
            "PASS",
    },

    "section_20_final_verification": {

        "checks":
            SECTION20_TOTAL_CHECKS,

        "passed":
            SECTION20_PASSED_CHECKS,

        "failed":
            SECTION20_FAILED_CHECKS,

        "status":
            SECTION20_FINAL_STATUS,
    },

    "coverage":
        STATISTICAL_COVERAGE_SUMMARY,

    "validation": {

        "final_verification":
            "PASS",

        "dataset_completeness":
            "PASS",

        "train_only_policy":
            "PASS",

        "identifier_policy":
            "PASS",

        "provenance_policy":
            "PASS",

        "target_policy":
            "PASS",

        "reference_guidance_consistency":
            "PASS",

        "artifact_completeness":
            "PASS",
    },

    "ready_for_downstream":
        True,

    "ready_for_notebook_04":
        True,

    "downstream_use": [

        "SPP-GAN architecture",

        "statistical conditioning/reference",

        "training-time statistical guidance",

        "synthetic-data evaluation",

        "comparative baseline analysis",
    ],
}


with open(
    COMPLETION_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        NOTEBOOK_03_COMPLETION_SUMMARY,
        f,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
        default=str,
    )


if not COMPLETION_SUMMARY_PATH.exists():

    raise RuntimeError(
        "Completion summary JSON was not successfully persisted."
    )


print(
    f"✓ Saved completion summary JSON:"
)

print(
    f"  {COMPLETION_SUMMARY_PATH}"
)


# ==============================================================================
# 21.18 — RELOAD AND VALIDATE ALL COMPLETION ARTIFACTS
# ==============================================================================
#
# This is the correct location for the artifact-existence gate.
# All artifacts have now been created.
# ==============================================================================

print("\n" + "-" * 100)
print("21.18 RELOAD AND VALIDATE COMPLETION ARTIFACTS")
print("-" * 100)


required_completion_files = {

    "completion_summary":
        COMPLETION_SUMMARY_PATH,

    "final_status":
        FINAL_STATUS_PATH,

    "dataset_completeness":
        DATASET_COMPLETENESS_PATH,

    "statistical_coverage":
        STATISTICAL_COVERAGE_PATH,

    "section_19_integrity":
        SECTION19_REPORT_PATH,
}


missing_completion_files = [

    name

    for name, path
    in required_completion_files.items()

    if not path.exists()
]


if missing_completion_files:

    raise RuntimeError(
        "Completion artifacts are still missing after persistence:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_completion_files
        )
    )


print(
    "✓ All required completion artifacts : PRESENT"
)


# ------------------------------------------------------------------------------
# Reload completion summary JSON
# ------------------------------------------------------------------------------

with open(
    COMPLETION_SUMMARY_PATH,
    "r",
    encoding="utf-8"
) as f:

    RELOADED_COMPLETION_SUMMARY = json.load(
        f
    )


if (
    RELOADED_COMPLETION_SUMMARY.get(
        "status"
    )
    != "COMPLETE"
):

    raise RuntimeError(
        "Reloaded completion summary does not report COMPLETE."
    )


if (
    RELOADED_COMPLETION_SUMMARY.get(
        "completion_gate"
    )
    != "PASS"
):

    raise RuntimeError(
        "Reloaded completion summary does not report PASS."
    )


# ------------------------------------------------------------------------------
# Reload final status
# ------------------------------------------------------------------------------

RELOADED_FINAL_STATUS_DF = pd.read_csv(
    FINAL_STATUS_PATH
)


if len(
    RELOADED_FINAL_STATUS_DF
) != 1:

    raise RuntimeError(
        "Reloaded final status must contain exactly one record."
    )


RELOADED_FINAL_STATUS = (
    RELOADED_FINAL_STATUS_DF.iloc[0]
)


if str(
    RELOADED_FINAL_STATUS["status"]
) != "COMPLETE":

    raise RuntimeError(
        "Reloaded final status is not COMPLETE."
    )


if str(
    RELOADED_FINAL_STATUS["completion_gate"]
) != "PASS":

    raise RuntimeError(
        "Reloaded final status completion gate is not PASS."
    )


# ------------------------------------------------------------------------------
# Safe Boolean parser
# ------------------------------------------------------------------------------

def parse_persisted_bool(
    value,
    field_name
):

    """
    Safely parse persisted Boolean values.

    This prevents the unsafe behavior:
        bool("False") == True
    """

    if isinstance(
        value,
        bool
    ):

        return value


    if isinstance(
        value,
        (np.bool_,)
    ):

        return bool(value)


    if pd.isna(value):

        raise RuntimeError(
            f"Persisted field '{field_name}' is missing."
        )


    normalized = (
        str(value)
        .strip()
        .lower()
    )


    if normalized in {
        "true",
        "1",
        "yes",
    }:

        return True


    if normalized in {
        "false",
        "0",
        "no",
    }:

        return False


    raise RuntimeError(
        f"Cannot interpret persisted field "
        f"'{field_name}' as Boolean: {value}"
    )


# ------------------------------------------------------------------------------
# Validate critical persisted Boolean policies
# ------------------------------------------------------------------------------

PERSISTED_BOOLEAN_FIELDS = {

    "train_only":
        True,

    "identifiers_excluded":
        True,

    "provenance_excluded":
        True,

    "target_retained":
        True,

    "target_excluded_from_preprocessing":
        True,

    "artifact_completeness":
        True,

    "ready_for_notebook_04":
        True,
}


for field_name, expected_value in (
    PERSISTED_BOOLEAN_FIELDS.items()
):

    if field_name not in (
        RELOADED_FINAL_STATUS.index
    ):

        raise RuntimeError(
            f"Required persisted field missing: "
            f"{field_name}"
        )


    actual_value = parse_persisted_bool(
        RELOADED_FINAL_STATUS[
            field_name
        ],
        field_name
    )


    if actual_value != expected_value:

        raise RuntimeError(
            f"Persisted field mismatch for "
            f"'{field_name}'.\n"
            f"Expected: {expected_value}\n"
            f"Actual  : {actual_value}"
        )


# ------------------------------------------------------------------------------
# Reload dataset completeness report
# ------------------------------------------------------------------------------

RELOADED_DATASET_COMPLETENESS_DF = (
    pd.read_csv(
        DATASET_COMPLETENESS_PATH
    )
)


if len(
    RELOADED_DATASET_COMPLETENESS_DF
) != len(DATASET_IDS):

    raise RuntimeError(
        "Reloaded dataset completeness report has "
        "an unexpected number of rows."
    )


if (
    RELOADED_DATASET_COMPLETENESS_DF[
        "status"
    ] != "PASS"
).any():

    raise RuntimeError(
        "Reloaded dataset completeness report contains "
        "a failed dataset."
    )


# ------------------------------------------------------------------------------
# Reload statistical coverage report
# ------------------------------------------------------------------------------

RELOADED_STATISTICAL_COVERAGE_DF = (
    pd.read_csv(
        STATISTICAL_COVERAGE_PATH
    )
)


if len(
    RELOADED_STATISTICAL_COVERAGE_DF
) != len(
    STATISTICAL_COVERAGE_SUMMARY
):

    raise RuntimeError(
        "Reloaded statistical coverage report has "
        "an unexpected number of rows."
    )


# ------------------------------------------------------------------------------
# Reload Section 19 persisted integrity report
# ------------------------------------------------------------------------------

RELOADED_SECTION19_DF = pd.read_csv(
    SECTION19_REPORT_PATH
)


if len(
    RELOADED_SECTION19_DF
) != SECTION19_EXPECTED_CHECKS:

    raise RuntimeError(
        "Reloaded Section 19 report does not contain "
        f"{SECTION19_EXPECTED_CHECKS} checks."
    )


if (
    RELOADED_SECTION19_DF[
        "status"
    ] != "PASS"
).any():

    raise RuntimeError(
        "Reloaded Section 19 report contains failed checks."
    )


# ------------------------------------------------------------------------------
# Final persistence validation
# ------------------------------------------------------------------------------

print(
    "✓ Completion summary reload       : PASS"
)

print(
    "✓ Final status reload             : PASS"
)

print(
    "✓ Dataset completeness reload     : PASS"
)

print(
    "✓ Statistical coverage reload     : PASS"
)

print(
    "✓ Section 19 integrity reload     : PASS"
)

print(
    "✓ Persisted policy validation     : PASS"
)


# ==============================================================================
# 21.19 — FINAL CONSOLE SUMMARY
# ==============================================================================

print("\n" + "-" * 100)
print("21.19 FINAL CONSOLE SUMMARY")
print("-" * 100)


print(
    "\nNotebook                              : "
    "03 — Statistical & Data Characterization"
)

print(
    "Status                                : "
    "COMPLETE"
)

print(
    "Completion gate                       : "
    "PASS"
)

print(
    "Datasets                              : "
    f"{len(DATASET_IDS)}"
)

print(
    "Feature characterization records      : "
    f"{len(FEATURE_TYPE_CHARACTERIZATION_DF)}"
)

print(
    "Numerical statistics records           : "
    f"{len(NUMERICAL_STATISTICS_DF)}"
)

print(
    "Categorical statistics records         : "
    f"{len(CATEGORICAL_STATISTICS_DF)}"
)

print(
    "Cardinality/entropy records            : "
    f"{len(CARDINALITY_ENTROPY_DF)}"
)

print(
    "Pearson correlation records             : "
    f"{len(PEARSON_DF)}"
)

print(
    "Spearman correlation records            : "
    f"{len(SPEARMAN_DF)}"
)

print(
    "Categorical dependency records          : "
    f"{len(CATEGORICAL_DEPENDENCY_DF)}"
)

print(
    "SPP-GAN statistical references          : "
    f"{len(SPP_GAN_STATISTICAL_REFERENCE)}"
)

print(
    "SPP-GAN statistical guidance artifacts  : "
    f"{len(SPP_GAN_STATISTICAL_GUIDANCE)}"
)

print(
    "Section 19 integrity checks             : "
    f"{SECTION19_PASSED}/"
    f"{SECTION19_TOTAL} PASS"
)

print(
    "Section 20 verification checks          : "
    f"{SECTION20_PASSED_CHECKS}/"
    f"{SECTION20_TOTAL_CHECKS} PASS"
)

print(
    "Statistical source                     : "
    "TRAIN ONLY"
)

print(
    "Validation used for statistics         : "
    "NO"
)

print(
    "Test used for statistics               : "
    "NO"
)

print(
    "Synthetic data used                    : "
    "NO"
)

print(
    "Identifiers included as features      : "
    "NO"
)

print(
    "Provenance included as features       : "
    "NO"
)

print(
    "Target retained in generative schema  : "
    "YES"
)

print(
    "Target excluded from preprocessing    : "
    "YES"
)

print("\n" + "-" * 100)

print(
    "Required canonical objects             : PASS"
)

print(
    "Section 20 final verification          : PASS"
)

print(
    "Dataset completeness                   : PASS"
)

print(
    "Train-only provenance                 : PASS"
)

print(
    "Identifier exclusion                  : PASS"
)

print(
    "Provenance exclusion                  : PASS"
)

print(
    "Target policy                         : PASS"
)

print(
    "Reference/guidance consistency        : PASS"
)

print(
    "Section 19 persisted integrity        : PASS"
)

print(
    "Completion artifacts                  : PASS"
)

print(
    "Completion artifact reload             : PASS"
)

print(
    "Downstream readiness                  : PASS"
)

print("\n" + "=" * 100)
print("NOTEBOOK 03 STATUS: COMPLETE")
print("=" * 100)

print(
    "\nNOTEBOOK 03 COMPLETION GATE PASSED"
)

print(
    "\nNotebook 03 is COMPLETE and READY FOR NOTEBOOK 04."
)

print(
    "\nCompletion summary:"
)

print(
    COMPLETION_SUMMARY_PATH
)

print(
    "\nFinal status report:"
)

print(
    FINAL_STATUS_PATH
)

print(
    "\nDataset completeness report:"
)

print(
    DATASET_COMPLETENESS_PATH
)

print(
    "\nStatistical coverage report:"
)

print(
    STATISTICAL_COVERAGE_PATH
)

print(
    "\nSection 19 integrity report:"
)

print(
    SECTION19_REPORT_PATH
)

print("=" * 100)

SECTION 21 — COMPLETION SUMMARY

----------------------------------------------------------------------------------------------------
21.1 REQUIRED CANONICAL OBJECTS
----------------------------------------------------------------------------------------------------
✓ Required canonical objects       : PASS

----------------------------------------------------------------------------------------------------
21.2 SECTION 20 FINAL VERIFICATION GATE
----------------------------------------------------------------------------------------------------
Section 20 checks              : 35
Section 20 passed              : 35
Section 20 failed              : 0
Section 20 final status        : PASS
✓ Section 20 final gate          : PASS

----------------------------------------------------------------------------------------------------
21.3 DATASET REGISTRY
----------------------------------------------------------------------------------------------------
✓ Dataset registry               : PAS